
# Stage 3 — Attack Scheduling & Injection

## Step 3.1 — Input Readiness & Schema Audit

### Objective
Verify that the approved Stage 2 assignments and the CERT r6.2 User-Day dataset are available and ready for attack scheduling.

### Input Datasets
- `validated_scenario_assignments_r62.parquet`
- `approved_scenario_catalog_r62.parquet`
- `user_day_r62_v1.parquet`
- `user_baseline_r62_v1.csv`

### Validation Scope
1. Verify input file availability.
2. Inspect dataset dimensions and schemas.
3. Check user and scenario identifiers.
4. Validate assignment-to-scenario relationships.
5. Check date coverage and scheduling requirements.

### Expected Outcome
Confirm that the input datasets are ready for attack scheduling without modifying the original data.


In [2]:

# Stage 3 — Step 3.1.1: Input File Discovery
# Goal: Locate the required input files before loading or validating them.
# Read-only: No source files will be modified.

from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration"
)

ATTACK_DATA = PROJECT_ROOT / "Attack_Simulation" / "data"

required_files = {
    "Stage 2 Assignments": "validated_scenario_assignments_r62.parquet",
    "Approved Scenario Catalog": "approved_scenario_catalog_r62.parquet",
    "User-Day Dataset": "user_day_r62_v1.parquet",
    "User Baseline": "user_baseline_r62_v1.csv",
}

print("=" * 65)
print("STAGE 3 — STEP 3.1.1: INPUT FILE DISCOVERY")
print("=" * 65)

print(f"\nProject root: {PROJECT_ROOT}")
print(f"Project root exists: {PROJECT_ROOT.is_dir()}")

found_files = {}
missing_files = []

for label, filename in required_files.items():
    matches = sorted(
        PROJECT_ROOT.rglob(filename)
    ) if PROJECT_ROOT.is_dir() else []

    print(f"\n[{label}]")
    print(f"Filename: {filename}")

    if len(matches) == 1:
        found_files[label] = matches[0]
        print("Status: FOUND")
        print(f"Path: {matches[0]}")
    elif len(matches) == 0:
        missing_files.append(label)
        print("Status: MISSING")
    else:
        print(f"Status: MULTIPLE MATCHES ({len(matches)})")
        for path in matches:
            print(f"  - {path}")

print("\n" + "=" * 65)
print("DISCOVERY SUMMARY")
print("=" * 65)
print(f"Unique files found: {len(found_files)}/{len(required_files)}")
print(f"Missing files: {len(missing_files)}")
print("No datasets were modified.")


STAGE 3 — STEP 3.1.1: INPUT FILE DISCOVERY

Project root: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration
Project root exists: True

[Stage 2 Assignments]
Filename: validated_scenario_assignments_r62.parquet
Status: FOUND
Path: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Attack_Simulation\data\validated_scenario_assignments_r62.parquet

[Approved Scenario Catalog]
Filename: approved_scenario_catalog_r62.parquet
Status: FOUND
Path: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Attack_Simulation\data\approved_scenario_catalog_r62.parquet

[User-Day Dataset]
Filename: user_day_r62_v1.parquet
Status: FOUND
Path: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Data_Engineering\Data_engineering\data\processed\user_day_r62_v1.parquet

[User Baseline]
Filename: user_baseline_r62_v1.csv
Status: FOUND
Path: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Data_Engineering\Data_engineering\data\features\user_baseline_r62_v1.csv

DISCOVERY SUMMARY
Unique files found: 4/4
Missing files: 0
No d


### Step 3.1.2 — Dataset Loading & Schema Inspection

**Objective:** Load the four required datasets and inspect their structures before attack scheduling.

**Validation Tasks:**
1. Load the approved scenario assignments and catalog.
2. Load the User-Day dataset and user baseline.
3. Inspect dataset dimensions and column data types.
4. Identify missing values in each dataset.
5. Display sample records to understand the available fields.

**Expected Outcome:** Establish the actual input schemas and identify any structural issues that must be addressed before scheduling.

**Note:** This step is read-only. No source data will be modified.


In [3]:

# Stage 3 — Step 3.1.2: Dataset Loading & Schema Inspection
# Goal: Load the four required datasets and inspect their actual schemas.
# Read-only: No source files or dataset values will be modified.

import pandas as pd
from IPython.display import display

# Use the verified file paths from Step 3.1.1.
assignments = pd.read_parquet(
    found_files["Stage 2 Assignments"]
)

scenario_catalog = pd.read_parquet(
    found_files["Approved Scenario Catalog"]
)

user_day = pd.read_parquet(
    found_files["User-Day Dataset"]
)

user_baseline = pd.read_csv(
    found_files["User Baseline"]
)

datasets = {
    "Stage 2 Assignments": assignments,
    "Approved Scenario Catalog": scenario_catalog,
    "User-Day Dataset": user_day,
    "User Baseline": user_baseline,
}

print("=" * 70)
print("STAGE 3 — STEP 3.1.2: DATASET LOADING & SCHEMA INSPECTION")
print("=" * 70)

summary = []

for name, df in datasets.items():

    summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Duplicate Column Names": int(
            df.columns.duplicated().sum()
        ),
        "Total Missing Values": int(
            df.isna().sum().sum()
        ),
    })

    print(f"\n{'=' * 70}")
    print(f"DATASET: {name}")
    print(f"{'=' * 70}")

    print(f"Shape: {df.shape}")

    print("\nCOLUMN SCHEMA:")
    schema = pd.DataFrame({
        "Column": df.columns,
        "Data Type": [str(t) for t in df.dtypes],
        "Missing Values": df.isna().sum().values,
    })

    display(schema)

    print("\nSAMPLE RECORDS:")
    display(df.head(3))

print("\n" + "=" * 70)
print("DATASET SUMMARY")
print("=" * 70)

display(pd.DataFrame(summary))

print("\nInspection completed. No source files were modified.")


STAGE 3 — STEP 3.1.2: DATASET LOADING & SCHEMA INSPECTION

DATASET: Stage 2 Assignments
Shape: (995, 9)

COLUMN SCHEMA:


,Column,Data Type,Missing Values
0,user,str,0
1,scenario_id,str,0
2,assignment_pool,str,0
3,duration,int64,0
4,stealth_level,str,0
5,assignment_group,str,0
6,eligible_workdays,int64,0
7,priority_rank,int64,0
8,assignment_type,str,0



SAMPLE RECORDS:


,user,scenario_id,assignment_pool,duration,stealth_level,assignment_group,eligible_workdays,priority_rank,assignment_type
0,AAC0610,SCN_0108,Short-duration pool,30,Low,Two-channel short-duration,105,24,Unique template
1,AAF0819,SCN_0007,Short-duration pool,5,High,Two-channel short-duration,105,25,Unique template
2,AAP0352,SCN_0016,Short-duration pool,10,High,Two-channel short-duration,105,26,Unique template



DATASET: Approved Scenario Catalog
Shape: (1143, 15)

COLUMN SCHEMA:


,Column,Data Type,Missing Values
0,scenario_id,str,0
1,duration,int64,0
2,off_days,int64,0
3,active_days,int64,0
4,switching_level,str,0
5,min_switches,int64,0
6,max_switches,int64,0
7,target_switches,int64,0
8,target_switching_rate,float64,0
9,two_channel_days,int64,0


... (9 outputs left out)

,user,baseline_period_start,baseline_period_end,baseline_workdays,baseline_first_day,baseline_last_day,email_active_days,email_active_rate,email_count_mean_active,email_count_p50,...,usb_copy_p95,usb_copy_mad,http_upload_active_days,http_upload_active_rate,http_upload_mean_active,http_upload_p50,http_upload_p75,http_upload_p90,http_upload_p95,http_upload_mad
0,AAB0162,2010-01-02,2010-12-31,251,2010-01-04,2010-12-30,153,0.609562,1.758170,2.0,...,NaN,NaN,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,AAB0398,2010-01-02,2010-12-31,251,2010-01-04,2010-12-30,199,0.792829,2.412060,2.0,...,NaN,NaN,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,AAC0610,2010-01-02,2010-12-31,251,2010-01-04,2010-12-30,97,0.386454,1.237113,1.0,...,4.0,1.0,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN



DATASET SUMMARY


,Dataset,Rows,Columns,Duplicate Column Names,Total Missing Values
0,Stage 2 Assignments,995,9,0,0
1,Approved Scenario Catalog,1143,15,0,0
2,User-Day Dataset,1394010,34,0,881
3,User Baseline,3995,36,0,39564



Inspection completed. No source files were modified.



### Step 3.1.3 — Key Integrity & Relationship Validation

**Objective:** Verify the integrity of dataset identifiers and relationships before attack scheduling.

**Validation Tasks:**
1. Check uniqueness of assigned users and scenario catalog IDs.
2. Verify that every assigned scenario exists in the approved catalog.
3. Verify that every assigned user exists in the User-Day dataset and User Baseline.
4. Check uniqueness of User-Day `(user, day)` records.
5. Check User-Day date validity.
6. Verify that scenario duration and stealth level match across datasets.

**Expected Outcome:** Confirm that the approved assignments can be reliably linked to the source data and scenario definitions.

**Note:** Read-only validation. No assignments, scenarios, or source records will be modified.


In [4]:

# Stage 3 — Step 3.1.3: Key Integrity & Relationship Validation
# Goal: Validate identifiers, dataset relationships, and scenario consistency.
# Read-only: No source files or datasets will be modified.

import pandas as pd
from IPython.display import display

print("=" * 70)
print("STAGE 3 — STEP 3.1.3: KEY INTEGRITY & RELATIONSHIP VALIDATION")
print("=" * 70)

checks = []

def add_check(name, passed, details):
    checks.append({
        "Check": name,
        "Status": "PASS" if passed else "FAIL",
        "Details": details
    })

# 1. Assigned user uniqueness
duplicate_users = int(assignments["user"].duplicated().sum())

add_check(
    "Unique Assigned Users",
    duplicate_users == 0,
    f"Duplicate users: {duplicate_users}"
)

# 2. Scenario catalog key uniqueness
duplicate_scenarios = int(
    scenario_catalog["scenario_id"].duplicated().sum()
)

add_check(
    "Unique Scenario Catalog IDs",
    duplicate_scenarios == 0,
    f"Duplicate scenario IDs: {duplicate_scenarios}"
)

# 3. User Baseline uniqueness
duplicate_baseline_users = int(
    user_baseline["user"].duplicated().sum()
)

add_check(
    "Unique Baseline Users",
    duplicate_baseline_users == 0,
    f"Duplicate baseline users: {duplicate_baseline_users}"
)

# 4. Assignment-to-catalog relationship
missing_scenarios = assignments.loc[
    ~assignments["scenario_id"].isin(
        scenario_catalog["scenario_id"]
    ),
    ["user", "scenario_id"]
]

add_check(
    "Assigned Scenarios Exist in Catalog",
    missing_scenarios.empty,
    f"Missing scenario references: {len(missing_scenarios)}"
)

# 5. Assignment-to-baseline relationship
missing_baseline_users = assignments.loc[
    ~assignments["user"].isin(user_baseline["user"]),
    ["user"]
]

add_check(
    "Assigned Users Exist in Baseline",
    missing_baseline_users.empty,
    f"Users missing from baseline: {len(missing_baseline_users)}"
)

# 6. Assignment-to-User-Day relationship
missing_user_day_users = assignments.loc[
    ~assignments["user"].isin(user_day["user"]),
    ["user"]
]

add_check(
    "Assigned Users Exist in User-Day",
    missing_user_day_users.empty,
    f"Users missing from User-Day: {len(missing_user_day_users)}"
)

# 7. User-Day primary key uniqueness
duplicate_user_days = int(
    user_day.duplicated(subset=["user", "day"]).sum()
)

add_check(
    "Unique User-Day Keys",
    duplicate_user_days == 0,
    f"Duplicate (user, day) records: {duplicate_user_days}"
)

# 8. Date validity
parsed_dates = pd.to_datetime(
    user_day["day"],
    errors="coerce"
)

invalid_dates = int(parsed_dates.isna().sum())

add_check(
    "Valid User-Day Dates",
    invalid_dates == 0,
    f"Invalid or missing dates: {invalid_dates}"
)

# 9. Scenario attribute consistency
if duplicate_scenarios == 0:

    scenario_reference = scenario_catalog.set_index(
        "scenario_id"
    )

    matched = assignments[
        assignments["scenario_id"].isin(
            scenario_reference.index
        )
    ].copy()

    matched["catalog_duration"] = matched[
        "scenario_id"
    ].map(scenario_reference["duration"])

    matched["catalog_stealth"] = matched[
        "scenario_id"
    ].map(scenario_reference["stealth_level"])

    duration_mismatch = int(
        (matched["duration"] != matched["catalog_duration"]).sum()
    )

    stealth_mismatch = int(
        (matched["stealth_level"] != matched["catalog_stealth"]).sum()
    )

    add_check(
        "Scenario Duration Consistency",
        duration_mismatch == 0 and missing_scenarios.empty,
        f"Duration mismatches: {duration_mismatch}"
    )

    add_check(
        "Scenario Stealth Consistency",
        stealth_mismatch == 0 and missing_scenarios.empty,
        f"Stealth mismatches: {stealth_mismatch}"
    )

else:
    add_check(
        "Scenario Attribute Consistency",
        False,
        "Cannot safely validate: duplicate catalog scenario IDs"
    )

# 10. Display results
validation_results_313 = pd.DataFrame(checks)

print("\nVALIDATION RESULTS")
display(validation_results_313)

passed = int((validation_results_313["Status"] == "PASS").sum())
failed = int((validation_results_313["Status"] == "FAIL").sum())

print("\n" + "=" * 70)
print("VALIDATION SUMMARY")
print("=" * 70)
print(f"Passed: {passed}")
print(f"Failed: {failed}")
print(f"Total checks: {len(validation_results_313)}")

if failed == 0:
    print("RESULT: PASS — Key integrity and relationships verified.")
else:
    print("RESULT: FAIL — Review failed checks before proceeding.")

print("\nNo source data was modified.")


STAGE 3 — STEP 3.1.3: KEY INTEGRITY & RELATIONSHIP VALIDATION

VALIDATION RESULTS


,Check,Status,Details
0,Unique Assigned Users,PASS,Duplicate users: 0
1,Unique Scenario Catalog IDs,PASS,Duplicate scenario IDs: 0
2,Unique Baseline Users,PASS,Duplicate baseline users: 0
3,Assigned Scenarios Exist in Catalog,PASS,Missing scenario references: 0
4,Assigned Users Exist in Baseline,PASS,Users missing from baseline: 0
5,Assigned Users Exist in User-Day,PASS,Users missing from User-Day: 0
6,Unique User-Day Keys,PASS,"Duplicate (user, day) records: 0"
7,Valid User-Day Dates,PASS,Invalid or missing dates: 0
8,Scenario Duration Consistency,PASS,Duration mismatches: 0
9,Scenario Stealth Consistency,PASS,Stealth mismatches: 0



VALIDATION SUMMARY
Passed: 10
Failed: 0
Total checks: 10
RESULT: PASS — Key integrity and relationships verified.

No source data was modified.



### Step 3.1.4 — Daily Channel Activity & Temporal Eligibility Audit

**Objective:** Examine the actual daily activity of the 995 assigned users to evaluate the initial requirements of Plan A.

**Plan A Rule:** Synthetic attack activity can only be injected into a channel that already has benign activity on the same day.

**Validation Tasks:**
1. Identify eligible workdays during the attack period.
2. Compare calculated workdays with Stage 2 assignments.
3. Count eligible days with existing Email, USB, and HTTP activity.
4. Count eligible days with one, two, or three active channels.
5. Identify users with limited daily channel activity.

**Expected Outcome:** Understand the availability of daily channel activity before attempting scenario scheduling.

**Note:** This is a preliminary feasibility audit, not the final scheduling feasibility test. All operations are read-only.


In [5]:

# Stage 3 — Step 3.1.4: Daily Channel Activity & Temporal Eligibility Audit
# Goal: Evaluate eligible workdays and existing channel activity for Plan A.
# Read-only: No datasets, assignments, or scenario definitions are modified.

import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.1.4: DAILY ACTIVITY & TEMPORAL AUDIT")
print("=" * 75)

# ------------------------------------------------------------
# 1. Define the attack-period boundaries.
# ------------------------------------------------------------

ATTACK_START = pd.Timestamp("2011-01-01")
ATTACK_END = pd.Timestamp("2011-06-01")

assigned_users = assignments["user"].drop_duplicates()

# Work with a temporary copy containing only required columns.
activity = user_day[
    [
        "user", "day", "is_weekend", "logon_count",
        "external_email_count", "usb_copy_count",
        "www_upload_count"
    ]
].copy()

activity["day"] = pd.to_datetime(activity["day"])

# Restrict analysis to assigned users and the attack period.
activity = activity.loc[
    activity["user"].isin(assigned_users)
    & activity["day"].between(ATTACK_START, ATTACK_END)
].copy()

print(f"\nAttack period: {ATTACK_START.date()} to {ATTACK_END.date()}")
print(f"Assigned users: {len(assigned_users):,}")
print(f"User-Day records in scope: {len(activity):,}")
print(f"Users with records in scope: {activity['user'].nunique():,}")

# ------------------------------------------------------------
# 2. Apply the approved workday eligibility rule.
# ------------------------------------------------------------

activity["eligible_day"] = (
    (activity["is_weekend"] == 0)
    & (activity["logon_count"] > 0)
)

eligible = activity.loc[activity["eligible_day"]].copy()

# ------------------------------------------------------------
# 3. Identify existing daily activity in each channel.
# ------------------------------------------------------------

eligible["email_active"] = (
    eligible["external_email_count"] > 0
)

eligible["usb_active"] = (
    eligible["usb_copy_count"] > 0
)

eligible["http_active"] = (
    eligible["www_upload_count"] > 0
)

eligible["active_channel_count"] = (
    eligible[
        ["email_active", "usb_active", "http_active"]
    ].sum(axis=1)
)

# ------------------------------------------------------------
# 4. Calculate per-user daily channel availability.
# ------------------------------------------------------------

profiles = eligible.groupby("user").agg(
    calculated_eligible_workdays=("day", "nunique"),
    email_active_days=("email_active", "sum"),
    usb_active_days=("usb_active", "sum"),
    http_active_days=("http_active", "sum"),
    no_channel_days=(
        "active_channel_count",
        lambda x: int((x == 0).sum())
    ),
    one_channel_days=(
        "active_channel_count",
        lambda x: int((x == 1).sum())
    ),
    two_channel_days=(
        "active_channel_count",
        lambda x: int((x == 2).sum())
    ),
    three_channel_days=(
        "active_channel_count",
        lambda x: int((x == 3).sum())
    )
)

# Include all assigned users, including any with zero eligible days.
profiles = profiles.reindex(assigned_users, fill_value=0)
profiles.index.name = "user"
profiles = profiles.reset_index()

profiles = assignments[
    ["user", "scenario_id", "duration", "eligible_workdays"]
].merge(
    profiles,
    on="user",
    how="left",
    validate="one_to_one"
)

profiles["workday_difference"] = (
    profiles["calculated_eligible_workdays"]
    - profiles["eligible_workdays"]
)

profiles["any_channel_days"] = (
    profiles["one_channel_days"]
    + profiles["two_channel_days"]
    + profiles["three_channel_days"]
)

# ------------------------------------------------------------
# 5. Report workday consistency with Stage 2.
# ------------------------------------------------------------

workday_mismatches = profiles.loc[
    profiles["workday_difference"] != 0
]

print("\n" + "=" * 75)
print("A. WORKDAY ELIGIBILITY CONSISTENCY")
print("=" * 75)

print(f"Users checked: {len(profiles):,}")
print(f"Matching Stage 2 workday counts: "
      f"{len(profiles) - len(workday_mismatches):,}")
print(f"Workday count mismatches: {len(workday_mismatches):,}")

if not workday_mismatches.empty:
    print("\nWorkday mismatch samples:")
    display(
        workday_mismatches[
            [
                "user", "eligible_workdays",
                "calculated_eligible_workdays",
                "workday_difference"
            ]
        ].head(10)
    )

# ------------------------------------------------------------
# 6. Report channel activity availability.
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("B. DAILY CHANNEL ACTIVITY AVAILABILITY")
print("=" * 75)

channel_summary = pd.DataFrame({
    "Channel": ["Email", "USB", "HTTP"],
    "Users With Active Days": [
        int((profiles["email_active_days"] > 0).sum()),
        int((profiles["usb_active_days"] > 0).sum()),
        int((profiles["http_active_days"] > 0).sum())
    ],
    "Total Active User-Days": [
        int(profiles["email_active_days"].sum()),
        int(profiles["usb_active_days"].sum()),
        int(profiles["http_active_days"].sum())
    ],
    "Median Active Days Per User": [
        float(profiles["email_active_days"].median()),
        float(profiles["usb_active_days"].median()),
        float(profiles["http_active_days"].median())
    ]
})

display(channel_summary)

print("\n" + "=" * 75)
print("C. SIMULTANEOUS CHANNEL AVAILABILITY")
print("=" * 75)

simultaneous_summary = pd.DataFrame({
    "Active Channels": [0, 1, 2, 3],
    "Total Eligible User-Days": [
        int(profiles["no_channel_days"].sum()),
        int(profiles["one_channel_days"].sum()),
        int(profiles["two_channel_days"].sum()),
        int(profiles["three_channel_days"].sum())
    ],
    "Users With At Least One Such Day": [
        int((profiles["no_channel_days"] > 0).sum()),
        int((profiles["one_channel_days"] > 0).sum()),
        int((profiles["two_channel_days"] > 0).sum()),
        int((profiles["three_channel_days"] > 0).sum())
    ]
})

display(simultaneous_summary)

# ------------------------------------------------------------
# 7. Preliminary availability indicators.
# These are NOT full scenario feasibility decisions.
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("D. PRELIMINARY PLAN A AVAILABILITY")
print("=" * 75)

preliminary = {
    "Users with no active-channel days":
        int((profiles["any_channel_days"] == 0).sum()),

    "Users with fewer active-channel days than duration":
        int((profiles["any_channel_days"] < profiles["duration"]).sum()),

    "Users with at least one two-channel day":
        int((profiles["two_channel_days"] > 0).sum()
            + (profiles["three_channel_days"] > 0).sum()
            - ((profiles["two_channel_days"] > 0)
               & (profiles["three_channel_days"] > 0)).sum()),

    "Users with at least one three-channel day":
        int((profiles["three_channel_days"] > 0).sum())
}

display(
    pd.DataFrame(
        preliminary.items(),
        columns=["Indicator", "Users"]
    )
)

print("\nPER-USER PROFILE SAMPLE:")
display(profiles.head(10))

print("\nAUDIT COMPLETED.")
print("No source files or assignments were modified.")
print("Final scenario feasibility has NOT yet been determined.")


STAGE 3 — STEP 3.1.4: DAILY ACTIVITY & TEMPORAL AUDIT

Attack period: 2011-01-01 to 2011-06-01
Assigned users: 995
User-Day records in scope: 109,225
Users with records in scope: 995

A. WORKDAY ELIGIBILITY CONSISTENCY
Users checked: 995
Matching Stage 2 workday counts: 995
Workday count mismatches: 0

B. DAILY CHANNEL ACTIVITY AVAILABILITY


,Channel,Users With Active Days,Total Active User-Days,Median Active Days Per User
0,Email,995,58710,70.0
1,USB,719,30018,21.0
2,HTTP,567,13898,18.0



C. SIMULTANEOUS CHANNEL AVAILABILITY


,Active Channels,Total Eligible User-Days,Users With At Least One Such Day
0,0,26804,943
1,1,51966,995
2,2,23215,985
3,3,1410,239



D. PRELIMINARY PLAN A AVAILABILITY


,Indicator,Users
0,Users with no active-channel days,0
1,Users with fewer active-channel days than dura...,20
2,Users with at least one two-channel day,985
3,Users with at least one three-channel day,239



PER-USER PROFILE SAMPLE:


,user,scenario_id,duration,eligible_workdays,calculated_eligible_workdays,email_active_days,usb_active_days,http_active_days,no_channel_days,one_channel_days,two_channel_days,three_channel_days,workday_difference,any_channel_days
0,AAC0610,SCN_0108,30,105,105,34,53,0,34,55,16,0,0,71
1,AAF0819,SCN_0007,5,105,105,73,89,0,6,36,63,0,0,99
2,AAP0352,SCN_0016,10,105,105,67,25,0,31,56,18,0,0,74
3,AAP1919,SCN_0040,20,105,105,84,25,0,17,67,21,0,0,88
4,AAS2987,SCN_0109,30,105,105,40,0,18,55,42,8,0,0,50
5,ABD3426,SCN_0402,40,106,106,92,105,20,1,10,78,17,0,105
6,ABH0349,SCN_0462,50,105,105,16,29,21,49,47,8,1,0,56
7,ABK3081,SCN_0009,5,106,106,83,23,0,16,74,16,0,0,90
8,ABM3203,SCN_0148,30,106,106,67,0,14,32,67,7,0,0,74
9,ABM3772,SCN_0008,5,105,105,87,47,0,9,58,38,0,0,96



AUDIT COMPLETED.
No source files or assignments were modified.
Final scenario feasibility has NOT yet been determined.



### Step 3.1.5 — Preliminary Plan A Scenario Feasibility

**Objective:** Evaluate whether each assigned user's available daily channel activity can satisfy the minimum requirements of their approved scenario under Plan A.

**Plan A Rule:** Inject activity only into channels that already have benign activity on the same eligible workday.

**Validation Tasks:**
1. Link each assigned user to their approved scenario requirements.
2. Compare available channel-active days with required attack-active days.
3. Check availability for two-channel and three-channel attack days.
4. Verify scenario day-count consistency.
5. Identify users who fail the minimum feasibility conditions.

**Important:** Passing these checks does not guarantee a valid attack schedule. Consecutive windows, off-day placement, channel switching, and temporal ordering will be evaluated separately.

**Expected Outcome:** Identify preliminary Plan A feasibility constraints without modifying the approved assignments or source datasets.


In [6]:

# Stage 3 — Step 3.1.5: Preliminary Plan A Scenario Feasibility
# Goal: Compare each assigned scenario's minimum daily channel
# activity requirements with the user's actual channel availability.
# Read-only: No original datasets or assignments will be modified.

import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.1.5: PRELIMINARY PLAN A FEASIBILITY")
print("=" * 75)

# ------------------------------------------------------------
# 1. Prepare approved scenario requirements.
# Rename overlapping columns BEFORE merging to avoid KeyError.
# ------------------------------------------------------------

scenario_requirements = scenario_catalog[
    [
        "scenario_id",
        "active_days",
        "off_days",
        "single_channel_days",
        "two_channel_days",
        "three_channel_days",
        "target_switches"
    ]
].copy()

scenario_requirements = scenario_requirements.rename(
    columns={
        "two_channel_days": "required_two_channel_days",
        "three_channel_days": "required_three_channel_days"
    }
)

# ------------------------------------------------------------
# 2. Combine user activity profiles with scenario requirements.
# ------------------------------------------------------------

plan_a_necessary = profiles.merge(
    scenario_requirements,
    on="scenario_id",
    how="left",
    validate="many_to_one"
)

assert len(plan_a_necessary) == len(assignments), (
    "Unexpected row count after scenario join."
)

required_columns = [
    "active_days",
    "off_days",
    "single_channel_days",
    "required_two_channel_days",
    "required_three_channel_days",
    "target_switches"
]

assert plan_a_necessary[required_columns].notna().all().all(), (
    "Missing scenario requirements after join."
)

assert plan_a_necessary["user"].is_unique, (
    "Duplicate users detected after scenario join."
)

# ------------------------------------------------------------
# 3. Calculate daily activity capacity.
# ------------------------------------------------------------

plan_a_necessary["available_multichannel_days"] = (
    plan_a_necessary["two_channel_days"]
    + plan_a_necessary["three_channel_days"]
)

plan_a_necessary["required_multichannel_days"] = (
    plan_a_necessary["required_two_channel_days"]
    + plan_a_necessary["required_three_channel_days"]
)

# A day with 3 active channels can support a 2-channel attack.
# A day with 2 or 3 active channels can support a 1-channel attack.
# These counts represent preliminary capacity, not a final schedule.

# ------------------------------------------------------------
# 4. Evaluate necessary feasibility conditions.
# ------------------------------------------------------------

checks = {
    "duration_capacity": (
        plan_a_necessary["calculated_eligible_workdays"]
        >= plan_a_necessary["duration"]
    ),

    "active_day_capacity": (
        plan_a_necessary["any_channel_days"]
        >= plan_a_necessary["active_days"]
    ),

    "multichannel_capacity": (
        plan_a_necessary["available_multichannel_days"]
        >= plan_a_necessary["required_multichannel_days"]
    ),

    "three_channel_capacity": (
        plan_a_necessary["three_channel_days"]
        >= plan_a_necessary["required_three_channel_days"]
    ),

    "scenario_day_partition": (
        plan_a_necessary["active_days"]
        + plan_a_necessary["off_days"]
        == plan_a_necessary["duration"]
    ),

    "scenario_channel_partition": (
        plan_a_necessary["single_channel_days"]
        + plan_a_necessary["required_two_channel_days"]
        + plan_a_necessary["required_three_channel_days"]
        == plan_a_necessary["active_days"]
    ),

    "switch_count_bounds": (
        (plan_a_necessary["target_switches"] >= 0)
        & (
            plan_a_necessary["target_switches"]
            <= plan_a_necessary["active_days"] - 1
        )
    )
}

for name, condition in checks.items():
    plan_a_necessary[f"pass_{name}"] = condition

check_columns = [f"pass_{name}" for name in checks]

plan_a_necessary["passes_necessary_conditions"] = (
    plan_a_necessary[check_columns].all(axis=1)
)

# ------------------------------------------------------------
# 5. Summarize individual condition results.
# ------------------------------------------------------------

condition_results = pd.DataFrame([
    {
        "Condition": name,
        "Passed Users": int(condition.sum()),
        "Failed Users": int((~condition).sum())
    }
    for name, condition in checks.items()
])

print("\nA. NECESSARY CONDITION RESULTS")
display(condition_results)

# ------------------------------------------------------------
# 6. Summarize overall preliminary feasibility.
# ------------------------------------------------------------

passed_users = int(
    plan_a_necessary["passes_necessary_conditions"].sum()
)

failed_users = len(plan_a_necessary) - passed_users

print("\n" + "=" * 75)
print("B. OVERALL PRELIMINARY FEASIBILITY")
print("=" * 75)

print(f"Assigned users: {len(plan_a_necessary):,}")
print(f"Passing necessary conditions: {passed_users:,}")
print(f"Failing necessary conditions: {failed_users:,}")

print(
    "\nNOTE: PASS means NOT RULED OUT."
    "\nA valid temporal attack schedule has NOT yet been established."
)

# ------------------------------------------------------------
# 7. Summarize feasibility by scenario duration.
# ------------------------------------------------------------

duration_summary = (
    plan_a_necessary
    .groupby("duration")
    .agg(
        Users=("user", "count"),
        Passing_Necessary_Conditions=(
            "passes_necessary_conditions", "sum"
        )
    )
    .reset_index()
)

duration_summary["Failing_Necessary_Conditions"] = (
    duration_summary["Users"]
    - duration_summary["Passing_Necessary_Conditions"]
)

print("\nC. FEASIBILITY BY SCENARIO DURATION")
display(duration_summary)

# ------------------------------------------------------------
# 8. Identify users failing necessary conditions.
# ------------------------------------------------------------

failed_cases = plan_a_necessary.loc[
    ~plan_a_necessary["passes_necessary_conditions"]
].copy()

failed_cases["failed_conditions"] = failed_cases.apply(
    lambda row: ", ".join(
        name for name in checks
        if not row[f"pass_{name}"]
    ),
    axis=1
)

print("\nD. USERS FAILING NECESSARY CONDITIONS")

if failed_cases.empty:
    print("No users failed the necessary conditions.")
else:
    display(
        failed_cases[
            [
                "user",
                "scenario_id",
                "duration",
                "active_days",
                "any_channel_days",
                "required_two_channel_days",
                "required_three_channel_days",
                "available_multichannel_days",
                "three_channel_days",
                "failed_conditions"
            ]
        ].head(20)
    )

# ------------------------------------------------------------
# 9. Final audit summary.
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("AUDIT COMPLETED")
print("=" * 75)
print("No source datasets or assignments were modified.")
print("Full temporal scheduling feasibility is NOT yet established.")


STAGE 3 — STEP 3.1.5: PRELIMINARY PLAN A FEASIBILITY

A. NECESSARY CONDITION RESULTS


,Condition,Passed Users,Failed Users
0,duration_capacity,995,0
1,active_day_capacity,989,6
2,multichannel_capacity,971,24
3,three_channel_capacity,933,62
4,scenario_day_partition,995,0
5,scenario_channel_partition,995,0
6,switch_count_bounds,995,0



B. OVERALL PRELIMINARY FEASIBILITY
Assigned users: 995
Passing necessary conditions: 915
Failing necessary conditions: 80

NOTE: PASS means NOT RULED OUT.
A valid temporal attack schedule has NOT yet been established.

C. FEASIBILITY BY SCENARIO DURATION


,duration,Users,Passing_Necessary_Conditions,Failing_Necessary_Conditions
0,5,176,176,0
1,10,176,171,5
2,20,177,173,4
3,30,177,169,8
4,40,144,118,26
5,50,145,108,37



D. USERS FAILING NECESSARY CONDITIONS


,user,scenario_id,duration,active_days,any_channel_days,required_two_channel_days,required_three_channel_days,available_multichannel_days,three_channel_days,failed_conditions
16,ACG3819,SCN_0996,50,40,69,5,1,18,0,three_channel_capacity
24,ADM0923,SCN_0473,50,42,96,4,1,23,0,three_channel_capacity
38,AGL1301,SCN_0474,50,42,76,4,1,18,0,three_channel_capacity
53,AKH1534,SCN_0213,40,34,41,4,1,12,0,three_channel_capacity
61,ANB2766,SCN_0489,50,42,50,4,2,9,0,three_channel_capacity
63,ANF3332,SCN_0502,50,42,39,5,1,4,0,"active_day_capacity, multichannel_capacity, th..."
70,APP3678,SCN_0049,20,18,30,2,0,0,0,multichannel_capacity
79,ASW1918,SCN_0514,50,42,82,5,1,18,0,three_channel_capacity
108,BDP3912,SCN_0032,10,8,19,1,0,0,0,multichannel_capacity
128,BMC3636,SCN_0447,40,32,44,4,1,7,0,three_channel_capacity



AUDIT COMPLETED
No source datasets or assignments were modified.
Full temporal scheduling feasibility is NOT yet established.



### Step 3.1.6 — Plan A Failure Diagnosis & Alternative Feasibility Audit

**Objective:** Investigate users who failed the preliminary Plan A conditions and assess whether alternative approved scenario templates may satisfy their daily channel activity constraints.

**Validation Tasks:**
1. Identify individual and overlapping failure reasons.
2. Analyze failures by scenario duration.
3. Search the approved scenario catalog for alternative templates.
4. Preserve the original scenario duration and stealth level.
5. Evaluate alternative templates against daily channel activity capacity.
6. Identify users with no preliminary alternatives.

**Constraints:**
- Plan A requires existing benign activity in every injected channel on the same day.
- Original Stage 2 assignments must remain unchanged.
- Alternative templates are diagnostic candidates only.
- Full Stage 2 compatibility and temporal scheduling are not established by this audit.

**Expected Outcome:** Understand whether the observed Plan A failures may be addressed through alternative approved scenarios before considering Plan B.

**Note:** Read-only analysis. No reassignment, injection, or file export.


In [7]:

# Stage 3 — Step 3.1.6: Plan A Failure Diagnosis & Alternative Feasibility Audit
# Goal: Diagnose preliminary Plan A failures and examine alternative
# approved templates with the same duration and stealth level.
# Read-only: No assignments, scenarios, or source datasets are modified.

import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.1.6: PLAN A FAILURE DIAGNOSIS")
print("=" * 75)

# ------------------------------------------------------------
# 1. Identify users failing the preliminary Plan A conditions.
# ------------------------------------------------------------

failed_audit = plan_a_necessary.loc[
    ~plan_a_necessary["passes_necessary_conditions"]
].copy()

assert failed_audit["user"].is_unique, (
    "Duplicate users found in failed cases."
)

print(f"\nUsers evaluated: {len(plan_a_necessary):,}")
print(f"Users failing necessary conditions: {len(failed_audit):,}")

if len(failed_audit) != 80:
    print(
        "WARNING: Failure count differs from the previously "
        "observed 80 users. Review Step 3.1.5 results."
    )

# ------------------------------------------------------------
# 2. Diagnose overlapping failures.
# ------------------------------------------------------------

failure_flags = {
    "Active-Day Capacity": "pass_active_day_capacity",
    "Multichannel Capacity": "pass_multichannel_capacity",
    "Three-Channel Capacity": "pass_three_channel_capacity",
    "Other Scenario Conditions": None
}

primary_flags = {
    key: column
    for key, column in failure_flags.items()
    if column is not None
}

def identify_failures(row):
    reasons = [
        name
        for name, column in primary_flags.items()
        if not bool(row[column])
    ]

    if not reasons:
        reasons = ["Other Scenario Conditions"]

    return " + ".join(reasons)

failed_audit["failure_combination"] = (
    failed_audit.apply(identify_failures, axis=1)
)

print("\nA. FAILURE CONDITION OVERLAP")

failure_overlap = (
    failed_audit["failure_combination"]
    .value_counts()
    .rename_axis("Failure Combination")
    .reset_index(name="Users")
)

display(failure_overlap)

print("\nB. FAILED USERS BY DURATION")

failed_by_duration = (
    failed_audit
    .groupby("duration")
    .size()
    .reset_index(name="Failed Users")
    .sort_values("duration")
)

display(failed_by_duration)

# ------------------------------------------------------------
# 3. Attach the original assigned stealth level.
# ------------------------------------------------------------

original_stealth = assignments[
    ["user", "stealth_level"]
].copy()

failed_audit = failed_audit.merge(
    original_stealth,
    on="user",
    how="left",
    validate="one_to_one"
)

assert failed_audit["stealth_level"].notna().all(), (
    "Missing stealth level for a failed user."
)

# ------------------------------------------------------------
# 4. Prepare candidate scenarios from the approved catalog.
# Rename scenario fields before merging.
# ------------------------------------------------------------

candidate_catalog = scenario_catalog[
    [
        "scenario_id",
        "duration",
        "stealth_level",
        "active_days",
        "off_days",
        "single_channel_days",
        "two_channel_days",
        "three_channel_days",
        "target_switches"
    ]
].copy()

candidate_catalog = candidate_catalog.rename(
    columns={
        "scenario_id": "candidate_scenario_id",
        "active_days": "candidate_active_days",
        "off_days": "candidate_off_days",
        "single_channel_days": "candidate_single_days",
        "two_channel_days": "candidate_two_days",
        "three_channel_days": "candidate_three_days",
        "target_switches": "candidate_switches"
    }
)

# ------------------------------------------------------------
# 5. Generate diagnostic candidate combinations.
# Keep original duration and stealth level unchanged.
# ------------------------------------------------------------

candidate_users = failed_audit[
    [
        "user",
        "scenario_id",
        "duration",
        "stealth_level",
        "calculated_eligible_workdays",
        "any_channel_days",
        "available_multichannel_days",
        "three_channel_days"
    ]
].copy()

candidates = candidate_users.merge(
    candidate_catalog,
    on=["duration", "stealth_level"],
    how="left",
    validate="many_to_many"
)

# Exclude the user's original assigned scenario.
candidates = candidates.loc[
    candidates["candidate_scenario_id"].notna()
    &
    (
        candidates["candidate_scenario_id"]
        != candidates["scenario_id"]
    )
].copy()

# ------------------------------------------------------------
# 6. Evaluate preliminary candidate capacity.
# These are necessary conditions, not full scheduling tests.
# ------------------------------------------------------------

candidates["candidate_required_multichannel_days"] = (
    candidates["candidate_two_days"]
    + candidates["candidate_three_days"]
)

candidate_checks = {
    "duration": (
        candidates["calculated_eligible_workdays"]
        >= candidates["duration"]
    ),

    "active_days": (
        candidates["any_channel_days"]
        >= candidates["candidate_active_days"]
    ),

    "multichannel": (
        candidates["available_multichannel_days"]
        >= candidates["candidate_required_multichannel_days"]
    ),

    "three_channel": (
        candidates["three_channel_days"]
        >= candidates["candidate_three_days"]
    ),

    "day_partition": (
        candidates["candidate_active_days"]
        + candidates["candidate_off_days"]
        == candidates["duration"]
    ),

    "channel_partition": (
        candidates["candidate_single_days"]
        + candidates["candidate_two_days"]
        + candidates["candidate_three_days"]
        == candidates["candidate_active_days"]
    ),

    "switch_bounds": (
        (candidates["candidate_switches"] >= 0)
        &
        (
            candidates["candidate_switches"]
            <= candidates["candidate_active_days"] - 1
        )
    )
}

for name, condition in candidate_checks.items():
    candidates[f"pass_{name}"] = condition

candidate_check_columns = [
    f"pass_{name}" for name in candidate_checks
]

candidates["passes_preliminary_checks"] = (
    candidates[candidate_check_columns].all(axis=1)
)

valid_candidates = candidates.loc[
    candidates["passes_preliminary_checks"]
].copy()

# ------------------------------------------------------------
# 7. Count preliminary alternatives for each failed user.
# ------------------------------------------------------------

alternative_counts = (
    valid_candidates
    .groupby("user")["candidate_scenario_id"]
    .nunique()
    .rename("potential_alternatives")
)

failed_audit["potential_alternatives"] = (
    failed_audit["user"]
    .map(alternative_counts)
    .fillna(0)
    .astype(int)
)

failed_audit["has_preliminary_alternative"] = (
    failed_audit["potential_alternatives"] > 0
)

with_alternatives = int(
    failed_audit["has_preliminary_alternative"].sum()
)

without_alternatives = int(
    (~failed_audit["has_preliminary_alternative"]).sum()
)

print("\n" + "=" * 75)
print("C. PRELIMINARY ALTERNATIVE AVAILABILITY")
print("=" * 75)

print(f"Failed users: {len(failed_audit):,}")
print(f"Users with potential alternatives: {with_alternatives:,}")
print(f"Users without preliminary alternatives: {without_alternatives:,}")
print(f"Total preliminary candidate templates: {len(valid_candidates):,}")

# ------------------------------------------------------------
# 8. Analyze potential recovery by scenario duration.
# ------------------------------------------------------------

duration_recovery = (
    failed_audit
    .groupby("duration")
    .agg(
        Failed_Users=("user", "count"),
        With_Potential_Alternatives=(
            "has_preliminary_alternative", "sum"
        )
    )
    .reset_index()
)

duration_recovery["Without_Potential_Alternatives"] = (
    duration_recovery["Failed_Users"]
    - duration_recovery["With_Potential_Alternatives"]
)

print("\nD. PRELIMINARY ALTERNATIVES BY DURATION")
display(duration_recovery)

# ------------------------------------------------------------
# 9. Display individual diagnostic results.
# ------------------------------------------------------------

diagnostic_columns = [
    "user",
    "scenario_id",
    "duration",
    "stealth_level",
    "failure_combination",
    "potential_alternatives"
]

print("\nE. FAILED USER DIAGNOSTIC SAMPLE")

display(
    failed_audit[diagnostic_columns]
    .sort_values(
        ["potential_alternatives", "duration", "user"]
    )
    .head(25)
)

print("\nF. USERS WITHOUT PRELIMINARY ALTERNATIVES")

no_alternatives = failed_audit.loc[
    ~failed_audit["has_preliminary_alternative"],
    diagnostic_columns
].copy()

if no_alternatives.empty:
    print(
        "Every failed user has at least one preliminary "
        "alternative candidate."
    )
else:
    display(
        no_alternatives.sort_values(
            ["duration", "user"]
        )
    )

# ------------------------------------------------------------
# 10. Final audit statement.
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 3.1.6 — AUDIT COMPLETED")
print("=" * 75)

print("Original Stage 2 assignments: UNCHANGED")
print("Approved scenario catalog: UNCHANGED")
print("Synthetic attack injection: NOT STARTED")
print("Plan B: NOT ACTIVATED")

print(
    "\nIMPORTANT: Candidate scenarios satisfy only preliminary "
    "daily activity conditions."
)
print(
    "Historical channel compatibility, Stage 2 assignment rules, "
    "temporal windows, and switching feasibility still require validation."
)


STAGE 3 — STEP 3.1.6: PLAN A FAILURE DIAGNOSIS

Users evaluated: 995
Users failing necessary conditions: 80

A. FAILURE CONDITION OVERLAP


,Failure Combination,Users
0,Three-Channel Capacity,55
1,Multichannel Capacity,16
2,Active-Day Capacity + Multichannel Capacity + ...,3
3,Multichannel Capacity + Three-Channel Capacity,3
4,Active-Day Capacity + Multichannel Capacity,2
5,Active-Day Capacity + Three-Channel Capacity,1



B. FAILED USERS BY DURATION


,duration,Failed Users
0,10,5
1,20,4
2,30,8
3,40,26
4,50,37



C. PRELIMINARY ALTERNATIVE AVAILABILITY
Failed users: 80
Users with potential alternatives: 10
Users without preliminary alternatives: 70
Total preliminary candidate templates: 1,140

D. PRELIMINARY ALTERNATIVES BY DURATION


,duration,Failed_Users,With_Potential_Alternatives,Without_Potential_Alternatives
0,10,5,0,5
1,20,4,0,4
2,30,8,0,8
3,40,26,0,26
4,50,37,10,27



E. FAILED USER DIAGNOSTIC SAMPLE


,user,scenario_id,duration,stealth_level,failure_combination,potential_alternatives
8,BDP3912,SCN_0032,10,Medium,Multichannel Capacity,0
28,FRG1576,SCN_0015,10,Low,Multichannel Capacity,0
38,JBB0847,SCN_0020,10,Medium,Multichannel Capacity,0
68,SDH1556,SCN_0027,10,Low,Multichannel Capacity,0
70,SOM2865,SCN_0015,10,Low,Multichannel Capacity,0
6,APP3678,SCN_0049,20,High,Multichannel Capacity,0
19,DFP3615,SCN_0102,20,Low,Multichannel Capacity,0
52,LYS3814,SCN_0054,20,Low,Multichannel Capacity,0
60,NTH2822,SCN_0071,20,Medium,Multichannel Capacity,0
10,BRD3314,SCN_0136,30,High,Active-Day Capacity + Multichannel Capacity,0



F. USERS WITHOUT PRELIMINARY ALTERNATIVES


,user,scenario_id,duration,stealth_level,failure_combination,potential_alternatives
8,BDP3912,SCN_0032,10,Medium,Multichannel Capacity,0
28,FRG1576,SCN_0015,10,Low,Multichannel Capacity,0
38,JBB0847,SCN_0020,10,Medium,Multichannel Capacity,0
68,SDH1556,SCN_0027,10,Low,Multichannel Capacity,0
70,SOM2865,SCN_0015,10,Low,Multichannel Capacity,0
...,...,...,...,...,...,...
69,SKS0339,SCN_1068,50,Low,Three-Channel Capacity,0
72,TDJ3138,SCN_1069,50,High,Three-Channel Capacity,0
74,UMF2894,SCN_0954,50,Low,Three-Channel Capacity,0
75,WBH1861,SCN_1084,50,High,Three-Channel Capacity,0



STEP 3.1.6 — AUDIT COMPLETED
Original Stage 2 assignments: UNCHANGED
Approved scenario catalog: UNCHANGED
Synthetic attack injection: NOT STARTED
Plan B: NOT ACTIVATED

IMPORTANT: Candidate scenarios satisfy only preliminary daily activity conditions.
Historical channel compatibility, Stage 2 assignment rules, temporal windows, and switching feasibility still require validation.



### Step 3.1.7 — Alternative Scenario Selection & Unified Preliminary Cohort

**Objective:** Select one suitable alternative scenario per recoverable user and combine those users with the original Plan A preliminary-pass cohort.

**Selection Rules:**
1. Use approved scenario templates only.
2. Preserve the original duration and stealth level.
3. Preserve Stage 2 assignment-group and historical channel eligibility.
4. Prioritize alternatives closest to the original scenario structure.
5. Apply the same seven preliminary feasibility conditions to both original and alternative assignments.
6. Defer users without suitable alternatives.

**Validation Scope:**
- Eligible workday capacity.
- Active-day capacity.
- Multichannel capacity.
- Three-channel capacity.
- Scenario day partition.
- Scenario channel partition.
- Theoretical switching-count bounds.

**Important Limitations:**
- Passing these checks does not establish temporal scheduling feasibility.
- Actual channel-switching sequences are not tested.
- The unresolved three-channel activity problem is deferred.
- Stage 2 global assignment diversity and reuse distributions may change and must be reviewed before approving any reassignment.

**Expected Outcome:** Create a provisional unified preliminary-pass cohort and a separate deferred-user cohort.

**Note:** Read-only analysis. Original Stage 2 assignments remain unchanged. No scheduling, injection, or file export.


In [8]:

# Stage 3 — Step 3.1.7: Alternative Selection & Unified Preliminary Cohort
# Goal: Select suitable preliminary alternatives, apply the same
# seven checks to all users, and form a provisional unified cohort.
# Read-only: No approved assignments or source files are modified.

import pandas as pd
import numpy as np
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.1.7: ALTERNATIVE SELECTION & UNIFICATION")
print("=" * 75)

# ------------------------------------------------------------
# 1. Verify previous audit results.
# ------------------------------------------------------------

assert len(assignments) == 995
assert assignments["user"].is_unique
assert len(plan_a_necessary) == 995
assert plan_a_necessary["user"].is_unique

original_pass_users = set(
    plan_a_necessary.loc[
        plan_a_necessary["passes_necessary_conditions"], "user"
    ]
)

failed_users = set(assignments["user"]) - original_pass_users

assert len(original_pass_users) == 915
assert len(failed_users) == 80

# ------------------------------------------------------------
# 2. Prepare original scenario characteristics.
# ------------------------------------------------------------

original_rules = scenario_catalog[
    [
        "scenario_id", "active_days", "off_days",
        "single_channel_days", "two_channel_days",
        "three_channel_days", "target_switches"
    ]
].rename(columns={
    "scenario_id": "original_scenario_id",
    "active_days": "original_active_days",
    "off_days": "original_off_days",
    "single_channel_days": "original_single_days",
    "two_channel_days": "original_two_days",
    "three_channel_days": "original_three_days",
    "target_switches": "original_switches"
})

# ------------------------------------------------------------
# 3. Prepare candidate scenarios from Step 3.1.6.
# Only candidates passing its preliminary conditions.
# ------------------------------------------------------------

options = valid_candidates.copy()

options = options.loc[
    options["user"].isin(failed_users)
].copy()

options = options.merge(
    assignments[
        [
            "user", "assignment_group",
            "assignment_pool", "eligible_workdays"
        ]
    ],
    on="user",
    how="left",
    validate="many_to_one"
)

options = options.merge(
    original_rules,
    left_on="scenario_id",
    right_on="original_scenario_id",
    how="left",
    validate="many_to_one"
)

assert options["original_scenario_id"].notna().all()

# ------------------------------------------------------------
# 4. Recheck Stage 2 user eligibility.
# ------------------------------------------------------------

baseline_channels = user_baseline[
    [
        "user", "email_active_days", "usb_active_days",
        "http_upload_active_days",
        "email_count_p95", "usb_copy_p95",
        "http_upload_p95"
    ]
].copy()

baseline_channels["historical_channel_count"] = (
    baseline_channels[
        [
            "email_active_days",
            "usb_active_days",
            "http_upload_active_days"
        ]
    ].gt(0).sum(axis=1)
)

options = options.merge(
    baseline_channels,
    on="user",
    how="left",
    validate="many_to_one"
)

# The approved Stage 2 policy requires 3 historical channels
# for long-duration scenarios and 2 for short-duration scenarios.

options["required_historical_channels"] = np.where(
    options["duration"].isin([40, 50]), 3, 2
)

options["stage2_group_compatible"] = (
    (
        options["assignment_group"].eq(
            "Three-channel long-duration"
        )
        & options["duration"].isin([40, 50])
        & options["assignment_pool"].eq("Long-duration pool")
    )
    |
    (
        options["assignment_group"].isin([
            "Two-channel short-duration",
            "Three-channel short-duration exception"
        ])
        & options["duration"].isin([5, 10, 20, 30])
        & options["assignment_pool"].eq("Short-duration pool")
    )
)

options["stage2_channel_compatible"] = (
    options["historical_channel_count"]
    >= options["required_historical_channels"]
)

options["stage2_duration_compatible"] = (
    options["eligible_workdays"] >= options["duration"]
)

# Verify valid historical P95 values on the channels
# required by the user's historical eligibility.

p95_valid = pd.Series(True, index=options.index)

channel_baselines = [
    ("email_active_days", "email_count_p95"),
    ("usb_active_days", "usb_copy_p95"),
    ("http_upload_active_days", "http_upload_p95")
]

for activity_col, p95_col in channel_baselines:
    historical_active = options[activity_col] > 0
    p95 = pd.to_numeric(options[p95_col], errors="coerce")

    p95_valid &= (
        ~historical_active
        | (p95.notna() & np.isfinite(p95) & (p95 > 0))
    )

options["stage2_baseline_compatible"] = p95_valid

options["stage2_candidate_compatible"] = options[
    [
        "stage2_group_compatible",
        "stage2_channel_compatible",
        "stage2_duration_compatible",
        "stage2_baseline_compatible"
    ]
].all(axis=1)

eligible_options = options.loc[
    options["stage2_candidate_compatible"]
].copy()

# ------------------------------------------------------------
# 5. Rank compatible candidates by structural similarity.
# Smaller differences receive higher priority.
# ------------------------------------------------------------

comparison_pairs = [
    ("candidate_active_days", "original_active_days"),
    ("candidate_off_days", "original_off_days"),
    ("candidate_single_days", "original_single_days"),
    ("candidate_two_days", "original_two_days"),
    ("candidate_three_days", "original_three_days"),
    ("candidate_switches", "original_switches")
]

eligible_options["structural_distance"] = sum(
    (
        eligible_options[candidate_col]
        - eligible_options[original_col]
    ).abs()
    for candidate_col, original_col in comparison_pairs
)

# Deterministic selection; scenario_id resolves equal scores.
eligible_options = eligible_options.sort_values(
    ["user", "structural_distance", "candidate_scenario_id"],
    kind="stable"
)

selected_alternatives = (
    eligible_options
    .drop_duplicates(subset=["user"], keep="first")
    .copy()
)

assert selected_alternatives["user"].is_unique

print("\nA. ALTERNATIVE SELECTION")

print(f"Original preliminary-pass users: {len(original_pass_users)}")
print(f"Original failed users: {len(failed_users)}")
print(f"Candidates after Stage 2 checks: {len(eligible_options)}")
print(f"Users with selected alternatives: {len(selected_alternatives)}")

display(
    selected_alternatives[
        [
            "user", "scenario_id", "candidate_scenario_id",
            "duration", "stealth_level", "structural_distance"
        ]
    ]
)

# ------------------------------------------------------------
# 6. Construct a provisional cohort without changing Stage 2.
# ------------------------------------------------------------

alternative_map = selected_alternatives.set_index(
    "user"
)["candidate_scenario_id"]

unified_plan_a_317 = assignments.loc[
    assignments["user"].isin(
        original_pass_users | set(alternative_map.index)
    )
].copy()

unified_plan_a_317["original_scenario_id"] = (
    unified_plan_a_317["scenario_id"]
)

unified_plan_a_317["proposed_scenario_id"] = (
    unified_plan_a_317["user"]
    .map(alternative_map)
    .fillna(unified_plan_a_317["original_scenario_id"])
)

unified_plan_a_317["selection_type"] = np.where(
    unified_plan_a_317["user"].isin(alternative_map.index),
    "Proposed alternative",
    "Original assignment"
)

# Preserve Stage 2 assignment metadata as historical provenance.
# Original assignment_type is not a new classification.
unified_plan_a_317 = unified_plan_a_317.rename(
    columns={"assignment_type": "original_assignment_type"}
)

# ------------------------------------------------------------
# 7. Join the selected scenario rules and daily capacities.
# ------------------------------------------------------------

effective_rules = scenario_catalog[
    [
        "scenario_id", "duration", "stealth_level",
        "active_days", "off_days", "single_channel_days",
        "two_channel_days", "three_channel_days",
        "target_switches"
    ]
].rename(columns={
    "scenario_id": "proposed_scenario_id",
    "duration": "effective_duration",
    "stealth_level": "effective_stealth",
    "active_days": "effective_active_days",
    "off_days": "effective_off_days",
    "single_channel_days": "effective_single_days",
    "two_channel_days": "effective_two_days",
    "three_channel_days": "effective_three_days",
    "target_switches": "effective_switches"
})

unified_plan_a_317 = unified_plan_a_317.merge(
    effective_rules,
    on="proposed_scenario_id",
    how="left",
    validate="many_to_one"
)

activity_capacity = plan_a_necessary[
    [
        "user", "calculated_eligible_workdays",
        "any_channel_days", "available_multichannel_days",
        "three_channel_days"
    ]
].copy()

unified_plan_a_317 = unified_plan_a_317.merge(
    activity_capacity,
    on="user",
    how="left",
    validate="one_to_one"
)

assert unified_plan_a_317["user"].is_unique
assert unified_plan_a_317["effective_duration"].notna().all()

# ------------------------------------------------------------
# 8. Reapply EXACTLY the same seven preliminary conditions
# to originals and proposed alternatives.
# ------------------------------------------------------------

u = unified_plan_a_317

unified_checks = {
    "duration_capacity": (
        u["calculated_eligible_workdays"]
        >= u["effective_duration"]
    ),
    "active_day_capacity": (
        u["any_channel_days"]
        >= u["effective_active_days"]
    ),
    "multichannel_capacity": (
        u["available_multichannel_days"]
        >= (
            u["effective_two_days"]
            + u["effective_three_days"]
        )
    ),
    "three_channel_capacity": (
        u["three_channel_days"]
        >= u["effective_three_days"]
    ),
    "scenario_day_partition": (
        u["effective_active_days"]
        + u["effective_off_days"]
        == u["effective_duration"]
    ),
    "scenario_channel_partition": (
        u["effective_single_days"]
        + u["effective_two_days"]
        + u["effective_three_days"]
        == u["effective_active_days"]
    ),
    "switch_count_bounds": (
        (u["effective_switches"] >= 0)
        & (
            u["effective_switches"]
            <= u["effective_active_days"] - 1
        )
    )
}

for check_name, result in unified_checks.items():
    unified_plan_a_317[f"pass_{check_name}"] = result

unified_plan_a_317["passes_preliminary_checks"] = (
    unified_plan_a_317[
        [f"pass_{name}" for name in unified_checks]
    ].all(axis=1)
)

unified_plan_a_317["preserves_duration_and_stealth"] = (
    unified_plan_a_317["duration"].eq(
        unified_plan_a_317["effective_duration"]
    )
    & unified_plan_a_317["stealth_level"].eq(
        unified_plan_a_317["effective_stealth"]
    )
)

assert unified_plan_a_317[
    "passes_preliminary_checks"
].all(), "Unified preliminary feasibility validation failed."

assert unified_plan_a_317[
    "preserves_duration_and_stealth"
].all(), "Duration or stealth level was changed."

print("\nB. UNIFIED COHORT VALIDATION")

unified_validation = pd.DataFrame([
    {
        "Condition": name,
        "Passed": int(result.sum()),
        "Failed": int((~result).sum())
    }
    for name, result in unified_checks.items()
])

display(unified_validation)

# ------------------------------------------------------------
# 9. Identify deferred users.
# ------------------------------------------------------------

selected_users = set(unified_plan_a_317["user"])

deferred_users_317 = assignments.loc[
    ~assignments["user"].isin(selected_users)
].copy()

assert len(unified_plan_a_317) + len(deferred_users_317) == 995
assert set(unified_plan_a_317["user"]).isdisjoint(
    set(deferred_users_317["user"])
)

print("\nC. COHORT COMPOSITION")

display(
    unified_plan_a_317["selection_type"]
    .value_counts()
    .rename_axis("Selection Type")
    .reset_index(name="Users")
)

print(f"\nUnified preliminary-pass users: {len(unified_plan_a_317)}")
print(f"Deferred users: {len(deferred_users_317)}")

# ------------------------------------------------------------
# 10. Report possible changes to Stage 2 global distributions.
# Diagnostic only; do not treat as Stage 2 reapproval.
# ------------------------------------------------------------

original_usage = assignments["scenario_id"].value_counts()
proposed_usage = (
    unified_plan_a_317["proposed_scenario_id"].value_counts()
)

print("\nD. STAGE 2 DISTRIBUTION IMPACT — DIAGNOSTIC")

print(
    "Original Stage 2 distinct templates:",
    assignments["scenario_id"].nunique()
)
print(
    "Proposed cohort distinct templates:",
    unified_plan_a_317["proposed_scenario_id"].nunique()
)
print(
    "Original maximum template usage:",
    int(original_usage.max())
)
print(
    "Proposed cohort maximum template usage:",
    int(proposed_usage.max())
)

print(
    "\nCAUTION: The proposed cohort excludes deferred users."
    "\nThese figures do not constitute Stage 2 distribution approval."
)

# ------------------------------------------------------------
# 11. Final status.
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 3.1.7 — COMPLETED")
print("=" * 75)

print("Original Stage 2 assignments: UNCHANGED")
print("Proposed alternatives: NOT YET APPROVED")
print("Temporal attack scheduling: NOT STARTED")
print("Injection: NOT STARTED")
print("Plan B: NOT ACTIVATED")
print("No files were exported.")


STAGE 3 — STEP 3.1.7: ALTERNATIVE SELECTION & UNIFICATION

A. ALTERNATIVE SELECTION
Original preliminary-pass users: 915
Original failed users: 80
Candidates after Stage 2 checks: 1140
Users with selected alternatives: 10


,user,scenario_id,candidate_scenario_id,duration,stealth_level,structural_distance
26,ELW1413,SCN_0620,SCN_0617,50,Medium,2
147,HCB1383,SCN_0663,SCN_0660,50,Low,2
266,HMR0098,SCN_0692,SCN_0689,50,Medium,2
384,JDC2176,SCN_0715,SCN_0712,50,High,2
500,JKK1422,SCN_0727,SCN_0724,50,High,2
615,JMV0602,SCN_0734,SCN_0731,50,Medium,2
733,KJS0119,SCN_0757,SCN_0754,50,High,2
853,MCB2802,SCN_0795,SCN_0792,50,Low,2
998,XSF3982,SCN_0981,SCN_0978,50,Low,2
1139,YCL0824,SCN_1141,SCN_1138,50,High,2



B. UNIFIED COHORT VALIDATION


,Condition,Passed,Failed
0,duration_capacity,925,0
1,active_day_capacity,925,0
2,multichannel_capacity,925,0
3,three_channel_capacity,925,0
4,scenario_day_partition,925,0
5,scenario_channel_partition,925,0
6,switch_count_bounds,925,0



C. COHORT COMPOSITION


,Selection Type,Users
0,Original assignment,915
1,Proposed alternative,10



Unified preliminary-pass users: 925
Deferred users: 70

D. STAGE 2 DISTRIBUTION IMPACT — DIAGNOSTIC
Original Stage 2 distinct templates: 490
Proposed cohort distinct templates: 436
Original maximum template usage: 20
Proposed cohort maximum template usage: 20

CAUTION: The proposed cohort excludes deferred users.
These figures do not constitute Stage 2 distribution approval.

STEP 3.1.7 — COMPLETED
Original Stage 2 assignments: UNCHANGED
Proposed alternatives: NOT YET APPROVED
Temporal attack scheduling: NOT STARTED
Injection: NOT STARTED
Plan B: NOT ACTIVATED
No files were exported.



### Step 3.1.8 — Stage 2 Assignment Impact & Provenance Audit

**Objective:** Evaluate the impact of the 10 proposed scenario replacements on the complete Stage 2 assignment population.

**Validation Tasks:**
1. Preserve all 995 original users.
2. Apply proposed alternatives to a temporary comparison only.
3. Confirm that only the intended 10 users change scenarios.
4. Verify that duration and stealth level remain unchanged.
5. Compare scenario diversity and template reuse across the full population.
6. Inspect structural differences between original and alternative scenarios.
7. Confirm that the 70 deferred users retain their original assignments.

**Expected Outcome:** Determine whether the proposed alternatives introduce changes requiring further review before approval.

**Note:** Read-only audit. Original Stage 2 files remain unchanged. No attack scheduling or injection.


In [9]:

# Stage 3 — Step 3.1.8: Stage 2 Assignment Impact & Provenance Audit
# Goal: Compare the complete original 995 assignments against a
# temporary 995-user version containing the 10 proposed alternatives.
# Read-only: No source files, assignments, or datasets are modified.

import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.1.8: ASSIGNMENT IMPACT AUDIT")
print("=" * 75)

# ------------------------------------------------------------
# 1. Create a temporary full-population comparison.
# ------------------------------------------------------------

comparison_318 = assignments.copy()

alternative_map_318 = selected_alternatives.set_index(
    "user"
)["candidate_scenario_id"]

comparison_318["proposed_scenario_id"] = (
    comparison_318["user"]
    .map(alternative_map_318)
    .fillna(comparison_318["scenario_id"])
)

comparison_318["scenario_changed"] = (
    comparison_318["scenario_id"]
    != comparison_318["proposed_scenario_id"]
)

assert len(comparison_318) == 995
assert comparison_318["user"].is_unique

changed_318 = comparison_318.loc[
    comparison_318["scenario_changed"]
].copy()

assert set(changed_318["user"]) == set(
    selected_alternatives["user"]
), "Unexpected scenario changes detected."

assert not comparison_318.loc[
    comparison_318["user"].isin(deferred_users_317["user"]),
    "scenario_changed"
].any(), "A deferred user's scenario was changed."

print("\nA. ASSIGNMENT PRESERVATION")

print(f"Original assigned users: {len(assignments)}")
print(f"Users in comparison: {len(comparison_318)}")
print(f"Proposed scenario changes: {len(changed_318)}")
print(f"Unchanged assignments: "
      f"{int((~comparison_318['scenario_changed']).sum())}")
print(f"Deferred users retained: {len(deferred_users_317)}")

# ------------------------------------------------------------
# 2. Compare original and proposed scenario attributes.
# ------------------------------------------------------------

attributes_318 = [
    "duration",
    "stealth_level",
    "active_days",
    "off_days",
    "single_channel_days",
    "two_channel_days",
    "three_channel_days",
    "target_switches",
    "structural_pattern_id"
]

scenario_lookup_318 = scenario_catalog.set_index("scenario_id")

original_attributes_318 = scenario_lookup_318[
    attributes_318
].add_prefix("original_")

proposed_attributes_318 = scenario_lookup_318[
    attributes_318
].add_prefix("proposed_")

comparison_318 = comparison_318.join(
    original_attributes_318,
    on="scenario_id",
    validate="many_to_one"
)

comparison_318 = comparison_318.join(
    proposed_attributes_318,
    on="proposed_scenario_id",
    validate="many_to_one"
)

assert comparison_318[
    ["original_duration", "proposed_duration",
     "original_stealth_level", "proposed_stealth_level"]
].notna().all().all()

duration_preserved = (
    comparison_318["original_duration"]
    == comparison_318["proposed_duration"]
).all()

stealth_preserved = (
    comparison_318["original_stealth_level"]
    == comparison_318["proposed_stealth_level"]
).all()

print("\nB. PRESERVED ASSIGNMENT DIMENSIONS")

print(f"Duration preserved: {duration_preserved}")
print(f"Stealth preserved: {stealth_preserved}")

assert duration_preserved and stealth_preserved

# ------------------------------------------------------------
# 3. Compare scenario diversity and reuse for all 995 users.
# ------------------------------------------------------------

original_usage_318 = (
    comparison_318["scenario_id"].value_counts()
)

proposed_usage_318 = (
    comparison_318["proposed_scenario_id"].value_counts()
)

distribution_318 = pd.DataFrame([
    {
        "Metric": "Assigned users",
        "Original Stage 2": len(comparison_318),
        "Proposed Full Population": len(comparison_318)
    },
    {
        "Metric": "Distinct scenario templates",
        "Original Stage 2": original_usage_318.size,
        "Proposed Full Population": proposed_usage_318.size
    },
    {
        "Metric": "Maximum template reuse",
        "Original Stage 2": int(original_usage_318.max()),
        "Proposed Full Population": int(proposed_usage_318.max())
    },
    {
        "Metric": "Distinct structural patterns",
        "Original Stage 2": comparison_318[
            "original_structural_pattern_id"
        ].nunique(),
        "Proposed Full Population": comparison_318[
            "proposed_structural_pattern_id"
        ].nunique()
    }
])

print("\nC. GLOBAL DISTRIBUTION COMPARISON")
display(distribution_318)

# ------------------------------------------------------------
# 4. Identify which attributes changed for the 10 users.
# ------------------------------------------------------------

changed_318 = comparison_318.loc[
    comparison_318["scenario_changed"]
].copy()

for attribute in attributes_318:
    changed_318[f"changed_{attribute}"] = (
        changed_318[f"original_{attribute}"]
        != changed_318[f"proposed_{attribute}"]
    )

attribute_changes_318 = pd.DataFrame([
    {
        "Attribute": attribute,
        "Changed Users": int(
            changed_318[f"changed_{attribute}"].sum()
        )
    }
    for attribute in attributes_318
])

print("\nD. STRUCTURAL CHANGES IN THE 10 ALTERNATIVES")
display(attribute_changes_318)

print("\nE. INDIVIDUAL ALTERNATIVE COMPARISON")

display(
    changed_318[
        [
            "user", "scenario_id", "proposed_scenario_id",
            "original_active_days", "proposed_active_days",
            "original_two_channel_days",
            "proposed_two_channel_days",
            "original_three_channel_days",
            "proposed_three_channel_days",
            "original_target_switches",
            "proposed_target_switches"
        ]
    ].sort_values("user")
)

# ------------------------------------------------------------
# 5. Identify possible template-reuse concerns.
# The observed Stage 2 maximum was 20; this is a diagnostic
# comparison, not an assumption that 20 is a formal quota.
# ------------------------------------------------------------

original_max_318 = int(original_usage_318.max())

usage_over_original_max_318 = proposed_usage_318[
    proposed_usage_318 > original_max_318
]

print("\nF. TEMPLATE REUSE DIAGNOSTIC")

print(f"Original maximum reuse: {original_max_318}")
print(f"Proposed maximum reuse: {int(proposed_usage_318.max())}")
print(
    "Templates exceeding original maximum:",
    len(usage_over_original_max_318)
)

if not usage_over_original_max_318.empty:
    display(usage_over_original_max_318)

# ------------------------------------------------------------
# 6. Final summary.
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 3.1.8 — AUDIT SUMMARY")
print("=" * 75)

print(f"Original preliminary-pass users: 915")
print(f"Proposed alternatives: {len(changed_318)}")
print(f"Provisional unified cohort: {len(unified_plan_a_317)}")
print(f"Deferred users: {len(deferred_users_317)}")

print("\nOriginal Stage 2 assignments: UNCHANGED")
print("Proposed alternatives: NOT YET APPROVED")
print("Temporal scheduling: NOT STARTED")
print("Plan B: NOT ACTIVATED")
print("No files were exported.")


STAGE 3 — STEP 3.1.8: ASSIGNMENT IMPACT AUDIT

A. ASSIGNMENT PRESERVATION
Original assigned users: 995
Users in comparison: 995
Proposed scenario changes: 10
Unchanged assignments: 985
Deferred users retained: 70

B. PRESERVED ASSIGNMENT DIMENSIONS
Duration preserved: True
Stealth preserved: True

C. GLOBAL DISTRIBUTION COMPARISON


,Metric,Original Stage 2,Proposed Full Population
0,Assigned users,995,995
1,Distinct scenario templates,490,490
2,Maximum template reuse,20,20
3,Distinct structural patterns,490,490



D. STRUCTURAL CHANGES IN THE 10 ALTERNATIVES


,Attribute,Changed Users
0,duration,0
1,stealth_level,0
2,active_days,0
3,off_days,0
4,single_channel_days,10
5,two_channel_days,0
6,three_channel_days,10
7,target_switches,0
8,structural_pattern_id,10



E. INDIVIDUAL ALTERNATIVE COMPARISON


,user,scenario_id,proposed_scenario_id,original_active_days,proposed_active_days,original_two_channel_days,proposed_two_channel_days,original_three_channel_days,proposed_three_channel_days,original_target_switches,proposed_target_switches
313,ELW1413,SCN_0620,SCN_0617,42,42,4,4,2,1,22,22
395,HCB1383,SCN_0663,SCN_0660,42,42,5,5,2,1,25,25
422,HMR0098,SCN_0692,SCN_0689,41,41,4,4,2,1,5,5
475,JDC2176,SCN_0715,SCN_0712,41,41,4,4,2,1,7,7
495,JKK1422,SCN_0727,SCN_0724,41,41,4,4,2,1,8,8
503,JMV0602,SCN_0734,SCN_0731,41,41,5,5,2,1,8,8
552,KJS0119,SCN_0757,SCN_0754,41,41,5,5,2,1,10,10
624,MCB2802,SCN_0795,SCN_0792,41,41,5,5,2,1,14,14
978,XSF3982,SCN_0981,SCN_0978,40,40,4,4,2,1,8,8
982,YCL0824,SCN_1141,SCN_1138,40,40,5,5,2,1,25,25



F. TEMPLATE REUSE DIAGNOSTIC
Original maximum reuse: 20
Proposed maximum reuse: 20
Templates exceeding original maximum: 0

STEP 3.1.8 — AUDIT SUMMARY
Original preliminary-pass users: 915
Proposed alternatives: 10
Provisional unified cohort: 925
Deferred users: 70

Original Stage 2 assignments: UNCHANGED
Proposed alternatives: NOT YET APPROVED
Temporal scheduling: NOT STARTED
Plan B: NOT ACTIVATED
No files were exported.



## Step 3.2 — Attack Window Selection

### Step 3.2.1 — Attack Window Feasibility Screening

**Objective:** Evaluate whether each user in the unified preliminary-pass cohort has at least one eligible attack window satisfying the minimum daily channel-activity requirements of their assigned or proposed scenario.

**Validation Tasks:**
1. Generate candidate windows based on eligible workdays.
2. Match each window to the scenario duration.
3. Check active-day capacity within each window.
4. Check multichannel and three-channel capacity.
5. Identify users with at least one feasible candidate window.
6. Diagnose users without a feasible window.

**Important Limitations:**
- No final attack window is selected.
- Active and off days are not yet assigned.
- Channel switching sequences are not yet constructed.
- Passing this screening does not guarantee a complete valid schedule.

**Expected Outcome:** Identify feasible candidate windows for the 925 users before attack scheduling.

**Note:** Read-only. No assignments or source datasets are modified.


In [10]:

# Stage 3 — Step 3.2.1: Attack Window Feasibility Screening
# Goal: Identify candidate eligible-workday windows satisfying the
# minimum channel-activity capacities for all 925 users.
# Read-only: No assignments, scheduling, injection, or exports.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.2.1: ATTACK WINDOW FEASIBILITY")
print("=" * 75)

# ------------------------------------------------------------
# 1. Validate the unified preliminary-pass cohort.
# ------------------------------------------------------------

cohort_321 = unified_plan_a_317.copy()

assert len(cohort_321) == 925, (
    "Unexpected cohort size. Review Step 3.1.7."
)

assert cohort_321["user"].is_unique

assert cohort_321["passes_preliminary_checks"].all()

# ------------------------------------------------------------
# 2. Prepare eligible User-Day activity.
# ------------------------------------------------------------

window_activity_321 = eligible[
    ["user", "day", "active_channel_count"]
].copy()

window_activity_321["day"] = pd.to_datetime(
    window_activity_321["day"]
)

assert not window_activity_321.duplicated(
    ["user", "day"]
).any(), "Duplicate eligible User-Day records."

activity_groups_321 = {
    user: group.sort_values("day").reset_index(drop=True)
    for user, group in window_activity_321.groupby("user")
}

# ------------------------------------------------------------
# 3. Helper: calculate counts for all rolling windows.
# ------------------------------------------------------------

def rolling_counts_321(values, window_size):
    values = np.asarray(values, dtype=np.int64)
    cumulative = np.concatenate(
        ([0], np.cumsum(values))
    )
    return (
        cumulative[window_size:]
        - cumulative[:-window_size]
    )

# ------------------------------------------------------------
# 4. Screen every possible eligible-workday window.
# ------------------------------------------------------------

window_results_321 = []

for row in cohort_321.itertuples(index=False):

    user = row.user
    duration = int(row.effective_duration)
    required_active = int(row.effective_active_days)

    required_multi = (
        int(row.effective_two_days)
        + int(row.effective_three_days)
    )

    required_three = int(row.effective_three_days)

    user_data = activity_groups_321.get(user)

    if user_data is None or len(user_data) < duration:
        window_results_321.append({
            "user": user,
            "scenario_id": row.proposed_scenario_id,
            "duration": duration,
            "selection_type": row.selection_type,
            "candidate_windows": 0,
            "passing_windows": 0,
            "max_active_in_window": 0,
            "max_multi_in_window": 0,
            "max_three_in_window": 0,
            "example_start": pd.NaT,
            "example_end": pd.NaT,
            "diagnosis": "Insufficient eligible workdays"
        })
        continue

    counts = user_data[
        "active_channel_count"
    ].to_numpy()

    active = rolling_counts_321(
        counts >= 1, duration
    )

    multi = rolling_counts_321(
        counts >= 2, duration
    )

    three = rolling_counts_321(
        counts == 3, duration
    )

    valid_windows = (
        (active >= required_active)
        & (multi >= required_multi)
        & (three >= required_three)
    )

    passing_count = int(valid_windows.sum())

    max_active = int(active.max())
    max_multi = int(multi.max())
    max_three = int(three.max())

    example_start = pd.NaT
    example_end = pd.NaT

    if passing_count > 0:
        first_valid = int(
            np.flatnonzero(valid_windows)[0]
        )

        example_start = user_data.iloc[
            first_valid
        ]["day"]

        example_end = user_data.iloc[
            first_valid + duration - 1
        ]["day"]

        diagnosis = "Preliminary window found"

    else:
        reasons = []

        if max_active < required_active:
            reasons.append("Active-day capacity")

        if max_multi < required_multi:
            reasons.append("Multichannel capacity")

        if max_three < required_three:
            reasons.append("Three-channel capacity")

        if not reasons:
            reasons.append(
                "Requirements not met together in one window"
            )

        diagnosis = " + ".join(reasons)

    window_results_321.append({
        "user": user,
        "scenario_id": row.proposed_scenario_id,
        "duration": duration,
        "selection_type": row.selection_type,
        "candidate_windows": len(valid_windows),
        "passing_windows": passing_count,
        "max_active_in_window": max_active,
        "max_multi_in_window": max_multi,
        "max_three_in_window": max_three,
        "example_start": example_start,
        "example_end": example_end,
        "diagnosis": diagnosis
    })

# ------------------------------------------------------------
# 5. Build the results table.
# ------------------------------------------------------------

window_audit_321 = pd.DataFrame(
    window_results_321
)

window_audit_321["has_feasible_window"] = (
    window_audit_321["passing_windows"] > 0
)

assert len(window_audit_321) == len(cohort_321)
assert window_audit_321["user"].is_unique

# ------------------------------------------------------------
# 6. Overall results.
# ------------------------------------------------------------

found = int(
    window_audit_321["has_feasible_window"].sum()
)

not_found = len(window_audit_321) - found

print("\nA. OVERALL WINDOW FEASIBILITY")

print(f"Users evaluated: {len(window_audit_321):,}")
print(f"Users with candidate windows: {found:,}")
print(f"Users without candidate windows: {not_found:,}")

# ------------------------------------------------------------
# 7. Results by scenario duration.
# ------------------------------------------------------------

print("\nB. WINDOW FEASIBILITY BY DURATION")

duration_results_321 = (
    window_audit_321
    .groupby("duration")
    .agg(
        Users=("user", "count"),
        With_Window=("has_feasible_window", "sum"),
        Total_Passing_Windows=("passing_windows", "sum")
    )
    .reset_index()
)

duration_results_321["Without_Window"] = (
    duration_results_321["Users"]
    - duration_results_321["With_Window"]
)

display(duration_results_321)

# ------------------------------------------------------------
# 8. Compare originals and alternatives.
# ------------------------------------------------------------

print("\nC. RESULTS BY SELECTION TYPE")

selection_results_321 = (
    window_audit_321
    .groupby("selection_type")
    .agg(
        Users=("user", "count"),
        With_Window=("has_feasible_window", "sum")
    )
    .reset_index()
)

selection_results_321["Without_Window"] = (
    selection_results_321["Users"]
    - selection_results_321["With_Window"]
)

display(selection_results_321)

# ------------------------------------------------------------
# 9. Show failure reasons and affected users.
# ------------------------------------------------------------

failed_windows_321 = window_audit_321.loc[
    ~window_audit_321["has_feasible_window"]
].copy()

print("\nD. WINDOW FAILURE DIAGNOSIS")

if failed_windows_321.empty:
    print("All users have at least one preliminary valid window.")
else:
    display(
        failed_windows_321["diagnosis"]
        .value_counts()
        .rename_axis("Diagnosis")
        .reset_index(name="Users")
    )

    print("\nFAILED USER SAMPLE:")
    display(
        failed_windows_321[
            [
                "user", "scenario_id", "duration",
                "selection_type", "candidate_windows",
                "max_active_in_window",
                "max_multi_in_window",
                "max_three_in_window",
                "diagnosis"
            ]
        ].head(20)
    )

# ------------------------------------------------------------
# 10. Preview example candidate windows (not final selections).
# ------------------------------------------------------------

print("\nE. EXAMPLE FEASIBLE WINDOWS")

display(
    window_audit_321.loc[
        window_audit_321["has_feasible_window"],
        [
            "user", "scenario_id", "duration",
            "selection_type", "passing_windows",
            "example_start", "example_end"
        ]
    ].head(10)
)

print("\n" + "=" * 75)
print("STEP 3.2.1 — SCREENING COMPLETED")
print("=" * 75)

print("No final attack windows were selected.")
print("Actual channel switching was not tested.")
print("Original Stage 2 assignments remain unchanged.")
print("No data injection or file export was performed.")
print("Plan B remains inactive.")


STAGE 3 — STEP 3.2.1: ATTACK WINDOW FEASIBILITY

A. OVERALL WINDOW FEASIBILITY
Users evaluated: 925
Users with candidate windows: 703
Users without candidate windows: 222

B. WINDOW FEASIBILITY BY DURATION


,duration,Users,With_Window,Total_Passing_Windows,Without_Window
0,5,176,157,5894,19
1,10,171,137,5872,34
2,20,173,131,6216,42
3,30,169,115,4635,54
4,40,118,79,3385,39
5,50,118,84,3228,34



C. RESULTS BY SELECTION TYPE


,selection_type,Users,With_Window,Without_Window
0,Original assignment,915,701,214
1,Proposed alternative,10,2,8



D. WINDOW FAILURE DIAGNOSIS


,Diagnosis,Users
0,Active-day capacity,191
1,Active-day capacity + Multichannel capacity,14
2,Requirements not met together in one window,12
3,Multichannel capacity,2
4,Three-channel capacity,2
5,Active-day capacity + Three-channel capacity,1



FAILED USER SAMPLE:


,user,scenario_id,duration,selection_type,candidate_windows,max_active_in_window,max_multi_in_window,max_three_in_window,diagnosis
0,AAC0610,SCN_0108,30,Original assignment,76,23,7,0,Active-day capacity
4,AAS2987,SCN_0109,30,Original assignment,76,19,4,0,Active-day capacity
6,ABH0349,SCN_0462,50,Original assignment,56,30,5,1,Active-day capacity
8,ABM3203,SCN_0148,30,Original assignment,77,24,3,0,Active-day capacity
10,ABO1173,SCN_0205,40,Original assignment,66,34,12,1,Requirements not met together in one window
12,ABR0397,SCN_0017,10,Original assignment,96,7,2,0,Active-day capacity
13,ABS0677,SCN_0041,20,Original assignment,86,17,6,0,Active-day capacity
14,ACB0701,SCN_0445,40,Original assignment,68,29,6,1,Active-day capacity
23,ADP1625,SCN_0111,30,Original assignment,76,17,4,0,Active-day capacity
24,ADP3667,SCN_0012,10,Original assignment,97,6,1,0,Active-day capacity



E. EXAMPLE FEASIBLE WINDOWS


,user,scenario_id,duration,selection_type,passing_windows,example_start,example_end
1,AAF0819,SCN_0007,5,Original assignment,71,2011-01-03,2011-01-07
2,AAP0352,SCN_0016,10,Original assignment,12,2011-02-03,2011-02-16
3,AAP1919,SCN_0040,20,Original assignment,36,2011-01-18,2011-02-14
5,ABD3426,SCN_0402,40,Original assignment,67,2011-01-03,2011-02-25
7,ABK3081,SCN_0009,5,Original assignment,42,2011-01-03,2011-01-07
9,ABM3772,SCN_0008,5,Original assignment,59,2011-01-03,2011-01-07
11,ABP2917,SCN_0041,20,Original assignment,14,2011-04-13,2011-05-10
15,ACB3123,SCN_0110,30,Original assignment,40,2011-01-17,2011-02-25
16,ACP2122,SCN_0003,5,Original assignment,26,2011-02-08,2011-02-14
17,ACS0128,SCN_0001,5,Original assignment,1,2011-01-03,2011-01-07



STEP 3.2.1 — SCREENING COMPLETED
No final attack windows were selected.
Actual channel switching was not tested.
Original Stage 2 assignments remain unchanged.
No data injection or file export was performed.
Plan B remains inactive.



### Step 3.2.2 — Temporal Constraint Diagnosis

**Objective:** Quantify the activity shortfalls preventing 222 users from satisfying the preliminary Plan A requirements within a single attack window.

**Validation Tasks:**
1. Calculate active-day shortfalls.
2. Calculate multichannel-day shortfalls.
3. Calculate three-channel-day shortfalls.
4. Identify cases where requirements are individually achievable but cannot be satisfied together in one window.
5. Analyze shortfalls by scenario duration and assignment type.
6. Identify users with the largest activity shortages.

**Important Notes:**
- Analysis is limited to the 222 users who failed Step 3.2.1.
- Individual maximum capacities may come from different windows.
- No scenario assignments or attack windows will be changed.
- Plan B remains inactive.

**Expected Outcome:** Understand the severity of temporal constraints before deciding how to address them.


In [11]:

# Stage 3 — Step 3.2.2: Temporal Constraint Diagnosis
# Goal: Quantify activity, multichannel, and three-channel shortfalls
# for users who failed the preliminary attack-window screening.
# Read-only: No original data, assignments, or files are modified.

import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.2.2: TEMPORAL CONSTRAINT DIAGNOSIS")
print("=" * 75)

# ------------------------------------------------------------
# 1. Verify previous results and select failed users.
# ------------------------------------------------------------

assert len(window_audit_321) == 925, (
    "Unexpected window audit size. Review Step 3.2.1."
)

assert window_audit_321["user"].is_unique, (
    "Duplicate users in window audit."
)

failed_windows_322 = window_audit_321.loc[
    ~window_audit_321["has_feasible_window"]
].copy()

assert len(failed_windows_322) == 222, (
    "Unexpected failed-user count. Review Step 3.2.1."
)

# ------------------------------------------------------------
# 2. Attach effective scenario requirements.
# ------------------------------------------------------------

requirements_322 = unified_plan_a_317[
    [
        "user",
        "effective_active_days",
        "effective_two_days",
        "effective_three_days"
    ]
].copy()

diagnosis_322 = failed_windows_322.merge(
    requirements_322,
    on="user",
    how="left",
    validate="one_to_one"
)

assert len(diagnosis_322) == 222
assert diagnosis_322["user"].is_unique

assert diagnosis_322[
    [
        "effective_active_days",
        "effective_two_days",
        "effective_three_days"
    ]
].notna().all().all(), (
    "Missing effective scenario requirements."
)

# ------------------------------------------------------------
# 3. Calculate required activity capacity.
# ------------------------------------------------------------

diagnosis_322["required_active"] = (
    diagnosis_322["effective_active_days"]
)

diagnosis_322["required_multi"] = (
    diagnosis_322["effective_two_days"]
    + diagnosis_322["effective_three_days"]
)

diagnosis_322["required_three"] = (
    diagnosis_322["effective_three_days"]
)

# ------------------------------------------------------------
# 4. Measure individual shortfalls.
# A maximum may be obtained from a different window
# for each requirement.
# ------------------------------------------------------------

diagnosis_322["active_shortfall"] = (
    diagnosis_322["required_active"]
    - diagnosis_322["max_active_in_window"]
).clip(lower=0)

diagnosis_322["multi_shortfall"] = (
    diagnosis_322["required_multi"]
    - diagnosis_322["max_multi_in_window"]
).clip(lower=0)

diagnosis_322["three_shortfall"] = (
    diagnosis_322["required_three"]
    - diagnosis_322["max_three_in_window"]
).clip(lower=0)

# ------------------------------------------------------------
# 5. Summarize the size of each shortfall.
# ------------------------------------------------------------

print("\nA. TEMPORAL SHORTFALL SUMMARY")

shortfall_metrics_322 = [
    ("Active Days", "active_shortfall"),
    ("Multichannel Days", "multi_shortfall"),
    ("Three-Channel Days", "three_shortfall")
]

shortfall_summary_322 = []

for name, column in shortfall_metrics_322:

    affected = diagnosis_322.loc[
        diagnosis_322[column] > 0,
        column
    ]

    shortfall_summary_322.append({
        "Requirement": name,
        "Affected Users": len(affected),
        "Median Shortfall": (
            float(affected.median())
            if not affected.empty else 0
        ),
        "Maximum Shortfall": (
            int(affected.max())
            if not affected.empty else 0
        )
    })

display(pd.DataFrame(shortfall_summary_322))

# ------------------------------------------------------------
# 6. Identify joint-window conflicts.
# Each individual requirement is achievable somewhere,
# but no single window satisfies all requirements.
# ------------------------------------------------------------

diagnosis_322["individual_maxima_sufficient"] = (
    (diagnosis_322["active_shortfall"] == 0)
    & (diagnosis_322["multi_shortfall"] == 0)
    & (diagnosis_322["three_shortfall"] == 0)
)

joint_conflicts_322 = diagnosis_322.loc[
    diagnosis_322["individual_maxima_sufficient"]
].copy()

print("\nB. JOINT-WINDOW CONSTRAINTS")

print(
    "Users with individually sufficient capacities "
    "but no jointly feasible window:",
    len(joint_conflicts_322)
)

if joint_conflicts_322.empty:
    print("No joint-window conflicts found.")
else:
    display(
        joint_conflicts_322[
            [
                "user",
                "scenario_id",
                "duration",
                "required_active",
                "required_multi",
                "required_three",
                "diagnosis"
            ]
        ]
    )

# ------------------------------------------------------------
# 7. Analyze shortfalls by scenario duration.
# ------------------------------------------------------------

print("\nC. SHORTFALLS BY SCENARIO DURATION")

duration_diagnosis_322 = (
    diagnosis_322
    .groupby("duration")
    .agg(
        Failed_Users=("user", "count"),
        Active_Shortfall_Users=(
            "active_shortfall",
            lambda s: int((s > 0).sum())
        ),
        Multichannel_Shortfall_Users=(
            "multi_shortfall",
            lambda s: int((s > 0).sum())
        ),
        Three_Channel_Shortfall_Users=(
            "three_shortfall",
            lambda s: int((s > 0).sum())
        ),
        Median_Active_Shortfall=(
            "active_shortfall", "median"
        )
    )
    .reset_index()
)

display(duration_diagnosis_322)

# ------------------------------------------------------------
# 8. Show users with the largest active-day shortages.
# ------------------------------------------------------------

print("\nD. LARGEST ACTIVE-DAY SHORTFALLS")

display(
    diagnosis_322[
        [
            "user",
            "scenario_id",
            "duration",
            "selection_type",
            "required_active",
            "max_active_in_window",
            "active_shortfall",
            "multi_shortfall",
            "three_shortfall"
        ]
    ]
    .sort_values(
        ["active_shortfall", "multi_shortfall"],
        ascending=False
    )
    .head(20)
)

# ------------------------------------------------------------
# 9. Compare original and alternative assignments.
# ------------------------------------------------------------

print("\nE. SHORTFALLS BY SELECTION TYPE")

selection_diagnosis_322 = (
    diagnosis_322
    .groupby("selection_type")
    .agg(
        Failed_Users=("user", "count"),
        Active_Shortfall_Users=(
            "active_shortfall",
            lambda s: int((s > 0).sum())
        ),
        Multichannel_Shortfall_Users=(
            "multi_shortfall",
            lambda s: int((s > 0).sum())
        ),
        Three_Channel_Shortfall_Users=(
            "three_shortfall",
            lambda s: int((s > 0).sum())
        )
    )
    .reset_index()
)

display(selection_diagnosis_322)

# ------------------------------------------------------------
# 10. Final audit summary.
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 3.2.2 — DIAGNOSIS COMPLETED")
print("=" * 75)

print(f"Users diagnosed: {len(diagnosis_322)}")
print(f"Joint-window conflicts: {len(joint_conflicts_322)}")
print("Original Stage 2 assignments: UNCHANGED")
print("Alternative assignments: PROVISIONAL")
print("Final attack scheduling: NOT STARTED")
print("Plan B: NOT ACTIVATED")
print("No files were modified or exported.")

print(
    "\nNOTE: Individual maxima may occur in different windows."
    "\nThis analysis does not construct attack schedules."
)


STAGE 3 — STEP 3.2.2: TEMPORAL CONSTRAINT DIAGNOSIS

A. TEMPORAL SHORTFALL SUMMARY


,Requirement,Affected Users,Median Shortfall,Maximum Shortfall
0,Active Days,206,3.0,17
1,Multichannel Days,16,1.0,1
2,Three-Channel Days,3,1.0,1



B. JOINT-WINDOW CONSTRAINTS
Users with individually sufficient capacities but no jointly feasible window: 12


,user,scenario_id,duration,required_active,required_multi,required_three,diagnosis
4,ABO1173,SCN_0205,40,34,5,1,Requirements not met together in one window
31,BLB1876,SCN_1010,50,40,7,2,Requirements not met together in one window
52,CIY1654,SCN_0239,40,34,4,1,Requirements not met together in one window
80,EDF2361,SCN_0197,30,24,3,0,Requirements not met together in one window
100,GKG3414,SCN_0155,30,25,3,0,Requirements not met together in one window
102,HCB1383,SCN_0660,50,42,6,1,Requirements not met together in one window
104,HKH0204,SCN_0685,50,41,7,2,Requirements not met together in one window
105,HKK2716,SCN_0287,40,34,4,1,Requirements not met together in one window
118,JKK1422,SCN_0724,50,41,5,1,Requirements not met together in one window
127,KCH3095,SCN_0742,50,41,6,1,Requirements not met together in one window



C. SHORTFALLS BY SCENARIO DURATION


,duration,Failed_Users,Active_Shortfall_Users,Multichannel_Shortfall_Users,Three_Channel_Shortfall_Users,Median_Active_Shortfall
0,5,19,19,0,0,1.0
1,10,34,34,0,0,2.0
2,20,42,40,3,0,3.0
3,30,54,51,12,0,4.0
4,40,39,36,1,0,3.0
5,50,34,26,0,3,2.5



D. LARGEST ACTIVE-DAY SHORTFALLS


,user,scenario_id,duration,selection_type,required_active,max_active_in_window,active_shortfall,multi_shortfall,three_shortfall
152,NBW2810,SCN_0822,50,Original assignment,41,24,17,0,0
49,CID3640,SCN_0238,40,Original assignment,34,18,16,0,0
33,BLL3910,SCN_0131,30,Original assignment,26,11,15,1,0
78,DZG0889,SCN_0192,30,Original assignment,24,9,15,1,0
200,TDM2859,SCN_0132,30,Original assignment,26,11,15,1,0
37,BPM3694,SCN_0134,30,Original assignment,26,12,14,1,0
177,RIB1582,SCN_0115,30,Original assignment,26,12,14,1,0
41,CAG0888,SCN_0142,30,Original assignment,25,11,14,0,0
29,BKF2204,SCN_0129,30,Original assignment,26,13,13,1,0
32,BLD2227,SCN_0130,30,Original assignment,26,13,13,0,0



E. SHORTFALLS BY SELECTION TYPE


,selection_type,Failed_Users,Active_Shortfall_Users,Multichannel_Shortfall_Users,Three_Channel_Shortfall_Users
0,Original assignment,214,200,16,3
1,Proposed alternative,8,6,0,0



STEP 3.2.2 — DIAGNOSIS COMPLETED
Users diagnosed: 222
Joint-window conflicts: 12
Original Stage 2 assignments: UNCHANGED
Alternative assignments: PROVISIONAL
Final attack scheduling: NOT STARTED
Plan B: NOT ACTIVATED
No files were modified or exported.

NOTE: Individual maxima may occur in different windows.
This analysis does not construct attack schedules.



### Step 3.2.3 — Window-Aware Alternative Feasibility Audit

**Objective:** Determine whether alternative approved scenarios can satisfy Plan A requirements within an actual eligible-workday window.

**Scope:**
- Evaluate users who failed Step 3.2.1.
- Defer users with a three-channel capacity shortfall.
- Preserve the existing scenario duration and stealth level.
- Compare candidate requirements against the same user's actual daily activity.

**Validation:**
1. Apply the seven preliminary feasibility conditions.
2. Require all activity-capacity conditions to hold within the same window.
3. Identify candidate alternatives without modifying assignments.
4. Report recoverable and unresolved users.

**Limitations:**
- No attack dates or active/off-day sequences are finalized.
- Actual channel switching is not yet validated.
- Stage 2 global assignment distributions require review before approving replacements.

**Expected Outcome:** Identify potential Plan A recoveries under the stricter temporal-window constraints.

**Note:** Read-only diagnostic analysis. No assignments or source files will be modified.


In [12]:

# Stage 3 — Step 3.2.3: Window-Aware Alternative Feasibility Audit
# Goal: Search for approved alternative scenarios that satisfy all
# preliminary activity constraints inside a single eligible-workday window.
# Three-channel shortfall cases are deferred as agreed.
# Read-only: No assignments, schedules, or source files are modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.2.3: WINDOW-AWARE ALTERNATIVE AUDIT")
print("=" * 75)

# ------------------------------------------------------------
# 1. Select failed users and defer three-channel shortages.
# ------------------------------------------------------------

assert len(diagnosis_322) == 222
assert diagnosis_322["user"].is_unique

three_channel_deferred_323 = diagnosis_322.loc[
    diagnosis_322["three_shortfall"] > 0
].copy()

search_users_323 = diagnosis_322.loc[
    diagnosis_322["three_shortfall"] == 0, "user"
]

search_cohort_323 = unified_plan_a_317.loc[
    unified_plan_a_317["user"].isin(search_users_323)
].copy()

assert len(search_cohort_323) == len(search_users_323)

print("\nA. AUDIT SCOPE")
print(f"Failed-window users: {len(diagnosis_322)}")
print(f"Three-channel cases deferred: {len(three_channel_deferred_323)}")
print(f"Users included in alternative search: {len(search_cohort_323)}")

# ------------------------------------------------------------
# 2. Prepare catalog and daily activity.
# ------------------------------------------------------------

catalog_groups_323 = {
    key: group.copy()
    for key, group in scenario_catalog.groupby(
        ["duration", "stealth_level"]
    )
}

daily_groups_323 = {
    user: group.sort_values("day").reset_index(drop=True)
    for user, group in eligible.groupby("user")
    if user in set(search_users_323)
}

# ------------------------------------------------------------
# 3. Search alternatives within real eligible-workday windows.
# ------------------------------------------------------------

candidate_results_323 = []

for row in search_cohort_323.itertuples(index=False):

    user = row.user
    duration = int(row.effective_duration)
    stealth = row.effective_stealth

    user_days = daily_groups_323.get(user)

    if user_days is None or len(user_days) < duration:
        continue

    counts = user_days["active_channel_count"].to_numpy()

    active_windows = rolling_counts_321(
        counts >= 1, duration
    )
    multi_windows = rolling_counts_321(
        counts >= 2, duration
    )
    three_windows = rolling_counts_321(
        counts == 3, duration
    )

    catalog_group = catalog_groups_323.get(
        (duration, stealth)
    )

    if catalog_group is None:
        continue

    for candidate in catalog_group.itertuples(index=False):

        if candidate.scenario_id == row.proposed_scenario_id:
            continue

        required_active = int(candidate.active_days)
        required_multi = (
            int(candidate.two_channel_days)
            + int(candidate.three_channel_days)
        )
        required_three = int(candidate.three_channel_days)

        # The same seven necessary conditions, with all daily
        # activity requirements checked jointly in one window.
        catalog_valid = (
            required_active + int(candidate.off_days)
            == duration
            and
            int(candidate.single_channel_days)
            + int(candidate.two_channel_days)
            + required_three
            == required_active
            and
            0 <= int(candidate.target_switches)
            <= required_active - 1
        )

        if not catalog_valid:
            continue

        valid_windows = (
            (active_windows >= required_active)
            & (multi_windows >= required_multi)
            & (three_windows >= required_three)
        )

        passing_windows = int(valid_windows.sum())

        if passing_windows == 0:
            continue

        # Compare the candidate with the user's CURRENT
        # effective scenario, not necessarily the Stage 2 original.
        structural_distance = (
            abs(required_active - int(row.effective_active_days))
            + abs(int(candidate.off_days) - int(row.effective_off_days))
            + abs(
                int(candidate.single_channel_days)
                - int(row.effective_single_days)
            )
            + abs(
                int(candidate.two_channel_days)
                - int(row.effective_two_days)
            )
            + abs(required_three - int(row.effective_three_days))
            + abs(
                int(candidate.target_switches)
                - int(row.effective_switches)
            )
        )

        candidate_results_323.append({
            "user": user,
            "current_scenario": row.proposed_scenario_id,
            "candidate_scenario": candidate.scenario_id,
            "duration": duration,
            "stealth_level": stealth,
            "structural_distance": structural_distance,
            "passing_windows": passing_windows,
            "candidate_active_days": required_active,
            "candidate_multi_days": required_multi,
            "candidate_three_days": required_three,
            "candidate_switches": int(candidate.target_switches),
        })

# ------------------------------------------------------------
# 4. Summarize candidate availability.
# ------------------------------------------------------------

candidate_results_323 = pd.DataFrame(
    candidate_results_323,
    columns=[
        "user", "current_scenario", "candidate_scenario",
        "duration", "stealth_level", "structural_distance",
        "passing_windows", "candidate_active_days",
        "candidate_multi_days", "candidate_three_days",
        "candidate_switches"
    ]
)

recovered_users_323 = set(candidate_results_323["user"])
searched_users_323 = set(search_cohort_323["user"])
unresolved_users_323 = searched_users_323 - recovered_users_323

print("\nB. WINDOW-AWARE ALTERNATIVE RESULTS")

print(f"Users evaluated: {len(searched_users_323)}")
print(f"Users with potential alternatives: {len(recovered_users_323)}")
print(f"Users without potential alternatives: {len(unresolved_users_323)}")
print(f"Total passing candidate templates: {len(candidate_results_323)}")

# ------------------------------------------------------------
# 5. Show the closest passing candidate for each recoverable user.
# Diagnostic suggestions only; no actual reassignment.
# ------------------------------------------------------------

print("\nC. CLOSEST POTENTIAL ALTERNATIVES")

if candidate_results_323.empty:
    closest_candidates_323 = candidate_results_323.copy()
    print("No passing window-aware alternatives were found.")
else:
    closest_candidates_323 = (
        candidate_results_323
        .sort_values(
            [
                "user",
                "structural_distance",
                "candidate_scenario"
            ],
            kind="stable"
        )
        .drop_duplicates("user")
    )

    display(closest_candidates_323.head(20))

# ------------------------------------------------------------
# 6. Summarize results by original scenario duration.
# ------------------------------------------------------------

print("\nD. ALTERNATIVE AVAILABILITY BY DURATION")

duration_recovery_323 = search_cohort_323[
    ["user", "effective_duration"]
].copy()

duration_recovery_323["has_alternative"] = (
    duration_recovery_323["user"].isin(recovered_users_323)
)

display(
    duration_recovery_323
    .groupby("effective_duration")
    .agg(
        Evaluated=("user", "count"),
        With_Alternative=("has_alternative", "sum")
    )
    .assign(
        Without_Alternative=lambda df:
        df["Evaluated"] - df["With_Alternative"]
    )
    .reset_index()
)

# ------------------------------------------------------------
# 7. Final status.
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 3.2.3 — AUDIT COMPLETED")
print("=" * 75)

print("Original Stage 2 assignments: UNCHANGED")
print("Current provisional scenarios: UNCHANGED")
print("Alternative scenarios: DIAGNOSTIC ONLY")
print("Three-channel shortages: DEFERRED")
print("Actual switching feasibility: NOT TESTED")
print("Plan B: NOT ACTIVATED")
print("No files were modified or exported.")


STAGE 3 — STEP 3.2.3: WINDOW-AWARE ALTERNATIVE AUDIT

A. AUDIT SCOPE
Failed-window users: 222
Three-channel cases deferred: 3
Users included in alternative search: 219

B. WINDOW-AWARE ALTERNATIVE RESULTS
Users evaluated: 219
Users with potential alternatives: 46
Users without potential alternatives: 173
Total passing candidate templates: 1387

C. CLOSEST POTENTIAL ALTERNATIVES


,user,current_scenario,candidate_scenario,duration,stealth_level,structural_distance,passing_windows,candidate_active_days,candidate_multi_days,candidate_three_days,candidate_switches
0,ABO1173,SCN_0205,SCN_0382,40,High,6,5,32,4,1,4
28,ABS0677,SCN_0041,SCN_0065,20,Medium,3,5,17,2,0,4
42,AGF1810,SCN_0037,SCN_0061,20,High,3,2,17,2,0,3
58,AJH2110,SCN_0212,SCN_0389,40,Medium,6,12,32,4,1,5
86,APB1122,SCN_0219,SCN_0396,40,Low,6,10,32,4,1,6
114,ARH2097,SCN_0223,SCN_0310,40,High,3,1,33,4,1,7
169,BDJ3326,SCN_0055,SCN_0100,20,High,8,1,16,2,0,9
178,BKB0853,SCN_0128,SCN_0197,30,Medium,6,16,24,3,0,13
193,BLB1876,SCN_1010,SCN_1007,50,Medium,2,6,40,6,1,12
244,BNF1110,SCN_0533,SCN_1013,50,Medium,6,9,40,5,1,13



D. ALTERNATIVE AVAILABILITY BY DURATION


,effective_duration,Evaluated,With_Alternative,Without_Alternative
0,5,19,0,19
1,10,34,7,27
2,20,42,11,31
3,30,54,8,46
4,40,39,10,29
5,50,31,10,21



STEP 3.2.3 — AUDIT COMPLETED
Original Stage 2 assignments: UNCHANGED
Current provisional scenarios: UNCHANGED
Alternative scenarios: DIAGNOSTIC ONLY
Three-channel shortages: DEFERRED
Actual switching feasibility: NOT TESTED
Plan B: NOT ACTIVATED
No files were modified or exported.



### Step 3.2.4 — Alternative Compatibility & Cohort Update

**Objective:** Validate window-aware alternative scenarios against the approved Stage 2 eligibility rules and form an updated provisional Plan A cohort.

**Validation Tasks:**
1. Check candidate compatibility with Stage 2 duration, stealth, assignment group, and historical channel eligibility.
2. Reapply the seven preliminary feasibility conditions.
3. Retain only alternatives that already passed the window feasibility screening.
4. Rank compatible alternatives by structural similarity to the original Stage 2 scenario.
5. Combine selected alternatives with the 703 existing window-pass users.
6. Preserve deferred users and inspect global scenario diversity and reuse.

**Important Limitations:**
- Full channel-switching feasibility remains untested.
- The selected attack windows are not final.
- Stage 2 global distribution constraints must be reviewed before final reassignment approval.
- No original files will be modified.

**Expected Outcome:** An updated provisional Plan A cohort and a clearly identified unresolved group.


In [13]:

# Stage 3 — Step 3.2.4: Alternative Compatibility & Cohort Update
# Goal: Validate window-aware candidates against Stage 2 rules,
# select closest eligible alternatives, and unify window-pass users.
# Read-only: No original datasets or assignments are modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.2.4: ALTERNATIVE COMPATIBILITY")
print("=" * 75)

# ------------------------------------------------------------
# 1. Verify input results.
# ------------------------------------------------------------

assert len(assignments) == 995
assert assignments["user"].is_unique
assert len(window_audit_321) == 925

existing_pass_users_324 = set(
    window_audit_321.loc[
        window_audit_321["has_feasible_window"], "user"
    ]
)

assert len(existing_pass_users_324) == 703

# ------------------------------------------------------------
# 2. Attach original Stage 2 assignment metadata.
# ------------------------------------------------------------

candidate_audit_324 = candidate_results_323.copy()

assignment_metadata_324 = assignments[
    [
        "user", "scenario_id", "duration", "stealth_level",
        "assignment_group", "assignment_pool",
        "eligible_workdays"
    ]
].rename(columns={
    "scenario_id": "stage2_scenario_id",
    "duration": "stage2_duration",
    "stealth_level": "stage2_stealth"
})

candidate_audit_324 = candidate_audit_324.merge(
    assignment_metadata_324,
    on="user",
    how="left",
    validate="many_to_one"
)

# Attach the full approved candidate scenario definition.
rule_columns_324 = [
    "scenario_id", "duration", "stealth_level",
    "active_days", "off_days", "single_channel_days",
    "two_channel_days", "three_channel_days",
    "target_switches"
]

candidate_rules_324 = scenario_catalog[
    rule_columns_324
].rename(columns={
    "scenario_id": "candidate_scenario",
    "duration": "catalog_duration",
    "stealth_level": "catalog_stealth",
    "active_days": "rule_active",
    "off_days": "rule_off",
    "single_channel_days": "rule_single",
    "two_channel_days": "rule_two",
    "three_channel_days": "rule_three",
    "target_switches": "rule_switches"
})

candidate_audit_324 = candidate_audit_324.merge(
    candidate_rules_324,
    on="candidate_scenario",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 3. Attach historical baseline and daily capacities.
# ------------------------------------------------------------

historical_324 = user_baseline[
    [
        "user", "email_active_days", "usb_active_days",
        "http_upload_active_days", "email_count_p95",
        "usb_copy_p95", "http_upload_p95"
    ]
].copy()

historical_324["historical_channel_count"] = (
    historical_324[
        [
            "email_active_days", "usb_active_days",
            "http_upload_active_days"
        ]
    ].gt(0).sum(axis=1)
)

candidate_audit_324 = candidate_audit_324.merge(
    historical_324,
    on="user",
    how="left",
    validate="many_to_one"
)

daily_capacity_324 = plan_a_necessary[
    [
        "user", "calculated_eligible_workdays",
        "any_channel_days", "available_multichannel_days",
        "three_channel_days"
    ]
].copy()

candidate_audit_324 = candidate_audit_324.merge(
    daily_capacity_324,
    on="user",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 4. Apply Stage 2 compatibility checks.
# ------------------------------------------------------------

c = candidate_audit_324

short_group_324 = (
    c["assignment_group"].isin([
        "Two-channel short-duration",
        "Three-channel short-duration exception"
    ])
    & c["assignment_pool"].eq("Short-duration pool")
    & c["catalog_duration"].isin([5, 10, 20, 30])
)

long_group_324 = (
    c["assignment_group"].eq("Three-channel long-duration")
    & c["assignment_pool"].eq("Long-duration pool")
    & c["catalog_duration"].isin([40, 50])
)

c["pass_stage2_group"] = short_group_324 | long_group_324

c["required_historical_channels"] = np.where(
    c["catalog_duration"].isin([40, 50]), 3, 2
)

c["pass_stage2_channels"] = (
    c["historical_channel_count"]
    >= c["required_historical_channels"]
)

c["pass_stage2_identity"] = (
    c["catalog_duration"].eq(c["stage2_duration"])
    & c["catalog_stealth"].eq(c["stage2_stealth"])
    & c["catalog_duration"].eq(c["duration"])
    & c["catalog_stealth"].eq(c["stealth_level"])
)

c["pass_stage2_workdays"] = (
    c["eligible_workdays"] >= c["catalog_duration"]
)

p95_valid_324 = pd.Series(True, index=c.index)

for active_col, p95_col in [
    ("email_active_days", "email_count_p95"),
    ("usb_active_days", "usb_copy_p95"),
    ("http_upload_active_days", "http_upload_p95")
]:
    p95 = pd.to_numeric(c[p95_col], errors="coerce")
    historical_active = c[active_col] > 0

    p95_valid_324 &= (
        ~historical_active
        | (p95.notna() & np.isfinite(p95) & (p95 > 0))
    )

c["pass_stage2_baseline"] = p95_valid_324

# ------------------------------------------------------------
# 5. Reapply the seven preliminary conditions.
# ------------------------------------------------------------

c["pass_duration_capacity"] = (
    c["calculated_eligible_workdays"] >= c["catalog_duration"]
)

c["pass_active_day_capacity"] = (
    c["any_channel_days"] >= c["rule_active"]
)

c["pass_multichannel_capacity"] = (
    c["available_multichannel_days"]
    >= c["rule_two"] + c["rule_three"]
)

c["pass_three_channel_capacity"] = (
    c["three_channel_days"] >= c["rule_three"]
)

c["pass_scenario_day_partition"] = (
    c["rule_active"] + c["rule_off"] == c["catalog_duration"]
)

c["pass_scenario_channel_partition"] = (
    c["rule_single"] + c["rule_two"] + c["rule_three"]
    == c["rule_active"]
)

c["pass_switch_count_bounds"] = (
    (c["rule_switches"] >= 0)
    & (c["rule_switches"] <= c["rule_active"] - 1)
)

# The candidate must also have passed the actual window
# screening performed in Step 3.2.3.
c["pass_window_screening"] = c["passing_windows"] > 0

validation_columns_324 = [
    "pass_stage2_group",
    "pass_stage2_channels",
    "pass_stage2_identity",
    "pass_stage2_workdays",
    "pass_stage2_baseline",
    "pass_duration_capacity",
    "pass_active_day_capacity",
    "pass_multichannel_capacity",
    "pass_three_channel_capacity",
    "pass_scenario_day_partition",
    "pass_scenario_channel_partition",
    "pass_switch_count_bounds",
    "pass_window_screening"
]

c["passes_all_checked_conditions"] = (
    c[validation_columns_324].all(axis=1)
)

print("\nA. CANDIDATE COMPATIBILITY RESULTS")

display(pd.DataFrame([
    {
        "Check": column,
        "Passed Candidates": int(c[column].sum()),
        "Failed Candidates": int((~c[column]).sum())
    }
    for column in validation_columns_324
]))

compatible_candidates_324 = c.loc[
    c["passes_all_checked_conditions"]
].copy()

print("\nTotal window-aware candidates:", len(c))
print("Candidates passing all checked conditions:",
      len(compatible_candidates_324))
print("Users with compatible candidates:",
      compatible_candidates_324["user"].nunique())

# ------------------------------------------------------------
# 6. Rank candidates against the ORIGINAL Stage 2 scenario.
# ------------------------------------------------------------

original_rules_324 = scenario_catalog[
    [
        "scenario_id", "active_days", "off_days",
        "single_channel_days", "two_channel_days",
        "three_channel_days", "target_switches"
    ]
].rename(columns={
    "scenario_id": "stage2_scenario_id",
    "active_days": "original_active",
    "off_days": "original_off",
    "single_channel_days": "original_single",
    "two_channel_days": "original_two",
    "three_channel_days": "original_three",
    "target_switches": "original_switches"
})

compatible_candidates_324 = compatible_candidates_324.merge(
    original_rules_324,
    on="stage2_scenario_id",
    how="left",
    validate="many_to_one"
)

distance_pairs_324 = [
    ("rule_active", "original_active"),
    ("rule_off", "original_off"),
    ("rule_single", "original_single"),
    ("rule_two", "original_two"),
    ("rule_three", "original_three"),
    ("rule_switches", "original_switches")
]

compatible_candidates_324["distance_to_stage2"] = sum(
    (
        compatible_candidates_324[a]
        - compatible_candidates_324[b]
    ).abs()
    for a, b in distance_pairs_324
)

selected_324 = (
    compatible_candidates_324
    .sort_values(
        ["user", "distance_to_stage2",
         "passing_windows", "candidate_scenario"],
        ascending=[True, True, False, True],
        kind="stable"
    )
    .drop_duplicates("user")
    .copy()
)

assert selected_324["user"].is_unique

print("\nB. SELECTED DIAGNOSTIC ALTERNATIVES")

display(
    selected_324[
        [
            "user", "stage2_scenario_id", "current_scenario",
            "candidate_scenario", "duration", "stealth_level",
            "distance_to_stage2", "passing_windows"
        ]
    ].head(20)
)

# ------------------------------------------------------------
# 7. Create a full-population PROVISIONAL comparison.
# Preserve previous Stage 3 proposals for existing users.
# ------------------------------------------------------------

proposal_324 = comparison_318[
    ["user", "scenario_id", "proposed_scenario_id"]
].copy()

new_map_324 = selected_324.set_index(
    "user"
)["candidate_scenario"]

replace_mask_324 = proposal_324["user"].isin(new_map_324.index)

proposal_324.loc[
    replace_mask_324, "proposed_scenario_id"
] = proposal_324.loc[
    replace_mask_324, "user"
].map(new_map_324)

assert len(proposal_324) == 995
assert proposal_324["user"].is_unique

# Users who passed Step 3.2.1 and newly selected users are
# kept at the SAME window-screening validation level.
unified_users_324 = (
    existing_pass_users_324 | set(selected_324["user"])
)

unified_window_pass_324 = proposal_324.loc[
    proposal_324["user"].isin(unified_users_324)
].copy()

unresolved_324 = proposal_324.loc[
    ~proposal_324["user"].isin(unified_users_324)
].copy()

assert len(unified_window_pass_324) + len(unresolved_324) == 995

# Preserve the original 70 deferred users unchanged.
original_deferred_mask_324 = proposal_324["user"].isin(
    deferred_users_317["user"]
)

assert (
    proposal_324.loc[
        original_deferred_mask_324, "scenario_id"
    ].values
    ==
    proposal_324.loc[
        original_deferred_mask_324, "proposed_scenario_id"
    ].values
).all()

# ------------------------------------------------------------
# 8. Report the updated cohort and distribution diagnostics.
# ------------------------------------------------------------

original_usage_324 = assignments["scenario_id"].value_counts()
proposed_usage_324 = proposal_324["proposed_scenario_id"].value_counts()

print("\nC. UPDATED PROVISIONAL COHORT")

print("Existing window-pass users:", len(existing_pass_users_324))
print("Newly selected alternative users:", len(selected_324))
print("Updated provisional window-pass cohort:",
      len(unified_window_pass_324))
print("Unresolved users:", len(unresolved_324))

print("\nD. GLOBAL SCENARIO DISTRIBUTION DIAGNOSTIC")

distribution_324 = pd.DataFrame([
    {
        "Metric": "Assigned users",
        "Original Stage 2": len(assignments),
        "Proposed": len(proposal_324)
    },
    {
        "Metric": "Distinct scenario templates",
        "Original Stage 2": original_usage_324.size,
        "Proposed": proposed_usage_324.size
    },
    {
        "Metric": "Maximum template usage",
        "Original Stage 2": int(original_usage_324.max()),
        "Proposed": int(proposed_usage_324.max())
    }
])

display(distribution_324)

print(
    "\nTemplates exceeding previous observed maximum:",
    int((proposed_usage_324 > original_usage_324.max()).sum())
)

print("\n" + "=" * 75)
print("STEP 3.2.4 — DIAGNOSTIC COMPLETED")
print("=" * 75)

print("Original Stage 2 files: UNCHANGED")
print("Proposed replacements: NOT FINALLY APPROVED")
print("Full scheduling and switching: NOT TESTED")
print("Plan B: NOT ACTIVATED")
print("No files were exported.")


STAGE 3 — STEP 3.2.4: ALTERNATIVE COMPATIBILITY

A. CANDIDATE COMPATIBILITY RESULTS


,Check,Passed Candidates,Failed Candidates
0,pass_stage2_group,1387,0
1,pass_stage2_channels,1387,0
2,pass_stage2_identity,1387,0
3,pass_stage2_workdays,1387,0
4,pass_stage2_baseline,1387,0
5,pass_duration_capacity,1387,0
6,pass_active_day_capacity,1387,0
7,pass_multichannel_capacity,1387,0
8,pass_three_channel_capacity,1387,0
9,pass_scenario_day_partition,1387,0



Total window-aware candidates: 1387
Candidates passing all checked conditions: 1387
Users with compatible candidates: 46

B. SELECTED DIAGNOSTIC ALTERNATIVES


,user,stage2_scenario_id,current_scenario,candidate_scenario,duration,stealth_level,distance_to_stage2,passing_windows
0,ABO1173,SCN_0205,SCN_0205,SCN_0382,40,High,6,5
28,ABS0677,SCN_0041,SCN_0041,SCN_0065,20,Medium,3,5
42,AGF1810,SCN_0037,SCN_0037,SCN_0061,20,High,3,2
58,AJH2110,SCN_0212,SCN_0212,SCN_0389,40,Medium,6,12
86,APB1122,SCN_0219,SCN_0219,SCN_0396,40,Low,6,10
114,ARH2097,SCN_0223,SCN_0223,SCN_0310,40,High,3,1
169,BDJ3326,SCN_0055,SCN_0055,SCN_0100,20,High,8,1
178,BKB0853,SCN_0128,SCN_0128,SCN_0197,30,Medium,6,16
193,BLB1876,SCN_1010,SCN_1010,SCN_1007,50,Medium,2,6
244,BNF1110,SCN_0533,SCN_0533,SCN_1013,50,Medium,6,9



C. UPDATED PROVISIONAL COHORT
Existing window-pass users: 703
Newly selected alternative users: 46
Updated provisional window-pass cohort: 749
Unresolved users: 246

D. GLOBAL SCENARIO DISTRIBUTION DIAGNOSTIC


,Metric,Original Stage 2,Proposed
0,Assigned users,995,995
1,Distinct scenario templates,490,486
2,Maximum template usage,20,20



Templates exceeding previous observed maximum: 0

STEP 3.2.4 — DIAGNOSTIC COMPLETED
Original Stage 2 files: UNCHANGED
Proposed replacements: NOT FINALLY APPROVED
Full scheduling and switching: NOT TESTED
Plan B: NOT ACTIVATED
No files were exported.



### Step 3.2.5 — Unified Cohort Integrity & Independent Window Validation

**Objective:** Independently validate the unified 749-user cohort before detailed attack scheduling.

**Validation Tasks:**
1. Verify unique user assignments and cohort membership.
2. Preserve the original scenarios of unresolved users.
3. Reconstruct effective scenario definitions.
4. Independently recalculate window-level activity capacities.
5. Verify the seven preliminary scenario conditions.
6. Confirm that each included user has at least one qualifying window.
7. Report the final provisional cohort and deferred users.

**Expected Outcome:** A consistent provisional cohort ready for detailed scheduling feasibility tests.

**Limitations:**
- No final attack windows or active/off-day sequences are selected.
- Actual channel switching remains untested.
- Proposed scenario replacements are not permanently approved.

**Note:** Read-only validation. No files will be modified or exported.


In [14]:

# Stage 3 — Step 3.2.5: Unified Cohort Integrity & Window Validation
# Goal: Independently validate all 749 provisional window-pass users
# and keep unresolved users on their original Stage 2 scenarios.
# Read-only: No files, assignments, or source data are modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.2.5: UNIFIED COHORT VALIDATION")
print("=" * 75)

# ------------------------------------------------------------
# 1. Reconstruct the full working population.
# ------------------------------------------------------------

assert len(assignments) == 995
assert assignments["user"].is_unique
assert len(unified_window_pass_324) == 749
assert unified_window_pass_324["user"].is_unique

working_assignments_325 = assignments.copy()

selected_map_325 = unified_window_pass_324.set_index(
    "user"
)["proposed_scenario_id"]

working_assignments_325["effective_scenario_id"] = (
    working_assignments_325["user"]
    .map(selected_map_325)
    .fillna(working_assignments_325["scenario_id"])
)

working_assignments_325["cohort_status"] = np.where(
    working_assignments_325["user"].isin(selected_map_325.index),
    "Window-pass",
    "Deferred"
)

working_assignments_325["scenario_changed"] = (
    working_assignments_325["effective_scenario_id"]
    != working_assignments_325["scenario_id"]
)

deferred_325 = working_assignments_325.loc[
    working_assignments_325["cohort_status"] == "Deferred"
]

assert len(deferred_325) == 246

assert (
    deferred_325["effective_scenario_id"]
    == deferred_325["scenario_id"]
).all(), "A deferred user's scenario was modified."

print("\nA. POPULATION INTEGRITY")
print(f"Total users: {len(working_assignments_325)}")
print(f"Window-pass users: {len(selected_map_325)}")
print(f"Deferred users: {len(deferred_325)}")
print(
    "Proposed scenario changes in working cohort:",
    int(working_assignments_325["scenario_changed"].sum())
)

# ------------------------------------------------------------
# 2. Join effective scenario requirements.
# ------------------------------------------------------------

cohort_325 = working_assignments_325.loc[
    working_assignments_325["cohort_status"] == "Window-pass"
].copy()

rules_325 = scenario_catalog[
    [
        "scenario_id", "duration", "stealth_level",
        "active_days", "off_days", "single_channel_days",
        "two_channel_days", "three_channel_days",
        "target_switches"
    ]
].rename(columns={
    "scenario_id": "effective_scenario_id",
    "duration": "effective_duration",
    "stealth_level": "effective_stealth",
    "active_days": "effective_active",
    "off_days": "effective_off",
    "single_channel_days": "effective_single",
    "two_channel_days": "effective_two",
    "three_channel_days": "effective_three",
    "target_switches": "effective_switches"
})

cohort_325 = cohort_325.merge(
    rules_325,
    on="effective_scenario_id",
    how="left",
    validate="many_to_one"
)

assert cohort_325["effective_duration"].notna().all()
assert cohort_325["duration"].eq(
    cohort_325["effective_duration"]
).all()
assert cohort_325["stealth_level"].eq(
    cohort_325["effective_stealth"]
).all()

# ------------------------------------------------------------
# 3. Independently reconstruct eligible daily channel activity.
# ------------------------------------------------------------

audit_daily_325 = user_day[
    [
        "user", "day", "is_weekend", "logon_count",
        "external_email_count", "usb_copy_count",
        "www_upload_count"
    ]
].copy()

audit_daily_325["day"] = pd.to_datetime(
    audit_daily_325["day"]
)

audit_daily_325 = audit_daily_325.loc[
    audit_daily_325["user"].isin(cohort_325["user"])
    & audit_daily_325["day"].between(
        pd.Timestamp("2011-01-01"),
        pd.Timestamp("2011-06-01")
    )
    & audit_daily_325["is_weekend"].eq(0)
    & audit_daily_325["logon_count"].gt(0)
].copy()

assert not audit_daily_325.duplicated(
    ["user", "day"]
).any()

audit_daily_325["channels_active"] = (
    audit_daily_325[
        [
            "external_email_count",
            "usb_copy_count",
            "www_upload_count"
        ]
    ].gt(0).sum(axis=1)
)

daily_groups_325 = {
    user: group.sort_values("day")
    for user, group in audit_daily_325.groupby("user")
}

# ------------------------------------------------------------
# 4. Independently recalculate rolling-window capacities.
# ------------------------------------------------------------

def rolling_sum_325(values, window):
    values = np.asarray(values, dtype=np.int64)
    cumulative = np.concatenate(([0], np.cumsum(values)))
    return cumulative[window:] - cumulative[:-window]

results_325 = []

for row in cohort_325.itertuples(index=False):

    days = daily_groups_325.get(row.user)
    duration = int(row.effective_duration)

    required_active = int(row.effective_active)
    required_multi = (
        int(row.effective_two) + int(row.effective_three)
    )
    required_three = int(row.effective_three)

    seven_checks = (
        int(row.eligible_workdays) >= duration
        and required_active + int(row.effective_off) == duration
        and (
            int(row.effective_single)
            + int(row.effective_two)
            + required_three
        ) == required_active
        and 0 <= int(row.effective_switches) <= required_active - 1
    )

    passing_windows = 0
    full_period_capacity_pass = False

    if days is not None:
        channel_counts = days["channels_active"].to_numpy()

        total_active = int((channel_counts >= 1).sum())
        total_multi = int((channel_counts >= 2).sum())
        total_three = int((channel_counts == 3).sum())

        full_period_capacity_pass = (
            len(days) >= duration
            and total_active >= required_active
            and total_multi >= required_multi
            and total_three >= required_three
        )

        if len(days) >= duration:
            active = rolling_sum_325(
                channel_counts >= 1, duration
            )
            multi = rolling_sum_325(
                channel_counts >= 2, duration
            )
            three = rolling_sum_325(
                channel_counts == 3, duration
            )

            valid = (
                (active >= required_active)
                & (multi >= required_multi)
                & (three >= required_three)
            )

            passing_windows = int(valid.sum())

    results_325.append({
        "user": row.user,
        "scenario_id": row.effective_scenario_id,
        "duration": duration,
        "scenario_changed": row.scenario_changed,
        "scenario_rules_pass": bool(seven_checks),
        "full_period_capacity_pass": bool(
            full_period_capacity_pass
        ),
        "passing_windows": passing_windows,
        "window_pass": passing_windows > 0
    })

validation_325 = pd.DataFrame(results_325)

validation_325["all_checked_pass"] = (
    validation_325[
        [
            "scenario_rules_pass",
            "full_period_capacity_pass",
            "window_pass"
        ]
    ].all(axis=1)
)

# ------------------------------------------------------------
# 5. Report independent validation.
# ------------------------------------------------------------

print("\nB. INDEPENDENT VALIDATION RESULTS")

validation_summary_325 = pd.DataFrame([
    {
        "Check": column,
        "Passed Users": int(validation_325[column].sum()),
        "Failed Users": int((~validation_325[column]).sum())
    }
    for column in [
        "scenario_rules_pass",
        "full_period_capacity_pass",
        "window_pass",
        "all_checked_pass"
    ]
])

display(validation_summary_325)

print("\nC. VALIDATION BY SCENARIO DURATION")

duration_validation_325 = (
    validation_325
    .groupby("duration")
    .agg(
        Users=("user", "count"),
        Passed=("all_checked_pass", "sum")
    )
    .reset_index()
)

duration_validation_325["Failed"] = (
    duration_validation_325["Users"]
    - duration_validation_325["Passed"]
)

display(duration_validation_325)

# ------------------------------------------------------------
# 6. Final consistency assertions.
# ------------------------------------------------------------

assert len(validation_325) == 749
assert validation_325["user"].is_unique
assert validation_325["all_checked_pass"].all(), (
    "Independent validation failed. Review results above."
)

assert set(cohort_325["user"]).isdisjoint(
    set(deferred_325["user"])
)

assert len(cohort_325) + len(deferred_325) == 995

print("\n" + "=" * 75)
print("STEP 3.2.5 — VALIDATION COMPLETED")
print("=" * 75)

print(f"Independently validated users: {len(validation_325)}")
print(f"Deferred users: {len(deferred_325)}")
print("Final attack windows: NOT SELECTED")
print("Active/off-day scheduling: NOT STARTED")
print("Actual channel switching: NOT TESTED")
print("Stage 2 files: UNCHANGED")
print("Plan B: NOT ACTIVATED")
print("No files were exported.")


STAGE 3 — STEP 3.2.5: UNIFIED COHORT VALIDATION

A. POPULATION INTEGRITY
Total users: 995
Window-pass users: 749
Deferred users: 246
Proposed scenario changes in working cohort: 48

B. INDEPENDENT VALIDATION RESULTS


,Check,Passed Users,Failed Users
0,scenario_rules_pass,749,0
1,full_period_capacity_pass,749,0
2,window_pass,749,0
3,all_checked_pass,749,0



C. VALIDATION BY SCENARIO DURATION


,duration,Users,Passed,Failed
0,5,157,157,0
1,10,144,144,0
2,20,142,142,0
3,30,123,123,0
4,40,89,89,0
5,50,94,94,0



STEP 3.2.5 — VALIDATION COMPLETED
Independently validated users: 749
Deferred users: 246
Final attack windows: NOT SELECTED
Active/off-day scheduling: NOT STARTED
Actual channel switching: NOT TESTED
Stage 2 files: UNCHANGED
Plan B: NOT ACTIVATED
No files were exported.



### Step 3.2.6 — Channel Dimension Distribution Audit

**Objective:** Analyze the channel-dimension requirements of the 749 independently validated scenarios.

**Validation Tasks:**
1. Count scenarios requiring only single-channel attack days.
2. Count scenarios requiring two-channel attack days but no three-channel days.
3. Count scenarios requiring at least one three-channel attack day.
4. Analyze channel requirements by scenario duration.
5. Display actual single-, two-, and three-channel day requirements.
6. Verify that all channel-day counts match the approved active-day requirements.

**Expected Outcome:** Understand the actual distribution of channel dimensions within the 749-user provisional cohort.

**Note:** Read-only analysis. No scenario assignments, attack windows, or source files will be modified.


In [15]:

# Stage 3 — Step 3.2.6: Channel Dimension Distribution Audit
# Goal: Count the actual single-, two-, and three-channel requirements
# of the 749 validated users and display their distributions.
# Read-only: No assignments, datasets, or files will be modified.

import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.2.6: CHANNEL DIMENSION DISTRIBUTION")
print("=" * 75)

# ------------------------------------------------------------
# 1. Prepare the independently validated 749-user cohort.
# ------------------------------------------------------------

channel_audit_326 = cohort_325[
    [
        "user",
        "effective_scenario_id",
        "effective_duration",
        "effective_active",
        "effective_single",
        "effective_two",
        "effective_three"
    ]
].copy()

assert len(channel_audit_326) == 749
assert channel_audit_326["user"].is_unique

# ------------------------------------------------------------
# 2. Classify each scenario by its highest daily channel dimension.
# Categories are mutually exclusive.
# ------------------------------------------------------------

def classify_channel_dimension(row):

    if row["effective_three"] > 0:
        return "Requires three-channel days"

    elif row["effective_two"] > 0:
        return "Requires two-channel days (no three)"

    else:
        return "Single-channel days only"


channel_audit_326["channel_category"] = (
    channel_audit_326.apply(
        classify_channel_dimension,
        axis=1
    )
)

category_order = [
    "Single-channel days only",
    "Requires two-channel days (no three)",
    "Requires three-channel days"
]

# ------------------------------------------------------------
# 3. Display overall channel-dimension distribution.
# ------------------------------------------------------------

print("\nA. CHANNEL DIMENSION DISTRIBUTION")

category_summary_326 = (
    channel_audit_326["channel_category"]
    .value_counts()
    .reindex(category_order, fill_value=0)
    .rename_axis("Channel Dimension")
    .reset_index(name="Users")
)

category_summary_326["Percentage"] = (
    category_summary_326["Users"] / 749 * 100
).round(2)

display(category_summary_326)

assert category_summary_326["Users"].sum() == 749

# ------------------------------------------------------------
# 4. Count users requiring each type of attack day.
# These categories can overlap.
# ------------------------------------------------------------

print("\nB. USERS REQUIRING EACH CHANNEL-DAY TYPE")

requirement_summary_326 = pd.DataFrame({
    "Requirement": [
        "At least one single-channel day",
        "At least one two-channel day",
        "At least one three-channel day"
    ],
    "Users": [
        int((channel_audit_326["effective_single"] > 0).sum()),
        int((channel_audit_326["effective_two"] > 0).sum()),
        int((channel_audit_326["effective_three"] > 0).sum())
    ]
})

display(requirement_summary_326)

# ------------------------------------------------------------
# 5. Analyze channel dimensions by scenario duration.
# ------------------------------------------------------------

print("\nC. CHANNEL DIMENSIONS BY SCENARIO DURATION")

duration_channels_326 = pd.crosstab(
    channel_audit_326["effective_duration"],
    channel_audit_326["channel_category"]
).reindex(columns=category_order, fill_value=0)

duration_channels_326["Total Users"] = (
    duration_channels_326.sum(axis=1)
)

display(duration_channels_326)

# ------------------------------------------------------------
# 6. Calculate total required attack days by channel dimension.
# ------------------------------------------------------------

print("\nD. TOTAL REQUIRED ATTACK DAYS")

day_summary_326 = pd.DataFrame({
    "Day Type": [
        "Single-channel attack days",
        "Two-channel attack days",
        "Three-channel attack days"
    ],
    "Total Required Days": [
        int(channel_audit_326["effective_single"].sum()),
        int(channel_audit_326["effective_two"].sum()),
        int(channel_audit_326["effective_three"].sum())
    ]
})

display(day_summary_326)

# ------------------------------------------------------------
# 7. Verify scenario day-count consistency.
# ------------------------------------------------------------

channel_audit_326["calculated_active_days"] = (
    channel_audit_326["effective_single"]
    + channel_audit_326["effective_two"]
    + channel_audit_326["effective_three"]
)

mismatches_326 = int(
    (
        channel_audit_326["calculated_active_days"]
        != channel_audit_326["effective_active"]
    ).sum()
)

print("\nE. VALIDATION")

print(f"Users checked: {len(channel_audit_326)}")
print(f"Channel-day partition mismatches: {mismatches_326}")

assert mismatches_326 == 0, (
    "Scenario channel-day partition mismatch detected."
)

# ------------------------------------------------------------
# 8. Display the actual scenario requirements.
# ------------------------------------------------------------

print("\nF. ACTUAL SCENARIO REQUIREMENTS — SAMPLE")

display(
    channel_audit_326[
        [
            "user",
            "effective_scenario_id",
            "effective_duration",
            "effective_single",
            "effective_two",
            "effective_three",
            "channel_category"
        ]
    ].sort_values(
        ["effective_three", "effective_two"],
        ascending=False
    ).head(20)
)

print("\n" + "=" * 75)
print("STEP 3.2.6 — COMPLETED")
print("=" * 75)
print("No source datasets or assignments were modified.")
print("No final attack windows were selected.")


STAGE 3 — STEP 3.2.6: CHANNEL DIMENSION DISTRIBUTION

A. CHANNEL DIMENSION DISTRIBUTION


,Channel Dimension,Users,Percentage
0,Single-channel days only,157,20.96
1,Requires two-channel days (no three),409,54.61
2,Requires three-channel days,183,24.43



B. USERS REQUIRING EACH CHANNEL-DAY TYPE


,Requirement,Users
0,At least one single-channel day,749
1,At least one two-channel day,592
2,At least one three-channel day,183



C. CHANNEL DIMENSIONS BY SCENARIO DURATION


channel_category,Single-channel days only,Requires two-channel days (no three),Requires three-channel days,Total Users
effective_duration,,,,
5,157,0,0,157
10,0,144,0,144
20,0,142,0,142
30,0,123,0,123
40,0,0,89,89
50,0,0,94,94



D. TOTAL REQUIRED ATTACK DAYS


,Day Type,Total Required Days
0,Single-channel attack days,12572
1,Two-channel attack days,1525
2,Three-channel attack days,220



E. VALIDATION
Users checked: 749
Channel-day partition mismatches: 0

F. ACTUAL SCENARIO REQUIREMENTS — SAMPLE


,user,effective_scenario_id,effective_duration,effective_single,effective_two,effective_three,channel_category
38,AKH0454,SCN_0997,50,33,5,2,Requires three-channel days
67,AYS1628,SCN_0518,50,35,5,2,Requires three-channel days
69,BBA1230,SCN_0519,50,35,5,2,Requires three-channel days
79,BDJ3001,SCN_0529,50,35,5,2,Requires three-channel days
183,DGS1079,SCN_0590,50,35,5,2,Requires three-channel days
186,DHG1830,SCN_0591,50,35,5,2,Requires three-channel days
464,MAH0438,SCN_0794,50,34,5,2,Requires three-channel days
484,MIC2325,SCN_0807,50,34,5,2,Requires three-channel days
535,NVH3726,SCN_0841,50,34,5,2,Requires three-channel days
553,PMF0768,SCN_0867,50,34,5,2,Requires three-channel days



STEP 3.2.6 — COMPLETED
No source datasets or assignments were modified.
No final attack windows were selected.



## Step 3.3 — Active & Off-Day Scheduling

### Step 3.3.1 — Candidate Window Registry

**Objective:** Register all preliminary feasible attack windows for the 749 validated users.

**Validation Tasks:**
1. Generate all eligible-workday windows matching each scenario duration.
2. Retain windows satisfying active-day, multichannel-day, and three-channel-day requirements.
3. Record the start and end dates of each window.
4. Validate each window's activity capacity.
5. Cross-check the registry against Step 3.2.5.
6. Preserve all qualifying windows for subsequent scheduling.

**Expected Outcome:** A validated candidate-window registry for all 749 users.

**Limitations:**
- No final attack windows are selected.
- Active and off days are not scheduled yet.
- Actual channel switching is not yet tested.
- Scenario replacements remain provisional.

**Note:** Read-only analysis. No source files or assignments will be modified.


In [16]:

# Stage 3 — Step 3.3.1: Candidate Window Registry
# Goal: Register all qualifying attack windows for the 749 users
# and independently verify their activity capacities.
# Read-only: No final scheduling, injection, or file export.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.3.1: CANDIDATE WINDOW REGISTRY")
print("=" * 75)

# ------------------------------------------------------------
# 1. Validate previously approved in-memory results.
# ------------------------------------------------------------

assert len(cohort_325) == 749
assert cohort_325["user"].is_unique
assert len(validation_325) == 749
assert validation_325["all_checked_pass"].all()

# ------------------------------------------------------------
# 2. Prepare eligible daily activity, ordered by date.
# ------------------------------------------------------------

daily_groups_331 = {
    user: group.sort_values("day").reset_index(drop=True)
    for user, group in audit_daily_325.groupby("user")
}

def rolling_sum_331(values, window_size):
    values = np.asarray(values, dtype=np.int64)
    cumulative = np.concatenate(
        ([0], np.cumsum(values))
    )
    return (
        cumulative[window_size:]
        - cumulative[:-window_size]
    )

# ------------------------------------------------------------
# 3. Enumerate all qualifying eligible-workday windows.
# ------------------------------------------------------------

window_records_331 = []

for row in cohort_325.itertuples(index=False):

    user = row.user
    duration = int(row.effective_duration)

    required_active = int(row.effective_active)
    required_multi = (
        int(row.effective_two)
        + int(row.effective_three)
    )
    required_three = int(row.effective_three)

    days = daily_groups_331.get(user)

    if days is None or len(days) < duration:
        continue

    counts = days["channels_active"].to_numpy()

    active_counts = rolling_sum_331(
        counts >= 1, duration
    )
    multi_counts = rolling_sum_331(
        counts >= 2, duration
    )
    three_counts = rolling_sum_331(
        counts == 3, duration
    )

    valid_indices = np.flatnonzero(
        (active_counts >= required_active)
        & (multi_counts >= required_multi)
        & (three_counts >= required_three)
    )

    for start_idx in valid_indices:

        start_idx = int(start_idx)
        end_idx = start_idx + duration - 1

        window_records_331.append({
            "user": user,
            "scenario_id": row.effective_scenario_id,
            "duration": duration,
            "window_start": days.iloc[start_idx]["day"],
            "window_end": days.iloc[end_idx]["day"],
            "required_active_days": required_active,
            "required_multichannel_days": required_multi,
            "required_three_channel_days": required_three,
            "available_active_days": int(active_counts[start_idx]),
            "available_multichannel_days": int(multi_counts[start_idx]),
            "available_three_channel_days": int(three_counts[start_idx])
        })

candidate_windows_331 = pd.DataFrame(window_records_331)

# ------------------------------------------------------------
# 4. Validate window uniqueness and channel capacity.
# ------------------------------------------------------------

assert not candidate_windows_331.empty, (
    "No qualifying windows were generated."
)

assert not candidate_windows_331.duplicated(
    ["user", "scenario_id", "window_start", "window_end"]
).any(), "Duplicate candidate windows detected."

capacity_pass_331 = (
    (candidate_windows_331["available_active_days"]
     >= candidate_windows_331["required_active_days"])
    &
    (candidate_windows_331["available_multichannel_days"]
     >= candidate_windows_331["required_multichannel_days"])
    &
    (candidate_windows_331["available_three_channel_days"]
     >= candidate_windows_331["required_three_channel_days"])
)

assert capacity_pass_331.all(), (
    "At least one candidate window violates channel capacity."
)

assert candidate_windows_331["user"].nunique() == 749, (
    "Some validated users have no registered windows."
)

# ------------------------------------------------------------
# 5. Cross-check against the independent window validation.
# ------------------------------------------------------------

registry_counts_331 = (
    candidate_windows_331.groupby("user")
    .size()
    .rename("registry_windows")
)

crosscheck_331 = validation_325[
    ["user", "passing_windows"]
].merge(
    registry_counts_331,
    on="user",
    how="left",
    validate="one_to_one"
)

assert (
    crosscheck_331["passing_windows"]
    == crosscheck_331["registry_windows"]
).all(), (
    "Registry counts disagree with Step 3.2.5."
)

# ------------------------------------------------------------
# 6. Report registry statistics.
# ------------------------------------------------------------

windows_per_user_331 = (
    candidate_windows_331.groupby("user").size()
)

print("\nA. REGISTRY SUMMARY")
print(f"Validated users: {len(cohort_325):,}")
print(f"Users with qualifying windows: "
      f"{candidate_windows_331['user'].nunique():,}")
print(f"Total candidate windows: {len(candidate_windows_331):,}")
print(f"Minimum per user: {windows_per_user_331.min():,}")
print(f"Median per user: {windows_per_user_331.median():.1f}")
print(f"Maximum per user: {windows_per_user_331.max():,}")

print("\nB. WINDOWS BY SCENARIO DURATION")

duration_summary_331 = (
    candidate_windows_331
    .groupby("duration")
    .agg(
        Users=("user", "nunique"),
        Candidate_Windows=("user", "size")
    )
    .reset_index()
)

duration_summary_331["Average_Windows_Per_User"] = (
    duration_summary_331["Candidate_Windows"]
    / duration_summary_331["Users"]
).round(2)

display(duration_summary_331)

print("\nC. USERS WITH FEWEST CANDIDATE WINDOWS")

display(
    windows_per_user_331
    .sort_values()
    .head(15)
    .reset_index(name="candidate_windows")
)

print("\nD. ACTUAL CANDIDATE WINDOW SAMPLE")

display(
    candidate_windows_331
    .sort_values(["user", "window_start"])
    .head(15)
)

# ------------------------------------------------------------
# 7. Final validation summary.
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 3.3.1 — VALIDATION SUMMARY")
print("=" * 75)

print("Unique window keys: PASS")
print("Window activity capacities: PASS")
print("All 749 users represented: PASS")
print("Independent window count cross-check: PASS")

print("\nFinal attack windows: NOT SELECTED")
print("Active/off-day schedules: NOT CREATED")
print("Actual channel switching: NOT TESTED")
print("Plan B: NOT ACTIVATED")
print("No source files were modified or exported.")


STAGE 3 — STEP 3.3.1: CANDIDATE WINDOW REGISTRY

A. REGISTRY SUMMARY
Validated users: 749
Users with qualifying windows: 749
Total candidate windows: 29,485
Minimum per user: 1
Median per user: 37.0
Maximum per user: 103

B. WINDOWS BY SCENARIO DURATION


,duration,Users,Candidate_Windows,Average_Windows_Per_User
0,5,157,5894,37.54
1,10,144,5897,40.95
2,20,142,6267,44.13
3,30,123,4685,38.09
4,40,89,3431,38.55
5,50,94,3311,35.22



C. USERS WITH FEWEST CANDIDATE WINDOWS


,user,candidate_windows
0,BDB2163,1
1,BDJ3326,1
2,BKF1545,1
3,ADH0975,1
4,BNS3233,1
5,BRW3552,1
6,RKH2877,1
7,HCB1383,1
8,BWM2339,1
9,JAB3656,1



D. ACTUAL CANDIDATE WINDOW SAMPLE


,user,scenario_id,duration,window_start,window_end,required_active_days,required_multichannel_days,required_three_channel_days,available_active_days,available_multichannel_days,available_three_channel_days
0,AAF0819,SCN_0007,5,2011-01-03,2011-01-07,5,0,0,5,4,0
1,AAF0819,SCN_0007,5,2011-01-04,2011-01-10,5,0,0,5,4,0
2,AAF0819,SCN_0007,5,2011-01-05,2011-01-11,5,0,0,5,4,0
3,AAF0819,SCN_0007,5,2011-01-06,2011-01-12,5,0,0,5,4,0
4,AAF0819,SCN_0007,5,2011-01-07,2011-01-13,5,0,0,5,4,0
5,AAF0819,SCN_0007,5,2011-01-10,2011-01-14,5,0,0,5,4,0
6,AAF0819,SCN_0007,5,2011-01-11,2011-01-17,5,0,0,5,4,0
7,AAF0819,SCN_0007,5,2011-01-12,2011-01-18,5,0,0,5,4,0
8,AAF0819,SCN_0007,5,2011-01-13,2011-01-19,5,0,0,5,3,0
9,AAF0819,SCN_0007,5,2011-01-14,2011-01-20,5,0,0,5,3,0



STEP 3.3.1 — VALIDATION SUMMARY
Unique window keys: PASS
Window activity capacities: PASS
All 749 users represented: PASS
Independent window count cross-check: PASS

Final attack windows: NOT SELECTED
Active/off-day schedules: NOT CREATED
Actual channel switching: NOT TESTED
Plan B: NOT ACTIVATED
No source files were modified or exported.



### Step 3.3.2 — Active & Off-Day Allocation Audit

**Objective:** Construct and validate a provisional active/off-day allocation for each of the 749 users under Plan A.

**Validation Tasks:**
1. Select one demonstration window per user from the validated registry.
2. Allocate the required three-channel days first.
3. Allocate the required two-channel days using remaining eligible dates.
4. Allocate single-channel days and mark remaining dates as attack-off days.
5. Verify exact scenario day counts and daily channel capacity.
6. Display sample provisional schedules.

**Important Limitations:**
- Demonstration windows are not final selections.
- Actual Email, USB, and HTTP channel combinations are not assigned.
- Primary-channel switching has not been tested.
- Off days mean no synthetic attack injection; benign activity may still occur.

**Expected Outcome:** Validate that the scenario's daily channel-count requirements can be represented by an explicit, non-overlapping day allocation.

**Note:** Read-only analysis. No attack injection or file export.


In [17]:

# Stage 3 — Step 3.3.2: Active & Off-Day Allocation Audit
# Goal: Construct one provisional day-type allocation per user
# and verify exact scenario counts and Plan A daily capacity.
# Read-only: No final scheduling, switching, injection, or export.

import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.3.2: ACTIVE & OFF-DAY ALLOCATION")
print("=" * 75)

# ------------------------------------------------------------
# 1. Verify previous results.
# ------------------------------------------------------------

assert len(cohort_325) == 749
assert cohort_325["user"].is_unique
assert candidate_windows_331["user"].nunique() == 749

# ------------------------------------------------------------
# 2. Select one DEMONSTRATION window per user.
# Prefer larger capacity margins, then earlier dates.
# This is NOT the final window-selection policy.
# ------------------------------------------------------------

ranked_windows_332 = candidate_windows_331.copy()

for kind in ["active", "multichannel", "three_channel"]:
    ranked_windows_332[f"{kind}_slack"] = (
        ranked_windows_332[f"available_{kind}_days"]
        - ranked_windows_332[f"required_{kind}_days"]
    )

demo_windows_332 = (
    ranked_windows_332
    .sort_values(
        [
            "user", "three_channel_slack",
            "multichannel_slack", "active_slack",
            "window_start"
        ],
        ascending=[True, False, False, False, True],
        kind="stable"
    )
    .drop_duplicates("user")
    .copy()
)

assert len(demo_windows_332) == 749

# ------------------------------------------------------------
# 3. Prepare lookup tables.
# ------------------------------------------------------------

scenario_lookup_332 = cohort_325.set_index("user")

daily_lookup_332 = {
    user: group.sort_values("day").reset_index(drop=True)
    for user, group in audit_daily_325.groupby("user")
}

records_332 = []

# ------------------------------------------------------------
# 4. Allocate exact daily channel-count requirements.
# Three-channel first, then two-channel, then single-channel.
# Days are disjoint; all remaining days become attack-off.
# ------------------------------------------------------------

for window in demo_windows_332.itertuples(index=False):

    user = window.user
    scenario = scenario_lookup_332.loc[user]

    days = daily_lookup_332[user]

    days = days.loc[
        days["day"].between(
            window.window_start,
            window.window_end
        )
    ].copy().reset_index(drop=True)

    duration = int(scenario["effective_duration"])

    assert len(days) == duration, (
        f"Invalid demonstration window length: {user}"
    )

    days["planned_channels"] = 0

    requirements = [
        (3, int(scenario["effective_three"])),
        (2, int(scenario["effective_two"])),
        (1, int(scenario["effective_single"]))
    ]

    for channel_count, required_days in requirements:

        available = days.loc[
            (days["planned_channels"] == 0)
            & (days["channels_active"] >= channel_count)
        ]

        # Choose less-flexible days first.
        chosen_indices = (
            available
            .sort_values(["channels_active", "day"])
            .index[:required_days]
        )

        assert len(chosen_indices) == required_days, (
            f"Insufficient distinct days for user {user}, "
            f"channel count {channel_count}"
        )

        days.loc[chosen_indices, "planned_channels"] = (
            channel_count
        )

    assert int((days["planned_channels"] == 0).sum()) == int(
        scenario["effective_off"]
    )

    assert (
        days["planned_channels"] <= days["channels_active"]
    ).all(), f"Plan A activity violation: {user}"

    for day in days.itertuples(index=False):

        records_332.append({
            "user": user,
            "scenario_id": window.scenario_id,
            "window_start": window.window_start,
            "window_end": window.window_end,
            "day": day.day,
            "natural_active_channels": int(day.channels_active),
            "planned_channels": int(day.planned_channels),
            "attack_day_type": (
                "Off" if day.planned_channels == 0
                else f"{day.planned_channels}-Channel"
            )
        })

# ------------------------------------------------------------
# 5. Build and validate provisional day allocations.
# ------------------------------------------------------------

provisional_days_332 = pd.DataFrame(records_332)

assert provisional_days_332["user"].nunique() == 749

assert not provisional_days_332.duplicated(
    ["user", "day"]
).any()

actual_counts_332 = (
    provisional_days_332
    .groupby(["user", "planned_channels"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=[0, 1, 2, 3], fill_value=0)
)

expected_counts_332 = cohort_325.set_index("user")[
    [
        "effective_off", "effective_single",
        "effective_two", "effective_three"
    ]
].rename(columns={
    "effective_off": 0,
    "effective_single": 1,
    "effective_two": 2,
    "effective_three": 3
})

expected_counts_332 = expected_counts_332[
    [0, 1, 2, 3]
]

actual_counts_332 = actual_counts_332.reindex(
    expected_counts_332.index
)

exact_match_332 = (
    actual_counts_332 == expected_counts_332
).all(axis=1)

capacity_match_332 = (
    provisional_days_332["planned_channels"]
    <= provisional_days_332["natural_active_channels"]
).all()

assert exact_match_332.all()
assert capacity_match_332

# ------------------------------------------------------------
# 6. Display validation summary.
# ------------------------------------------------------------

print("\nA. DAY ALLOCATION VALIDATION")

print(f"Users evaluated: {len(exact_match_332):,}")
print(f"Users with exact day-count matches: "
      f"{int(exact_match_332.sum()):,}")
print(f"Users with mismatches: "
      f"{int((~exact_match_332).sum()):,}")
print(f"Daily channel-capacity check: "
      f"{'PASS' if capacity_match_332 else 'FAIL'}")

print("\nB. TOTAL PROVISIONAL DAY TYPES")

day_totals_332 = (
    provisional_days_332["attack_day_type"]
    .value_counts()
    .reindex(
        ["Off", "1-Channel", "2-Channel", "3-Channel"],
        fill_value=0
    )
    .rename_axis("Day Type")
    .reset_index(name="Days")
)

display(day_totals_332)

# ------------------------------------------------------------
# 7. Display an actual provisional 3-channel scenario.
# ------------------------------------------------------------

three_channel_users_332 = cohort_325.loc[
    cohort_325["effective_three"] > 0,
    "user"
]

print("\nC. THREE-CHANNEL SCENARIO EXAMPLE")

if not three_channel_users_332.empty:

    sample_user_332 = three_channel_users_332.iloc[0]

    print(f"User: {sample_user_332}")
    print("Showing provisional days from this user's window.")

    display(
        provisional_days_332.loc[
            provisional_days_332["user"] == sample_user_332,
            [
                "day", "natural_active_channels",
                "planned_channels", "attack_day_type"
            ]
        ].head(20)
    )

# ------------------------------------------------------------
# 8. Final status.
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 3.3.2 — ALLOCATION AUDIT COMPLETED")
print("=" * 75)

print("Day-count allocation: VERIFIED")
print("Demonstration windows: PROVISIONAL")
print("Specific attack channels: NOT SELECTED")
print("Primary-channel switching: NOT TESTED")
print("Final attack schedules: NOT APPROVED")
print("Plan B: NOT ACTIVATED")
print("No files were modified or exported.")


STAGE 3 — STEP 3.3.2: ACTIVE & OFF-DAY ALLOCATION

A. DAY ALLOCATION VALIDATION
Users evaluated: 749
Users with exact day-count matches: 749
Users with mismatches: 0
Daily channel-capacity check: PASS

B. TOTAL PROVISIONAL DAY TYPES


,Day Type,Days
0,Off,2698
1,1-Channel,12572
2,2-Channel,1525
3,3-Channel,220



C. THREE-CHANNEL SCENARIO EXAMPLE
User: ABD3426
Showing provisional days from this user's window.


,day,natural_active_channels,planned_channels,attack_day_type
35,2011-03-31,2,2,2-Channel
36,2011-04-01,3,3,3-Channel
37,2011-04-04,2,2,2-Channel
38,2011-04-05,2,2,2-Channel
39,2011-04-06,1,1,1-Channel
40,2011-04-07,2,1,1-Channel
41,2011-04-08,3,0,Off
42,2011-04-11,2,1,1-Channel
43,2011-04-12,1,1,1-Channel
44,2011-04-13,2,1,1-Channel



STEP 3.3.2 — ALLOCATION AUDIT COMPLETED
Day-count allocation: VERIFIED
Demonstration windows: PROVISIONAL
Specific attack channels: NOT SELECTED
Primary-channel switching: NOT TESTED
Final attack schedules: NOT APPROVED
Plan B: NOT ACTIVATED
No files were modified or exported.



### Step 3.3.3 — Primary-Channel Switching Feasibility Audit

**Objective:** Determine whether the provisional active/off-day allocations can achieve each user's exact target number of primary-channel switches under Plan A.

**Validation Tasks:**
1. Identify naturally active Email, USB, and HTTP channels on each scheduled attack day.
2. Determine the possible primary channels for each active day.
3. Calculate reachable switching counts across consecutive attack-active days.
4. Check whether the exact `target_switches` value is achievable.
5. Diagnose users whose demonstration allocations cannot satisfy switching requirements.

**Switching Definition:** A switch occurs when the primary channel changes between consecutive attack-active days. Attack-off days are excluded from the switching sequence.

**Expected Outcome:** Identify which provisional allocations can meet the exact switching target before constructing final channel schedules.

**Limitations:**
- This test uses one demonstration window and day allocation per user.
- Failure does not establish that every candidate window or allocation is infeasible.
- Specific channel combinations and final primary-channel sequences are not selected.

**Note:** Read-only. No source datasets, assignments, or files will be modified.


In [18]:

# Stage 3 — Step 3.3.3: Primary-Channel Switching Feasibility Audit
# Goal: Test whether each provisional attack-day allocation can
# achieve its exact target number of primary-channel switches.
# Read-only: No schedules, assignments, or files are modified.

import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.3.3: SWITCHING FEASIBILITY AUDIT")
print("=" * 75)

# ------------------------------------------------------------
# 1. Validate the provisional day allocations.
# ------------------------------------------------------------

assert provisional_days_332["user"].nunique() == 749
assert cohort_325["user"].is_unique

assert not provisional_days_332.duplicated(
    ["user", "day"]
).any()

# ------------------------------------------------------------
# 2. Attach actual daily channel availability.
# ------------------------------------------------------------

channel_columns_333 = [
    "external_email_count",
    "usb_copy_count",
    "www_upload_count"
]

natural_activity_333 = audit_daily_325[
    ["user", "day"] + channel_columns_333
].copy()

switching_days_333 = provisional_days_332.merge(
    natural_activity_333,
    on=["user", "day"],
    how="left",
    validate="one_to_one"
)

assert switching_days_333[channel_columns_333].notna().all().all(), (
    "Missing natural channel activity after merge."
)

switching_days_333["verified_active_channels"] = (
    switching_days_333[channel_columns_333].gt(0).sum(axis=1)
)

assert (
    switching_days_333["verified_active_channels"]
    == switching_days_333["natural_active_channels"]
).all(), "Natural channel activity mismatch."

assert (
    switching_days_333["planned_channels"]
    <= switching_days_333["verified_active_channels"]
).all(), "Plan A daily channel availability violation."

# ------------------------------------------------------------
# 3. Determine reachable switching counts.
# Dynamic programming tracks ALL achievable switch counts,
# not only minimum and maximum counts.
# ------------------------------------------------------------

def reachable_switch_counts_333(active_days):

    states = {}

    for record in active_days.itertuples(index=False):

        available = []

        if record.external_email_count > 0:
            available.append("Email")

        if record.usb_copy_count > 0:
            available.append("USB")

        if record.www_upload_count > 0:
            available.append("HTTP")

        planned = int(record.planned_channels)

        assert planned >= 1
        assert len(available) >= planned

        # Any naturally active channel may be primary.
        # The remaining planned channels can be chosen
        # from the other naturally active channels.
        if not states:
            states = {channel: 1 for channel in available}
            continue

        next_states = {}

        for current in available:

            reachable_mask = 0

            for previous, previous_mask in states.items():

                if previous == current:
                    reachable_mask |= previous_mask
                else:
                    reachable_mask |= previous_mask << 1

            next_states[current] = reachable_mask

        states = next_states

    combined_mask = 0

    for mask in states.values():
        combined_mask |= mask

    reachable = [
        count
        for count in range(combined_mask.bit_length())
        if (combined_mask >> count) & 1
    ]

    return reachable

# ------------------------------------------------------------
# 4. Evaluate all 749 provisional schedules.
# ------------------------------------------------------------

scenario_lookup_333 = cohort_325.set_index("user")

switching_results_333 = []

for user, group in switching_days_333.groupby("user"):

    group = group.sort_values("day")

    active_days = group.loc[
        group["planned_channels"] > 0
    ].copy()

    scenario = scenario_lookup_333.loc[user]

    required_switches = int(
        scenario["effective_switches"]
    )

    reachable = reachable_switch_counts_333(active_days)

    exact_target_pass = required_switches in reachable

    switching_results_333.append({
        "user": user,
        "scenario_id": scenario["effective_scenario_id"],
        "duration": int(scenario["effective_duration"]),
        "target_switches": required_switches,
        "active_days": len(active_days),
        "minimum_possible_switches": min(reachable),
        "maximum_possible_switches": max(reachable),
        "reachable_switch_count_options": len(reachable),
        "exact_switching_feasible": exact_target_pass
    })

switching_audit_333 = pd.DataFrame(switching_results_333)

assert len(switching_audit_333) == 749
assert switching_audit_333["user"].is_unique

# ------------------------------------------------------------
# 5. Overall switching feasibility.
# ------------------------------------------------------------

passed_333 = int(
    switching_audit_333["exact_switching_feasible"].sum()
)

failed_333 = len(switching_audit_333) - passed_333

print("\nA. EXACT SWITCHING FEASIBILITY")

print(f"Users evaluated: {len(switching_audit_333):,}")
print(f"Exact switching target achievable: {passed_333:,}")
print(f"Exact switching target not achievable: {failed_333:,}")

# ------------------------------------------------------------
# 6. Results by scenario duration.
# ------------------------------------------------------------

print("\nB. SWITCHING FEASIBILITY BY DURATION")

duration_switching_333 = (
    switching_audit_333
    .groupby("duration")
    .agg(
        Users=("user", "count"),
        Passing=("exact_switching_feasible", "sum")
    )
    .reset_index()
)

duration_switching_333["Failing"] = (
    duration_switching_333["Users"]
    - duration_switching_333["Passing"]
)

display(duration_switching_333)

# ------------------------------------------------------------
# 7. Diagnose allocations that cannot achieve the target.
# ------------------------------------------------------------

print("\nC. USERS FAILING EXACT SWITCHING")

failed_switching_333 = switching_audit_333.loc[
    ~switching_audit_333["exact_switching_feasible"]
].copy()

if failed_switching_333.empty:
    print("All demonstration allocations can achieve their switching targets.")
else:
    display(
        failed_switching_333[
            [
                "user",
                "scenario_id",
                "duration",
                "active_days",
                "target_switches",
                "minimum_possible_switches",
                "maximum_possible_switches",
                "reachable_switch_count_options"
            ]
        ].head(25)
    )

# ------------------------------------------------------------
# 8. Preview successful cases.
# ------------------------------------------------------------

print("\nD. SUCCESSFUL SWITCHING EXAMPLES")

display(
    switching_audit_333.loc[
        switching_audit_333["exact_switching_feasible"],
        [
            "user",
            "scenario_id",
            "duration",
            "target_switches",
            "minimum_possible_switches",
            "maximum_possible_switches"
        ]
    ].head(10)
)

# ------------------------------------------------------------
# 9. Final status.
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 3.3.3 — AUDIT COMPLETED")
print("=" * 75)

print("Demonstration day allocations: UNCHANGED")
print("Final attack windows: NOT SELECTED")
print("Exact primary-channel sequences: NOT CONSTRUCTED")
print("Alternative windows and allocations: NOT YET TESTED")
print("Original Stage 2 files: UNCHANGED")
print("Plan B: NOT ACTIVATED")
print("No files were exported.")


STAGE 3 — STEP 3.3.3: SWITCHING FEASIBILITY AUDIT

A. EXACT SWITCHING FEASIBILITY
Users evaluated: 749
Exact switching target achievable: 544
Exact switching target not achievable: 205

B. SWITCHING FEASIBILITY BY DURATION


,duration,Users,Passing,Failing
0,5,157,141,16
1,10,144,100,44
2,20,142,91,51
3,30,123,75,48
4,40,89,66,23
5,50,94,71,23



C. USERS FAILING EXACT SWITCHING


,user,scenario_id,duration,active_days,target_switches,minimum_possible_switches,maximum_possible_switches,reachable_switch_count_options
6,ABO1173,SCN_0382,40,32,4,10,23,14
13,ACV1946,SCN_0472,50,42,6,10,27,17
14,ADG1737,SCN_0206,40,34,4,8,31,24
15,ADH0975,SCN_0018,10,9,3,4,4,1
17,AEB3249,SCN_0207,40,34,4,6,17,12
23,AFB3228,SCN_0020,10,9,4,0,2,2
27,AGW1590,SCN_0211,40,34,5,15,27,13
28,AHB0648,SCN_0021,10,9,4,2,2,1
31,AIJ3995,SCN_0022,10,9,5,0,1,2
33,AJH2110,SCN_0389,40,32,5,11,24,14



D. SUCCESSFUL SWITCHING EXAMPLES


,user,scenario_id,duration,target_switches,minimum_possible_switches,maximum_possible_switches
0,AAF0819,SCN_0007,5,3,0,4
1,AAP0352,SCN_0016,10,3,1,7
2,AAP1919,SCN_0040,20,4,2,11
3,ABD3426,SCN_0402,40,7,0,28
4,ABK3081,SCN_0009,5,3,0,3
5,ABM3772,SCN_0008,5,3,0,4
7,ABP2917,SCN_0041,20,4,4,10
8,ABS0677,SCN_0065,20,4,2,8
9,ACB3123,SCN_0110,30,5,0,7
10,ACP2122,SCN_0003,5,1,1,3



STEP 3.3.3 — AUDIT COMPLETED
Demonstration day allocations: UNCHANGED
Final attack windows: NOT SELECTED
Exact primary-channel sequences: NOT CONSTRUCTED
Alternative windows and allocations: NOT YET TESTED
Original Stage 2 files: UNCHANGED
Plan B: NOT ACTIVATED
No files were exported.



### Step 3.3.4 — Switching Failure Diagnosis

**Objective:** Diagnose why 205 provisional attack-day allocations cannot achieve their exact primary-channel switching targets.

**Validation Tasks:**
1. Identify users whose minimum switching count exceeds the target.
2. Identify users whose maximum switching count is below the target.
3. Identify unreachable targets between the minimum and maximum.
4. Measure switching gaps and classify affected users by scenario duration.
5. Preserve all original assignments and provisional schedules.

**Expected Outcome:** Determine which switching constraints require alternative windows or day allocations before final scheduling.

**Important:** This audit diagnoses the current demonstration allocations only. It does not establish full Plan A infeasibility.

**Note:** Read-only. No source datasets or files will be modified.


In [19]:

# Stage 3 — Step 3.3.4: Switching Failure Diagnosis
# Goal: Classify the 205 switching failures and measure how far
# each provisional allocation is from its exact switching target.
# Read-only: No assignments, schedules, or files are modified.

import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.3.4: SWITCHING FAILURE DIAGNOSIS")
print("=" * 75)

# ------------------------------------------------------------
# 1. Validate Step 3.3.3 results.
# ------------------------------------------------------------

assert len(switching_audit_333) == 749
assert switching_audit_333["user"].is_unique

failed_switching_334 = switching_audit_333.loc[
    ~switching_audit_333["exact_switching_feasible"]
].copy()

assert len(failed_switching_334) == 205, (
    "Unexpected failure count. Review Step 3.3.3."
)

# ------------------------------------------------------------
# 2. Classify the reason for switching failure.
# ------------------------------------------------------------

def classify_switching_failure_334(row):

    target = int(row["target_switches"])
    minimum = int(row["minimum_possible_switches"])
    maximum = int(row["maximum_possible_switches"])

    if target < minimum:
        return "Too many unavoidable switches"

    if target > maximum:
        return "Insufficient possible switches"

    return "Unreachable target within bounds"


failed_switching_334["failure_type"] = (
    failed_switching_334.apply(
        classify_switching_failure_334,
        axis=1
    )
)

# ------------------------------------------------------------
# 3. Calculate the gap to the reachable range.
# A zero gap may still be infeasible if the target is missing
# from the exact reachable switching counts.
# ------------------------------------------------------------

failed_switching_334["switching_gap"] = (
    failed_switching_334.apply(
        lambda row: max(
            int(row["minimum_possible_switches"])
            - int(row["target_switches"]),
            int(row["target_switches"])
            - int(row["maximum_possible_switches"]),
            0
        ),
        axis=1
    )
)

# ------------------------------------------------------------
# 4. Summarize switching failure types.
# ------------------------------------------------------------

print("\nA. SWITCHING FAILURE TYPES")

failure_summary_334 = (
    failed_switching_334["failure_type"]
    .value_counts()
    .rename_axis("Failure Type")
    .reset_index(name="Users")
)

display(failure_summary_334)

assert failure_summary_334["Users"].sum() == 205

# ------------------------------------------------------------
# 5. Analyze failures by scenario duration.
# ------------------------------------------------------------

print("\nB. FAILURE TYPES BY SCENARIO DURATION")

duration_failure_334 = pd.crosstab(
    failed_switching_334["duration"],
    failed_switching_334["failure_type"]
)

duration_failure_334["Total"] = (
    duration_failure_334.sum(axis=1)
)

display(duration_failure_334)

# ------------------------------------------------------------
# 6. Summarize gap severity for out-of-range targets.
# ------------------------------------------------------------

print("\nC. SWITCHING GAP SEVERITY")

gap_summary_334 = (
    failed_switching_334
    .groupby("failure_type")
    .agg(
        Users=("user", "count"),
        Median_Gap=("switching_gap", "median"),
        Maximum_Gap=("switching_gap", "max")
    )
    .reset_index()
)

display(gap_summary_334)

# ------------------------------------------------------------
# 7. Display the most constrained demonstration allocations.
# ------------------------------------------------------------

print("\nD. LARGEST SWITCHING GAPS")

display(
    failed_switching_334[
        [
            "user",
            "scenario_id",
            "duration",
            "target_switches",
            "minimum_possible_switches",
            "maximum_possible_switches",
            "failure_type",
            "switching_gap"
        ]
    ]
    .sort_values(
        ["switching_gap", "user"],
        ascending=[False, True]
    )
    .head(20)
)

# ------------------------------------------------------------
# 8. Final reconciliation and audit status.
# ------------------------------------------------------------

passed_334 = int(
    switching_audit_333["exact_switching_feasible"].sum()
)

assert passed_334 + len(failed_switching_334) == 749

print("\n" + "=" * 75)
print("STEP 3.3.4 — DIAGNOSIS COMPLETED")
print("=" * 75)

print(f"Switching-feasible demonstration allocations: {passed_334}")
print(f"Failed demonstration allocations: {len(failed_switching_334)}")
print(f"Total users accounted for: {passed_334 + len(failed_switching_334)}")

print("\nAlternative windows: NOT YET TESTED")
print("Alternative day allocations: NOT YET TESTED")
print("Original Stage 2 assignments: UNCHANGED")
print("Plan B: NOT ACTIVATED")
print("No files were modified or exported.")


STAGE 3 — STEP 3.3.4: SWITCHING FAILURE DIAGNOSIS

A. SWITCHING FAILURE TYPES


,Failure Type,Users
0,Insufficient possible switches,94
1,Too many unavoidable switches,71
2,Unreachable target within bounds,40



B. FAILURE TYPES BY SCENARIO DURATION


failure_type,Insufficient possible switches,Too many unavoidable switches,Unreachable target within bounds,Total
duration,,,,
5,9,3,4,16
10,18,12,14,44
20,26,15,10,51
30,27,9,12,48
40,8,15,0,23
50,6,17,0,23



C. SWITCHING GAP SEVERITY


,failure_type,Users,Median_Gap,Maximum_Gap
0,Insufficient possible switches,94,2.0,15
1,Too many unavoidable switches,71,2.0,13
2,Unreachable target within bounds,40,0.0,0



D. LARGEST SWITCHING GAPS


,user,scenario_id,duration,target_switches,minimum_possible_switches,maximum_possible_switches,failure_type,switching_gap
742,YNW2855,SCN_1098,50,22,0,7,Insufficient possible switches,15
303,HKH0204,SCN_0682,50,4,17,32,Too many unavoidable switches,13
34,AJS3021,SCN_1096,50,22,3,10,Insufficient possible switches,12
27,AGW1590,SCN_0211,40,5,15,27,Too many unavoidable switches,10
41,AMB3100,SCN_0170,30,15,1,5,Insufficient possible switches,10
108,BSM2844,SCN_0138,30,16,2,6,Insufficient possible switches,10
106,BRW3552,SCN_0200,30,14,2,6,Insufficient possible switches,8
610,SCE1877,SCN_0450,40,19,8,11,Insufficient possible switches,8
141,CHM1561,SCN_0082,20,10,0,3,Insufficient possible switches,7
586,RKH2877,SCN_0163,30,13,2,6,Insufficient possible switches,7



STEP 3.3.4 — DIAGNOSIS COMPLETED
Switching-feasible demonstration allocations: 544
Failed demonstration allocations: 205
Total users accounted for: 749

Alternative windows: NOT YET TESTED
Alternative day allocations: NOT YET TESTED
Original Stage 2 assignments: UNCHANGED
Plan B: NOT ACTIVATED
No files were modified or exported.



### Step 3.3.5 — Alternative Window & Day Allocation Search

**Objective:** Attempt to recover the 205 users whose initial demonstration schedules could not achieve their exact switching targets.

**Search Strategy:**
1. Examine all registered candidate windows for each failed user.
2. Try six deterministic active/off-day allocation strategies.
3. Preserve the exact single-, two-, and three-channel day requirements.
4. Enforce Plan A daily channel activity.
5. Evaluate the exact reachable primary-channel switching counts.
6. Record the first successful window and allocation strategy per recovered user.

**Validation:**
- No changes to scenario duration, stealth, or target switches.
- No violations of daily channel availability.
- No overlapping active/off-day assignments.
- Exact switching target must be achievable.

**Important:** This is a multi-strategy search, not an exhaustive search of every possible day allocation. Users without a solution are unresolved, not proven infeasible.

**Expected Outcome:** Identify additional users whose switching requirements can be satisfied without modifying their scenarios.

**Note:** Read-only. No final schedules, injections, or file exports.


In [20]:

# Stage 3 — Step 3.3.5: Alternative Window & Day Allocation Search
# Goal: Retry the 205 switching failures using all registered
# windows and six deterministic day-allocation strategies.
# Read-only: No assignments, source data, or files are modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.3.5: SWITCHING RECOVERY SEARCH")
print("=" * 75)

# ------------------------------------------------------------
# 1. Validate input datasets.
# ------------------------------------------------------------

assert len(failed_switching_334) == 205
assert failed_switching_334["user"].is_unique
assert candidate_windows_331["user"].nunique() == 749

failed_users_335 = set(failed_switching_334["user"])

rules_335 = cohort_325.set_index("user")

window_groups_335 = {
    user: group.sort_values("window_start")
    for user, group in candidate_windows_331.loc[
        candidate_windows_331["user"].isin(failed_users_335)
    ].groupby("user")
}

daily_groups_335 = {}

for user, group in audit_daily_325.loc[
    audit_daily_325["user"].isin(failed_users_335)
].groupby("user"):

    group = group.sort_values("day").reset_index(drop=True)

    masks = (
        group["external_email_count"].gt(0).to_numpy(dtype=np.int8)
        + 2 * group["usb_copy_count"].gt(0).to_numpy(dtype=np.int8)
        + 4 * group["www_upload_count"].gt(0).to_numpy(dtype=np.int8)
    )

    natural = (
        group["external_email_count"].gt(0).to_numpy(dtype=np.int8)
        + group["usb_copy_count"].gt(0).to_numpy(dtype=np.int8)
        + group["www_upload_count"].gt(0).to_numpy(dtype=np.int8)
    )

    daily_groups_335[user] = {
        "dates": group["day"].to_numpy(dtype="datetime64[ns]"),
        "masks": masks,
        "natural": natural
    }

# ------------------------------------------------------------
# 2. Alternative day-allocation strategies.
# Allocate 3-channel days first, then 2-channel, then 1.
# ------------------------------------------------------------

strategies_335 = [
    "scarce_early",
    "scarce_late",
    "flexible_early",
    "flexible_late",
    "chronological",
    "reverse_chronological"
]

def allocate_days_335(natural, requirements, strategy):

    planned = np.zeros(len(natural), dtype=np.int8)

    for channel_count in (3, 2, 1):

        required = int(requirements[channel_count])

        if required == 0:
            continue

        indices = np.flatnonzero(
            (planned == 0) & (natural >= channel_count)
        )

        if len(indices) < required:
            return None

        if strategy == "scarce_early":
            order = indices[
                np.lexsort((indices, natural[indices]))
            ]

        elif strategy == "scarce_late":
            order = indices[
                np.lexsort((-indices, natural[indices]))
            ]

        elif strategy == "flexible_early":
            order = indices[
                np.lexsort((indices, -natural[indices]))
            ]

        elif strategy == "flexible_late":
            order = indices[
                np.lexsort((-indices, -natural[indices]))
            ]

        elif strategy == "chronological":
            order = indices

        elif strategy == "reverse_chronological":
            order = indices[::-1]

        else:
            raise ValueError("Unknown allocation strategy.")

        planned[order[:required]] = channel_count

    if np.any(planned > natural):
        return None

    for channel_count in (1, 2, 3):
        if np.count_nonzero(planned == channel_count) != requirements[channel_count]:
            return None

    return planned

# ------------------------------------------------------------
# 3. Exact switching feasibility via dynamic programming.
# Off days are excluded from the primary-channel sequence.
# ------------------------------------------------------------

def exact_switch_feasible_335(planned, masks, target):

    states = {}

    for index in np.flatnonzero(planned > 0):

        mask = int(masks[index])

        allowed = [
            channel for channel in range(3)
            if mask & (1 << channel)
        ]

        if len(allowed) < int(planned[index]):
            return False

        if not states:
            states = {channel: 1 for channel in allowed}
            continue

        next_states = {}

        for current in allowed:

            reachable_mask = 0

            for previous, previous_mask in states.items():
                reachable_mask |= (
                    previous_mask
                    if previous == current
                    else previous_mask << 1
                )

            next_states[current] = reachable_mask

        states = next_states

    combined_mask = 0

    for reachable_mask in states.values():
        combined_mask |= reachable_mask

    return bool(combined_mask & (1 << int(target)))

# ------------------------------------------------------------
# 4. Search every registered window for each failed user.
# Stop at the first successful allocation for that user.
# ------------------------------------------------------------

recovered_records_335 = []

windows_tested_335 = 0
allocations_tested_335 = 0

for user in sorted(failed_users_335):

    rule = rules_335.loc[user]
    day_data = daily_groups_335[user]

    duration = int(rule["effective_duration"])
    target = int(rule["effective_switches"])

    requirements = {
        1: int(rule["effective_single"]),
        2: int(rule["effective_two"]),
        3: int(rule["effective_three"])
    }

    expected_off = int(rule["effective_off"])

    found = False

    for window in window_groups_335[user].itertuples(index=False):

        start = np.searchsorted(
            day_data["dates"],
            np.datetime64(window.window_start)
        )

        end = start + duration

        if end > len(day_data["dates"]):
            raise AssertionError(f"Invalid window for {user}")

        assert day_data["dates"][end - 1] == np.datetime64(
            window.window_end
        ), f"Window date mismatch for {user}"

        natural = day_data["natural"][start:end]
        masks = day_data["masks"][start:end]

        windows_tested_335 += 1

        for strategy in strategies_335:

            planned = allocate_days_335(
                natural, requirements, strategy
            )

            if planned is None:
                continue

            assert np.count_nonzero(planned == 0) == expected_off

            allocations_tested_335 += 1

            if exact_switch_feasible_335(
                planned, masks, target
            ):

                recovered_records_335.append({
                    "user": user,
                    "scenario_id": rule["effective_scenario_id"],
                    "duration": duration,
                    "target_switches": target,
                    "window_start": window.window_start,
                    "window_end": window.window_end,
                    "allocation_strategy": strategy
                })

                found = True
                break

        if found:
            break

# ------------------------------------------------------------
# 5. Reconcile recovered and unresolved users.
# ------------------------------------------------------------

recovered_switching_335 = pd.DataFrame(
    recovered_records_335,
    columns=[
        "user", "scenario_id", "duration",
        "target_switches", "window_start", "window_end",
        "allocation_strategy"
    ]
)

recovered_users_335 = set(recovered_switching_335["user"])

unresolved_users_335 = failed_users_335 - recovered_users_335

assert recovered_switching_335["user"].is_unique
assert recovered_users_335.isdisjoint(unresolved_users_335)
assert len(recovered_users_335) + len(unresolved_users_335) == 205

original_pass_335 = int(
    switching_audit_333["exact_switching_feasible"].sum()
)

print("\nA. SWITCHING RECOVERY RESULTS")

print(f"Initially passing users: {original_pass_335}")
print(f"Initially failing users: {len(failed_users_335)}")
print(f"Recovered by alternative search: {len(recovered_users_335)}")
print(f"Still unresolved: {len(unresolved_users_335)}")
print(f"Total users with a feasible demonstration: "
      f"{original_pass_335 + len(recovered_users_335)}")

print("\nB. SEARCH COVERAGE")

print(f"Candidate windows examined: {windows_tested_335:,}")
print(f"Valid day allocations evaluated: {allocations_tested_335:,}")
print(f"Allocation strategies available: {len(strategies_335)}")

# ------------------------------------------------------------
# 6. Report recovery by original failure type.
# ------------------------------------------------------------

diagnosis_summary_335 = failed_switching_334[
    ["user", "failure_type"]
].copy()

diagnosis_summary_335["recovered"] = (
    diagnosis_summary_335["user"].isin(recovered_users_335)
)

print("\nC. RECOVERY BY ORIGINAL FAILURE TYPE")

recovery_by_type_335 = (
    diagnosis_summary_335
    .groupby("failure_type")
    .agg(
        Initially_Failed=("user", "count"),
        Recovered=("recovered", "sum")
    )
    .reset_index()
)

recovery_by_type_335["Unresolved"] = (
    recovery_by_type_335["Initially_Failed"]
    - recovery_by_type_335["Recovered"]
)

display(recovery_by_type_335)

# ------------------------------------------------------------
# 7. Report recovery by duration.
# ------------------------------------------------------------

print("\nD. RECOVERY BY DURATION")

recovery_by_duration_335 = failed_switching_334[
    ["user", "duration"]
].copy()

recovery_by_duration_335["recovered"] = (
    recovery_by_duration_335["user"].isin(recovered_users_335)
)

display(
    recovery_by_duration_335
    .groupby("duration")
    .agg(
        Initially_Failed=("user", "count"),
        Recovered=("recovered", "sum")
    )
    .assign(
        Unresolved=lambda df:
        df["Initially_Failed"] - df["Recovered"]
    )
    .reset_index()
)

# ------------------------------------------------------------
# 8. Preview successful alternative-window choices.
# ------------------------------------------------------------

print("\nE. RECOVERED USER EXAMPLES")

if recovered_switching_335.empty:
    print("No additional switching-feasible allocations were found.")
else:
    display(recovered_switching_335.head(20))

print("\n" + "=" * 75)
print("STEP 3.3.5 — SEARCH COMPLETED")
print("=" * 75)

print("Recovered solutions: PROVISIONAL")
print("Unresolved cases: NOT PROVEN INFEASIBLE")
print("Exact primary-channel sequences: NOT YET CONSTRUCTED")
print("Original Stage 2 assignments: UNCHANGED")
print("Plan B: NOT ACTIVATED")
print("No files were modified or exported.")


STAGE 3 — STEP 3.3.5: SWITCHING RECOVERY SEARCH

A. SWITCHING RECOVERY RESULTS
Initially passing users: 544
Initially failing users: 205
Recovered by alternative search: 116
Still unresolved: 89
Total users with a feasible demonstration: 660

B. SEARCH COVERAGE
Candidate windows examined: 2,050
Valid day allocations evaluated: 11,812
Allocation strategies available: 6

C. RECOVERY BY ORIGINAL FAILURE TYPE


,failure_type,Initially_Failed,Recovered,Unresolved
0,Insufficient possible switches,94,42,52
1,Too many unavoidable switches,71,38,33
2,Unreachable target within bounds,40,36,4



D. RECOVERY BY DURATION


,duration,Initially_Failed,Recovered,Unresolved
0,5,16,9,7
1,10,44,31,13
2,20,51,29,22
3,30,48,22,26
4,40,23,13,10
5,50,23,12,11



E. RECOVERED USER EXAMPLES


,user,scenario_id,duration,target_switches,window_start,window_end,allocation_strategy
0,ADG1737,SCN_0206,40,4,2011-03-25,2011-05-20,flexible_late
1,AEB3249,SCN_0207,40,4,2011-02-01,2011-03-28,scarce_early
2,AFB3228,SCN_0020,10,4,2011-02-11,2011-02-24,scarce_early
3,AHB0648,SCN_0021,10,4,2011-01-10,2011-01-21,scarce_early
4,AKH0454,SCN_0997,50,9,2011-01-03,2011-03-11,flexible_early
5,AME1490,SCN_0103,30,3,2011-01-03,2011-02-11,flexible_late
6,AOM1457,SCN_0503,50,8,2011-01-03,2011-03-11,scarce_early
7,APB1122,SCN_0396,40,6,2011-03-11,2011-05-06,scarce_early
8,AVH0566,SCN_0027,10,1,2011-01-04,2011-01-17,scarce_early
9,AXW3243,SCN_0028,10,3,2011-02-04,2011-02-17,scarce_early



STEP 3.3.5 — SEARCH COMPLETED
Recovered solutions: PROVISIONAL
Unresolved cases: NOT PROVEN INFEASIBLE
Exact primary-channel sequences: NOT YET CONSTRUCTED
Original Stage 2 assignments: UNCHANGED
Plan B: NOT ACTIVATED
No files were modified or exported.



### Step 3.3.6 — Exact Switching Sequence Construction & Validation

**Objective:** Construct and independently validate explicit primary-channel sequences for the 660 provisionally switching-feasible users.

**Validation Tasks:**
1. Reconstruct the 544 initially feasible day allocations.
2. Reconstruct the 116 allocations recovered through alternative-window search.
3. Assign actual Email, USB, and HTTP attack channels based on existing benign daily activity.
4. Construct a primary-channel sequence achieving the exact switching target.
5. Verify single-, two-, three-channel, and off-day requirements.
6. Independently count the realized channel switches.

**Expected Outcome:** A reproducible provisional daily attack-channel schedule for all 660 users.

**Limitations:**
- Dates and allocations remain provisional.
- The 89 unresolved switching cases are not modified.
- The 246 previously deferred users remain unchanged.
- No data injection, final approval, or file export.

**Note:** Read-only analysis. Original CERT and Stage 2 outputs remain unchanged.


In [21]:

# Stage 3 — Step 3.3.6: Exact Switching Sequence Construction
# Goal: Reconstruct all 660 feasible day allocations, assign actual
# attack channels, and independently verify exact primary switches.
# Read-only: No source datasets, assignments, or files are modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.3.6: SWITCHING SEQUENCE CONSTRUCTION")
print("=" * 75)

# ------------------------------------------------------------
# 1. Identify the 544 initial successes and 116 recoveries.
# ------------------------------------------------------------

initial_users_336 = set(
    switching_audit_333.loc[
        switching_audit_333["exact_switching_feasible"], "user"
    ]
)

recovered_users_336 = set(recovered_switching_335["user"])

assert len(initial_users_336) == 544
assert len(recovered_users_336) == 116
assert initial_users_336.isdisjoint(recovered_users_336)

feasible_users_336 = initial_users_336 | recovered_users_336

assert len(feasible_users_336) == 660

rules_336 = cohort_325.set_index("user")

# ------------------------------------------------------------
# 2. Preserve the original successful demonstration allocations.
# ------------------------------------------------------------

initial_days_336 = provisional_days_332.loc[
    provisional_days_332["user"].isin(initial_users_336),
    [
        "user", "scenario_id", "window_start",
        "window_end", "day", "planned_channels"
    ]
].copy()

initial_days_336["solution_source"] = "Initial demonstration"

# ------------------------------------------------------------
# 3. Reconstruct the 116 recovered allocations from
# their recorded windows and allocation strategies.
# ------------------------------------------------------------

recovered_parts_336 = []

for solution in recovered_switching_335.itertuples(index=False):

    user = solution.user
    rule = rules_336.loc[user]

    days = audit_daily_325.loc[
        (audit_daily_325["user"] == user)
        & audit_daily_325["day"].between(
            solution.window_start, solution.window_end
        )
    ].sort_values("day").copy()

    assert len(days) == int(rule["effective_duration"]), (
        f"Invalid recovered window: {user}"
    )

    natural_counts = days[
        [
            "external_email_count",
            "usb_copy_count",
            "www_upload_count"
        ]
    ].gt(0).sum(axis=1).to_numpy()

    requirements = {
        1: int(rule["effective_single"]),
        2: int(rule["effective_two"]),
        3: int(rule["effective_three"])
    }

    planned = allocate_days_335(
        natural_counts,
        requirements,
        solution.allocation_strategy
    )

    assert planned is not None, (
        f"Could not reconstruct allocation: {user}"
    )

    recovered_parts_336.append(pd.DataFrame({
        "user": user,
        "scenario_id": solution.scenario_id,
        "window_start": solution.window_start,
        "window_end": solution.window_end,
        "day": days["day"].to_numpy(),
        "planned_channels": planned,
        "solution_source": "Recovered allocation"
    }))

recovered_days_336 = pd.concat(
    recovered_parts_336, ignore_index=True
)

# ------------------------------------------------------------
# 4. Combine both groups and attach actual natural channels.
# ------------------------------------------------------------

combined_days_336 = pd.concat(
    [initial_days_336, recovered_days_336],
    ignore_index=True
)

assert combined_days_336["user"].nunique() == 660

assert not combined_days_336.duplicated(
    ["user", "day"]
).any()

natural_columns_336 = [
    "external_email_count",
    "usb_copy_count",
    "www_upload_count"
]

combined_days_336 = combined_days_336.merge(
    audit_daily_325[
        ["user", "day"] + natural_columns_336
    ],
    on=["user", "day"],
    how="left",
    validate="one_to_one"
)

assert combined_days_336[
    natural_columns_336
].notna().all().all()

# ------------------------------------------------------------
# 5. Construct a witness primary-channel sequence.
# Dynamic programming retains a valid sequence for each
# reachable (last_channel, switch_count) state.
# ------------------------------------------------------------

channel_names_336 = [
    ("Email", "external_email_count"),
    ("USB", "usb_copy_count"),
    ("HTTP", "www_upload_count")
]

def available_channels_336(record):
    return [
        name for name, col in channel_names_336
        if getattr(record, col) > 0
    ]

def find_primary_sequence_336(active_records, target):

    states = {(None, 0): ()}

    for record in active_records:

        available = available_channels_336(record)
        planned = int(record.planned_channels)

        if len(available) < planned:
            return None

        next_states = {}

        for (previous, switches), sequence in states.items():
            for current in available:

                new_switches = switches + int(
                    previous is not None
                    and previous != current
                )

                if new_switches > target:
                    continue

                key = (current, new_switches)

                if key not in next_states:
                    next_states[key] = sequence + (current,)

        states = next_states

        if not states:
            return None

    for (last_channel, switches), sequence in states.items():
        if switches == target:
            return sequence

    return None

# ------------------------------------------------------------
# 6. Construct daily channel assignments and verify switches.
# ------------------------------------------------------------

witness_records_336 = []
user_results_336 = []

for user, group in combined_days_336.groupby("user"):

    group = group.sort_values("day")
    rule = rules_336.loc[user]

    expected = {
        0: int(rule["effective_off"]),
        1: int(rule["effective_single"]),
        2: int(rule["effective_two"]),
        3: int(rule["effective_three"])
    }

    assert len(group) == int(rule["effective_duration"])

    for day_type, count in expected.items():
        assert int(
            (group["planned_channels"] == day_type).sum()
        ) == count, f"Day-count mismatch: {user}"

    active = group.loc[
        group["planned_channels"] > 0
    ]

    target = int(rule["effective_switches"])

    sequence = find_primary_sequence_336(
        list(active.itertuples(index=False)),
        target
    )

    assert sequence is not None, (
        f"Exact switching sequence not found: {user}"
    )

    active_position = 0
    previous_primary = None
    realized_switches = 0

    for record in group.itertuples(index=False):

        planned_count = int(record.planned_channels)
        available = available_channels_336(record)

        if planned_count == 0:
            primary = None
            selected = []

        else:
            primary = sequence[active_position]
            active_position += 1

            selected = [primary] + [
                ch for ch in available
                if ch != primary
            ][:planned_count - 1]

            assert len(selected) == planned_count
            assert set(selected).issubset(set(available))

            if (
                previous_primary is not None
                and previous_primary != primary
            ):
                realized_switches += 1

            previous_primary = primary

        witness_records_336.append({
            "user": user,
            "scenario_id": record.scenario_id,
            "day": record.day,
            "window_start": record.window_start,
            "window_end": record.window_end,
            "planned_channels": planned_count,
            "primary_channel": primary,
            "attack_channels": "|".join(selected),
            "solution_source": record.solution_source
        })

    assert realized_switches == target, (
        f"Switching mismatch for {user}"
    )

    assert active_position == len(active)

    user_results_336.append({
        "user": user,
        "scenario_id": rule["effective_scenario_id"],
        "target_switches": target,
        "realized_switches": realized_switches,
        "exact_match": realized_switches == target
    })

# ------------------------------------------------------------
# 7. Build and validate the provisional schedule witness.
# ------------------------------------------------------------

switching_witness_days_336 = pd.DataFrame(
    witness_records_336
)

switching_witness_users_336 = pd.DataFrame(
    user_results_336
)

assert switching_witness_users_336["user"].nunique() == 660
assert switching_witness_users_336["exact_match"].all()

assert not switching_witness_days_336.duplicated(
    ["user", "day"]
).any()

# ------------------------------------------------------------
# 8. Display results.
# ------------------------------------------------------------

print("\nA. EXACT SWITCHING WITNESS VALIDATION")

print(f"Initial successful users: {len(initial_users_336)}")
print(f"Recovered users: {len(recovered_users_336)}")
print(f"Users with constructed sequences: "
      f"{len(switching_witness_users_336)}")
print(f"Exact switching matches: "
      f"{int(switching_witness_users_336['exact_match'].sum())}")
print(f"Switching mismatches: "
      f"{int((~switching_witness_users_336['exact_match']).sum())}")

print("\nB. PROVISIONAL DAILY SCHEDULE TOTALS")

display(
    switching_witness_days_336[
        "planned_channels"
    ].value_counts()
    .reindex([0, 1, 2, 3], fill_value=0)
    .rename_axis("Planned Channel Count")
    .reset_index(name="Days")
)

print("\nC. THREE-CHANNEL SCENARIO WITNESS EXAMPLE")

three_channel_users_336 = cohort_325.loc[
    cohort_325["effective_three"] > 0,
    "user"
]

example_users_336 = [
    user for user in three_channel_users_336
    if user in feasible_users_336
]

if example_users_336:
    example_user = example_users_336[0]

    print(f"User: {example_user}")

    display(
        switching_witness_days_336.loc[
            switching_witness_days_336["user"] == example_user,
            [
                "day", "planned_channels",
                "primary_channel", "attack_channels"
            ]
        ].head(20)
    )

print("\nD. POPULATION RECONCILIATION")

print(f"Users with constructed switching sequences: "
      f"{len(feasible_users_336)}")
print(f"Unresolved switching cases: {len(unresolved_users_335)}")
print(f"Previously deferred users: {len(deferred_325)}")

assert (
    len(feasible_users_336)
    + len(unresolved_users_335)
    + len(deferred_325)
    == 995
)

print("\n" + "=" * 75)
print("STEP 3.3.6 — COMPLETED")
print("=" * 75)

print("Primary-channel sequences: CONSTRUCTED PROVISIONALLY")
print("Exact switching counts: VERIFIED")
print("Final attack schedules: NOT APPROVED")
print("Synthetic injection: NOT STARTED")
print("Plan B: NOT ACTIVATED")
print("No source files were modified or exported.")


STAGE 3 — STEP 3.3.6: SWITCHING SEQUENCE CONSTRUCTION

A. EXACT SWITCHING WITNESS VALIDATION
Initial successful users: 544
Recovered users: 116
Users with constructed sequences: 660
Exact switching matches: 660
Switching mismatches: 0

B. PROVISIONAL DAILY SCHEDULE TOTALS


,Planned Channel Count,Days
0,0,2319
1,1,10858
2,2,1307
3,3,196



C. THREE-CHANNEL SCENARIO WITNESS EXAMPLE
User: ABD3426


,day,planned_channels,primary_channel,attack_channels
35,2011-03-31,2,Email,Email|USB
36,2011-04-01,3,Email,Email|USB|HTTP
37,2011-04-04,2,Email,Email|USB
38,2011-04-05,2,Email,Email|USB
39,2011-04-06,1,USB,USB
40,2011-04-07,1,Email,Email
41,2011-04-08,0,NaN,
42,2011-04-11,1,Email,Email
43,2011-04-12,1,USB,USB
44,2011-04-13,1,Email,Email



D. POPULATION RECONCILIATION
Users with constructed switching sequences: 660
Unresolved switching cases: 89
Previously deferred users: 246

STEP 3.3.6 — COMPLETED
Primary-channel sequences: CONSTRUCTED PROVISIONALLY
Exact switching counts: VERIFIED
Final attack schedules: NOT APPROVED
Synthetic injection: NOT STARTED
Plan B: NOT ACTIVATED
No source files were modified or exported.



### Step 3.3.7 — Exact Scheduling Feasibility for Unresolved Users

**Objective:** Search for exact Plan A scheduling solutions for the 89 users unresolved after the six-strategy recovery search.

**Validation Requirements:**
1. Preserve the assigned scenario duration and stealth level.
2. Preserve exact active, off, single-, two-, and three-channel day counts.
3. Use only naturally active channels on each attack day.
4. Achieve the exact target number of primary-channel switches.
5. Examine alternative eligible-workday windows.
6. Independently validate any recovered schedule.

**Search Method:** Dynamic Programming with state caching and feasibility pruning.

**Result Classification:**
- FOUND: A valid schedule was constructed.
- INFEASIBLE: Every registered candidate window was fully searched without a solution.
- INCOMPLETE: The computational search limit was reached, so infeasibility is not established.

**Expected Outcome:** Recover additional users or distinguish unresolved computational cases from proven scheduling constraints.

**Note:** Read-only. No injection, reassignment, or file export.


In [22]:

# Stage 3 — Step 3.3.7: Exact Scheduling Feasibility
# Goal: Search the remaining 89 cases using dynamic programming
# over daily attack types and primary-channel switching sequences.
# Read-only: No original data, assignments, or files are modified.

import numpy as np
import pandas as pd
from functools import lru_cache
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.3.7: EXACT SCHEDULING FEASIBILITY")
print("=" * 75)

# ------------------------------------------------------------
# 1. Verify unresolved cases and prepare source activity.
# ------------------------------------------------------------

users_337 = sorted(unresolved_users_335)

assert len(users_337) == 89
assert set(users_337).issubset(set(cohort_325["user"]))

rules_337 = cohort_325.set_index("user")

daily_337 = {}
windows_337 = {}

for user in users_337:

    days = audit_daily_325.loc[
        audit_daily_325["user"] == user
    ].sort_values("day").reset_index(drop=True)

    masks = (
        days["external_email_count"].gt(0).to_numpy(dtype=np.int8)
        + 2 * days["usb_copy_count"].gt(0).to_numpy(dtype=np.int8)
        + 4 * days["www_upload_count"].gt(0).to_numpy(dtype=np.int8)
    )

    daily_337[user] = {
        "dates": days["day"].to_numpy(dtype="datetime64[ns]"),
        "masks": masks
    }

    windows_337[user] = (
        candidate_windows_331.loc[
            candidate_windows_331["user"] == user
        ]
        .sort_values("window_start")
        .copy()
    )

    assert len(windows_337[user]) > 0

# ------------------------------------------------------------
# 2. Define a controlled search budget.
# A budget interruption is NOT an infeasibility result.
# ------------------------------------------------------------

MAX_STATES_PER_USER_337 = 150_000

class SearchBudgetExceeded337(Exception):
    pass

CHANNEL_NAMES_337 = ("Email", "USB", "HTTP")

# ------------------------------------------------------------
# 3. Exact solver for one candidate window.
# State tracks:
# - current day
# - remaining 1/2/3-channel days
# - previous primary channel
# - remaining required switches
# Off days are inferred from remaining window length.
# ------------------------------------------------------------

def solve_window_337(masks, requirements, target, budget):

    masks = tuple(int(x) for x in masks)
    n = len(masks)

    suffix = [[0] * (n + 1) for _ in range(3)]

    for i in range(n - 1, -1, -1):
        count = masks[i].bit_count()

        for k in range(3):
            suffix[k][i] = (
                suffix[k][i + 1] + int(count >= k + 1)
            )

    choices = {}

    @lru_cache(maxsize=None)
    def solve(i, r1, r2, r3, last, switches_left):

        budget["used"] += 1

        if budget["used"] > budget["limit"]:
            raise SearchBudgetExceeded337()

        active_left = r1 + r2 + r3

        if min(r1, r2, r3, switches_left) < 0:
            return False

        # Necessary remaining daily-capacity conditions.
        if (
            active_left > n - i
            or active_left > suffix[0][i]
            or r2 + r3 > suffix[1][i]
            or r3 > suffix[2][i]
        ):
            return False

        # Upper bound on future switches.
        max_future_switches = (
            active_left
            if last != -1
            else max(0, active_left - 1)
        )

        if switches_left > max_future_switches:
            return False

        if i == n:
            return active_left == 0 and switches_left == 0

        available = [
            c for c in range(3)
            if masks[i] & (1 << c)
        ]

        for planned_count in (3, 2, 1, 0):

            if planned_count > len(available):
                continue

            if planned_count == 3 and r3 == 0:
                continue
            if planned_count == 2 and r2 == 0:
                continue
            if planned_count == 1 and r1 == 0:
                continue

            if planned_count == 0:

                if solve(
                    i + 1, r1, r2, r3,
                    last, switches_left
                ):
                    choices[
                        (i, r1, r2, r3, last, switches_left)
                    ] = (0, -1)

                    return True

                continue

            for primary in available:

                switch = int(
                    last != -1 and last != primary
                )

                if solve(
                    i + 1,
                    r1 - int(planned_count == 1),
                    r2 - int(planned_count == 2),
                    r3 - int(planned_count == 3),
                    primary,
                    switches_left - switch
                ):

                    choices[
                        (i, r1, r2, r3, last, switches_left)
                    ] = (planned_count, primary)

                    return True

        return False

    r1, r2, r3 = requirements

    feasible = solve(
        0, r1, r2, r3, -1, int(target)
    )

    if not feasible:
        return None

    # Reconstruct an explicit valid schedule.
    schedule = []
    i = 0
    last = -1
    switches_left = int(target)

    while i < n:

        state = (i, r1, r2, r3, last, switches_left)

        planned_count, primary = choices[state]

        schedule.append((planned_count, primary))

        if planned_count > 0:

            switches_left -= int(
                last != -1 and last != primary
            )

            last = primary

            r1 -= int(planned_count == 1)
            r2 -= int(planned_count == 2)
            r3 -= int(planned_count == 3)

        i += 1

    return schedule

# ------------------------------------------------------------
# 4. Search all registered windows for each unresolved user.
# ------------------------------------------------------------

results_337 = []
witness_records_337 = []

for position, user in enumerate(users_337, start=1):

    rule = rules_337.loc[user]
    data = daily_337[user]

    duration = int(rule["effective_duration"])
    target = int(rule["effective_switches"])

    requirements = (
        int(rule["effective_single"]),
        int(rule["effective_two"]),
        int(rule["effective_three"])
    )

    budget = {
        "used": 0,
        "limit": MAX_STATES_PER_USER_337
    }

    status = "INFEASIBLE"
    examined = 0

    for window in windows_337[user].itertuples(index=False):

        start = np.searchsorted(
            data["dates"],
            np.datetime64(window.window_start)
        )

        end = start + duration

        assert end <= len(data["dates"])

        assert (
            data["dates"][end - 1]
            == np.datetime64(window.window_end)
        ), f"Window mismatch: {user}"

        masks = data["masks"][start:end]

        examined += 1

        try:
            solution = solve_window_337(
                masks, requirements, target, budget
            )

        except SearchBudgetExceeded337:
            status = "INCOMPLETE"
            break

        if solution is None:
            continue

        # Independently validate the constructed schedule.
        realized = 0
        previous = -1
        day_types = []

        for idx, (planned, primary) in enumerate(solution):

            mask = int(masks[idx])

            assert planned <= mask.bit_count()

            if planned > 0:
                assert mask & (1 << primary)

                if previous != -1 and previous != primary:
                    realized += 1

                previous = primary

            day_types.append(planned)

            witness_records_337.append({
                "user": user,
                "scenario_id": rule["effective_scenario_id"],
                "day": pd.Timestamp(data["dates"][start + idx]),
                "planned_channels": planned,
                "primary_channel": (
                    CHANNEL_NAMES_337[primary]
                    if primary >= 0 else None
                ),
                "window_start": window.window_start,
                "window_end": window.window_end
            })

        assert realized == target
        assert day_types.count(1) == requirements[0]
        assert day_types.count(2) == requirements[1]
        assert day_types.count(3) == requirements[2]
        assert day_types.count(0) == int(rule["effective_off"])

        status = "FOUND"
        break

    results_337.append({
        "user": user,
        "scenario_id": rule["effective_scenario_id"],
        "duration": duration,
        "target_switches": target,
        "status": status,
        "windows_examined": examined,
        "total_candidate_windows": len(windows_337[user]),
        "dp_states_evaluated": budget["used"]
    })

    if position % 10 == 0 or position == len(users_337):
        print(f"Processed {position}/{len(users_337)} users")

# ------------------------------------------------------------
# 5. Summarize exact search results.
# ------------------------------------------------------------

exact_search_audit_337 = pd.DataFrame(results_337)
exact_recovery_days_337 = pd.DataFrame(witness_records_337)

status_order_337 = ["FOUND", "INFEASIBLE", "INCOMPLETE"]

status_summary_337 = (
    exact_search_audit_337["status"]
    .value_counts()
    .reindex(status_order_337, fill_value=0)
    .rename_axis("Status")
    .reset_index(name="Users")
)

print("\nA. EXACT SEARCH RESULTS")
display(status_summary_337)

print("\nB. SEARCH COVERAGE")

print(f"Users evaluated: {len(exact_search_audit_337)}")
print(
    "Total windows examined:",
    f"{exact_search_audit_337['windows_examined'].sum():,}"
)
print(
    "Total DP states evaluated:",
    f"{exact_search_audit_337['dp_states_evaluated'].sum():,}"
)

# ------------------------------------------------------------
# 6. Show recovered users and unresolved status.
# ------------------------------------------------------------

print("\nC. RECOVERED USER SAMPLE")

found_337 = exact_search_audit_337.loc[
    exact_search_audit_337["status"] == "FOUND"
]

if found_337.empty:
    print("No additional schedules were found.")
else:
    display(found_337.head(20))

print("\nD. UNRESOLVED OR INFEASIBLE CASES")

remaining_337 = exact_search_audit_337.loc[
    exact_search_audit_337["status"] != "FOUND"
]

if remaining_337.empty:
    print("All 89 users received exact scheduling solutions.")
else:
    display(
        remaining_337.groupby(
            ["duration", "status"]
        ).size().reset_index(name="Users")
    )

# ------------------------------------------------------------
# 7. Population reconciliation.
# ------------------------------------------------------------

newly_found_337 = int(
    (exact_search_audit_337["status"] == "FOUND").sum()
)

print("\n" + "=" * 75)
print("STEP 3.3.7 — SEARCH SUMMARY")
print("=" * 75)

print(f"Previously constructed schedules: 660")
print(f"Additional exact solutions: {newly_found_337}")
print(f"Total provisional solutions: {660 + newly_found_337}")
print(f"Still without a constructed solution: {89 - newly_found_337}")
print(f"Previously deferred users: 246")

assert len(exact_search_audit_337) == 89

print("\nOriginal Stage 2 files: UNCHANGED")
print("Plan B: NOT ACTIVATED")
print("Synthetic injection: NOT STARTED")
print("No files were modified or exported.")


STAGE 3 — STEP 3.3.7: EXACT SCHEDULING FEASIBILITY
Processed 10/89 users
Processed 20/89 users
Processed 30/89 users
Processed 40/89 users
Processed 50/89 users
Processed 60/89 users
Processed 70/89 users
Processed 80/89 users
Processed 89/89 users

A. EXACT SEARCH RESULTS


,Status,Users
0,FOUND,10
1,INFEASIBLE,78
2,INCOMPLETE,1



B. SEARCH COVERAGE
Users evaluated: 89
Total windows examined: 1,065
Total DP states evaluated: 783,782

C. RECOVERED USER SAMPLE


,user,scenario_id,duration,target_switches,status,windows_examined,total_candidate_windows,dp_states_evaluated
32,DNB1999,SCN_0180,30,5,FOUND,5,13,545
44,HQF1912,SCN_0693,50,5,FOUND,4,9,12833
49,KAA2150,SCN_0034,20,2,FOUND,7,11,360
50,KBG2061,SCN_0037,20,3,FOUND,10,18,1231
60,LNM1879,SCN_0204,40,4,FOUND,31,63,51001
65,NIB1197,SCN_0105,30,3,FOUND,6,22,1493
69,RFB2959,SCN_0080,20,9,FOUND,27,49,4428
75,SLC1588,SCN_0167,30,14,FOUND,10,43,3922
81,TNR0319,SCN_0385,40,4,FOUND,1,66,43
85,VAB1526,SCN_0952,50,6,FOUND,6,15,7156



D. UNRESOLVED OR INFEASIBLE CASES


,duration,status,Users
0,5,INFEASIBLE,7
1,10,INFEASIBLE,13
2,20,INFEASIBLE,19
3,30,INFEASIBLE,23
4,40,INFEASIBLE,8
5,50,INCOMPLETE,1
6,50,INFEASIBLE,8



STEP 3.3.7 — SEARCH SUMMARY
Previously constructed schedules: 660
Additional exact solutions: 10
Total provisional solutions: 670
Still without a constructed solution: 79
Previously deferred users: 246

Original Stage 2 files: UNCHANGED
Plan B: NOT ACTIVATED
Synthetic injection: NOT STARTED
No files were modified or exported.



### Step 3.3.9 — Schedule Consolidation & Final Structural Validation

**Objective:** Consolidate all 670 successful provisional schedules and independently validate their compliance with approved scenario requirements.

**Inputs:**
- 660 schedules from Step 3.3.6.
- 10 additional schedules from Step 3.3.7.
- Approved scenario definitions and original User-Day activity.

**Validation Tasks:**
1. Combine both scheduling outputs.
2. Complete channel combinations for the 10 DP-recovered schedules.
3. Verify unique user-day records and scenario duration.
4. Validate exact active, off, single-, two-, and three-channel day counts.
5. Verify all selected channels had pre-existing benign activity.
6. Independently recount primary-channel switches.
7. Reconcile the complete population of 995 users.

**Expected Outcome:** One validated provisional daily attack schedule for all 670 successful users.

**Deferred Cases:** 78 INFEASIBLE, 1 INCOMPLETE, and 246 previously deferred users remain unchanged.

**Note:** Read-only validation. No final approval, injection, or file export.


In [23]:

# Stage 3 — Step 3.3.9: Schedule Consolidation & Structural Validation
# Goal: Consolidate 660 existing schedules with 10 DP recoveries,
# normalize date comparisons, and independently validate every
# daily attack schedule against its effective scenario and Plan A.
# Read-only: No source files or original assignments are modified.

import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.3.9: SCHEDULE CONSOLIDATION")
print("=" * 75)

# ------------------------------------------------------------
# 1. Validate the two successful scheduling sources.
# ------------------------------------------------------------

initial_339 = switching_witness_days_336.copy()

dp_found_users_339 = set(
    exact_search_audit_337.loc[
        exact_search_audit_337["status"] == "FOUND",
        "user"
    ]
)

dp_339 = exact_recovery_days_337.loc[
    exact_recovery_days_337["user"].isin(dp_found_users_339)
].copy()

assert initial_339["user"].nunique() == 660
assert dp_339["user"].nunique() == 10

assert set(initial_339["user"]).isdisjoint(
    set(dp_339["user"])
)

dp_339["attack_channels"] = None
dp_339["solution_source"] = "Exact DP recovery"

common_columns_339 = [
    "user",
    "scenario_id",
    "day",
    "window_start",
    "window_end",
    "planned_channels",
    "primary_channel",
    "attack_channels",
    "solution_source"
]

schedule_339 = pd.concat(
    [
        initial_339[common_columns_339],
        dp_339[common_columns_339]
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# 2. Normalize timestamps in local working copies.
# This avoids false mismatches between datetime units.
# ------------------------------------------------------------

for column in ["day", "window_start", "window_end"]:
    schedule_339[column] = (
        pd.to_datetime(schedule_339[column], errors="raise")
        .dt.normalize()
        .astype("datetime64[ns]")
    )

natural_columns_339 = [
    "external_email_count",
    "usb_copy_count",
    "www_upload_count"
]

natural_days_339 = audit_daily_325[
    ["user", "day"] + natural_columns_339
].copy()

natural_days_339["day"] = (
    pd.to_datetime(natural_days_339["day"], errors="raise")
    .dt.normalize()
    .astype("datetime64[ns]")
)

assert not natural_days_339.duplicated(
    ["user", "day"]
).any(), "Duplicate natural User-Day keys."

assert not schedule_339.duplicated(
    ["user", "day"]
).any(), "Duplicate scheduled User-Day keys."

assert schedule_339["user"].nunique() == 670

# ------------------------------------------------------------
# 3. Attach original natural channel activity.
# ------------------------------------------------------------

schedule_339 = schedule_339.merge(
    natural_days_339,
    on=["user", "day"],
    how="left",
    validate="one_to_one"
)

assert schedule_339[natural_columns_339].notna().all().all(), (
    "Missing original natural channel activity."
)

channel_mapping_339 = {
    "Email": "external_email_count",
    "USB": "usb_copy_count",
    "HTTP": "www_upload_count"
}

# ------------------------------------------------------------
# 4. Complete attack-channel combinations for DP recoveries.
# Preserve their primary-channel sequences.
# ------------------------------------------------------------

def complete_channels_339(row):

    planned = int(row["planned_channels"])

    if planned == 0:
        return ""

    primary = row["primary_channel"]

    assert isinstance(primary, str)
    assert primary in channel_mapping_339

    if row["solution_source"] != "Exact DP recovery":
        return row["attack_channels"]

    available = [
        name
        for name, column in channel_mapping_339.items()
        if row[column] > 0
    ]

    assert primary in available
    assert len(available) >= planned

    selected = [primary] + [
        channel
        for channel in available
        if channel != primary
    ][:planned - 1]

    return "|".join(selected)


schedule_339["attack_channels"] = schedule_339.apply(
    complete_channels_339,
    axis=1
)

# ------------------------------------------------------------
# 5. Prepare scenario lookup.
# ------------------------------------------------------------

rules_339 = cohort_325.set_index("user")

assert rules_339.index.is_unique

validation_records_339 = []

# ------------------------------------------------------------
# 6. Independently validate all 670 schedules.
# ------------------------------------------------------------

for user, group in schedule_339.groupby("user"):

    group = group.sort_values("day").reset_index(drop=True)

    rule = rules_339.loc[user]

    duration = int(rule["effective_duration"])

    expected_counts = {
        0: int(rule["effective_off"]),
        1: int(rule["effective_single"]),
        2: int(rule["effective_two"]),
        3: int(rule["effective_three"])
    }

    target_switches = int(rule["effective_switches"])

    # --------------------------------------------------------
    # 6.1 Validate scenario identity and duration.
    # --------------------------------------------------------

    assert len(group) == duration, (
        f"Duration mismatch for user {user}"
    )

    assert group["scenario_id"].eq(
        rule["effective_scenario_id"]
    ).all(), (
        f"Scenario mismatch for user {user}"
    )

    assert group["window_start"].nunique() == 1, (
        f"Multiple start dates for user {user}"
    )

    assert group["window_end"].nunique() == 1, (
        f"Multiple end dates for user {user}"
    )

    start_date = group["window_start"].iloc[0]
    end_date = group["window_end"].iloc[0]

    assert group["day"].between(
        start_date, end_date
    ).all(), (
        f"Scheduled day outside window for user {user}"
    )

    # --------------------------------------------------------
    # 6.2 Validate exact eligible-workday sequence.
    # Compare normalized calendar dates, not pandas Series
    # metadata or datetime storage precision.
    # --------------------------------------------------------

    expected_days = (
        natural_days_339.loc[
            (natural_days_339["user"] == user)
            & natural_days_339["day"].between(
                start_date, end_date
            ),
            "day"
        ]
        .sort_values()
        .reset_index(drop=True)
    )

    actual_days = (
        group["day"]
        .sort_values()
        .reset_index(drop=True)
    )

    expected_dates = expected_days.dt.date.tolist()
    actual_dates = actual_days.dt.date.tolist()

    if actual_dates != expected_dates:

        print("\n" + "!" * 75)
        print("ELIGIBLE-WORKDAY SEQUENCE MISMATCH")
        print("!" * 75)

        print(f"User: {user}")
        print(f"Scenario: {rule['effective_scenario_id']}")
        print(f"Window: {start_date.date()} to {end_date.date()}")

        print(f"Scheduled days: {len(actual_dates)}")
        print(f"Expected eligible days: {len(expected_dates)}")

        print("\nActual scheduled dates:")
        print(actual_dates)

        print("\nExpected eligible dates:")
        print(expected_dates)

        print("\nDates missing from schedule:")
        print(sorted(set(expected_dates) - set(actual_dates)))

        print("\nUnexpected scheduled dates:")
        print(sorted(set(actual_dates) - set(expected_dates)))

        raise AssertionError(
            f"Actual eligible-workday mismatch for user {user}"
        )

    # --------------------------------------------------------
    # 6.3 Validate exact off/single/two/three-day counts.
    # --------------------------------------------------------

    actual_counts = (
        group["planned_channels"]
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
    )

    for day_type, required in expected_counts.items():

        assert int(actual_counts[day_type]) == required, (
            f"Day-type mismatch for user {user}, "
            f"type {day_type}"
        )

    # --------------------------------------------------------
    # 6.4 Validate chosen channels against Plan A.
    # --------------------------------------------------------

    previous_primary = None
    realized_switches = 0

    for record in group.itertuples(index=False):

        planned = int(record.planned_channels)

        available = {
            channel
            for channel, column in channel_mapping_339.items()
            if getattr(record, column) > 0
        }

        selected = (
            record.attack_channels.split("|")
            if isinstance(record.attack_channels, str)
            and record.attack_channels
            else []
        )

        assert len(selected) == planned, (
            f"Selected channel count mismatch: "
            f"{user}, {record.day}"
        )

        assert len(selected) == len(set(selected)), (
            f"Duplicate selected channels: "
            f"{user}, {record.day}"
        )

        assert set(selected).issubset(available), (
            f"Plan A channel violation: "
            f"{user}, {record.day}"
        )

        if planned == 0:

            assert pd.isna(record.primary_channel), (
                f"Primary channel assigned on Off day: "
                f"{user}, {record.day}"
            )

            continue

        primary = record.primary_channel

        assert primary in selected, (
            f"Invalid primary channel: "
            f"{user}, {record.day}"
        )

        if (
            previous_primary is not None
            and previous_primary != primary
        ):
            realized_switches += 1

        previous_primary = primary

    # --------------------------------------------------------
    # 6.5 Validate exact switching count.
    # --------------------------------------------------------

    assert realized_switches == target_switches, (
        f"Switching mismatch for user {user}: "
        f"expected {target_switches}, "
        f"actual {realized_switches}"
    )

    validation_records_339.append({
        "user": user,
        "scenario_id": rule["effective_scenario_id"],
        "duration": duration,
        "target_switches": target_switches,
        "realized_switches": realized_switches,
        "structural_validation": "PASS"
    })

# ------------------------------------------------------------
# 7. Store validated provisional schedules in memory.
# ------------------------------------------------------------

validated_users_339 = pd.DataFrame(
    validation_records_339
)

validated_schedule_339 = (
    schedule_339
    .sort_values(["user", "day"])
    .reset_index(drop=True)
)

assert len(validated_users_339) == 670
assert validated_users_339["user"].is_unique
assert validated_schedule_339["user"].nunique() == 670

# ------------------------------------------------------------
# 8. Display consolidation results.
# ------------------------------------------------------------

print("\nA. CONSOLIDATION SUMMARY")

print("Previously constructed schedules: 660")
print("Additional exact DP schedules: 10")

print(
    "Total consolidated users:",
    validated_schedule_339["user"].nunique()
)

print(
    "Total scheduled User-Day records:",
    f"{len(validated_schedule_339):,}"
)

print("\nB. INDEPENDENT VALIDATION")

print(f"Users checked: {len(validated_users_339)}")
print(
    "Users passing all checks:",
    int(
        validated_users_339[
            "structural_validation"
        ].eq("PASS").sum()
    )
)

print("Eligible-workday sequences: PASS")
print("Scenario duration and day counts: PASS")
print("Natural channel availability: PASS")
print("Exact primary-channel switching: PASS")

print("\nC. SCHEDULED DAY DISTRIBUTION")

display(
    validated_schedule_339["planned_channels"]
    .value_counts()
    .reindex([0, 1, 2, 3], fill_value=0)
    .rename_axis("Planned Channels")
    .reset_index(name="Days")
)

print("\nD. PROVISIONAL SCHEDULE SAMPLE")

display(
    validated_schedule_339[
        [
            "user",
            "scenario_id",
            "day",
            "planned_channels",
            "primary_channel",
            "attack_channels",
            "solution_source"
        ]
    ].head(20)
)

# ------------------------------------------------------------
# 9. Reconcile all 995 original assigned users.
# ------------------------------------------------------------

found_339 = set(validated_users_339["user"])

infeasible_339 = set(
    exact_search_audit_337.loc[
        exact_search_audit_337["status"] == "INFEASIBLE",
        "user"
    ]
)

incomplete_339 = set(
    exact_search_audit_337.loc[
        exact_search_audit_337["status"] == "INCOMPLETE",
        "user"
    ]
)

previously_deferred_339 = set(deferred_325["user"])

groups_339 = [
    found_339,
    infeasible_339,
    incomplete_339,
    previously_deferred_339
]

assert all(
    groups_339[i].isdisjoint(groups_339[j])
    for i in range(len(groups_339))
    for j in range(i + 1, len(groups_339))
), "Population categories overlap."

assert set().union(*groups_339) == set(
    assignments["user"]
), "Population reconciliation mismatch."

print("\nE. FULL POPULATION RECONCILIATION")

print(f"Validated schedules: {len(found_339)}")
print(f"INFEASIBLE within current model: {len(infeasible_339)}")
print(f"INCOMPLETE — deferred: {len(incomplete_339)}")
print(f"Previously deferred: {len(previously_deferred_339)}")
print(f"Total accounted for: {sum(map(len, groups_339))}")

print("\n" + "=" * 75)
print("STEP 3.3.9 — VALIDATION COMPLETED")
print("=" * 75)

print("Provisional schedules: CONSOLIDATED AND VALIDATED")
print("Original Stage 2 assignments: UNCHANGED")
print("Synthetic injection: NOT STARTED")
print("Plan B: NOT ACTIVATED")
print("No files were modified or exported.")


STAGE 3 — STEP 3.3.9: SCHEDULE CONSOLIDATION

A. CONSOLIDATION SUMMARY
Previously constructed schedules: 660
Additional exact DP schedules: 10
Total consolidated users: 670
Total scheduled User-Day records: 15,010

B. INDEPENDENT VALIDATION
Users checked: 670
Users passing all checks: 670
Eligible-workday sequences: PASS
Scenario duration and day counts: PASS
Natural channel availability: PASS
Exact primary-channel switching: PASS

C. SCHEDULED DAY DISTRIBUTION


,Planned Channels,Days
0,0,2374
1,1,11098
2,2,1337
3,3,201



D. PROVISIONAL SCHEDULE SAMPLE


,user,scenario_id,day,planned_channels,primary_channel,attack_channels,solution_source
0,AAF0819,SCN_0007,2011-03-03,1,Email,Email,Initial demonstration
1,AAF0819,SCN_0007,2011-03-04,1,Email,Email,Initial demonstration
2,AAF0819,SCN_0007,2011-03-07,1,USB,USB,Initial demonstration
3,AAF0819,SCN_0007,2011-03-08,1,Email,Email,Initial demonstration
4,AAF0819,SCN_0007,2011-03-09,1,USB,USB,Initial demonstration
5,AAP0352,SCN_0016,2011-04-06,1,USB,USB,Initial demonstration
6,AAP0352,SCN_0016,2011-04-07,1,Email,Email,Initial demonstration
7,AAP0352,SCN_0016,2011-04-08,1,Email,Email,Initial demonstration
8,AAP0352,SCN_0016,2011-04-11,2,Email,Email|USB,Initial demonstration
9,AAP0352,SCN_0016,2011-04-12,1,Email,Email,Initial demonstration



E. FULL POPULATION RECONCILIATION
Validated schedules: 670
INFEASIBLE within current model: 78
INCOMPLETE — deferred: 1
Previously deferred: 246
Total accounted for: 995

STEP 3.3.9 — VALIDATION COMPLETED
Provisional schedules: CONSOLIDATED AND VALIDATED
Original Stage 2 assignments: UNCHANGED
Synthetic injection: NOT STARTED
Plan B: NOT ACTIVATED
No files were modified or exported.


In [52]:

# Stage 3.5 — One-Time Setup: Save Validated Schedule
# Goal: Create an independent schedule reference for QA.
# Safety: Never overwrite an existing reference file.

from pathlib import Path
import pandas as pd

ROOT = Path(
    r"C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration"
)

reference_dir = ROOT / "Attack_Simulation" / "validation_inputs"
reference_path = reference_dir / "validated_schedule_339.parquet"

if reference_path.exists():
    print("Reference file already exists:")
    print(reference_path)
    print("No file was overwritten.")
else:
    assert "validated_schedule_339" in globals(), (
        "Run Step 3.3.9 in the original notebook first."
    )

    schedule_reference = validated_schedule_339.copy(deep=True)

    assert len(schedule_reference) == 15010
    assert schedule_reference["user"].nunique() == 670
    assert not schedule_reference.duplicated(
        ["user", "day"]
    ).any()

    reference_dir.mkdir(parents=True, exist_ok=True)

    schedule_reference.to_parquet(
        reference_path,
        index=False
    )

    reloaded = pd.read_parquet(reference_path)

    pd.testing.assert_frame_equal(
        schedule_reference.reset_index(drop=True),
        reloaded.reset_index(drop=True),
        check_exact=True
    )

    print("Schedule reference saved and verified.")
    print("Rows:", len(reloaded))
    print("Users:", reloaded["user"].nunique())
    print("Path:", reference_path)


Schedule reference saved and verified.
Rows: 15010
Users: 670
Path: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Attack_Simulation\validation_inputs\validated_schedule_339.parquet



## Step 3.4 — Attack Injection Planning

### Step 3.4.1 — Injection Baseline Coverage Audit

**Objective:** Assess historical baseline coverage for every scheduled attack channel before calculating synthetic injection quantities.

**Validation Tasks:**
1. Expand the validated schedule into individual channel-day assignments.
2. Confirm that every selected channel has natural activity on its scheduled day.
3. Check historical channel activity for each user.
4. Audit the availability and positivity of candidate P95 baseline values.
5. Identify channel assignments requiring baseline-methodology review.

**Important:** Baseline field names are preliminary candidates. Their exact feature definitions must be verified against Data Engineering before they are used to calculate injection quantities.

**Expected Outcome:** A baseline-coverage report for all scheduled Email, USB, and HTTP channel-day assignments.

**Note:** Read-only. No injection, reassignment, or file export.


In [24]:

# Stage 3 — Step 3.4.1: Injection Baseline Coverage Audit
# Goal: Verify baseline availability for each scheduled channel-day
# before defining stealth-aware synthetic injection quantities.
# Read-only: No source data, schedule, or files are modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.4.1: INJECTION BASELINE COVERAGE")
print("=" * 75)

# ------------------------------------------------------------
# 1. Validate the consolidated schedule and baseline schema.
# ------------------------------------------------------------

assert validated_schedule_339["user"].nunique() == 670
assert not validated_schedule_339.duplicated(
    ["user", "day"]
).any()

baseline_fields_341 = [
    "user",
    "email_active_days", "email_count_p95",
    "usb_active_days", "usb_copy_p95",
    "http_upload_active_days", "http_upload_p95"
]

missing_fields_341 = (
    set(baseline_fields_341) - set(user_baseline.columns)
)

assert not missing_fields_341, (
    f"Missing baseline columns: {missing_fields_341}"
)

assert user_baseline["user"].is_unique

# ------------------------------------------------------------
# 2. Expand planned attack days into channel-day records.
# ------------------------------------------------------------

active_schedule_341 = validated_schedule_339.loc[
    validated_schedule_339["planned_channels"] > 0
].copy()

channel_days_341 = (
    active_schedule_341.assign(
        channel=active_schedule_341[
            "attack_channels"
        ].str.split("|", regex=False)
    )
    .explode("channel")
    .reset_index(drop=True)
)

assert channel_days_341["channel"].isin(
    ["Email", "USB", "HTTP"]
).all()

assert not channel_days_341.duplicated(
    ["user", "day", "channel"]
).any()

assert len(channel_days_341) == int(
    active_schedule_341["planned_channels"].sum()
)

# ------------------------------------------------------------
# 3. Attach historical baseline values.
# ------------------------------------------------------------

channel_days_341 = channel_days_341.merge(
    user_baseline[baseline_fields_341],
    on="user",
    how="left",
    validate="many_to_one"
)

mapping_341 = {
    "Email": (
        "external_email_count",
        "email_active_days",
        "email_count_p95"
    ),
    "USB": (
        "usb_copy_count",
        "usb_active_days",
        "usb_copy_p95"
    ),
    "HTTP": (
        "www_upload_count",
        "http_upload_active_days",
        "http_upload_p95"
    )
}

channel_days_341["natural_count"] = np.nan
channel_days_341["historical_active_days"] = np.nan
channel_days_341["candidate_p95"] = np.nan

for channel, (
    natural_column, history_column, p95_column
) in mapping_341.items():

    mask = channel_days_341["channel"].eq(channel)

    channel_days_341.loc[
        mask, "natural_count"
    ] = channel_days_341.loc[
        mask, natural_column
    ].to_numpy()

    channel_days_341.loc[
        mask, "historical_active_days"
    ] = channel_days_341.loc[
        mask, history_column
    ].to_numpy()

    channel_days_341.loc[
        mask, "candidate_p95"
    ] = channel_days_341.loc[
        mask, p95_column
    ].to_numpy()

# ------------------------------------------------------------
# 4. Evaluate daily availability and baseline coverage.
# ------------------------------------------------------------

channel_days_341["natural_activity_pass"] = (
    channel_days_341["natural_count"] > 0
)

channel_days_341["historical_activity_pass"] = (
    channel_days_341["historical_active_days"] > 0
)

channel_days_341["p95_missing"] = (
    ~np.isfinite(
        pd.to_numeric(
            channel_days_341["candidate_p95"],
            errors="coerce"
        ).to_numpy(dtype=float)
    )
)

channel_days_341["p95_nonpositive"] = (
    channel_days_341["candidate_p95"].le(0)
    & ~channel_days_341["p95_missing"]
)

channel_days_341["p95_usable_candidate"] = (
    ~channel_days_341["p95_missing"]
    & ~channel_days_341["p95_nonpositive"]
)

assert channel_days_341["natural_activity_pass"].all(), (
    "Selected attack channel lacks same-day natural activity."
)

# ------------------------------------------------------------
# 5. Report coverage by channel.
# ------------------------------------------------------------

print("\nA. CHANNEL-LEVEL BASELINE COVERAGE")

coverage_341 = (
    channel_days_341
    .groupby("channel")
    .agg(
        Planned_Channel_Days=("user", "size"),
        Unique_Users=("user", "nunique"),
        Missing_P95=("p95_missing", "sum"),
        Nonpositive_P95=("p95_nonpositive", "sum"),
        No_Historical_Activity=(
            "historical_activity_pass",
            lambda s: int((~s).sum())
        ),
        Usable_P95_Candidates=(
            "p95_usable_candidate", "sum"
        )
    )
    .reindex(["Email", "USB", "HTTP"])
    .reset_index()
)

display(coverage_341)

# ------------------------------------------------------------
# 6. Identify users requiring additional baseline review.
# ------------------------------------------------------------

review_needed_341 = channel_days_341.loc[
    ~channel_days_341["p95_usable_candidate"]
    | ~channel_days_341["historical_activity_pass"]
].copy()

print("\nB. USERS REQUIRING BASELINE REVIEW")

print(
    "Unique users requiring review:",
    review_needed_341["user"].nunique()
)

print(
    "Affected channel-day assignments:",
    len(review_needed_341)
)

if review_needed_341.empty:
    print("No missing or nonpositive candidate P95 values.")
else:
    display(
        review_needed_341[
            [
                "user", "scenario_id", "day", "channel",
                "natural_count", "historical_active_days",
                "candidate_p95"
            ]
        ].head(20)
    )

# ------------------------------------------------------------
# 7. Overall audit summary.
# ------------------------------------------------------------

print("\nC. AUDIT SUMMARY")

print(
    "Scheduled users:",
    validated_schedule_339["user"].nunique()
)
print("Scheduled attack channel-days:", len(channel_days_341))
print(
    "Days with no synthetic attack:",
    int((validated_schedule_339["planned_channels"] == 0).sum())
)
print(
    "Same-day natural activity violations:",
    int((~channel_days_341["natural_activity_pass"]).sum())
)

print("\n" + "=" * 75)
print("STEP 3.4.1 — AUDIT COMPLETED")
print("=" * 75)

print("Baseline feature semantics: NOT YET VERIFIED")
print("Injection quantities: NOT CALCULATED")
print("Synthetic data injection: NOT STARTED")
print("Original Stage 2 assignments: UNCHANGED")
print("No files were modified or exported.")


STAGE 3 — STEP 3.4.1: INJECTION BASELINE COVERAGE

A. CHANNEL-LEVEL BASELINE COVERAGE


,channel,Planned_Channel_Days,Unique_Users,Missing_P95,Nonpositive_P95,No_Historical_Activity,Usable_P95_Candidates
0,Email,9343,670,0,0,0,9343
1,USB,3499,499,0,0,0,3499
2,HTTP,1533,338,0,0,0,1533



B. USERS REQUIRING BASELINE REVIEW
Unique users requiring review: 0
Affected channel-day assignments: 0
No missing or nonpositive candidate P95 values.

C. AUDIT SUMMARY
Scheduled users: 670
Scheduled attack channel-days: 14375
Days with no synthetic attack: 2374
Same-day natural activity violations: 0

STEP 3.4.1 — AUDIT COMPLETED
Baseline feature semantics: NOT YET VERIFIED
Injection quantities: NOT CALCULATED
Synthetic data injection: NOT STARTED
Original Stage 2 assignments: UNCHANGED
No files were modified or exported.



### Step 3.4.2 — Baseline Feature Semantics & Metric Alignment

**Objective:** Verify the original Data Engineering definitions of Email, USB, and HTTP baseline percentiles before using them in synthetic injection calculations.

**Validation Tasks:**
1. Locate the original baseline-building notebook.
2. Inspect the source definitions of the three P95 features.
3. Review relevant entries in the feature and schema dictionaries.
4. Identify the corresponding User-Day activity columns.
5. Determine whether the percentiles represent active-day activity or all workdays.

**Expected Outcome:** Establish a verified mapping between baseline features and planned injection metrics.

**Note:** Read-only source inspection. No injection quantities will be calculated or applied.


In [25]:

# Stage 3 — Step 3.4.2: Baseline Feature Semantics Audit
# Goal: Inspect the original Data Engineering notebook and dictionaries
# to verify the exact definitions of the P95 baseline features.
# Read-only: No source files, datasets, or schedules are modified.

from pathlib import Path
import pandas as pd
import json
import re
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.4.2: BASELINE FEATURE SEMANTICS")
print("=" * 75)

# ------------------------------------------------------------
# 1. Locate Data Engineering source files.
# ------------------------------------------------------------

project_root_342 = Path(
    r"C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration"
)

de_root_342 = project_root_342 / "Data_Engineering"

assert de_root_342.exists(), (
    "Data Engineering folder not found."
)

notebooks_342 = sorted(
    [
        p for p in de_root_342.rglob("*.ipynb")
        if "baseline" in p.stem.lower()
        and "malicious" not in p.stem.lower()
    ],
    key=lambda p: (
        not p.stem.startswith("03_"),
        str(p)
    )
)

feature_dict_files_342 = list(
    de_root_342.rglob("feature_dictionary_r62.csv")
)

schema_dict_files_342 = list(
    de_root_342.rglob("schema_dictionary_r62.csv")
)

print("\nA. SOURCE DISCOVERY")

print("Baseline-building notebook candidates:")
for p in notebooks_342:
    print(" -", p.relative_to(project_root_342))

print("\nFeature dictionaries found:",
      len(feature_dict_files_342))

print("Schema dictionaries found:",
      len(schema_dict_files_342))

# ------------------------------------------------------------
# 2. Identify User-Day fields relevant to injection.
# ------------------------------------------------------------

print("\nB. RELEVANT USER-DAY COLUMNS")

terms_342 = [
    "email", "usb", "upload", "http", "sent"
]

matched_columns_342 = [
    col for col in user_day.columns
    if any(term in col.lower() for term in terms_342)
]

for col in matched_columns_342:
    print(f" - {col}: {user_day[col].dtype}")

# ------------------------------------------------------------
# 3. Inspect the three baseline features.
# ------------------------------------------------------------

print("\nC. BASELINE FEATURE DEFINITIONS TO VERIFY")

baseline_targets_342 = [
    "email_count_p95",
    "usb_copy_p95",
    "http_upload_p95"
]

for feature in baseline_targets_342:
    print(
        f"{feature}: dtype={user_baseline[feature].dtype}, "
        f"missing={user_baseline[feature].isna().sum()}"
    )

# ------------------------------------------------------------
# 4. Display relevant source-code excerpts.
# ------------------------------------------------------------

print("\nD. BASELINE-BUILDING SOURCE CODE")

search_pattern_342 = re.compile(
    r"email_count|email_send|external_email|"
    r"usb_copy|http_upload|www_upload|"
    r"quantile|percentile|p95|active_day",
    re.IGNORECASE
)

if not notebooks_342:
    print("No baseline notebook was found automatically.")
else:
    notebook_path_342 = notebooks_342[0]

    print("Inspecting:", notebook_path_342)

    with notebook_path_342.open(
        "r", encoding="utf-8"
    ) as f:
        notebook_342 = json.load(f)

    shown_342 = 0

    for cell_index, cell in enumerate(
        notebook_342.get("cells", [])
    ):
        if cell.get("cell_type") != "code":
            continue

        source = cell.get("source", [])
        source = (
            "".join(source)
            if isinstance(source, list)
            else str(source)
        )

        lines = source.splitlines()

        hit_indices = [
            i for i, line in enumerate(lines)
            if search_pattern_342.search(line)
        ]

        if not hit_indices:
            continue

        # Show neighboring lines to retain calculation context.
        indices = set()

        for hit in hit_indices:
            indices.update(
                range(
                    max(0, hit - 4),
                    min(len(lines), hit + 6)
                )
            )

        print(f"\n--- Notebook Cell {cell_index} ---")

        previous = -2

        for i in sorted(indices):
            if i > previous + 1:
                print("    ...")

            print(f"{i + 1:03d}: {lines[i]}")
            previous = i

        shown_342 += 1

        if shown_342 >= 12:
            print("\nFurther matching cells omitted.")
            break

# ------------------------------------------------------------
# 5. Inspect feature and schema dictionary entries.
# ------------------------------------------------------------

def inspect_dictionary_342(paths, label, targets):

    print(f"\nE. {label.upper()}")

    if not paths:
        print("Dictionary not found.")
        return

    path = paths[0]

    print("Source:", path)

    try:
        dictionary = pd.read_csv(path)
    except UnicodeDecodeError:
        dictionary = pd.read_csv(
            path, encoding="latin-1"
        )

    text_view = dictionary.astype(str)

    matched = text_view.apply(
        lambda column: column.str.contains(
            "|".join(re.escape(t) for t in targets),
            case=False,
            na=False
        )
    ).any(axis=1)

    results = dictionary.loc[matched]

    print("Matching entries:", len(results))

    if results.empty:
        print("No matching entries found.")
    else:
        display(results.head(30))


inspect_dictionary_342(
    feature_dict_files_342,
    "Feature Dictionary",
    baseline_targets_342
)

inspect_dictionary_342(
    schema_dict_files_342,
    "Schema Dictionary",
    [
        "external_email_count",
        "email_send_count",
        "usb_copy_count",
        "www_upload_count"
    ]
)

# ------------------------------------------------------------
# 6. End the audit without making unverified assumptions.
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 3.4.2 — SOURCE INSPECTION COMPLETED")
print("=" * 75)

print("P95-to-User-Day metric mapping: PENDING REVIEW")
print("P95 active-day methodology: PENDING REVIEW")
print("Stealth multipliers: NOT APPLIED")
print("Injection quantities: NOT CALCULATED")
print("Synthetic injection: NOT STARTED")
print("No files were modified.")


STAGE 3 — STEP 3.4.2: BASELINE FEATURE SEMANTICS

A. SOURCE DISCOVERY
Baseline-building notebook candidates:
 - Data_Engineering\Data_engineering\notebooks\03_build_baseline_r62.ipynb

Feature dictionaries found: 1
Schema dictionaries found: 1

B. RELEVANT USER-DAY COLUMNS
 - email_event_count: int64
 - email_send_count: float64
 - email_view_count: float64
 - external_email_count: float64
 - internal_email_count: float64
 - total_sent_email_bytes: float64
 - external_email_bytes: float64
 - emails_with_attachments: float64
 - usb_copy_count: float64
 - unique_usb_files: int64
 - unique_usb_extensions: int64
 - http_event_count: int64
 - www_upload_count: float64
 - unique_upload_domains: int64

C. BASELINE FEATURE DEFINITIONS TO VERIFY
email_count_p95: dtype=float64, missing=1
usb_copy_p95: dtype=float64, missing=3212
http_upload_p95: dtype=float64, missing=3380

D. BASELINE-BUILDING SOURCE CODE
Inspecting: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Data_Engineering\Data_engineer

,column_order,dataset,feature_name,data_type,nullable,category,feature_role,description,calculation,time_scope,source_columns,notes
39,40,behavior_features_r62_v1,email_count_p95,DOUBLE,YES,Baseline,User Baseline,95th percentile of external email count on act...,"QUANTILE_CONT(external_email_count, 0.95)",Baseline period before 2011-01-01; weekdays wi...,external_email_count,Zero-activity days are excluded from the distr...
49,50,behavior_features_r62_v1,usb_copy_p95,DOUBLE,YES,Baseline,User Baseline,95th percentile of USB copy count on active US...,"QUANTILE_CONT(usb_copy_count, 0.95)",Baseline period before 2011-01-01; weekdays wi...,usb_copy_count,NaN
55,56,behavior_features_r62_v1,http_upload_p95,DOUBLE,YES,Baseline,User Baseline,95th percentile of WWW Upload count on active ...,"QUANTILE_CONT(www_upload_count, 0.95)",Baseline period before 2011-01-01; weekdays wi...,www_upload_count,NaN
61,62,behavior_features_r62_v1,email_count_above_p95,INTEGER,YES,Baseline-Relative,Anomaly Indicator,Indicates whether today's external email count...,1 if external_email_count > email_count_p95 el...,Future period from 2011-01-01 onward; weekday ...,"external_email_count, email_count_p95",NaN
63,64,behavior_features_r62_v1,usb_copy_above_p95,INTEGER,YES,Baseline-Relative,Anomaly Indicator,Indicates whether today's USB copy count excee...,1 if usb_copy_count > usb_copy_p95 else 0,Future period from 2011-01-01 onward; weekday ...,"usb_copy_count, usb_copy_p95",NaN
64,65,behavior_features_r62_v1,http_upload_above_p95,INTEGER,YES,Baseline-Relative,Anomaly Indicator,Indicates whether today's WWW Upload count exc...,1 if www_upload_count > http_upload_p95 else 0,Future period from 2011-01-01 onward; weekday ...,"www_upload_count, http_upload_p95",NaN
65,66,behavior_features_r62_v1,email_count_to_p95_ratio,DOUBLE,YES,Baseline-Relative,Relative Magnitude,Ratio of today's external email count to the u...,external_email_count / email_count_p95,Future period from 2011-01-01 onward; weekday ...,"external_email_count, email_count_p95",NULL when the user's P95 is missing or zero.
67,68,behavior_features_r62_v1,usb_copy_to_p95_ratio,DOUBLE,YES,Baseline-Relative,Relative Magnitude,Ratio of today's USB copy count to the user's ...,usb_copy_count / usb_copy_p95,Future period from 2011-01-01 onward; weekday ...,"usb_copy_count, usb_copy_p95",NULL when the user's P95 is missing or zero.
68,69,behavior_features_r62_v1,http_upload_to_p95_ratio,DOUBLE,YES,Baseline-Relative,Relative Magnitude,Ratio of today's WWW Upload count to the user'...,www_upload_count / http_upload_p95,Future period from 2011-01-01 onward; weekday ...,"www_upload_count, http_upload_p95",NULL when the user's P95 is missing or zero.



E. SCHEMA DICTIONARY
Source: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Data_Engineering\Data_engineering\results\schema_dictionary_r62.csv
Matching entries: 16


,dataset,column_name,data_type,nullable
3,email_daily_r62,email_send_count,DOUBLE,YES
5,email_daily_r62,external_email_count,DOUBLE,YES
20,usb_daily_r62,usb_copy_count,DOUBLE,YES
29,http_daily_r62,www_upload_count,DOUBLE,YES
43,user_day_r62_v1,email_send_count,DOUBLE,YES
45,user_day_r62_v1,external_email_count,DOUBLE,YES
58,user_day_r62_v1,usb_copy_count,DOUBLE,YES
65,user_day_r62_v1,www_upload_count,DOUBLE,YES
113,behavior_features_r62_v1,email_send_count,DOUBLE,YES
115,behavior_features_r62_v1,external_email_count,DOUBLE,YES



STEP 3.4.2 — SOURCE INSPECTION COMPLETED
P95-to-User-Day metric mapping: PENDING REVIEW
P95 active-day methodology: PENDING REVIEW
Stealth multipliers: NOT APPLIED
Injection quantities: NOT CALCULATED
Synthetic injection: NOT STARTED
No files were modified.



### Step 3.4.3 — Stealth-Aware Injection Quantity Analysis

**Objective:** Compare three candidate injection-quantity formulas using the verified channel-specific P95 baselines, existing daily activity, and assigned stealth multipliers.

**Candidate Formulas:**
1. **P95-Based Addition:** Increment scaled from the historical active-day P95.
2. **Natural-Activity Addition:** Increment scaled from existing same-day activity.
3. **P95 Headroom:** Nonnegative increment within a multiplier-scaled P95 reference level.

**Validation Tasks:**
1. Preserve High, Medium, and Low stealth multipliers (0.5, 1.0, 1.5).
2. Compute integer-valued candidate injection increments.
3. Measure zero-injection cases.
4. Measure original and simulated-final activity relative to P95.
5. Identify newly created P95 exceedances.
6. Compare results by channel and stealth level.
7. Inspect examples with large relative increases.

**Important Limitations:**
- P95 is a historical percentile, not a mandatory upper limit.
- The formulas are experimental and have not been approved.
- Event counts are not equivalent to exfiltrated data volume in bytes.
- No injection is executed and no source files are modified.

**Expected Outcome:** A quantitative comparison supporting selection or refinement of an injection-quantity policy.


In [26]:

# Stage 3 — Step 3.4.3: Stealth-Aware Injection Quantity Analysis
# Goal: Compare three candidate quantity formulas against real
# natural activity, user-specific P95, and assigned stealth levels.
# Read-only: No original User-Day data, assignments, or files modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.4.3: INJECTION QUANTITY ANALYSIS")
print("=" * 75)

# ------------------------------------------------------------
# 1. Verify source data and baseline coverage.
# ------------------------------------------------------------

assert validated_schedule_339["user"].nunique() == 670

assert len(channel_days_341) == 14375, (
    "Unexpected channel-day count. Review Step 3.4.1."
)

assert not channel_days_341.duplicated(
    ["user", "day", "channel"]
).any()

assert channel_days_341[
    "natural_activity_pass"
].all(), "Missing natural activity on a selected attack channel."

assert channel_days_341[
    "p95_usable_candidate"
].all(), "Missing or invalid candidate P95."

# ------------------------------------------------------------
# 2. Prepare a separate experimental working dataset.
# ------------------------------------------------------------

quantity_base_343 = channel_days_341[
    [
        "user",
        "scenario_id",
        "day",
        "channel",
        "natural_count",
        "candidate_p95"
    ]
].copy()

stealth_metadata_343 = cohort_325[
    ["user", "effective_stealth"]
].copy()

assert stealth_metadata_343["user"].is_unique

quantity_base_343 = quantity_base_343.merge(
    stealth_metadata_343,
    on="user",
    how="left",
    validate="many_to_one"
)

STEALTH_MULTIPLIERS_343 = {
    "High": 0.5,
    "Medium": 1.0,
    "Low": 1.5
}

assert quantity_base_343[
    "effective_stealth"
].isin(STEALTH_MULTIPLIERS_343).all()

quantity_base_343["stealth_multiplier"] = (
    quantity_base_343["effective_stealth"]
    .map(STEALTH_MULTIPLIERS_343)
)

# ------------------------------------------------------------
# 3. Validate that original channel counts are positive
# integers, even though stored as floating-point values.
# ------------------------------------------------------------

for column in ["natural_count", "candidate_p95"]:

    quantity_base_343[column] = pd.to_numeric(
        quantity_base_343[column],
        errors="raise"
    )

    assert np.isfinite(
        quantity_base_343[column].to_numpy()
    ).all(), f"Non-finite values in {column}"

    assert quantity_base_343[column].gt(0).all(), (
        f"Nonpositive values in {column}"
    )

natural_values_343 = quantity_base_343[
    "natural_count"
].to_numpy(dtype=float)

assert np.allclose(
    natural_values_343,
    np.rint(natural_values_343),
    rtol=0,
    atol=1e-9
), "Natural activity contains non-integer counts."

quantity_base_343["natural_count"] = (
    np.rint(natural_values_343).astype(np.int64)
)

# ------------------------------------------------------------
# 4. Calculate three candidate injection-quantity formulas.
#
# A: P95-Based Addition
# B: Natural-Activity Addition
# C: P95 Headroom
#
# Values represent ADDED events, not final totals.
# ------------------------------------------------------------

natural_343 = quantity_base_343[
    "natural_count"
].to_numpy(dtype=np.int64)

p95_343 = quantity_base_343[
    "candidate_p95"
].to_numpy(dtype=float)

multiplier_343 = quantity_base_343[
    "stealth_multiplier"
].to_numpy(dtype=float)

candidate_increments_343 = {
    "A_P95_Addition": np.maximum(
        1,
        np.ceil(p95_343 * multiplier_343)
    ).astype(np.int64),

    "B_Natural_Addition": np.maximum(
        1,
        np.ceil(natural_343 * multiplier_343)
    ).astype(np.int64),

    "C_P95_Headroom": np.maximum(
        0,
        np.floor(
            p95_343 * multiplier_343 + 1e-9
        ).astype(np.int64) - natural_343
    )
}

# ------------------------------------------------------------
# 5. Build an experimental result for each formula.
# Original activity is always preserved.
# ------------------------------------------------------------

experiment_parts_343 = []

for formula_name, increments in candidate_increments_343.items():

    result = quantity_base_343.copy()

    result["formula"] = formula_name
    result["injected_count"] = increments

    result["simulated_final_count"] = (
        result["natural_count"]
        + result["injected_count"]
    )

    # Relative historical activity comparison.
    result["original_to_p95_ratio"] = (
        result["natural_count"]
        / result["candidate_p95"]
    )

    result["final_to_p95_ratio"] = (
        result["simulated_final_count"]
        / result["candidate_p95"]
    )

    result["original_above_p95"] = (
        result["natural_count"]
        > result["candidate_p95"]
    )

    result["final_above_p95"] = (
        result["simulated_final_count"]
        > result["candidate_p95"]
    )

    result["newly_above_p95"] = (
        ~result["original_above_p95"]
        & result["final_above_p95"]
    )

    result["zero_injection"] = (
        result["injected_count"] == 0
    )

    result["final_to_natural_ratio"] = (
        result["simulated_final_count"]
        / result["natural_count"]
    )

    experiment_parts_343.append(result)

quantity_experiment_343 = pd.concat(
    experiment_parts_343,
    ignore_index=True
)

# ------------------------------------------------------------
# 6. Validate experimental results.
# ------------------------------------------------------------

assert len(quantity_experiment_343) == 3 * len(
    quantity_base_343
)

assert quantity_experiment_343[
    "injected_count"
].ge(0).all()

assert quantity_experiment_343[
    "simulated_final_count"
].ge(
    quantity_experiment_343["natural_count"]
).all()

assert (
    quantity_experiment_343["simulated_final_count"]
    == quantity_experiment_343["natural_count"]
    + quantity_experiment_343["injected_count"]
).all()

assert not quantity_experiment_343.duplicated(
    ["user", "day", "channel", "formula"]
).any()

print("\nA. EXPERIMENT INPUT SUMMARY")

print(f"Scheduled users: {quantity_base_343['user'].nunique()}")
print(f"Original planned channel-days: {len(quantity_base_343):,}")
print("Candidate formulas:", len(candidate_increments_343))
print(f"Experimental comparison rows: {len(quantity_experiment_343):,}")

# ------------------------------------------------------------
# 7. Summarize each formula.
# ------------------------------------------------------------

print("\nB. OVERALL FORMULA COMPARISON")

formula_summary_343 = (
    quantity_experiment_343
    .groupby("formula")
    .agg(
        Channel_Days=("user", "size"),
        Zero_Injections=("zero_injection", "sum"),
        Total_Injected_Events=("injected_count", "sum"),
        Median_Injected_Events=("injected_count", "median"),
        P95_Injected_Events=(
            "injected_count",
            lambda s: s.quantile(0.95)
        ),
        Already_Above_P95=("original_above_p95", "sum"),
        Newly_Above_P95=("newly_above_p95", "sum"),
        Final_Above_P95=("final_above_p95", "sum"),
        Median_Final_To_P95=(
            "final_to_p95_ratio", "median"
        )
    )
    .reset_index()
)

formula_summary_343["Zero_Injection_Rate_%"] = (
    100
    * formula_summary_343["Zero_Injections"]
    / formula_summary_343["Channel_Days"]
).round(2)

formula_summary_343["New_P95_Exceedance_Rate_%"] = (
    100
    * formula_summary_343["Newly_Above_P95"]
    / formula_summary_343["Channel_Days"]
).round(2)

display(formula_summary_343.round(2))

# ------------------------------------------------------------
# 8. Compare results across channels.
# ------------------------------------------------------------

print("\nC. FORMULA COMPARISON BY CHANNEL")

channel_summary_343 = (
    quantity_experiment_343
    .groupby(["formula", "channel"])
    .agg(
        Channel_Days=("user", "size"),
        Zero_Injections=("zero_injection", "sum"),
        Median_Injection=("injected_count", "median"),
        Total_Injected_Events=("injected_count", "sum"),
        Newly_Above_P95=("newly_above_p95", "sum")
    )
    .reset_index()
)

display(channel_summary_343.round(2))

# ------------------------------------------------------------
# 9. Compare results across assigned stealth levels.
# Different levels correspond to different user populations.
# This is descriptive, not a causal stealth evaluation.
# ------------------------------------------------------------

print("\nD. FORMULA COMPARISON BY STEALTH LEVEL")

stealth_summary_343 = (
    quantity_experiment_343
    .groupby(
        ["formula", "effective_stealth"],
        observed=True
    )
    .agg(
        Channel_Days=("user", "size"),
        Unique_Users=("user", "nunique"),
        Zero_Injections=("zero_injection", "sum"),
        Median_Injection=("injected_count", "median"),
        Total_Injected_Events=("injected_count", "sum"),
        Newly_Above_P95=("newly_above_p95", "sum"),
        Median_Final_To_P95=(
            "final_to_p95_ratio", "median"
        )
    )
    .reset_index()
)

stealth_summary_343["effective_stealth"] = (
    pd.Categorical(
        stealth_summary_343["effective_stealth"],
        categories=["High", "Medium", "Low"],
        ordered=True
    )
)

stealth_summary_343 = stealth_summary_343.sort_values(
    ["formula", "effective_stealth"]
)

display(stealth_summary_343.round(2))

# ------------------------------------------------------------
# 10. Examine examples with large P95-relative final counts.
# These are diagnostics, not automatic policy violations.
# ------------------------------------------------------------

print("\nE. HIGHEST FINAL-TO-P95 RATIOS — SAMPLE")

display(
    quantity_experiment_343[
        [
            "user",
            "day",
            "channel",
            "effective_stealth",
            "formula",
            "natural_count",
            "candidate_p95",
            "injected_count",
            "simulated_final_count",
            "final_to_p95_ratio"
        ]
    ]
    .sort_values(
        "final_to_p95_ratio",
        ascending=False
    )
    .head(15)
    .round(2)
)

# ------------------------------------------------------------
# 11. Review zero-injection cases.
# These matter because every scheduled attack channel-day
# is expected to receive positive synthetic activity.
# ------------------------------------------------------------

print("\nF. ZERO-INJECTION DIAGNOSTIC")

zero_summary_343 = (
    quantity_experiment_343
    .loc[quantity_experiment_343["zero_injection"]]
    .groupby(["formula", "channel"])
    .size()
    .rename("Zero_Channel_Days")
    .reset_index()
)

if zero_summary_343.empty:
    print("All formulas produced positive injection counts.")
else:
    display(zero_summary_343)

# ------------------------------------------------------------
# 12. Final interpretation safeguards.
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 3.4.3 — FORMULA COMPARISON COMPLETED")
print("=" * 75)

print("Injection formulas: EXPERIMENTAL")
print("Chosen injection policy: NOT YET APPROVED")
print("Injection quantities: NOT APPLIED")
print("Historical P95: NOT TREATED AS A HARD SAFETY LIMIT")
print("Counts do not establish bytes exfiltrated.")
print("Original User-Day data: UNCHANGED")
print("Final scenario assignments: UNCHANGED")
print("No files were modified or exported.")


STAGE 3 — STEP 3.4.3: INJECTION QUANTITY ANALYSIS

A. EXPERIMENT INPUT SUMMARY
Scheduled users: 670
Original planned channel-days: 14,375
Candidate formulas: 3
Experimental comparison rows: 43,125

B. OVERALL FORMULA COMPARISON


,formula,Channel_Days,Zero_Injections,Total_Injected_Events,Median_Injected_Events,P95_Injected_Events,Already_Above_P95,Newly_Above_P95,Final_Above_P95,Median_Final_To_P95,Zero_Injection_Rate_%,New_P95_Exceedance_Rate_%
0,A_P95_Addition,14375,0,75623,5.0,11.0,544,11186,11730,1.57,0.00,77.82
1,B_Natural_Addition,14375,0,42095,2.0,8.0,544,5541,6085,1.00,0.00,38.55
2,C_P95_Headroom,14375,3722,35951,2.0,7.0,544,4696,5240,1.00,25.89,32.67



C. FORMULA COMPARISON BY CHANNEL


,formula,channel,Channel_Days,Zero_Injections,Median_Injection,Total_Injected_Events,Newly_Above_P95
0,A_P95_Addition,Email,9343,0,4.0,44336,7086
1,A_P95_Addition,HTTP,1533,0,6.0,11252,1275
2,A_P95_Addition,USB,3499,0,5.0,20035,2825
3,B_Natural_Addition,Email,9343,0,2.0,23880,3402
4,B_Natural_Addition,HTTP,1533,0,3.0,6793,702
5,B_Natural_Addition,USB,3499,0,2.0,11422,1437
6,C_P95_Headroom,Email,9343,2267,2.0,22513,3141
7,C_P95_Headroom,HTTP,1533,449,2.0,4788,563
8,C_P95_Headroom,USB,3499,1006,2.0,8650,992



D. FORMULA COMPARISON BY STEALTH LEVEL


,formula,effective_stealth,Channel_Days,Unique_Users,Zero_Injections,Median_Injection,Total_Injected_Events,Newly_Above_P95,Median_Final_To_P95
0,A_P95_Addition,High,4413,217,0,2.0,12137,1611,1.00
2,A_P95_Addition,Medium,5023,226,0,4.0,25952,4832,1.50
1,A_P95_Addition,Low,4939,227,0,6.0,37534,4743,2.00
3,B_Natural_Addition,High,4413,217,0,1.0,7236,986,0.75
5,B_Natural_Addition,Medium,5023,226,0,2.0,13759,1793,1.00
4,B_Natural_Addition,Low,4939,227,0,3.0,21100,2762,1.25
6,C_P95_Headroom,High,4413,217,3004,0.0,1633,0,0.50
8,C_P95_Headroom,Medium,5023,226,606,2.0,11924,0,1.00
7,C_P95_Headroom,Low,4939,227,112,4.0,22394,4696,1.50


... (2 outputs left out)

,user,day,channel,effective_stealth,formula,natural_count,candidate_p95,injected_count,simulated_final_count,final_to_p95_ratio
16563,CAM1323,2011-04-12,USB,Low,B_Natural_Addition,3,1.10,5,8,7.27
20606,HNC0074,2011-05-03,Email,Low,B_Natural_Addition,8,4.00,12,20,5.00
17093,CGM2036,2011-04-27,Email,Low,B_Natural_Addition,6,3.00,9,15,5.00
21281,JEM0546,2011-04-22,Email,Low,B_Natural_Addition,6,3.00,9,15,5.00
15365,ARC3034,2011-05-04,USB,Low,B_Natural_Addition,5,2.75,8,13,4.73
16572,CAM1323,2011-04-28,USB,Low,B_Natural_Addition,2,1.10,3,5,4.55
2188,CAM1323,2011-04-12,USB,Low,A_P95_Addition,3,1.10,2,5,4.55
14589,ADM0158,2011-02-16,Email,Low,B_Natural_Addition,7,4.00,11,18,4.50
14816,AHC1994,2011-02-25,Email,Low,B_Natural_Addition,7,4.00,11,18,4.50
19705,GEH3406,2011-03-31,USB,Low,B_Natural_Addition,7,4.00,11,18,4.50



F. ZERO-INJECTION DIAGNOSTIC


,formula,channel,Zero_Channel_Days
0,C_P95_Headroom,Email,2267
1,C_P95_Headroom,HTTP,449
2,C_P95_Headroom,USB,1006



STEP 3.4.3 — FORMULA COMPARISON COMPLETED
Injection formulas: EXPERIMENTAL
Chosen injection policy: NOT YET APPROVED
Injection quantities: NOT APPLIED
Historical P95: NOT TREATED AS A HARD SAFETY LIMIT
Counts do not establish bytes exfiltrated.
Original User-Day data: UNCHANGED
Final scenario assignments: UNCHANGED
No files were modified or exported.



### Step 3.4.4 — Pre-Injection Natural P95 Exceedance Audit

**Objective:** Identify users whose original activity already exceeds their personal channel-specific P95 during their scheduled attack windows, before any synthetic injection.

**Scope:**
- 670 users with validated schedules.
- All eligible days inside each user's selected attack window, including Off days.
- Email, USB, and HTTP, whether selected for injection or not.

**Validation Tasks:**
1. Compare original daily channel counts against historical P95.
2. Count affected users, user-days, and channel-day records.
3. Identify the exact user, date, channel, source dataset, and source column.
4. Measure exceedance magnitude and Final/Original-to-P95 ratio.
5. Distinguish channels selected for injection from other natural activity.
6. Compare exceedances across channels and stealth levels.
7. Identify missing historical baselines separately.

**Expected Outcome:** Quantify pre-existing behavioral exceedances and determine how they may affect stealth-aware injection planning.

**Important:** This is a read-only audit of original behavior. No injection formulas are applied, no synthetic activity is added, and no source files are modified.


In [27]:

# Stage 3 — Step 3.4.4: Pre-Injection Natural P95 Exceedance Audit
# Goal: Identify which scheduled users naturally exceed channel-specific
# P95, on which dates, in which channels, and from which source fields.
# Read-only: No injection or source-data modification.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 80)
print("STAGE 3 — STEP 3.4.4: NATURAL P95 EXCEEDANCE AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# 1. Validate the selected attack windows and baseline.
# ------------------------------------------------------------

assert validated_schedule_339["user"].nunique() == 670
assert len(validated_schedule_339) == 15010

assert not validated_schedule_339.duplicated(
    ["user", "day"]
).any()

assert user_baseline["user"].is_unique

# ------------------------------------------------------------
# 2. Read original activity from the validated schedule.
#    Include ALL three channels, including on Off days.
# ------------------------------------------------------------

source_mapping_344 = {
    "external_email_count": {
        "channel": "Email",
        "source_dataset": "email_daily_r62",
        "baseline_column": "email_count_p95"
    },
    "usb_copy_count": {
        "channel": "USB",
        "source_dataset": "usb_daily_r62",
        "baseline_column": "usb_copy_p95"
    },
    "www_upload_count": {
        "channel": "HTTP",
        "source_dataset": "http_daily_r62",
        "baseline_column": "http_upload_p95"
    }
}

source_columns_344 = list(source_mapping_344)

audit_natural_344 = validated_schedule_339[
    [
        "user", "scenario_id", "day",
        "planned_channels", "attack_channels"
    ] + source_columns_344
].copy()

audit_natural_344["day"] = pd.to_datetime(
    audit_natural_344["day"]
)

audit_natural_344 = audit_natural_344.melt(
    id_vars=[
        "user", "scenario_id", "day",
        "planned_channels", "attack_channels"
    ],
    value_vars=source_columns_344,
    var_name="source_column",
    value_name="natural_count"
)

audit_natural_344["channel"] = (
    audit_natural_344["source_column"]
    .map(lambda x: source_mapping_344[x]["channel"])
)

audit_natural_344["source_dataset"] = (
    audit_natural_344["source_column"]
    .map(lambda x: source_mapping_344[x]["source_dataset"])
)

# ------------------------------------------------------------
# 3. Attach the original historical P95 baselines.
# ------------------------------------------------------------

baseline_columns_344 = [
    "user",
    "email_count_p95",
    "usb_copy_p95",
    "http_upload_p95"
]

audit_natural_344 = audit_natural_344.merge(
    user_baseline[baseline_columns_344],
    on="user",
    how="left",
    validate="many_to_one"
)

audit_natural_344["baseline_p95"] = np.nan

for source_column, info in source_mapping_344.items():

    mask = audit_natural_344[
        "source_column"
    ].eq(source_column)

    audit_natural_344.loc[
        mask, "baseline_p95"
    ] = audit_natural_344.loc[
        mask, info["baseline_column"]
    ].to_numpy()

# ------------------------------------------------------------
# 4. Identify whether each channel-day was selected
#    for attack injection.
# ------------------------------------------------------------

def channel_selected_344(row):

    if row["planned_channels"] == 0:
        return False

    channels = str(row["attack_channels"]).split("|")

    return row["channel"] in channels

audit_natural_344["selected_for_injection"] = (
    audit_natural_344.apply(
        channel_selected_344,
        axis=1
    )
)

audit_natural_344["day_type"] = np.where(
    audit_natural_344["planned_channels"].eq(0),
    "Off Day",
    "Attack Day"
)

# ------------------------------------------------------------
# 5. Calculate pre-injection P95 deviation.
# ------------------------------------------------------------

audit_natural_344["natural_count"] = pd.to_numeric(
    audit_natural_344["natural_count"],
    errors="raise"
)

audit_natural_344["baseline_p95"] = pd.to_numeric(
    audit_natural_344["baseline_p95"],
    errors="coerce"
)

assert np.isfinite(
    audit_natural_344["natural_count"]
).all()

assert audit_natural_344["natural_count"].ge(0).all()

audit_natural_344["p95_available"] = (
    np.isfinite(audit_natural_344["baseline_p95"])
    & audit_natural_344["baseline_p95"].gt(0)
)

audit_natural_344["naturally_above_p95"] = (
    audit_natural_344["p95_available"]
    & (
        audit_natural_344["natural_count"]
        > audit_natural_344["baseline_p95"]
    )
)

audit_natural_344["natural_to_p95_ratio"] = (
    audit_natural_344["natural_count"]
    / audit_natural_344["baseline_p95"]
)

audit_natural_344["excess_events"] = (
    audit_natural_344["natural_count"]
    - audit_natural_344["baseline_p95"]
)

audit_natural_344["excess_percentage"] = (
    (
        audit_natural_344["natural_to_p95_ratio"] - 1
    ) * 100
)

# ------------------------------------------------------------
# 6. Independently validate coverage and planned selection.
# ------------------------------------------------------------

assert len(audit_natural_344) == 15010 * 3

assert not audit_natural_344.duplicated(
    ["user", "day", "channel"]
).any()

assert int(
    audit_natural_344["selected_for_injection"].sum()
) == 14375

assert audit_natural_344.loc[
    audit_natural_344["selected_for_injection"],
    "p95_available"
].all()

# ------------------------------------------------------------
# 7. Keep all original exceedances for detailed inspection.
# ------------------------------------------------------------

exceedances_344 = audit_natural_344.loc[
    audit_natural_344["naturally_above_p95"]
].copy()

exceedances_344 = exceedances_344.sort_values(
    ["user", "day", "channel"]
).reset_index(drop=True)

planned_exceedances_344 = exceedances_344.loc[
    exceedances_344["selected_for_injection"]
].copy()

# ------------------------------------------------------------
# 8. Overall population summary.
# ------------------------------------------------------------

print("\nA. OVERALL PRE-INJECTION EXCEEDANCE SUMMARY")

summary_344 = pd.DataFrame({
    "Metric": [
        "Scheduled Users",
        "Scheduled User-Days",
        "All Channel-Day Records",
        "Records With Valid P95",
        "Records Without Valid P95",
        "Users Naturally Above P95",
        "User-Days With Any Natural Exceedance",
        "Channel-Days Naturally Above P95",
        "Planned Injection Channel-Days",
        "Planned Channel-Days Already Above P95",
        "Users Affected On Planned Injection Channels"
    ],
    "Value": [
        audit_natural_344["user"].nunique(),
        len(validated_schedule_339),
        len(audit_natural_344),
        int(audit_natural_344["p95_available"].sum()),
        int((~audit_natural_344["p95_available"]).sum()),
        exceedances_344["user"].nunique(),
        len(exceedances_344[
            ["user", "day"]
        ].drop_duplicates()),
        len(exceedances_344),
        int(audit_natural_344[
            "selected_for_injection"
        ].sum()),
        len(planned_exceedances_344),
        planned_exceedances_344["user"].nunique()
    ]
})

display(summary_344)

# ------------------------------------------------------------
# 9. Exceedance summary by channel and source.
# ------------------------------------------------------------

print("\nB. EXCEEDANCES BY CHANNEL AND SOURCE")

channel_summary_344 = (
    audit_natural_344
    .groupby(
        ["channel", "source_dataset", "source_column"]
    )
    .agg(
        Channel_Days=("user", "size"),
        Valid_P95=("p95_available", "sum"),
        Missing_P95=(
            "p95_available",
            lambda s: int((~s).sum())
        ),
        Naturally_Above_P95=("naturally_above_p95", "sum")
    )
    .reset_index()
)

users_by_channel_344 = (
    exceedances_344
    .groupby("channel")["user"]
    .nunique()
    .rename("Affected_Users")
)

channel_summary_344 = channel_summary_344.merge(
    users_by_channel_344,
    on="channel",
    how="left"
)

channel_summary_344["Affected_Users"] = (
    channel_summary_344["Affected_Users"]
    .fillna(0)
    .astype(int)
)

channel_summary_344["Exceedance_Rate_%"] = (
    100
    * channel_summary_344["Naturally_Above_P95"]
    / channel_summary_344["Valid_P95"].replace(0, np.nan)
).round(2)

display(
    channel_summary_344
    .set_index("channel")
    .reindex(["Email", "USB", "HTTP"])
    .reset_index()
)

# ------------------------------------------------------------
# 10. Distinguish planned injection channels, other
#     channels, and Off days.
# ------------------------------------------------------------

print("\nC. EXCEEDANCE LOCATION RELATIVE TO ATTACK SCHEDULE")

audit_natural_344["schedule_context"] = np.select(
    [
        audit_natural_344["selected_for_injection"],
        audit_natural_344["day_type"].eq("Off Day")
    ],
    [
        "Selected Injection Channel",
        "Off Day"
    ],
    default="Unselected Channel On Attack Day"
)

context_summary_344 = (
    audit_natural_344
    .groupby("schedule_context")
    .agg(
        Channel_Days=("user", "size"),
        Valid_P95=("p95_available", "sum"),
        Natural_Exceedances=("naturally_above_p95", "sum")
    )
    .reset_index()
)

display(context_summary_344)

# ------------------------------------------------------------
# 11. Rank users with pre-existing exceedances.
# ------------------------------------------------------------

print("\nD. USERS WITH THE MOST NATURAL EXCEEDANCES")

user_summary_344 = (
    exceedances_344
    .groupby("user")
    .agg(
        Exceeding_Channel_Days=("channel", "size"),
        Exceeding_Dates=("day", "nunique"),
        Channels_Affected=("channel", "nunique"),
        Maximum_Natural_P95_Ratio=(
            "natural_to_p95_ratio", "max"
        )
    )
    .sort_values(
        "Exceeding_Channel_Days",
        ascending=False
    )
    .reset_index()
)

display(user_summary_344.head(30).round(2))

# ------------------------------------------------------------
# 12. Show exact user, date, channel, source, and deviation.
# ------------------------------------------------------------

print("\nE. EXACT NATURAL EXCEEDANCE RECORDS")

detail_columns_344 = [
    "user",
    "scenario_id",
    "day",
    "channel",
    "source_dataset",
    "source_column",
    "day_type",
    "selected_for_injection",
    "natural_count",
    "baseline_p95",
    "excess_events",
    "natural_to_p95_ratio",
    "excess_percentage"
]

print(
    "Total detailed exceedance records:",
    len(exceedances_344)
)

print("\nFirst 60 records, sorted by user and date:")

display(
    exceedances_344[
        detail_columns_344
    ].head(60).round(2)
)

print("\nTop 30 largest relative exceedances:")

display(
    exceedances_344[
        detail_columns_344
    ]
    .sort_values(
        "natural_to_p95_ratio",
        ascending=False
    )
    .head(30)
    .round(2)
)

# ------------------------------------------------------------
# 13. Focus on records scheduled for injection.
# ------------------------------------------------------------

print("\nF. PRE-EXISTING EXCEEDANCES ON INJECTION CHANNELS")

print(
    "Affected Users:",
    planned_exceedances_344["user"].nunique()
)

print(
    "Affected Planned Channel-Days:",
    len(planned_exceedances_344)
)

display(
    planned_exceedances_344[
        detail_columns_344
    ].head(50).round(2)
)

# ------------------------------------------------------------
# 14. Final status.
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 3.4.4 — NATURAL P95 AUDIT COMPLETED")
print("=" * 80)

print("Original natural activity: ANALYZED")
print("Synthetic injection: NOT STARTED")
print("Injection policy: NOT APPROVED")
print("No source files were modified or exported.")


STAGE 3 — STEP 3.4.4: NATURAL P95 EXCEEDANCE AUDIT

A. OVERALL PRE-INJECTION EXCEEDANCE SUMMARY


,Metric,Value
0,Scheduled Users,670
1,Scheduled User-Days,15010
2,All Channel-Day Records,45030
3,Records With Valid P95,37560
4,Records Without Valid P95,7470
5,Users Naturally Above P95,371
6,User-Days With Any Natural Exceedance,781
7,Channel-Days Naturally Above P95,792
8,Planned Injection Channel-Days,14375
9,Planned Channel-Days Already Above P95,544



B. EXCEEDANCES BY CHANNEL AND SOURCE


,channel,source_dataset,source_column,Channel_Days,Valid_P95,Missing_P95,Naturally_Above_P95,Affected_Users,Exceedance_Rate_%
0,Email,email_daily_r62,external_email_count,15010,15010,0,422,259,2.81
1,USB,usb_daily_r62,usb_copy_count,15010,12655,2355,248,169,1.96
2,HTTP,http_daily_r62,www_upload_count,15010,9895,5115,122,93,1.23



C. EXCEEDANCE LOCATION RELATIVE TO ATTACK SCHEDULE


,schedule_context,Channel_Days,Valid_P95,Natural_Exceedances
0,Off Day,7122,6097,85
1,Selected Injection Channel,14375,14375,544
2,Unselected Channel On Attack Day,23533,17088,163


... (8 outputs left out)


F. PRE-EXISTING EXCEEDANCES ON INJECTION CHANNELS
Affected Users: 306
Affected Planned Channel-Days: 544


C:\Users\hp\AppData\Local\Temp\ipykernel_14440\4189539059.py:448: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].head(50).round(2)


,user,scenario_id,day,channel,source_dataset,source_column,day_type,selected_for_injection,natural_count,baseline_p95,excess_events,natural_to_p95_ratio,excess_percentage
0,ABD3426,SCN_0402,2011-04-06,USB,usb_daily_r62,usb_copy_count,Attack Day,True,10.0,9.00,1.00,1.11,11.11
1,ABD3426,SCN_0402,2011-04-28,Email,email_daily_r62,external_email_count,Attack Day,True,7.0,6.00,1.00,1.17,16.67
9,ADM0158,SCN_0042,2011-02-16,Email,email_daily_r62,external_email_count,Attack Day,True,7.0,4.00,3.00,1.75,75.00
10,AEB3249,SCN_0207,2011-03-03,Email,email_daily_r62,external_email_count,Attack Day,True,5.0,4.00,1.00,1.25,25.00
16,AFP2709,SCN_0044,2011-05-13,USB,usb_daily_r62,usb_copy_count,Attack Day,True,4.0,3.00,1.00,1.33,33.33
17,AHC1994,SCN_0150,2011-02-25,Email,email_daily_r62,external_email_count,Attack Day,True,7.0,4.00,3.00,1.75,75.00
18,AHC1994,SCN_0150,2011-03-08,Email,email_daily_r62,external_email_count,Attack Day,True,6.0,4.00,2.00,1.50,50.00
19,AHC1994,SCN_0150,2011-03-18,Email,email_daily_r62,external_email_count,Attack Day,True,5.0,4.00,1.00,1.25,25.00
20,AIP1614,SCN_0487,2011-01-25,USB,usb_daily_r62,usb_copy_count,Attack Day,True,11.0,9.00,2.00,1.22,22.22
23,AKB2312,SCN_0040,2011-05-02,USB,usb_daily_r62,usb_copy_count,Attack Day,True,11.0,8.00,3.00,1.38,37.50



STEP 3.4.4 — NATURAL P95 AUDIT COMPLETED
Original natural activity: ANALYZED
Synthetic injection: NOT STARTED
Injection policy: NOT APPROVED
No source files were modified or exported.



### Step 3.4.5 — High Stealth Injection Feasibility Audit

**Objective:** Determine how many scheduled High Stealth channel-days can receive positive synthetic activity while remaining at or below the user's historical P95.

**Validation Tasks:**
1. Separate natural activity below, equal to, and above P95.
2. Calculate available integer headroom below P95.
3. Identify channel-days where any positive injection is impossible without exceeding P95.
4. Test whether the original Formula A increment (ceil(P95 × 0.5)) fits within the available headroom.
5. Report affected users, channels, and exact dates.
6. Quantify conflicts requiring possible quantity adjustments or schedule reconsideration.

**Important:** No injection policy is approved. No dates, channels, or assignments are changed.

**Expected Outcome:** Establish the actual feasibility constraints for High Stealth before deciding how to handle pre-existing elevated natural activity.


In [28]:

# Stage 3 — Step 3.4.5: High Stealth Injection Feasibility
# Goal: Identify High Stealth channel-days that can receive
# positive injection while keeping final activity <= P95.
# Read-only: No source data or schedules are modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.4.5: HIGH STEALTH FEASIBILITY")
print("=" * 75)

# ------------------------------------------------------------
# 1. Validate the planned injection population.
# ------------------------------------------------------------

assert len(quantity_base_343) == 14375
assert quantity_base_343["user"].nunique() == 670

assert not quantity_base_343.duplicated(
    ["user", "day", "channel"]
).any()

natural_exceedances_345 = (
    quantity_base_343["natural_count"]
    > quantity_base_343["candidate_p95"]
)

assert int(natural_exceedances_345.sum()) == 544, (
    "Natural exceedance count differs from Step 3.4.4."
)

# ------------------------------------------------------------
# 2. Isolate only scheduled High Stealth channel-days.
# ------------------------------------------------------------

high_feasibility_345 = quantity_base_343.loc[
    quantity_base_343["effective_stealth"].eq("High")
].copy()

assert len(high_feasibility_345) == 4413
assert high_feasibility_345["user"].nunique() == 217

natural_345 = high_feasibility_345[
    "natural_count"
].to_numpy(dtype=float)

p95_345 = high_feasibility_345[
    "candidate_p95"
].to_numpy(dtype=float)

assert np.isfinite(natural_345).all()
assert np.isfinite(p95_345).all()
assert np.all(natural_345 > 0)
assert np.all(p95_345 > 0)

# ------------------------------------------------------------
# 3. Classify natural activity relative to P95.
# ------------------------------------------------------------

tolerance_345 = 1e-9

high_feasibility_345["natural_below_p95"] = (
    natural_345 < p95_345 - tolerance_345
)

high_feasibility_345["natural_equal_p95"] = (
    np.abs(natural_345 - p95_345) <= tolerance_345
)

high_feasibility_345["natural_above_p95"] = (
    natural_345 > p95_345 + tolerance_345
)

# ------------------------------------------------------------
# 4. Calculate maximum positive INTEGER headroom.
# ------------------------------------------------------------

headroom_345 = np.maximum(
    0,
    np.floor(
        p95_345 - natural_345 + tolerance_345
    )
).astype(np.int64)

high_feasibility_345["integer_headroom"] = headroom_345

high_feasibility_345["can_inject_at_least_one"] = (
    headroom_345 >= 1
)

# ------------------------------------------------------------
# 5. Test the original Formula A for High Stealth.
# Formula A = max(1, ceil(P95 * 0.5)).
# ------------------------------------------------------------

a_increment_345 = np.maximum(
    1,
    np.ceil(p95_345 * 0.5)
).astype(np.int64)

high_feasibility_345["formula_A_increment"] = (
    a_increment_345
)

high_feasibility_345["formula_A_final"] = (
    natural_345 + a_increment_345
)

high_feasibility_345["formula_A_fits_p95"] = (
    high_feasibility_345["formula_A_final"]
    <= p95_345 + tolerance_345
)

# ------------------------------------------------------------
# 6. Classify reasons for failure.
# ------------------------------------------------------------

high_feasibility_345["feasibility_status"] = np.select(
    [
        high_feasibility_345["natural_above_p95"],
        high_feasibility_345["natural_equal_p95"],
        ~high_feasibility_345["can_inject_at_least_one"],
        ~high_feasibility_345["formula_A_fits_p95"]
    ],
    [
        "Natural Already Above P95",
        "Natural Equal To P95",
        "No Integer Headroom",
        "Formula A Exceeds Available Headroom"
    ],
    default="Formula A Fits"
)

# ------------------------------------------------------------
# 7. Overall summary.
# ------------------------------------------------------------

print("\nA. HIGH STEALTH FEASIBILITY SUMMARY")

summary_345 = pd.DataFrame({
    "Metric": [
        "High Stealth Users",
        "High Stealth Channel-Days",
        "Natural Below P95",
        "Natural Equal To P95",
        "Natural Above P95",
        "Can Inject At Least One Within P95",
        "Cannot Inject Positively Within P95",
        "Formula A Fits Within P95",
        "Formula A Does Not Fit Within P95"
    ],
    "Value": [
        high_feasibility_345["user"].nunique(),
        len(high_feasibility_345),
        int(high_feasibility_345["natural_below_p95"].sum()),
        int(high_feasibility_345["natural_equal_p95"].sum()),
        int(high_feasibility_345["natural_above_p95"].sum()),
        int(high_feasibility_345["can_inject_at_least_one"].sum()),
        int((~high_feasibility_345["can_inject_at_least_one"]).sum()),
        int(high_feasibility_345["formula_A_fits_p95"].sum()),
        int((~high_feasibility_345["formula_A_fits_p95"]).sum())
    ]
})

display(summary_345)

# ------------------------------------------------------------
# 8. Reasons for Formula A incompatibility.
# ------------------------------------------------------------

print("\nB. FEASIBILITY STATUS")

status_345 = (
    high_feasibility_345["feasibility_status"]
    .value_counts()
    .rename_axis("Status")
    .reset_index(name="Channel_Days")
)

status_345["Percentage"] = (
    status_345["Channel_Days"]
    / len(high_feasibility_345)
    * 100
).round(2)

display(status_345)

# ------------------------------------------------------------
# 9. Group failure cases by channel.
# ------------------------------------------------------------

print("\nC. HIGH STEALTH CONSTRAINTS BY CHANNEL")

channel_audit_345 = (
    high_feasibility_345
    .groupby("channel")
    .agg(
        Channel_Days=("user", "size"),
        Users=("user", "nunique"),
        Natural_Above_P95=("natural_above_p95", "sum"),
        No_Positive_Headroom=(
            "can_inject_at_least_one",
            lambda s: int((~s).sum())
        ),
        Formula_A_Fits=("formula_A_fits_p95", "sum"),
        Formula_A_Fails=(
            "formula_A_fits_p95",
            lambda s: int((~s).sum())
        )
    )
    .reindex(["Email", "USB", "HTTP"])
    .reset_index()
)

display(channel_audit_345)

# ------------------------------------------------------------
# 10. Identify affected users and exact channel-days.
# ------------------------------------------------------------

print("\nD. USERS AFFECTED BY HIGH STEALTH CONSTRAINTS")

failed_345 = high_feasibility_345.loc[
    ~high_feasibility_345["formula_A_fits_p95"]
].copy()

print(
    "Users for whom Formula A fails on at least one channel-day:",
    failed_345["user"].nunique()
)

print(
    "Users without positive headroom on at least one channel-day:",
    high_feasibility_345.loc[
        ~high_feasibility_345["can_inject_at_least_one"],
        "user"
    ].nunique()
)

print("\nFirst 40 incompatible channel-days:")

display(
    failed_345[
        [
            "user", "scenario_id", "day", "channel",
            "natural_count", "candidate_p95",
            "integer_headroom", "formula_A_increment",
            "formula_A_final", "feasibility_status"
        ]
    ]
    .sort_values(["user", "day", "channel"])
    .head(40)
)

# ------------------------------------------------------------
# 11. Verify the feasibility classification.
# ------------------------------------------------------------

assert (
    high_feasibility_345[
        [
            "natural_below_p95",
            "natural_equal_p95",
            "natural_above_p95"
        ]
    ].sum(axis=1) == 1
).all()

assert (
    high_feasibility_345["formula_A_fits_p95"]
    <= high_feasibility_345["can_inject_at_least_one"]
).all()

assert (
    high_feasibility_345["formula_A_fits_p95"].sum()
    + (~high_feasibility_345["formula_A_fits_p95"]).sum()
    == len(high_feasibility_345)
)

print("\n" + "=" * 75)
print("STEP 3.4.5 — HIGH STEALTH AUDIT COMPLETED")
print("=" * 75)

print("Schedule changes: NONE")
print("Injection formula changes: NONE")
print("Synthetic injection: NOT STARTED")
print("No source files modified or exported.")


STAGE 3 — STEP 3.4.5: HIGH STEALTH FEASIBILITY

A. HIGH STEALTH FEASIBILITY SUMMARY


,Metric,Value
0,High Stealth Users,217
1,High Stealth Channel-Days,4413
2,Natural Below P95,3934
3,Natural Equal To P95,322
4,Natural Above P95,157
5,Can Inject At Least One Within P95,3887
6,Cannot Inject Positively Within P95,526
7,Formula A Fits Within P95,2645
8,Formula A Does Not Fit Within P95,1768



B. FEASIBILITY STATUS


,Status,Channel_Days,Percentage
0,Formula A Fits,2645,59.94
1,Formula A Exceeds Available Headroom,1242,28.14
2,Natural Equal To P95,322,7.30
3,Natural Already Above P95,157,3.56
4,No Integer Headroom,47,1.07



C. HIGH STEALTH CONSTRAINTS BY CHANNEL


,channel,Channel_Days,Users,Natural_Above_P95,No_Positive_Headroom,Formula_A_Fits,Formula_A_Fails
0,Email,2949,217,87,347,1914,1035
1,USB,1038,158,45,123,540,498
2,HTTP,426,106,25,56,191,235



D. USERS AFFECTED BY HIGH STEALTH CONSTRAINTS
Users for whom Formula A fails on at least one channel-day: 212
Users without positive headroom on at least one channel-day: 172

First 40 incompatible channel-days:


,user,scenario_id,day,channel,natural_count,candidate_p95,integer_headroom,formula_A_increment,formula_A_final,feasibility_status
3,AAF0819,SCN_0007,2011-03-08,Email,3,4.0,1,2,5.0,Formula A Exceeds Available Headroom
5,AAP0352,SCN_0016,2011-04-06,USB,2,2.1,0,2,4.0,No Integer Headroom
6,AAP0352,SCN_0016,2011-04-07,Email,3,4.0,1,2,5.0,Formula A Exceeds Available Headroom
8,AAP0352,SCN_0016,2011-04-11,Email,3,4.0,1,2,5.0,Formula A Exceeds Available Headroom
9,AAP0352,SCN_0016,2011-04-11,USB,1,2.1,1,2,3.0,Formula A Exceeds Available Headroom
13,AAP0352,SCN_0016,2011-04-15,USB,2,2.1,0,2,4.0,No Integer Headroom
21,AAP1919,SCN_0040,2011-02-04,Email,3,5.0,2,3,6.0,Formula A Exceeds Available Headroom
25,AAP1919,SCN_0040,2011-02-10,Email,3,5.0,2,3,6.0,Formula A Exceeds Available Headroom
28,AAP1919,SCN_0040,2011-02-15,Email,3,5.0,2,3,6.0,Formula A Exceeds Available Headroom
157,ACS0128,SCN_0001,2011-01-05,Email,3,5.0,2,3,6.0,Formula A Exceeds Available Headroom



STEP 3.4.5 — HIGH STEALTH AUDIT COMPLETED
Schedule changes: NONE
Injection formula changes: NONE
Synthetic injection: NOT STARTED
No source files modified or exported.



### Step 3.4.6 — Same-Day Channel Replacement Feasibility

**Objective:** Investigate whether High Stealth channel-days without positive P95 headroom have alternative naturally active channels on the same scheduled day.

**Validation Tasks:**
1. Identify attack days containing at least one selected channel without positive integer headroom.
2. Evaluate all three naturally active channels against their historical P95 values.
3. Check whether the number of feasible channels can satisfy the scheduled channel count.
4. Determine whether the original primary channel remains feasible.
5. Report cases with potential same-day alternatives versus cases requiring further scheduling analysis.

**Important:** Same-day channel capacity is only a necessary condition. It does not prove that primary-channel switches, scenario structural constraints, or other scheduling requirements remain satisfied.

**Expected Outcome:** Quantify potentially recoverable High Stealth attack days without modifying the validated schedules.

**Read-only:** No injection, reassignment, or file export.


In [29]:

# Stage 3 — Step 3.4.6: Same-Day Channel Replacement Feasibility
# Goal: Assess whether High Stealth days with zero injection headroom
# have other naturally active channels that can accept positive injection.
# Read-only: No schedule changes, injection, or file exports.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.4.6: SAME-DAY CHANNEL FEASIBILITY")
print("=" * 75)

# ------------------------------------------------------------
# 1. Identify high-stealth days with zero headroom in
#    at least one selected attack channel.
# ------------------------------------------------------------

high_conflicts_346 = high_feasibility_345.loc[
    ~high_feasibility_345["can_inject_at_least_one"]
].copy()

assert len(high_conflicts_346) == 526

high_conflicts_346["day"] = pd.to_datetime(
    high_conflicts_346["day"]
).dt.normalize()

conflict_days_346 = (
    high_conflicts_346[["user", "day"]]
    .drop_duplicates()
)

# ------------------------------------------------------------
# 2. Load all three original natural channel counts
#    for the affected scheduled attack days.
# ------------------------------------------------------------

source_day_346 = validated_schedule_339.copy()

source_day_346["day"] = pd.to_datetime(
    source_day_346["day"]
).dt.normalize()

affected_days_346 = source_day_346.merge(
    conflict_days_346,
    on=["user", "day"],
    how="inner",
    validate="one_to_one"
)

assert len(affected_days_346) == len(conflict_days_346)

assert affected_days_346["planned_channels"].gt(0).all()

baseline_346 = user_baseline[
    [
        "user",
        "email_count_p95",
        "usb_copy_p95",
        "http_upload_p95"
    ]
].copy()

affected_days_346 = affected_days_346.merge(
    baseline_346,
    on="user",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 3. Evaluate all naturally available channels.
#    A channel must have at least one natural event and
#    enough P95 headroom for one additional integer event.
# ------------------------------------------------------------

channel_definitions_346 = {
    "Email": ("external_email_count", "email_count_p95"),
    "USB": ("usb_copy_count", "usb_copy_p95"),
    "HTTP": ("www_upload_count", "http_upload_p95")
}

for channel, (natural_col, p95_col) in channel_definitions_346.items():

    natural = pd.to_numeric(
        affected_days_346[natural_col],
        errors="raise"
    )

    p95 = pd.to_numeric(
        affected_days_346[p95_col],
        errors="coerce"
    )

    affected_days_346[
        f"{channel}_feasible"
    ] = (
        natural.gt(0)
        & p95.notna()
        & np.isfinite(p95)
        & (natural + 1 <= p95 + 1e-9)
    )

feasible_cols_346 = [
    f"{channel}_feasible"
    for channel in channel_definitions_346
]

affected_days_346["feasible_channel_count"] = (
    affected_days_346[feasible_cols_346]
    .sum(axis=1)
)

# ------------------------------------------------------------
# 4. Test the current selected channel set against all
#    feasible channels. Do not change any selected channels.
# ------------------------------------------------------------

def assess_day_346(row):

    selected = [
        c.strip()
        for c in str(row["attack_channels"]).split("|")
        if c.strip() in channel_definitions_346
    ]

    feasible = [
        c for c in channel_definitions_346
        if bool(row[f"{c}_feasible"])
    ]

    assert len(selected) == int(row["planned_channels"]), (
        "Selected channel count does not match schedule."
    )

    blocked = [c for c in selected if c not in feasible]
    alternatives = [c for c in feasible if c not in selected]

    return pd.Series({
        "selected_channels": "|".join(selected),
        "feasible_channels": "|".join(feasible),
        "blocked_selected_channels": "|".join(blocked),
        "alternative_channels": "|".join(alternatives),
        "blocked_count": len(blocked),
        "original_primary_feasible": (
            row["primary_channel"] in feasible
        )
    })

assessment_346 = affected_days_346.apply(
    assess_day_346,
    axis=1
)

affected_days_346 = pd.concat(
    [
        affected_days_346.reset_index(drop=True),
        assessment_346.reset_index(drop=True)
    ],
    axis=1
)

assert affected_days_346["blocked_count"].ge(1).all()

# ------------------------------------------------------------
# 5. Preliminary same-day feasibility.
#    This does NOT validate primary switches or other
#    attack scenario constraints.
# ------------------------------------------------------------

affected_days_346["potential_same_day_replacement"] = (
    affected_days_346["feasible_channel_count"]
    >= affected_days_346["planned_channels"]
)

affected_days_346["replacement_keeps_primary"] = (
    affected_days_346["potential_same_day_replacement"]
    & affected_days_346["original_primary_feasible"]
)

affected_days_346["assessment_status"] = np.select(
    [
        affected_days_346["replacement_keeps_primary"],
        affected_days_346["potential_same_day_replacement"]
    ],
    [
        "Candidate Replacement - Primary Preserved",
        "Candidate Replacement - Primary Change Needed"
    ],
    default="Insufficient Same-Day Channel Capacity"
)

# ------------------------------------------------------------
# 6. Summary.
# ------------------------------------------------------------

print("\nA. SAME-DAY RECOVERY SCREEN")

summary_346 = pd.DataFrame({
    "Metric": [
        "High Users",
        "Selected Channel-Days Without Headroom",
        "Affected Attack Days",
        "Affected Users",
        "Days With Potential Same-Day Replacement",
        "Days With Original Primary Preserved",
        "Days Requiring Primary Change",
        "Days With Insufficient Same-Day Capacity"
    ],
    "Value": [
        high_feasibility_345["user"].nunique(),
        len(high_conflicts_346),
        len(affected_days_346),
        affected_days_346["user"].nunique(),
        int(affected_days_346[
            "potential_same_day_replacement"
        ].sum()),
        int(affected_days_346[
            "replacement_keeps_primary"
        ].sum()),
        int((
            affected_days_346["potential_same_day_replacement"]
            & ~affected_days_346["original_primary_feasible"]
        ).sum()),
        int((
            ~affected_days_346["potential_same_day_replacement"]
        ).sum())
    ]
})

display(summary_346)

# ------------------------------------------------------------
# 7. Count outcomes by required channel count.
# ------------------------------------------------------------

print("\nB. FEASIBILITY BY REQUIRED CHANNEL COUNT")

capacity_346 = (
    affected_days_346
    .groupby(
        ["planned_channels", "assessment_status"]
    )
    .agg(
        Attack_Days=("user", "size"),
        Affected_Users=("user", "nunique")
    )
    .reset_index()
)

display(capacity_346)

# ------------------------------------------------------------
# 8. Show exact affected users, days, selected channels,
#    available replacements, and original primary.
# ------------------------------------------------------------

print("\nC. SAME-DAY REPLACEMENT DETAILS")

detail_cols_346 = [
    "user",
    "scenario_id",
    "day",
    "planned_channels",
    "primary_channel",
    "selected_channels",
    "blocked_selected_channels",
    "feasible_channels",
    "alternative_channels",
    "feasible_channel_count",
    "assessment_status"
]

display(
    affected_days_346[detail_cols_346]
    .sort_values(["user", "day"])
    .head(50)
)

# ------------------------------------------------------------
# 9. Confirm that no synthetic injection was applied.
# ------------------------------------------------------------

assert (
    affected_days_346[
        "replacement_keeps_primary"
    ].sum()
    <= affected_days_346[
        "potential_same_day_replacement"
    ].sum()
)

print("\n" + "=" * 75)
print("STEP 3.4.6 — SCREENING COMPLETED")
print("=" * 75)

print("Same-day alternatives: SCREENED ONLY")
print("Primary channel switching: NOT REVALIDATED")
print("Attack schedules: UNCHANGED")
print("Injection quantities: NOT APPLIED")
print("Source datasets: UNCHANGED")
print("No files were modified or exported.")


STAGE 3 — STEP 3.4.6: SAME-DAY CHANNEL FEASIBILITY

A. SAME-DAY RECOVERY SCREEN


,Metric,Value
0,High Users,217
1,Selected Channel-Days Without Headroom,526
2,Affected Attack Days,514
3,Affected Users,172
4,Days With Potential Same-Day Replacement,131
5,Days With Original Primary Preserved,1
6,Days Requiring Primary Change,130
7,Days With Insufficient Same-Day Capacity,383



B. FEASIBILITY BY REQUIRED CHANNEL COUNT


,planned_channels,assessment_status,Attack_Days,Affected_Users
0,1,Candidate Replacement - Primary Change Needed,129,92
1,1,Insufficient Same-Day Channel Capacity,267,130
2,2,Candidate Replacement - Primary Change Needed,1,1
3,2,Candidate Replacement - Primary Preserved,1,1
4,2,Insufficient Same-Day Channel Capacity,97,76
5,3,Insufficient Same-Day Channel Capacity,19,17



C. SAME-DAY REPLACEMENT DETAILS


,user,scenario_id,day,planned_channels,primary_channel,selected_channels,blocked_selected_channels,feasible_channels,alternative_channels,feasible_channel_count,assessment_status
0,AAP0352,SCN_0016,2011-04-06,1,USB,USB,USB,,,0,Insufficient Same-Day Channel Capacity
1,AAP0352,SCN_0016,2011-04-15,1,USB,USB,USB,Email,Email,1,Candidate Replacement - Primary Change Needed
2,AED1421,SCN_0043,2011-01-05,2,Email,Email|USB,Email,USB,,1,Insufficient Same-Day Channel Capacity
3,AED1421,SCN_0043,2011-01-18,1,USB,USB,USB,,,0,Insufficient Same-Day Channel Capacity
4,AED1421,SCN_0043,2011-01-21,1,Email,Email,Email,,,0,Insufficient Same-Day Channel Capacity
5,AED1421,SCN_0043,2011-01-27,1,Email,Email,Email,USB,USB,1,Candidate Replacement - Primary Change Needed
6,AFB0904,SCN_0112,2011-03-21,1,USB,USB,USB,,,0,Insufficient Same-Day Channel Capacity
7,AFB0904,SCN_0112,2011-03-29,2,Email,Email|USB,Email,USB,,1,Insufficient Same-Day Channel Capacity
8,AFB0904,SCN_0112,2011-04-07,2,Email,Email|USB,Email,USB,,1,Insufficient Same-Day Channel Capacity
9,AFB0904,SCN_0112,2011-04-18,2,Email,Email|USB,Email,USB,,1,Insufficient Same-Day Channel Capacity



STEP 3.4.6 — SCREENING COMPLETED
Same-day alternatives: SCREENED ONLY
Primary channel switching: NOT REVALIDATED
Attack schedules: UNCHANGED
Injection quantities: NOT APPLIED
Source datasets: UNCHANGED
No files were modified or exported.


طفشت من محاولة اختيار افضل حقن لذلك بنسوي المعادلات الثلاث ثم نقارن نتايجها



### Step 3.4.7 — Controlled Experimental Injection Sandbox

**Objective:** Apply the three candidate injection formulas to independent, count-only experimental views without changing the original CERT r6.2 User-Day dataset.

**Candidate Formulas:**
- A: P95-Based Addition
- B: Natural-Activity Addition
- C: P95 Headroom (diagnostic candidate with known zero-injection cases)

**Validation Tasks:**
1. Preserve the same 670 users and validated attack schedule.
2. Apply each formula independently to Email, USB, and HTTP counts.
3. Verify that the original daily counts match the previously audited natural activity.
4. Ensure no channel with zero natural activity becomes active through injection.
5. Verify that all resulting counts equal natural activity plus synthetic increments.
6. Verify that days and channels outside the injection plan remain unchanged.
7. Report injection quantities, changed records, and zero-injection cases.

**Scope:** Count-only experimental views. Related event totals and behavioral features are not recalculated in this step.

**Expected Outcome:** Three isolated experimental datasets, ready for subsequent consistency checks and feature engineering.

**Important:** No original files, scenario assignments, or validated schedules will be modified or exported.


In [31]:

# Stage 3 — Step 3.4.7: Controlled Experimental Injection Sandbox
# Goal: Apply candidate A/B/C increments to independent count-only
# User-Day views, verifying exact source values and all injection deltas.
# Corrected: Compare user-day keys independently of Pandas row indices.
# Read-only: No original files or schedules are modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.4.7: EXPERIMENTAL INJECTION")
print("=" * 75)

# ------------------------------------------------------------
# 1. Define source columns and verify experimental inputs.
# ------------------------------------------------------------

channel_columns_347 = {
    "Email": "external_email_count",
    "USB": "usb_copy_count",
    "HTTP": "www_upload_count"
}

formulas_347 = [
    "A_P95_Addition",
    "B_Natural_Addition",
    "C_P95_Headroom"
]

assert set(
    quantity_experiment_343["formula"].unique()
) == set(formulas_347)

assert quantity_base_343["user"].nunique() == 670
assert len(quantity_base_343) == 14375

key_columns_347 = ["user", "day"]

# ------------------------------------------------------------
# 2. Prepare an independent original count-only view.
#    Include all available days for the scheduled users.
# ------------------------------------------------------------

original_counts_347 = user_day.loc[
    user_day["user"].isin(quantity_base_343["user"]),
    key_columns_347 + list(channel_columns_347.values())
].copy()

original_counts_347["day"] = pd.to_datetime(
    original_counts_347["day"]
).dt.normalize()

# Normalize row indices and establish deterministic ordering.
original_counts_347 = (
    original_counts_347
    .sort_values(key_columns_347)
    .reset_index(drop=True)
)

assert not original_counts_347.duplicated(
    key_columns_347
).any()

assert original_counts_347["user"].nunique() == 670

for col in channel_columns_347.values():

    values = pd.to_numeric(
        original_counts_347[col],
        errors="raise"
    ).to_numpy(dtype=float)

    assert np.isfinite(values).all(), (
        f"Invalid or missing values in {col}"
    )

    assert np.all(values >= 0), (
        f"Negative values in {col}"
    )

    assert np.allclose(
        values,
        np.rint(values),
        rtol=0,
        atol=1e-9
    ), f"Non-integer activity counts in {col}"

# ------------------------------------------------------------
# 3. Verify that selected natural counts match the original
#    User-Day data before applying any synthetic increments.
# ------------------------------------------------------------

plan_reference_347 = quantity_base_343[
    ["user", "day", "channel", "natural_count"]
].copy()

plan_reference_347["day"] = pd.to_datetime(
    plan_reference_347["day"]
).dt.normalize()

assert not plan_reference_347.duplicated(
    ["user", "day", "channel"]
).any()

for channel, source_col in channel_columns_347.items():

    planned = plan_reference_347.loc[
        plan_reference_347["channel"].eq(channel),
        ["user", "day", "natural_count"]
    ]

    checked = planned.merge(
        original_counts_347[
            ["user", "day", source_col]
        ],
        on=key_columns_347,
        how="left",
        validate="one_to_one",
        indicator=True
    )

    assert checked["_merge"].eq("both").all(), (
        f"Missing original records in {channel}"
    )

    assert np.allclose(
        checked["natural_count"].to_numpy(dtype=float),
        checked[source_col].to_numpy(dtype=float),
        rtol=0,
        atol=1e-9
    ), f"Original activity mismatch in {channel}"

    assert checked["natural_count"].gt(0).all(), (
        f"Plan A violation: inactive channel in {channel}"
    )

print("\nA. ORIGINAL SOURCE VALIDATION")
print("Original users:", original_counts_347["user"].nunique())
print("Original user-days:", len(original_counts_347))
print("Planned channel-days:", len(plan_reference_347))
print("Original activity matching: PASS")

# ------------------------------------------------------------
# 4. Prepare original keyed values for independent
#    comparison against every experimental variant.
# ------------------------------------------------------------

original_indexed_347 = (
    original_counts_347
    .set_index(key_columns_347)
    .sort_index()
)

original_keys_347 = original_indexed_347.index

# ------------------------------------------------------------
# 5. Apply each experimental formula in a separate view.
# ------------------------------------------------------------

experiment_counts_347 = {}
experiment_plans_347 = []
summary_rows_347 = []

for formula in formulas_347:

    print(f"\nProcessing formula: {formula}")

    plan = quantity_experiment_343.loc[
        quantity_experiment_343["formula"].eq(formula),
        [
            "user",
            "day",
            "channel",
            "injected_count",
            "natural_count"
        ]
    ].copy()

    plan["day"] = pd.to_datetime(
        plan["day"]
    ).dt.normalize()

    assert len(plan) == 14375

    assert not plan.duplicated(
        ["user", "day", "channel"]
    ).any()

    assert plan["injected_count"].notna().all()
    assert plan["injected_count"].ge(0).all()

    increment_values = plan[
        "injected_count"
    ].to_numpy(dtype=float)

    assert np.isfinite(increment_values).all()

    assert np.allclose(
        increment_values,
        np.rint(increment_values),
        rtol=0,
        atol=1e-9
    ), "Injection increments must be integers."

    plan["injected_count"] = np.rint(
        increment_values
    ).astype(np.int64)

    # --------------------------------------------------------
    # 6. Pivot channel-level increments to daily deltas.
    # --------------------------------------------------------

    deltas = plan.pivot(
        index=key_columns_347,
        columns="channel",
        values="injected_count"
    ).fillna(0)

    for channel in channel_columns_347:

        if channel not in deltas.columns:
            deltas[channel] = 0

    deltas = (
        deltas[list(channel_columns_347)]
        .astype(np.int64)
        .reset_index()
    )

    deltas.columns.name = None

    # --------------------------------------------------------
    # 7. Apply deltas only to a fresh independent copy.
    # --------------------------------------------------------

    variant = original_counts_347.copy(deep=True)

    variant = variant.merge(
        deltas,
        on=key_columns_347,
        how="left",
        validate="one_to_one"
    )

    for channel, source_col in channel_columns_347.items():

        delta = (
            variant[channel]
            .fillna(0)
            .astype(np.int64)
        )

        natural_values = variant[
            source_col
        ].to_numpy(dtype=float)

        # Do not create activity where natural activity is zero.
        assert not (
            (natural_values == 0)
            & delta.gt(0).to_numpy()
        ).any(), (
            f"Plan A zero-activity violation: {channel}"
        )

        variant[source_col] = (
            natural_values
            + delta.to_numpy(dtype=np.int64)
        )

    # Remove temporary channel-delta columns.
    variant = variant.drop(
        columns=list(channel_columns_347.keys())
    )

    # --------------------------------------------------------
    # 8. Validate exact equality of user-day key sets.
    #
    # IMPORTANT FIX:
    # Compare canonical keys, not the original Pandas Index.
    # --------------------------------------------------------

    assert len(variant) == len(original_counts_347)

    assert not variant.duplicated(
        key_columns_347
    ).any()

    variant_indexed = (
        variant
        .set_index(key_columns_347)
        .sort_index()
    )

    assert variant_indexed.index.equals(
        original_keys_347
    ), (
        f"User-day keys changed in {formula}"
    )

    # --------------------------------------------------------
    # 9. Verify every channel's exact observed delta
    #    against the planned synthetic injection.
    # --------------------------------------------------------

    expected_deltas = (
        deltas
        .set_index(key_columns_347)
        .reindex(original_keys_347)
        .fillna(0)
    )

    total_changed = 0
    changed_channel_days = 0

    for channel, source_col in channel_columns_347.items():

        before = original_indexed_347[
            source_col
        ].to_numpy(dtype=float)

        after = variant_indexed[
            source_col
        ].to_numpy(dtype=float)

        observed_delta = after - before

        expected_delta = expected_deltas[
            channel
        ].to_numpy(dtype=float)

        assert np.allclose(
            observed_delta,
            expected_delta,
            rtol=0,
            atol=1e-9
        ), (
            f"Unexpected activity change in {channel} "
            f"under {formula}"
        )

        assert np.all(observed_delta >= 0)

        total_changed += int(
            np.rint(observed_delta.sum())
        )

        changed_channel_days += int(
            np.count_nonzero(observed_delta)
        )

    assert total_changed == int(
        plan["injected_count"].sum()
    )

    assert changed_channel_days == int(
        plan["injected_count"].gt(0).sum()
    )

    # --------------------------------------------------------
    # 10. Save experimental results IN MEMORY ONLY.
    # --------------------------------------------------------

    experiment_counts_347[formula] = (
        variant_indexed.reset_index()
    )

    experiment_plans_347.append(
        plan.assign(formula=formula)
    )

    summary_rows_347.append({
        "Formula": formula,
        "Users": variant["user"].nunique(),
        "Original_User_Days": len(variant),
        "Planned_Channel_Days": len(plan),
        "Positive_Injection_Channel_Days": int(
            plan["injected_count"].gt(0).sum()
        ),
        "Zero_Injection_Channel_Days": int(
            plan["injected_count"].eq(0).sum()
        ),
        "Total_Injected_Events": total_changed
    })

    print(
        f"PASS — {formula}: "
        f"{total_changed:,} injected events"
    )

# ------------------------------------------------------------
# 11. Summarize all experimental variants.
# ------------------------------------------------------------

experiment_injection_plan_347 = pd.concat(
    experiment_plans_347,
    ignore_index=True
)

summary_347 = pd.DataFrame(summary_rows_347)

print("\nB. EXPERIMENTAL INJECTION COMPARISON")

display(summary_347)

# ------------------------------------------------------------
# 12. Validate experiment-wide consistency.
# ------------------------------------------------------------

assert len(experiment_counts_347) == 3

assert summary_347[
    "Users"
].eq(670).all()

assert summary_347[
    "Original_User_Days"
].nunique() == 1

assert summary_347[
    "Planned_Channel_Days"
].eq(14375).all()

expected_totals_347 = {
    "A_P95_Addition": 75623,
    "B_Natural_Addition": 42095,
    "C_P95_Headroom": 35951
}

for formula, expected_total in expected_totals_347.items():

    actual_total = summary_347.loc[
        summary_347["Formula"].eq(formula),
        "Total_Injected_Events"
    ].iloc[0]

    assert int(actual_total) == expected_total, (
        f"Unexpected total for {formula}"
    )

# Confirm that the source view was never altered.
pd.testing.assert_frame_equal(
    original_counts_347.set_index(key_columns_347).sort_index(),
    original_indexed_347,
    check_exact=True
)

print("\n" + "=" * 75)
print("STEP 3.4.7 — EXPERIMENTAL INJECTION COMPLETED")
print("=" * 75)

print("Count-only experimental variants:", len(experiment_counts_347))
print("Original User-Day source: UNCHANGED")
print("Stage 2 assignments: UNCHANGED")
print("Validated schedules: UNCHANGED")
print("Related event counts: NOT YET RECONCILED")
print("Behavioral features: NOT YET RECALCULATED")
print("ML evaluation: NOT STARTED")
print("No files were modified or exported.")


STAGE 3 — STEP 3.4.7: EXPERIMENTAL INJECTION

A. ORIGINAL SOURCE VALIDATION
Original users: 670
Original user-days: 252273
Planned channel-days: 14375
Original activity matching: PASS

Processing formula: A_P95_Addition
PASS — A_P95_Addition: 75,623 injected events

Processing formula: B_Natural_Addition
PASS — B_Natural_Addition: 42,095 injected events

Processing formula: C_P95_Headroom
PASS — C_P95_Headroom: 35,951 injected events

B. EXPERIMENTAL INJECTION COMPARISON


,Formula,Users,Original_User_Days,Planned_Channel_Days,Positive_Injection_Channel_Days,Zero_Injection_Channel_Days,Total_Injected_Events
0,A_P95_Addition,670,252273,14375,14375,0,75623
1,B_Natural_Addition,670,252273,14375,14375,0,42095
2,C_P95_Headroom,670,252273,14375,10653,3722,35951



STEP 3.4.7 — EXPERIMENTAL INJECTION COMPLETED
Count-only experimental variants: 3
Original User-Day source: UNCHANGED
Stage 2 assignments: UNCHANGED
Validated schedules: UNCHANGED
Related event counts: NOT YET RECONCILED
Behavioral features: NOT YET RECALCULATED
ML evaluation: NOT STARTED
No files were modified or exported.



### Step 3.4.8 — Post-Injection Consistency & Ground Truth Audit

**Objective:** Evaluate the structural consistency and effective attack coverage of the three experimental count-only injection variants before rebuilding behavioral features.

**Validation Tasks:**
1. Verify all original User-Day keys remain unchanged.
2. Compare injected channel counts against their corresponding event totals.
3. Distinguish pre-existing source inconsistencies from newly introduced ones.
4. Measure how many scheduled attack days actually received positive synthetic activity.
5. Identify fully inactive injection days in Candidate C.
6. Report affected users and examples requiring reconciliation.

**Important:**
- This audit does not assume USB copy counts must equal general file copy counts.
- Event-total relationships are evaluated as consistency candidates; original source behavior is also measured.
- Injection quantities remain unchanged.
- No source files or schedules are modified.
- No new behavioral features or ML training are performed.

**Expected Outcome:** Identify the adjustments required before producing internally consistent experimental User-Day datasets and rebuilding features.



### Step 3.4.8 — Post-Injection Consistency & Ground Truth Audit

**Objective:** Evaluate the structural consistency and effective attack coverage of the three experimental count-only injection variants before rebuilding behavioral features.

**Validation Tasks:**
1. Verify all original User-Day keys remain unchanged.
2. Compare injected channel counts against their corresponding event totals.
3. Distinguish pre-existing source inconsistencies from newly introduced ones.
4. Measure how many scheduled attack days actually received positive synthetic activity.
5. Identify fully inactive injection days in Candidate C.
6. Report affected users and examples requiring reconciliation.

**Important:**
- This audit does not assume USB copy counts must equal general file copy counts.
- Event-total relationships are evaluated as consistency candidates; original source behavior is also measured.
- Injection quantities remain unchanged.
- No source files or schedules are modified.
- No new behavioral features or ML training are performed.

**Expected Outcome:** Identify the adjustments required before producing internally consistent experimental User-Day datasets and rebuilding features.


In [32]:

# Stage 3 — Step 3.4.8: Post-Injection Consistency & Ground Truth Audit
# Goal: Detect new count-total inconsistencies and fully zero-injection
# attack days across the three experimental formula variants.
# Read-only: No datasets, schedules, or source files are modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 78)
print("STAGE 3 — STEP 3.4.8: POST-INJECTION CONSISTENCY AUDIT")
print("=" * 78)

# ------------------------------------------------------------
# 1. Verify all required in-memory experimental outputs.
# ------------------------------------------------------------

formulas_348 = [
    "A_P95_Addition",
    "B_Natural_Addition",
    "C_P95_Headroom"
]

assert set(experiment_counts_347) == set(formulas_348)
assert len(original_counts_347) == 252273

assert validated_schedule_339["user"].nunique() == 670

keys_348 = ["user", "day"]

# ------------------------------------------------------------
# 2. Build a read-only source view containing related totals.
# ------------------------------------------------------------

required_cols_348 = [
    "user",
    "day",
    "external_email_count",
    "email_send_count",
    "email_event_count",
    "usb_copy_count",
    "file_copy_count",
    "www_upload_count",
    "http_event_count"
]

missing_cols_348 = [
    c for c in required_cols_348
    if c not in user_day.columns
]

assert not missing_cols_348, (
    f"Missing User-Day columns: {missing_cols_348}"
)

source_348 = user_day.loc[
    user_day["user"].isin(
        original_counts_347["user"].unique()
    ),
    required_cols_348
].copy()

source_348["day"] = pd.to_datetime(
    source_348["day"]
).dt.normalize()

source_348 = (
    source_348
    .set_index(keys_348)
    .sort_index()
)

assert len(source_348) == 252273
assert source_348.index.is_unique

# ------------------------------------------------------------
# 3. Define count-total relationships to audit.
#
# Email external <= email_send_count
# Email external <= email_event_count
# HTTP upload <= http_event_count
#
# USB and general file-copy logs are separate activity sources.
# Their counts will be reviewed descriptively only.
# ------------------------------------------------------------

relationships_348 = [
    (
        "Email",
        "external_email_count",
        "email_send_count",
        "External Email vs Total Email Sends"
    ),
    (
        "Email",
        "external_email_count",
        "email_event_count",
        "External Email vs All Email Events"
    ),
    (
        "HTTP",
        "www_upload_count",
        "http_event_count",
        "HTTP Uploads vs All HTTP Events"
    )
]

# ------------------------------------------------------------
# 4. Compare original and experimentally injected counts.
# ------------------------------------------------------------

relationship_rows_348 = []
example_parts_348 = []

for formula in formulas_348:

    variant_348 = experiment_counts_347[
        formula
    ].copy()

    variant_348["day"] = pd.to_datetime(
        variant_348["day"]
    ).dt.normalize()

    variant_348 = (
        variant_348
        .set_index(keys_348)
        .sort_index()
    )

    assert variant_348.index.equals(source_348.index)

    for channel, activity_col, total_col, label in relationships_348:

        natural = source_348[
            activity_col
        ].to_numpy(dtype=float)

        final = variant_348[
            activity_col
        ].to_numpy(dtype=float)

        total = source_348[
            total_col
        ].to_numpy(dtype=float)

        changed = final > natural + 1e-9

        valid = (
            np.isfinite(natural)
            & np.isfinite(final)
            & np.isfinite(total)
        )

        originally_exceeds_total = (
            valid & (natural > total + 1e-9)
        )

        finally_exceeds_total = (
            valid & (final > total + 1e-9)
        )

        newly_exceeds_total = (
            changed
            & valid
            & ~originally_exceeds_total
            & finally_exceeds_total
        )

        relationship_rows_348.append({
            "Formula": formula,
            "Relationship": label,
            "Changed_Channel_Days": int(changed.sum()),
            "Changed_With_Missing_Total": int(
                (changed & ~valid).sum()
            ),
            "Originally_Exceeding_Total": int(
                (changed & originally_exceeds_total).sum()
            ),
            "Newly_Exceeding_Total": int(
                newly_exceeds_total.sum()
            ),
            "Finally_Exceeding_Total": int(
                (changed & finally_exceeds_total).sum()
            )
        })

        if newly_exceeds_total.any():

            locations = np.flatnonzero(
                newly_exceeds_total
            )[:10]

            examples = pd.DataFrame({
                "user": source_348.index.get_level_values(
                    "user"
                )[locations],
                "day": source_348.index.get_level_values(
                    "day"
                )[locations],
                "Formula": formula,
                "Channel": channel,
                "Natural_Count": natural[locations],
                "Injected_Count": (
                    final[locations] - natural[locations]
                ),
                "Final_Count": final[locations],
                "Related_Total": total[locations],
                "Related_Total_Column": total_col
            })

            example_parts_348.append(examples)

relationship_audit_348 = pd.DataFrame(
    relationship_rows_348
)

print("\nA. RELATED EVENT-TOTAL CONSISTENCY")
display(relationship_audit_348)

# ------------------------------------------------------------
# 5. Assess scheduled attack days against actual injection.
# ------------------------------------------------------------

scheduled_attack_days_348 = (
    validated_schedule_339.loc[
        validated_schedule_339["planned_channels"] > 0,
        ["user", "day"]
    ]
    .copy()
)

scheduled_attack_days_348["day"] = pd.to_datetime(
    scheduled_attack_days_348["day"]
).dt.normalize()

assert not scheduled_attack_days_348.duplicated(
    keys_348
).any()

attack_day_rows_348 = []
zero_day_parts_348 = []

for formula in formulas_348:

    plan = experiment_injection_plan_347.loc[
        experiment_injection_plan_347["formula"].eq(
            formula
        )
    ].copy()

    plan["day"] = pd.to_datetime(
        plan["day"]
    ).dt.normalize()

    assert len(plan) == 14375

    daily = (
        plan.groupby(keys_348, as_index=False)
        .agg(
            Injected_Events=("injected_count", "sum"),
            Positive_Channels=(
                "injected_count",
                lambda s: int((s > 0).sum())
            ),
            Planned_Channels=("channel", "size")
        )
    )

    assert len(daily) == len(scheduled_attack_days_348)

    checked = scheduled_attack_days_348.merge(
        daily,
        on=keys_348,
        how="left",
        validate="one_to_one",
        indicator=True
    )

    assert checked["_merge"].eq("both").all()

    zero_days = checked.loc[
        checked["Injected_Events"].eq(0)
    ].copy()

    zero_days["Formula"] = formula

    zero_day_parts_348.append(zero_days)

    attack_day_rows_348.append({
        "Formula": formula,
        "Scheduled_Attack_Days": len(checked),
        "Days_With_Positive_Injection": int(
            checked["Injected_Events"].gt(0).sum()
        ),
        "Days_With_Zero_Injection": len(zero_days),
        "Users_With_Zero_Injection_Days": (
            zero_days["user"].nunique()
        ),
        "Partially_Executed_Days": int((
            checked["Positive_Channels"].gt(0)
            & (
                checked["Positive_Channels"]
                < checked["Planned_Channels"]
            )
        ).sum())
    })

attack_day_audit_348 = pd.DataFrame(
    attack_day_rows_348
)

zero_attack_days_348 = pd.concat(
    zero_day_parts_348,
    ignore_index=True
)

print("\nB. SCHEDULED ATTACK-DAY COVERAGE")
display(attack_day_audit_348)

# ------------------------------------------------------------
# 6. Diagnostic review of USB vs general file copies.
# Separate sources: observational comparison only.
# ------------------------------------------------------------

print("\nC. USB / FILE COUNT RELATIONSHIP — DIAGNOSTIC ONLY")

usb_rows_348 = []

natural_usb = source_348[
    "usb_copy_count"
].to_numpy(dtype=float)

file_copies = source_348[
    "file_copy_count"
].to_numpy(dtype=float)

for formula in formulas_348:

    variant = experiment_counts_347[
        formula
    ].copy()

    variant["day"] = pd.to_datetime(
        variant["day"]
    ).dt.normalize()

    variant = (
        variant.set_index(keys_348).sort_index()
    )

    final_usb = variant[
        "usb_copy_count"
    ].to_numpy(dtype=float)

    changed = final_usb > natural_usb + 1e-9

    valid = (
        np.isfinite(natural_usb)
        & np.isfinite(final_usb)
        & np.isfinite(file_copies)
    )

    usb_rows_348.append({
        "Formula": formula,
        "Changed_USB_Days": int(changed.sum()),
        "Original_USB_Greater_Than_File_Copy": int(
            (
                changed & valid
                & (natural_usb > file_copies)
            ).sum()
        ),
        "Final_USB_Greater_Than_File_Copy": int(
            (
                changed & valid
                & (final_usb > file_copies)
            ).sum()
        )
    })

display(pd.DataFrame(usb_rows_348))

print(
    "Note: USB copy and general file-copy metrics are "
    "not assumed to represent the same event population."
)

# ------------------------------------------------------------
# 7. Show concrete examples requiring reconciliation.
# ------------------------------------------------------------

print("\nD. EXAMPLES OF NEW COUNT-TOTAL DISCREPANCIES")

if example_parts_348:

    consistency_examples_348 = pd.concat(
        example_parts_348,
        ignore_index=True
    )

    display(
        consistency_examples_348.head(30)
    )

else:

    consistency_examples_348 = pd.DataFrame()

    print(
        "No new count-total exceedances found "
        "for the audited relationships."
    )

print("\nE. EXAMPLES OF ATTACK DAYS WITHOUT ANY INJECTION")

if zero_attack_days_348.empty:

    print("Every scheduled attack day received positive activity.")

else:

    display(
        zero_attack_days_348[
            [
                "user",
                "day",
                "Formula",
                "Planned_Channels",
                "Positive_Channels",
                "Injected_Events"
            ]
        ]
        .sort_values(["Formula", "user", "day"])
        .head(30)
    )

# ------------------------------------------------------------
# 8. Final integrity assertions and status.
# ------------------------------------------------------------

assert attack_day_audit_348[
    "Scheduled_Attack_Days"
].nunique() == 1

assert attack_day_audit_348[
    "Days_With_Positive_Injection"
].add(
    attack_day_audit_348[
        "Days_With_Zero_Injection"
    ]
).eq(
    attack_day_audit_348[
        "Scheduled_Attack_Days"
    ]
).all()

assert len(experiment_counts_347) == 3

print("\n" + "=" * 78)
print("STEP 3.4.8 — AUDIT COMPLETED")
print("=" * 78)

print("Source totals: INSPECTED, NOT MODIFIED")
print("Candidate formulas: UNCHANGED")
print("Experimental count-only views: UNCHANGED")
print("Schedule: UNCHANGED")
print("Ground Truth validity: PENDING REVIEW")
print("Behavioral features: NOT RECALCULATED")
print("No files modified or exported.")


STAGE 3 — STEP 3.4.8: POST-INJECTION CONSISTENCY AUDIT

A. RELATED EVENT-TOTAL CONSISTENCY


,Formula,Relationship,Changed_Channel_Days,Changed_With_Missing_Total,Originally_Exceeding_Total,Newly_Exceeding_Total,Finally_Exceeding_Total
0,A_P95_Addition,External Email vs Total Email Sends,9343,0,0,7964,7964
1,A_P95_Addition,External Email vs All Email Events,9343,0,0,753,753
2,A_P95_Addition,HTTP Uploads vs All HTTP Events,1533,0,0,0,0
3,B_Natural_Addition,External Email vs Total Email Sends,9343,0,0,5075,5075
4,B_Natural_Addition,External Email vs All Email Events,9343,0,0,546,546
5,B_Natural_Addition,HTTP Uploads vs All HTTP Events,1533,0,0,0,0
6,C_P95_Headroom,External Email vs Total Email Sends,7076,0,0,4634,4634
7,C_P95_Headroom,External Email vs All Email Events,7076,0,0,66,66
8,C_P95_Headroom,HTTP Uploads vs All HTTP Events,1084,0,0,0,0



B. SCHEDULED ATTACK-DAY COVERAGE


,Formula,Scheduled_Attack_Days,Days_With_Positive_Injection,Days_With_Zero_Injection,Users_With_Zero_Injection_Days,Partially_Executed_Days
0,A_P95_Addition,12636,12636,0,0,0
1,B_Natural_Addition,12636,12636,0,0,0
2,C_P95_Headroom,12636,9591,3045,434,374



C. USB / FILE COUNT RELATIONSHIP — DIAGNOSTIC ONLY


,Formula,Changed_USB_Days,Original_USB_Greater_Than_File_Copy,Final_USB_Greater_Than_File_Copy
0,A_P95_Addition,3499,0,2822
1,B_Natural_Addition,3499,0,1646
2,C_P95_Headroom,2493,0,1269


Note: USB copy and general file-copy metrics are not assumed to represent the same event population.

D. EXAMPLES OF NEW COUNT-TOTAL DISCREPANCIES


,user,day,Formula,Channel,Natural_Count,Injected_Count,Final_Count,Related_Total,Related_Total_Column
0,AAF0819,2011-03-08,A_P95_Addition,Email,3.0,2.0,5.0,3.0,email_send_count
1,AAP0352,2011-04-07,A_P95_Addition,Email,3.0,2.0,5.0,3.0,email_send_count
2,AAP0352,2011-04-08,A_P95_Addition,Email,2.0,2.0,4.0,3.0,email_send_count
3,AAP0352,2011-04-12,A_P95_Addition,Email,1.0,2.0,3.0,2.0,email_send_count
4,AAP1919,2011-01-31,A_P95_Addition,Email,1.0,3.0,4.0,2.0,email_send_count
5,AAP1919,2011-02-01,A_P95_Addition,Email,1.0,3.0,4.0,3.0,email_send_count
6,AAP1919,2011-02-03,A_P95_Addition,Email,1.0,3.0,4.0,2.0,email_send_count
7,AAP1919,2011-02-04,A_P95_Addition,Email,3.0,3.0,6.0,3.0,email_send_count
8,AAP1919,2011-02-07,A_P95_Addition,Email,1.0,3.0,4.0,1.0,email_send_count
9,AAP1919,2011-02-08,A_P95_Addition,Email,1.0,3.0,4.0,2.0,email_send_count



E. EXAMPLES OF ATTACK DAYS WITHOUT ANY INJECTION


,user,day,Formula,Planned_Channels,Positive_Channels,Injected_Events
0,AAF0819,2011-03-03,C_P95_Headroom,1,0,0
1,AAF0819,2011-03-08,C_P95_Headroom,1,0,0
2,AAP0352,2011-04-06,C_P95_Headroom,1,0,0
3,AAP0352,2011-04-07,C_P95_Headroom,1,0,0
4,AAP0352,2011-04-08,C_P95_Headroom,1,0,0
5,AAP0352,2011-04-11,C_P95_Headroom,2,0,0
6,AAP0352,2011-04-15,C_P95_Headroom,1,0,0
7,AAP0352,2011-04-19,C_P95_Headroom,1,0,0
8,AAP1919,2011-02-02,C_P95_Headroom,1,0,0
9,AAP1919,2011-02-04,C_P95_Headroom,1,0,0



STEP 3.4.8 — AUDIT COMPLETED
Source totals: INSPECTED, NOT MODIFIED
Candidate formulas: UNCHANGED
Experimental count-only views: UNCHANGED
Schedule: UNCHANGED
Ground Truth validity: PENDING REVIEW
Behavioral features: NOT RECALCULATED
No files modified or exported.



### Step 3.4.9 — Original Event-Count Dependency Audit

**Objective:** Examine relationships between the original User-Day event counts to identify how synthetic channel activity should be reflected in related aggregate features.

**Relationships to investigate:**
- External + internal email counts versus total email sends.
- Email sends + views versus total email events.
- WWW visits + downloads + uploads versus total HTTP events.
- USB copies versus general file-copy counts (diagnostic only).
- External email bytes versus total sent email bytes.

**Validation Tasks:**
1. Analyze original data before injection.
2. Measure equality, lower-than, and greater-than relationships.
3. Repeat the audit on channel-days selected for injection.
4. Inspect source examples and identify reliable relationships.
5. Separate event-count consistency from unknown payload volume.

**Expected Outcome:** Establish which related fields can be updated consistently and which require further semantic verification.

**Important:** No injection quantities, schedules, or source files are changed.


In [33]:

# Stage 3 — Step 3.4.9: Original Event-Count Dependency Audit
# Goal: Investigate how Email, HTTP, USB, and related totals
# behave in the ORIGINAL User-Day data before any injection.
# Read-only: No data, schedules, or files are modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 78)
print("STAGE 3 — STEP 3.4.9: ORIGINAL COUNT DEPENDENCIES")
print("=" * 78)

# ------------------------------------------------------------
# 1. Identify all columns needed for source verification.
# ------------------------------------------------------------

required_349 = [
    "user", "day",
    "email_event_count",
    "email_send_count",
    "email_view_count",
    "external_email_count",
    "internal_email_count",
    "total_sent_email_bytes",
    "external_email_bytes",
    "http_event_count",
    "www_visit_count",
    "www_download_count",
    "www_upload_count",
    "file_event_count",
    "file_copy_count",
    "file_open_count",
    "file_write_count",
    "file_delete_count",
    "usb_copy_count"
]

missing_349 = [
    col for col in required_349
    if col not in user_day.columns
]

assert not missing_349, (
    f"Missing source columns: {missing_349}"
)

source_349 = user_day.loc[
    user_day["user"].isin(
        quantity_base_343["user"].unique()
    ),
    required_349
].copy()

source_349["day"] = pd.to_datetime(
    source_349["day"]
).dt.normalize()

source_349 = (
    source_349
    .sort_values(["user", "day"])
    .reset_index(drop=True)
)

assert len(source_349) == 252273
assert source_349["user"].nunique() == 670

assert not source_349.duplicated(
    ["user", "day"]
).any()

# ------------------------------------------------------------
# 2. Identify days selected for injection in each channel.
#    The source observations themselves remain unchanged.
# ------------------------------------------------------------

for channel in ["Email", "USB", "HTTP"]:

    keys = quantity_base_343.loc[
        quantity_base_343["channel"].eq(channel),
        ["user", "day"]
    ].copy()

    keys["day"] = pd.to_datetime(
        keys["day"]
    ).dt.normalize()

    keys = pd.MultiIndex.from_frame(keys)

    source_keys = pd.MultiIndex.from_frame(
        source_349[["user", "day"]]
    )

    source_349[
        f"planned_{channel}"
    ] = source_keys.isin(keys)

assert int(source_349["planned_Email"].sum()) == 9343
assert int(source_349["planned_USB"].sum()) == 3499
assert int(source_349["planned_HTTP"].sum()) == 1533

# ------------------------------------------------------------
# 3. Define observational relationships.
#    These are hypotheses to test, not assumed identities.
# ------------------------------------------------------------

relationships_349 = [
    {
        "name": "Email Sends vs External + Internal",
        "channel": "Email",
        "left": ["email_send_count"],
        "right": [
            "external_email_count",
            "internal_email_count"
        ]
    },
    {
        "name": "Email Events vs Sends + Views",
        "channel": "Email",
        "left": ["email_event_count"],
        "right": [
            "email_send_count",
            "email_view_count"
        ]
    },
    {
        "name": "HTTP Events vs Visit + Download + Upload",
        "channel": "HTTP",
        "left": ["http_event_count"],
        "right": [
            "www_visit_count",
            "www_download_count",
            "www_upload_count"
        ]
    },
    {
        "name": "File Events vs File Action Counts",
        "channel": "USB",
        "left": ["file_event_count"],
        "right": [
            "file_copy_count",
            "file_open_count",
            "file_write_count",
            "file_delete_count"
        ]
    },
    {
        "name": "File Copies vs USB Copies (Diagnostic)",
        "channel": "USB",
        "left": ["file_copy_count"],
        "right": ["usb_copy_count"]
    },
    {
        "name": "Total Email Bytes vs External Email Bytes",
        "channel": "Email",
        "left": ["total_sent_email_bytes"],
        "right": ["external_email_bytes"]
    }
]

# ------------------------------------------------------------
# 4. Evaluate relationships across both:
#    - All original user-days
#    - The channel-days selected for experimental injection
# ------------------------------------------------------------

results_349 = []
tolerance_349 = 1e-9

for relation in relationships_349:

    channel = relation["channel"]

    for scope, subset in [
        ("All User-Days", source_349),
        (
            "Planned Channel-Days",
            source_349.loc[
                source_349[f"planned_{channel}"]
            ]
        )
    ]:

        left = subset[
            relation["left"]
        ].sum(axis=1, min_count=len(relation["left"]))

        right = subset[
            relation["right"]
        ].sum(axis=1, min_count=len(relation["right"]))

        lhs = left.to_numpy(dtype=float)
        rhs = right.to_numpy(dtype=float)

        valid = (
            np.isfinite(lhs)
            & np.isfinite(rhs)
        )

        equal = (
            valid
            & np.isclose(
                lhs, rhs, rtol=0, atol=tolerance_349
            )
        )

        left_greater = (
            valid & (lhs > rhs + tolerance_349)
        )

        right_greater = (
            valid & (rhs > lhs + tolerance_349)
        )

        # Exclude all-zero pairs from a second,
        # more informative active-record comparison.
        active = (
            valid
            & ((lhs > 0) | (rhs > 0))
        )

        results_349.append({
            "Relationship": relation["name"],
            "Scope": scope,
            "Records": len(subset),
            "Valid": int(valid.sum()),
            "Active_Records": int(active.sum()),
            "Equal_All": int(equal.sum()),
            "Equal_Active": int((equal & active).sum()),
            "Left_Greater_Active": int(
                (left_greater & active).sum()
            ),
            "Right_Greater_Active": int(
                (right_greater & active).sum()
            ),
            "Equality_Rate_Active_%": round(
                100 * (equal & active).sum()
                / max(int(active.sum()), 1),
                2
            )
        })

dependency_audit_349 = pd.DataFrame(results_349)

# ------------------------------------------------------------
# 5. Show all observed relationships.
# ------------------------------------------------------------

print("\nA. ORIGINAL COUNT RELATIONSHIPS")

display(
    dependency_audit_349.loc[
        dependency_audit_349[
            "Scope"
        ].eq("All User-Days")
    ]
)

print("\nB. RELATIONSHIPS ON PLANNED INJECTION CHANNEL-DAYS")

display(
    dependency_audit_349.loc[
        dependency_audit_349[
            "Scope"
        ].eq("Planned Channel-Days")
    ]
)

# ------------------------------------------------------------
# 6. Show concrete original examples for Email and HTTP.
# ------------------------------------------------------------

print("\nC. ORIGINAL EMAIL EXAMPLES")

display(
    source_349.loc[
        source_349["planned_Email"],
        [
            "user", "day",
            "external_email_count",
            "internal_email_count",
            "email_send_count",
            "email_view_count",
            "email_event_count",
            "external_email_bytes",
            "total_sent_email_bytes"
        ]
    ]
    .head(15)
)

print("\nD. ORIGINAL HTTP EXAMPLES")

display(
    source_349.loc[
        source_349["planned_HTTP"],
        [
            "user", "day",
            "www_visit_count",
            "www_download_count",
            "www_upload_count",
            "http_event_count"
        ]
    ]
    .head(15)
)

# ------------------------------------------------------------
# 7. Inspect USB and file relationships without asserting
#    that the two event sources are interchangeable.
# ------------------------------------------------------------

print("\nE. ORIGINAL USB / FILE EXAMPLES")

display(
    source_349.loc[
        source_349["planned_USB"],
        [
            "user", "day",
            "usb_copy_count",
            "file_copy_count",
            "file_event_count",
            "file_open_count",
            "file_write_count",
            "file_delete_count"
        ]
    ]
    .head(15)
)

# ------------------------------------------------------------
# 8. Final status.
# ------------------------------------------------------------

assert len(dependency_audit_349) == 12
assert source_349["user"].nunique() == 670

print("\n" + "=" * 78)
print("STEP 3.4.9 — ORIGINAL DEPENDENCY AUDIT COMPLETED")
print("=" * 78)

print("Original count relationships: INSPECTED")
print("Related-field update rules: NOT YET APPROVED")
print("Synthetic data volume: NOT INFERRED")
print("Experimental variants: UNCHANGED")
print("Original datasets: UNCHANGED")
print("No files modified or exported.")


STAGE 3 — STEP 3.4.9: ORIGINAL COUNT DEPENDENCIES

A. ORIGINAL COUNT RELATIONSHIPS


,Relationship,Scope,Records,Valid,Active_Records,Equal_All,Equal_Active,Left_Greater_Active,Right_Greater_Active,Equality_Rate_Active_%
0,Email Sends vs External + Internal,All User-Days,252273,252273,232097,252273,232097,0,0,100.00
2,Email Events vs Sends + Views,All User-Days,252273,252273,250000,252273,250000,0,0,100.00
4,HTTP Events vs Visit + Download + Upload,All User-Days,252273,252273,251081,252273,251081,0,0,100.00
6,File Events vs File Action Counts,All User-Days,252273,252273,163058,252273,163058,0,0,100.00
8,File Copies vs USB Copies (Diagnostic),All User-Days,252273,252273,110677,155974,14378,96299,0,12.99
10,Total Email Bytes vs External Email Bytes,All User-Days,252273,252273,232097,60982,40806,191291,0,17.58



B. RELATIONSHIPS ON PLANNED INJECTION CHANNEL-DAYS


,Relationship,Scope,Records,Valid,Active_Records,Equal_All,Equal_Active,Left_Greater_Active,Right_Greater_Active,Equality_Rate_Active_%
1,Email Sends vs External + Internal,Planned Channel-Days,9343,9343,9343,9343,9343,0,0,100.00
3,Email Events vs Sends + Views,Planned Channel-Days,9343,9343,9343,9343,9343,0,0,100.00
5,HTTP Events vs Visit + Download + Upload,Planned Channel-Days,1533,1533,1533,1533,1533,0,0,100.00
7,File Events vs File Action Counts,Planned Channel-Days,3499,3499,3499,3499,3499,0,0,100.00
9,File Copies vs USB Copies (Diagnostic),Planned Channel-Days,3499,3499,3499,597,597,2902,0,17.06
11,Total Email Bytes vs External Email Bytes,Planned Channel-Days,9343,9343,9343,2260,2260,7083,0,24.19



C. ORIGINAL EMAIL EXAMPLES


,user,day,external_email_count,internal_email_count,email_send_count,email_view_count,email_event_count,external_email_bytes,total_sent_email_bytes
294,AAF0819,2011-03-03,2.0,2.0,4.0,5.0,9,557485.0,4154895.0
295,AAF0819,2011-03-04,1.0,2.0,3.0,6.0,9,1386232.0,1488010.0
297,AAF0819,2011-03-08,3.0,0.0,3.0,6.0,9,739874.0,739874.0
675,AAP0352,2011-04-07,3.0,0.0,3.0,5.0,8,2313343.0,2313343.0
676,AAP0352,2011-04-08,2.0,1.0,3.0,6.0,9,1732674.0,1748905.0
677,AAP0352,2011-04-11,3.0,2.0,5.0,4.0,9,5990613.0,6060251.0
678,AAP0352,2011-04-12,1.0,1.0,2.0,7.0,9,30976.0,68937.0
679,AAP0352,2011-04-13,1.0,2.0,3.0,6.0,9,24312.0,1697177.0
680,AAP0352,2011-04-14,1.0,5.0,6.0,3.0,9,42198.0,6529437.0
683,AAP0352,2011-04-19,2.0,3.0,5.0,4.0,9,60465.0,1003494.0



D. ORIGINAL HTTP EXAMPLES


,user,day,www_visit_count,www_download_count,www_upload_count,http_event_count
1480,ABD3426,2011-04-01,161.0,0.0,1.0,162
3014,ABS0677,2011-01-17,87.0,7.0,1.0,95
3017,ABS0677,2011-01-20,86.0,8.0,1.0,95
3025,ABS0677,2011-02-01,88.0,5.0,2.0,95
3028,ABS0677,2011-02-04,85.0,8.0,2.0,95
3029,ABS0677,2011-02-07,87.0,6.0,2.0,95
3030,ABS0677,2011-02-08,92.0,2.0,1.0,95
3399,ACB3123,2011-02-23,83.0,5.0,2.0,90
3408,ACB3123,2011-03-08,85.0,4.0,1.0,90
3416,ACB3123,2011-03-18,83.0,6.0,1.0,90



E. ORIGINAL USB / FILE EXAMPLES


,user,day,usb_copy_count,file_copy_count,file_event_count,file_open_count,file_write_count,file_delete_count
296,AAF0819,2011-03-07,1.0,1.0,2,0.0,0.0,1.0
298,AAF0819,2011-03-09,1.0,3.0,7,2.0,0.0,2.0
674,AAP0352,2011-04-06,2.0,2.0,4,0.0,1.0,1.0
677,AAP0352,2011-04-11,1.0,3.0,5,2.0,0.0,0.0
681,AAP0352,2011-04-15,2.0,2.0,6,1.0,1.0,2.0
983,AAP1919,2011-01-31,1.0,2.0,3,0.0,0.0,1.0
985,AAP1919,2011-02-02,1.0,1.0,1,0.0,0.0,0.0
986,AAP1919,2011-02-03,1.0,2.0,2,0.0,0.0,0.0
998,AAP1919,2011-02-21,1.0,2.0,3,0.0,0.0,1.0
1479,ABD3426,2011-03-31,2.0,10.0,27,7.0,4.0,6.0



STEP 3.4.9 — ORIGINAL DEPENDENCY AUDIT COMPLETED
Original count relationships: INSPECTED
Related-field update rules: NOT YET APPROVED
Synthetic data volume: NOT INFERRED
Experimental variants: UNCHANGED
Original datasets: UNCHANGED
No files modified or exported.



### Step 3.4.10 — Aggregate Count Reconciliation

**Objective:** Reconstruct the three experimental User-Day variants with internally consistent event-count relationships.

**Reconciliation Rules:**
- Email: increase external emails, total sends, and total email events by the same injected count.
- USB: increase USB copies, total file copies, and total file events by the same injected count.
- HTTP: increase WWW uploads and total HTTP events by the same injected count.

**Validation Tasks:**
1. Preserve all original user-day keys and records.
2. Apply only the selected synthetic channel-day increments.
3. Preserve exact Email, File, and HTTP count identities.
4. Preserve the USB Copy subset relationship.
5. Verify all unmodified columns remain unchanged.
6. Verify that total injected channel activities match each experimental plan.
7. Keep the three formulas in separate datasets.

**Limitations:**
- This step reconciles aggregate event counts only.
- Email bytes and all unobserved payload sizes are not reconstructed.
- No underlying Email, File, or HTTP event rows are synthesized.
- Zero-injection cases in Formula C remain unresolved.
- The resulting datasets are not yet approved for ML training.

**Expected Outcome:** Three validated count-reconciled experimental User-Day variants, maintained in memory only.


In [34]:

# Stage 3 — Step 3.4.10: Aggregate Count Reconciliation
# Goal: Build independent complete User-Day experimental variants
# and preserve verified Email, USB/File, and HTTP count identities.
# Read-only source: No original files or schedules are modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 78)
print("STAGE 3 — STEP 3.4.10: AGGREGATE COUNT RECONCILIATION")
print("=" * 78)

# ------------------------------------------------------------
# 1. Define formulas and event-count dependencies.
# ------------------------------------------------------------

formulas_3410 = [
    "A_P95_Addition",
    "B_Natural_Addition",
    "C_P95_Headroom"
]

dependencies_3410 = {
    "Email": [
        "external_email_count",
        "email_send_count",
        "email_event_count"
    ],
    "USB": [
        "usb_copy_count",
        "file_copy_count",
        "file_event_count"
    ],
    "HTTP": [
        "www_upload_count",
        "http_event_count"
    ]
}

modified_cols_3410 = sorted(
    set(
        col
        for cols in dependencies_3410.values()
        for col in cols
    )
)

keys_3410 = ["user", "day"]

assert set(experiment_counts_347) == set(formulas_3410)

# ------------------------------------------------------------
# 2. Create a complete original User-Day view.
# ------------------------------------------------------------

original_full_3410 = user_day.loc[
    user_day["user"].isin(
        quantity_base_343["user"].unique()
    )
].copy(deep=True)

original_full_3410["day"] = pd.to_datetime(
    original_full_3410["day"]
).dt.normalize()

original_full_3410 = (
    original_full_3410
    .sort_values(keys_3410)
    .reset_index(drop=True)
)

assert len(original_full_3410) == 252273
assert original_full_3410["user"].nunique() == 670
assert not original_full_3410.duplicated(keys_3410).any()

original_indexed_3410 = (
    original_full_3410
    .set_index(keys_3410)
    .sort_index()
)

# ------------------------------------------------------------
# 3. Define exact aggregate-count validation rules.
# ------------------------------------------------------------

def validate_counts_3410(frame):

    def same(left, right):
        return np.allclose(
            frame[left].to_numpy(dtype=float),
            right.to_numpy(dtype=float),
            rtol=0,
            atol=1e-9
        )

    assert same(
        "email_send_count",
        frame["external_email_count"]
        + frame["internal_email_count"]
    ), "Email send identity failed"

    assert same(
        "email_event_count",
        frame["email_send_count"]
        + frame["email_view_count"]
    ), "Email event identity failed"

    assert same(
        "http_event_count",
        frame["www_visit_count"]
        + frame["www_download_count"]
        + frame["www_upload_count"]
    ), "HTTP event identity failed"

    assert same(
        "file_event_count",
        frame["file_copy_count"]
        + frame["file_open_count"]
        + frame["file_write_count"]
        + frame["file_delete_count"]
    ), "File event identity failed"

    assert (
        frame["file_copy_count"].to_numpy(dtype=float)
        + 1e-9
        >= frame["usb_copy_count"].to_numpy(dtype=float)
    ).all(), "USB Copy subset relationship failed"

validate_counts_3410(original_indexed_3410)

print("\nA. ORIGINAL SOURCE VALIDATION")
print("Users:", original_full_3410["user"].nunique())
print("User-days:", len(original_full_3410))
print("Original count identities: PASS")

# ------------------------------------------------------------
# 4. Independently build and validate each variant.
# ------------------------------------------------------------

full_variants_3410 = {}
reconciliation_rows_3410 = []

for formula in formulas_3410:

    plan = experiment_injection_plan_347.loc[
        experiment_injection_plan_347["formula"].eq(formula),
        ["user", "day", "channel", "injected_count"]
    ].copy()

    plan["day"] = pd.to_datetime(
        plan["day"]
    ).dt.normalize()

    assert len(plan) == 14375
    assert not plan.duplicated(
        ["user", "day", "channel"]
    ).any()

    deltas = plan.pivot(
        index=keys_3410,
        columns="channel",
        values="injected_count"
    ).fillna(0)

    deltas = (
        deltas.reindex(
            columns=["Email", "USB", "HTTP"],
            fill_value=0
        )
        .reindex(original_indexed_3410.index)
        .fillna(0)
        .astype(np.int64)
    )

    assert int(deltas.to_numpy().sum()) == int(
        plan["injected_count"].sum()
    )

    variant = original_indexed_3410.copy(deep=True)

    # Apply each synthetic channel increment to all corresponding
    # aggregate-count dependencies.
    for channel, columns in dependencies_3410.items():

        delta = deltas[channel].to_numpy(dtype=np.int64)

        for col in columns:
            variant[col] = (
                variant[col].to_numpy(dtype=float) + delta
            )

    # --------------------------------------------------------
    # 5. Verify source keys and exact aggregate identities.
    # --------------------------------------------------------

    assert variant.index.equals(original_indexed_3410.index)

    validate_counts_3410(variant)

    expected_deltas_3410 = {}

    for channel, columns in dependencies_3410.items():
        for col in columns:
            expected_deltas_3410[col] = (
                deltas[channel].to_numpy(dtype=float)
            )

    for col, expected in expected_deltas_3410.items():

        observed = (
            variant[col].to_numpy(dtype=float)
            - original_indexed_3410[col].to_numpy(dtype=float)
        )

        assert np.allclose(
            observed, expected, rtol=0, atol=1e-9
        ), f"Unexpected change in {col} for {formula}"

    # All columns not identified as count dependencies
    # must remain identical to the original.
    unchanged_cols = [
        c for c in original_indexed_3410.columns
        if c not in modified_cols_3410
    ]

    pd.testing.assert_frame_equal(
        variant[unchanged_cols],
        original_indexed_3410[unchanged_cols],
        check_exact=True
    )

    full_variants_3410[formula] = variant.reset_index()

    reconciliation_rows_3410.append({
        "Formula": formula,
        "Users": variant.index.get_level_values("user").nunique(),
        "User_Days": len(variant),
        "Injected_Email": int(deltas["Email"].sum()),
        "Injected_USB": int(deltas["USB"].sum()),
        "Injected_HTTP": int(deltas["HTTP"].sum()),
        "Total_Injected_Channel_Events": int(
            deltas.to_numpy().sum()
        ),
        "Zero_Injection_Channel_Days": int(
            plan["injected_count"].eq(0).sum()
        ),
        "Count_Identities": "PASS",
        "Other_Columns_Preserved": "PASS"
    })

    print(f"PASS — {formula}")

# ------------------------------------------------------------
# 6. Display reconciliation results.
# ------------------------------------------------------------

reconciliation_summary_3410 = pd.DataFrame(
    reconciliation_rows_3410
)

print("\nB. RECONCILED EXPERIMENTAL VARIANTS")
display(reconciliation_summary_3410)

assert len(full_variants_3410) == 3

assert (
    reconciliation_summary_3410["Total_Injected_Channel_Events"]
    .to_numpy()
    == np.array([75623, 42095, 35951])
).all()

# ------------------------------------------------------------
# 7. Explicitly document unresolved semantic dependencies.
# ------------------------------------------------------------

print("\nC. UNRESOLVED SEMANTIC DEPENDENCIES")

semantic_limitations_3410 = pd.DataFrame([
    {
        "Area": "Email",
        "Unresolved": "External and total email bytes",
        "Status": "NOT RECONSTRUCTED"
    },
    {
        "Area": "Email",
        "Unresolved": "Recipients, domains and attachments",
        "Status": "ORIGINAL VALUES PRESERVED"
    },
    {
        "Area": "USB",
        "Unresolved": "Unique files, extensions and payload size",
        "Status": "NOT SYNTHESIZED"
    },
    {
        "Area": "HTTP",
        "Unresolved": "Upload domains and payload size",
        "Status": "NOT SYNTHESIZED"
    }
])

display(semantic_limitations_3410)

print("\n" + "=" * 78)
print("STEP 3.4.10 — COUNT RECONCILIATION COMPLETED")
print("=" * 78)

print("Aggregate event-count identities: VALIDATED")
print("Event-level reconstruction: NOT PERFORMED")
print("Payload bytes: NOT RECONSTRUCTED")
print("Behavioral features: NOT YET RECALCULATED")
print("ML-ready datasets: NOT YET APPROVED")
print("Original source and schedules: UNCHANGED")
print("No files modified or exported.")


STAGE 3 — STEP 3.4.10: AGGREGATE COUNT RECONCILIATION

A. ORIGINAL SOURCE VALIDATION
Users: 670
User-days: 252273
Original count identities: PASS
PASS — A_P95_Addition
PASS — B_Natural_Addition
PASS — C_P95_Headroom

B. RECONCILED EXPERIMENTAL VARIANTS


,Formula,Users,User_Days,Injected_Email,Injected_USB,Injected_HTTP,Total_Injected_Channel_Events,Zero_Injection_Channel_Days,Count_Identities,Other_Columns_Preserved
0,A_P95_Addition,670,252273,44336,20035,11252,75623,0,PASS,PASS
1,B_Natural_Addition,670,252273,23880,11422,6793,42095,0,PASS,PASS
2,C_P95_Headroom,670,252273,22513,8650,4788,35951,3722,PASS,PASS



C. UNRESOLVED SEMANTIC DEPENDENCIES


,Area,Unresolved,Status
0,Email,External and total email bytes,NOT RECONSTRUCTED
1,Email,"Recipients, domains and attachments",ORIGINAL VALUES PRESERVED
2,USB,"Unique files, extensions and payload size",NOT SYNTHESIZED
3,HTTP,Upload domains and payload size,NOT SYNTHESIZED



STEP 3.4.10 — COUNT RECONCILIATION COMPLETED
Aggregate event-count identities: VALIDATED
Event-level reconstruction: NOT PERFORMED
Payload bytes: NOT RECONSTRUCTED
Behavioral features: NOT YET RECALCULATED
ML-ready datasets: NOT YET APPROVED
Original source and schedules: UNCHANGED
No files modified or exported.



### Step 3.4.11 — Contextual Feature Consistency Audit

**Objective:** Verify that preserving original contextual features remains logically consistent after experimental injection under Formulas A, B, and C.

**Features Under Review:**
- Email: attachments, external recipients, external domains, and byte-related fields.
- USB: unique files, file extensions, and removable-media activity.
- HTTP: unique domains and upload domains.

**Validation Tasks:**
1. Confirm contextual features remain identical to the original User-Day dataset.
2. Compare logical relationships before and after injection.
3. Separate pre-existing inconsistencies from newly introduced inconsistencies.
4. Measure how unchanged contextual values affect ratios after activity counts increase.
5. Quantify the effect of preserving email bytes despite increasing email counts.
6. Identify any affected channel-days requiring additional review.

**Approved Decisions:**
- Original byte values are retained as final recorded values.
- No synthetic recipients, domains, filenames, extensions, or attachments are created.
- The original historical baseline remains unchanged.
- New temporal features will later use 5, 10, 20, and 30 workday windows.

**Expected Outcome:** Establish contextual consistency and identify any limitations before recalculating behavioral features.

**Important:** Read-only audit. No source files, schedules, or experimental variants are modified.


In [35]:

# Stage 3 — Step 3.4.11: Contextual Feature Consistency Audit
# Goal: Verify preserved contextual fields, audit logical constraints,
# and quantify changes in context-to-activity ratios after injection.
# Read-only: No datasets, schedules, or files are modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 78)
print("STAGE 3 — STEP 3.4.11: CONTEXTUAL CONSISTENCY AUDIT")
print("=" * 78)

# ------------------------------------------------------------
# 1. Verify experimental inputs.
# ------------------------------------------------------------

formulas_3411 = [
    "A_P95_Addition",
    "B_Natural_Addition",
    "C_P95_Headroom"
]

keys_3411 = ["user", "day"]

assert set(full_variants_3410) == set(formulas_3411)
assert len(original_full_3410) == 252273
assert original_full_3410["user"].nunique() == 670

# ------------------------------------------------------------
# 2. Define contextual fields that must remain unchanged.
# ------------------------------------------------------------

context_fields_3411 = {
    "Email": [
        "emails_with_attachments",
        "attachment_count",
        "unique_external_recipients",
        "unique_external_domains",
        "external_email_bytes",
        "total_sent_email_bytes"
    ],
    "USB": [
        "unique_usb_files",
        "unique_usb_extensions",
        "from_removable_count"
    ],
    "HTTP": [
        "unique_domains",
        "unique_upload_domains"
    ]
}

all_context_3411 = sorted({
    col
    for cols in context_fields_3411.values()
    for col in cols
})

required_3411 = all_context_3411 + [
    "email_send_count",
    "external_email_count",
    "usb_copy_count",
    "www_upload_count"
]

missing_3411 = [
    col for col in required_3411
    if col not in original_full_3410.columns
]

assert not missing_3411, (
    f"Missing required columns: {missing_3411}"
)

# ------------------------------------------------------------
# 3. Prepare original and experimental views.
# ------------------------------------------------------------

original_3411 = original_full_3410.copy(deep=True)

original_3411["day"] = pd.to_datetime(
    original_3411["day"]
).dt.normalize()

original_3411 = (
    original_3411
    .set_index(keys_3411)
    .sort_index()
)

assert original_3411.index.is_unique

variants_3411 = {}

for formula in formulas_3411:

    variant = full_variants_3410[formula].copy()

    variant["day"] = pd.to_datetime(
        variant["day"]
    ).dt.normalize()

    variant = (
        variant
        .set_index(keys_3411)
        .sort_index()
    )

    assert variant.index.equals(original_3411.index)

    variants_3411[formula] = variant

# ------------------------------------------------------------
# 4. Verify complete preservation of contextual fields.
# ------------------------------------------------------------

preservation_rows_3411 = []

for formula, variant in variants_3411.items():

    for channel, cols in context_fields_3411.items():

        pd.testing.assert_frame_equal(
            variant[cols],
            original_3411[cols],
            check_exact=True
        )

        preservation_rows_3411.append({
            "Formula": formula,
            "Channel": channel,
            "Context_Columns": len(cols),
            "Changed_Context_Values": 0,
            "Status": "PASS"
        })

print("\nA. CONTEXTUAL FIELD PRESERVATION")

preservation_audit_3411 = pd.DataFrame(
    preservation_rows_3411
)

display(preservation_audit_3411)

# ------------------------------------------------------------
# 5. Define logical count/context relationships.
#
# The first field should be >= the second field.
# These relationships are audited, not assumed to be
# universally valid before source data is checked.
# ------------------------------------------------------------

constraints_3411 = [
    (
        "Email",
        "email_send_count",
        "emails_with_attachments",
        "Sent Emails >= Emails With Attachments"
    ),
    (
        "Email",
        "attachment_count",
        "emails_with_attachments",
        "Attachments >= Emails With Attachments"
    ),
    (
        "Email",
        "unique_external_recipients",
        "unique_external_domains",
        "Recipients >= External Domains"
    ),
    (
        "Email",
        "total_sent_email_bytes",
        "external_email_bytes",
        "Total Email Bytes >= External Email Bytes"
    ),
    (
        "USB",
        "usb_copy_count",
        "unique_usb_files",
        "USB Copies >= Unique USB Files"
    ),
    (
        "USB",
        "unique_usb_files",
        "unique_usb_extensions",
        "Unique USB Files >= Unique Extensions"
    ),
    (
        "HTTP",
        "www_upload_count",
        "unique_upload_domains",
        "Uploads >= Unique Upload Domains"
    ),
    (
        "HTTP",
        "unique_domains",
        "unique_upload_domains",
        "Web Domains >= Upload Domains"
    )
]

# ------------------------------------------------------------
# 6. Identify channel-days receiving positive injection.
# ------------------------------------------------------------

audit_rows_3411 = []
new_violation_examples_3411 = []

for formula, variant in variants_3411.items():

    plan = experiment_injection_plan_347.loc[
        experiment_injection_plan_347["formula"].eq(formula)
    ].copy()

    plan["day"] = pd.to_datetime(
        plan["day"]
    ).dt.normalize()

    assert len(plan) == 14375

    for channel, upper_col, lower_col, label in constraints_3411:

        selected_keys = plan.loc[
            plan["channel"].eq(channel)
            & plan["injected_count"].gt(0),
            keys_3411
        ]

        selected_index = pd.MultiIndex.from_frame(
            selected_keys
        )

        assert selected_index.is_unique
        assert selected_index.isin(original_3411.index).all()

        before_upper = pd.to_numeric(
            original_3411.loc[selected_index, upper_col],
            errors="coerce"
        ).to_numpy(dtype=float)

        before_lower = pd.to_numeric(
            original_3411.loc[selected_index, lower_col],
            errors="coerce"
        ).to_numpy(dtype=float)

        after_upper = pd.to_numeric(
            variant.loc[selected_index, upper_col],
            errors="coerce"
        ).to_numpy(dtype=float)

        after_lower = pd.to_numeric(
            variant.loc[selected_index, lower_col],
            errors="coerce"
        ).to_numpy(dtype=float)

        valid = (
            np.isfinite(before_upper)
            & np.isfinite(before_lower)
            & np.isfinite(after_upper)
            & np.isfinite(after_lower)
        )

        before_invalid = (
            valid & (before_lower > before_upper + 1e-9)
        )

        after_invalid = (
            valid & (after_lower > after_upper + 1e-9)
        )

        new_invalid = after_invalid & ~before_invalid

        audit_rows_3411.append({
            "Formula": formula,
            "Channel": channel,
            "Constraint": label,
            "Positive_Injection_Channel_Days": len(selected_index),
            "Missing_Values": int((~valid).sum()),
            "Original_Violations": int(before_invalid.sum()),
            "Final_Violations": int(after_invalid.sum()),
            "New_Violations": int(new_invalid.sum())
        })

        if new_invalid.any():

            positions = np.flatnonzero(new_invalid)[:5]

            new_violation_examples_3411.append(
                pd.DataFrame({
                    "user": selected_index.get_level_values(
                        "user"
                    )[positions],
                    "day": selected_index.get_level_values(
                        "day"
                    )[positions],
                    "formula": formula,
                    "constraint": label,
                    "original_upper": before_upper[positions],
                    "original_lower": before_lower[positions],
                    "final_upper": after_upper[positions],
                    "final_lower": after_lower[positions]
                })
            )

constraint_audit_3411 = pd.DataFrame(
    audit_rows_3411
)

print("\nB. CONTEXTUAL CONSTRAINT AUDIT")

display(constraint_audit_3411)

print("\nC. NEWLY INTRODUCED CONSTRAINT VIOLATIONS")

new_total_3411 = int(
    constraint_audit_3411["New_Violations"].sum()
)

print("New violations:", new_total_3411)

if new_violation_examples_3411:
    display(
        pd.concat(
            new_violation_examples_3411,
            ignore_index=True
        )
    )
else:
    print("No new violations detected in the audited constraints.")

# ------------------------------------------------------------
# 7. Examine context-to-activity ratios.
#
# Even unchanged contextual values can produce different
# ratios after the denominator (activity count) increases.
# These are diagnostic signals, not automatic errors.
# ------------------------------------------------------------

ratio_definitions_3411 = [
    (
        "Email",
        "external_email_bytes",
        "external_email_count",
        "External Email Bytes per External Email"
    ),
    (
        "Email",
        "unique_external_recipients",
        "external_email_count",
        "Unique Recipients per External Email"
    ),
    (
        "USB",
        "unique_usb_files",
        "usb_copy_count",
        "Unique USB Files per Copy"
    ),
    (
        "HTTP",
        "unique_upload_domains",
        "www_upload_count",
        "Unique Upload Domains per Upload"
    )
]

ratio_rows_3411 = []

for formula, variant in variants_3411.items():

    for channel, numerator, denominator, label in ratio_definitions_3411:

        source_plan = experiment_injection_plan_347.loc[
            experiment_injection_plan_347["formula"].eq(formula)
            & experiment_injection_plan_347["channel"].eq(channel)
            & experiment_injection_plan_347["injected_count"].gt(0),
            keys_3411
        ]

        idx = pd.MultiIndex.from_frame(source_plan)

        num = pd.to_numeric(
            original_3411.loc[idx, numerator],
            errors="coerce"
        ).to_numpy(dtype=float)

        original_den = pd.to_numeric(
            original_3411.loc[idx, denominator],
            errors="coerce"
        ).to_numpy(dtype=float)

        final_den = pd.to_numeric(
            variant.loc[idx, denominator],
            errors="coerce"
        ).to_numpy(dtype=float)

        valid = (
            np.isfinite(num)
            & np.isfinite(original_den)
            & np.isfinite(final_den)
            & (original_den > 0)
            & (final_den > 0)
        )

        informative = valid & (num > 0)

        before_ratio = (
            num[informative] / original_den[informative]
        )

        after_ratio = (
            num[informative] / final_den[informative]
        )

        if informative.any():

            relative_drop = (
                1 - after_ratio / before_ratio
            )

            median_before = float(np.median(before_ratio))
            median_after = float(np.median(after_ratio))

            half_reduction = int(
                (relative_drop >= 0.5 - 1e-9).sum()
            )

        else:
            median_before = np.nan
            median_after = np.nan
            half_reduction = 0

        ratio_rows_3411.append({
            "Formula": formula,
            "Channel": channel,
            "Ratio": label,
            "Positive_Injection_Days": len(idx),
            "Valid_Nonzero_Numerator_Days": int(
                informative.sum()
            ),
            "Median_Ratio_Before": median_before,
            "Median_Ratio_After": median_after,
            "Days_With_50pct_Or_More_Ratio_Drop": half_reduction
        })

ratio_audit_3411 = pd.DataFrame(ratio_rows_3411)

print("\nD. CONTEXT-TO-ACTIVITY RATIO CHANGES")

display(ratio_audit_3411.round(3))

# ------------------------------------------------------------
# 8. Explicitly verify the approved frozen-byte policy.
# ------------------------------------------------------------

print("\nE. EMAIL BYTE PRESERVATION POLICY")

byte_columns_3411 = [
    "external_email_bytes",
    "total_sent_email_bytes"
]

for formula, variant in variants_3411.items():

    pd.testing.assert_frame_equal(
        variant[byte_columns_3411],
        original_3411[byte_columns_3411],
        check_exact=True
    )

    print(f"{formula}: original byte values preserved — PASS")

print(
    "\nNote: Bytes are treated as the final recorded values, "
    "not as newly simulated payload volume."
)

# ------------------------------------------------------------
# 9. Final safeguards and status.
# ------------------------------------------------------------

assert len(preservation_audit_3411) == 9
assert len(constraint_audit_3411) == 24
assert len(ratio_audit_3411) == 12

assert preservation_audit_3411["Status"].eq("PASS").all()

print("\n" + "=" * 78)
print("STEP 3.4.11 — CONTEXTUAL AUDIT COMPLETED")
print("=" * 78)

print("Contextual fields: ORIGINAL VALUES PRESERVED")
print("Byte values: ORIGINAL VALUES PRESERVED")
print("New contextual inconsistencies:", new_total_3411)
print("Context-to-activity ratios: AUDITED")
print("Experimental variants: UNCHANGED")
print("New temporal features: NOT YET CALCULATED")
print("Approved rolling windows: 5 / 10 / 20 / 30 WD")
print("Original datasets: UNCHANGED")
print("No files modified or exported.")


STAGE 3 — STEP 3.4.11: CONTEXTUAL CONSISTENCY AUDIT

A. CONTEXTUAL FIELD PRESERVATION


,Formula,Channel,Context_Columns,Changed_Context_Values,Status
0,A_P95_Addition,Email,6,0,PASS
1,A_P95_Addition,USB,3,0,PASS
2,A_P95_Addition,HTTP,2,0,PASS
3,B_Natural_Addition,Email,6,0,PASS
4,B_Natural_Addition,USB,3,0,PASS
5,B_Natural_Addition,HTTP,2,0,PASS
6,C_P95_Headroom,Email,6,0,PASS
7,C_P95_Headroom,USB,3,0,PASS
8,C_P95_Headroom,HTTP,2,0,PASS



B. CONTEXTUAL CONSTRAINT AUDIT


,Formula,Channel,Constraint,Positive_Injection_Channel_Days,Missing_Values,Original_Violations,Final_Violations,New_Violations
0,A_P95_Addition,Email,Sent Emails >= Emails With Attachments,9343,0,0,0,0
1,A_P95_Addition,Email,Attachments >= Emails With Attachments,9343,0,0,0,0
2,A_P95_Addition,Email,Recipients >= External Domains,9343,0,0,0,0
3,A_P95_Addition,Email,Total Email Bytes >= External Email Bytes,9343,0,0,0,0
4,A_P95_Addition,USB,USB Copies >= Unique USB Files,3499,0,0,0,0
5,A_P95_Addition,USB,Unique USB Files >= Unique Extensions,3499,0,0,0,0
6,A_P95_Addition,HTTP,Uploads >= Unique Upload Domains,1533,0,0,0,0
7,A_P95_Addition,HTTP,Web Domains >= Upload Domains,1533,0,0,0,0
8,B_Natural_Addition,Email,Sent Emails >= Emails With Attachments,9343,0,0,0,0
9,B_Natural_Addition,Email,Attachments >= Emails With Attachments,9343,0,0,0,0



C. NEWLY INTRODUCED CONSTRAINT VIOLATIONS
New violations: 0
No new violations detected in the audited constraints.

D. CONTEXT-TO-ACTIVITY RATIO CHANGES


,Formula,Channel,Ratio,Positive_Injection_Days,Valid_Nonzero_Numerator_Days,Median_Ratio_Before,Median_Ratio_After,Days_With_50pct_Or_More_Ratio_Drop
0,A_P95_Addition,Email,External Email Bytes per External Email,9343,9343,51462.00,19888.000,8441
1,A_P95_Addition,Email,Unique Recipients per External Email,9343,9343,1.75,0.571,8441
2,A_P95_Addition,USB,Unique USB Files per Copy,3499,3499,1.00,0.286,3168
3,A_P95_Addition,HTTP,Unique Upload Domains per Upload,1533,1533,1.00,0.296,1366
4,B_Natural_Addition,Email,External Email Bytes per External Email,9343,9343,51462.00,26540.000,7496
5,B_Natural_Addition,Email,Unique Recipients per External Email,9343,9343,1.75,0.800,7496
6,B_Natural_Addition,USB,Unique USB Files per Copy,3499,3499,1.00,0.500,2802
7,B_Natural_Addition,HTTP,Unique Upload Domains per Upload,1533,1533,1.00,0.429,1237
8,C_P95_Headroom,Email,External Email Bytes per External Email,7076,7076,40951.00,18984.250,5778
9,C_P95_Headroom,Email,Unique Recipients per External Email,7076,7076,2.00,0.667,5778



E. EMAIL BYTE PRESERVATION POLICY
A_P95_Addition: original byte values preserved — PASS
B_Natural_Addition: original byte values preserved — PASS
C_P95_Headroom: original byte values preserved — PASS

Note: Bytes are treated as the final recorded values, not as newly simulated payload volume.

STEP 3.4.11 — CONTEXTUAL AUDIT COMPLETED
Contextual fields: ORIGINAL VALUES PRESERVED
Byte values: ORIGINAL VALUES PRESERVED
New contextual inconsistencies: 0
Context-to-activity ratios: AUDITED
Experimental variants: UNCHANGED
New temporal features: NOT YET CALCULATED
Approved rolling windows: 5 / 10 / 20 / 30 WD
Original datasets: UNCHANGED
No files modified or exported.



### Step 3.4.12 — Rebuild Existing Behavioral Features

**Objective:** Recalculate the original 85-column behavioral feature structure independently for the three reconciled injection variants.

**Source Methodology:** Reproduce the baseline-relative and causal rolling-feature calculations from `04_build_behavior_features_r62.ipynb`.

**Validation Tasks:**
1. Preserve the historical user baseline.
2. Use eligible future workdays only: date on or after 2011-01-01, non-weekend, and positive logon count.
3. Recalculate channel activity indicators, P95 flags, and P95 ratios.
4. Recalculate original rolling activity and count features over 5, 10, and 20 workdays.
5. Verify byte-based features remain unchanged.
6. Verify active-day indicators remain unchanged under Plan A.
7. Compare each injected feature dataset against an independently rebuilt benign reference.
8. Identify the number of feature values changed by each injection formula.

**Approved Future Work:** Additional behavioral features will use 5, 10, 20, and 30 workday windows.

**Limitations:** No new feature engineering, ground-truth labeling, ML training, or file export occurs in this step.

**Expected Outcome:** Three validated experimental feature datasets with the existing 85-column structure.


In [36]:

# Stage 3 — Step 3.4.12: Rebuild Existing Behavioral Features
# Goal: Recalculate original baseline-relative and rolling features
# from each reconciled User-Day variant, using the verified
# Data Engineering methodology.
# Read-only: No original files or schedules are modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 78)
print("STAGE 3 — STEP 3.4.12: EXISTING FEATURE REBUILD")
print("=" * 78)

# ------------------------------------------------------------
# 1. Define the approved experimental inputs.
# ------------------------------------------------------------

formulas_3412 = [
    "A_P95_Addition",
    "B_Natural_Addition",
    "C_P95_Headroom"
]

assert set(full_variants_3410) == set(formulas_3412)

CUTOFF_3412 = pd.Timestamp("2011-01-01")

keys_3412 = ["user", "day"]

# Historical baseline fields included in the original
# 85-column behavior-feature dataset.

baseline_cols_3412 = [
    "baseline_workdays",

    "email_active_days",
    "email_active_rate",
    "email_count_p50",
    "email_count_p90",
    "email_count_p95",
    "email_count_mad",
    "email_bytes_p50",
    "email_bytes_p90",
    "email_bytes_p95",
    "email_bytes_mad",

    "usb_active_days",
    "usb_active_rate",
    "usb_copy_p50",
    "usb_copy_p90",
    "usb_copy_p95",
    "usb_copy_mad",

    "http_upload_active_days",
    "http_upload_active_rate",
    "http_upload_p50",
    "http_upload_p90",
    "http_upload_p95",
    "http_upload_mad"
]

assert len(baseline_cols_3412) == 23

assert user_baseline["user"].is_unique

assert set(baseline_cols_3412).issubset(
    user_baseline.columns
)

baseline_view_3412 = user_baseline[
    ["user"] + baseline_cols_3412
].copy()

# ------------------------------------------------------------
# 2. Build a reusable reconstruction function.
#    The function never changes its input dataframe.
# ------------------------------------------------------------

def rebuild_existing_features_3412(user_day_frame):

    d = user_day_frame.copy(deep=True)

    d["day"] = pd.to_datetime(
        d["day"]
    ).dt.normalize()

    assert not d.duplicated(keys_3412).any()

    # Match the original Data Engineering workday filter.
    d = d.loc[
        d["day"].ge(CUTOFF_3412)
        & d["is_weekend"].eq(0)
        & d["logon_count"].gt(0)
    ].copy()

    d = d.merge(
        baseline_view_3412,
        on="user",
        how="inner",
        validate="many_to_one"
    )

    d = d.sort_values(
        keys_3412
    ).reset_index(drop=True)

    assert not d.duplicated(keys_3412).any()
    assert d["baseline_workdays"].notna().all()

    # --------------------------------------------------------
    # 3. Channel activity indicators.
    # --------------------------------------------------------

    indicators = {
        "email_active_today": "external_email_count",
        "usb_active_today": "usb_copy_count",
        "http_upload_active_today": "www_upload_count"
    }

    for indicator, count_col in indicators.items():

        d[indicator] = (
            d[count_col].gt(0).astype(np.int64)
        )

    d["active_channel_count"] = (
        d["email_active_today"]
        + d["usb_active_today"]
        + d["http_upload_active_today"]
    )

    # --------------------------------------------------------
    # 4. Baseline-relative P95 flags and ratios.
    # --------------------------------------------------------

    feature_definitions = {
        "email_count": (
            "external_email_count",
            "email_count_p95"
        ),
        "email_bytes": (
            "external_email_bytes",
            "email_bytes_p95"
        ),
        "usb_copy": (
            "usb_copy_count",
            "usb_copy_p95"
        ),
        "http_upload": (
            "www_upload_count",
            "http_upload_p95"
        )
    }

    for prefix, (count_col, p95_col) in (
        feature_definitions.items()
    ):

        # Match the original SQL CASE logic.
        d[f"{prefix}_above_p95"] = (
            d[p95_col].notna()
            & d[count_col].gt(d[p95_col])
        ).astype(np.int64)

        d[f"{prefix}_to_p95_ratio"] = (
            d[count_col].div(
                d[p95_col].where(
                    d[p95_col].gt(0)
                )
            )
        )

    # --------------------------------------------------------
    # 5. Causal rolling features.
    #
    # Each window includes the current eligible workday
    # and the preceding N-1 eligible workdays.
    # No future observations enter the calculation.
    # --------------------------------------------------------

    rolling_specs = [
        ("email_active_today", "email_active_days"),
        ("usb_active_today", "usb_active_days"),
        ("http_upload_active_today", "http_active_days")
    ]

    for source_col, output_prefix in rolling_specs:

        for window in [5, 10, 20]:

            output_col = (
                f"{output_prefix}_{window}wd"
            )

            d[output_col] = (
                d.groupby(
                    "user", sort=False
                )[source_col]
                .transform(
                    lambda s, n=window: (
                        s.rolling(
                            window=n,
                            min_periods=1
                        ).sum()
                    )
                )
                .astype(np.int64)
            )

    # Original Data Engineering count windows:
    # 5 and 10 eligible workdays.

    count_specs = {
        "external_email_count": "external_email_count",
        "usb_copy_count": "usb_copy_count",
        "www_upload_count": "http_upload_count"
    }

    for source_col, output_prefix in count_specs.items():

        for window in [5, 10]:

            output_col = (
                f"{output_prefix}_{window}wd"
            )

            d[output_col] = (
                d.groupby(
                    "user", sort=False
                )[source_col]
                .transform(
                    lambda s, n=window: (
                        s.rolling(
                            window=n,
                            min_periods=1
                        ).sum()
                    )
                )
            )

    # Original multi-channel rolling feature.

    d["_multichannel_today_3412"] = (
        d["active_channel_count"].ge(2)
        .astype(np.int64)
    )

    d["multi_channel_days_10wd"] = (
        d.groupby(
            "user", sort=False
        )["_multichannel_today_3412"]
        .transform(
            lambda s: s.rolling(
                window=10,
                min_periods=1
            ).sum()
        )
        .astype(np.int64)
    )

    d = d.drop(
        columns=["_multichannel_today_3412"]
    )

    assert not d.duplicated(keys_3412).any()

    return d


# ------------------------------------------------------------
# 6. Independently rebuild the ORIGINAL feature reference.
# ------------------------------------------------------------

original_features_3412 = (
    rebuild_existing_features_3412(
        original_full_3410
    )
)

assert original_features_3412["user"].nunique() == 670

assert len(original_features_3412.columns) == 85, (
    "Original 85-column feature schema mismatch."
)

print("\nA. ORIGINAL FEATURE REBUILD")

print(
    "Users:",
    original_features_3412["user"].nunique()
)

print(
    "Eligible user-days:",
    len(original_features_3412)
)

print(
    "Feature dataset columns:",
    len(original_features_3412.columns)
)

print("Original feature rebuild: PASS")

# ------------------------------------------------------------
# 7. Define feature groups for the comparison.
# ------------------------------------------------------------

expected_changed_features_3412 = [
    "email_count_above_p95",
    "usb_copy_above_p95",
    "http_upload_above_p95",

    "email_count_to_p95_ratio",
    "usb_copy_to_p95_ratio",
    "http_upload_to_p95_ratio",

    "external_email_count_5wd",
    "external_email_count_10wd",

    "usb_copy_count_5wd",
    "usb_copy_count_10wd",

    "http_upload_count_5wd",
    "http_upload_count_10wd"
]

expected_unchanged_features_3412 = [
    "email_bytes_above_p95",
    "email_bytes_to_p95_ratio",
    "email_active_today",
    "usb_active_today",
    "http_upload_active_today",
    "active_channel_count",
    "multi_channel_days_10wd"
]

for channel in ["email", "usb", "http"]:

    prefix = (
        "http_active_days"
        if channel == "http"
        else f"{channel}_active_days"
    )

    for window in [5, 10, 20]:

        expected_unchanged_features_3412.append(
            f"{prefix}_{window}wd"
        )

# ------------------------------------------------------------
# 8. Rebuild each formula independently and compare.
# ------------------------------------------------------------

feature_variants_3412 = {}
feature_summary_rows_3412 = []
changed_details_3412 = []

original_idx_3412 = (
    original_features_3412
    .set_index(keys_3412)
    .sort_index()
)

for formula in formulas_3412:

    features = rebuild_existing_features_3412(
        full_variants_3410[formula]
    )

    assert len(features.columns) == 85

    assert set(features.columns) == set(
        original_features_3412.columns
    )

    feature_idx = (
        features
        .set_index(keys_3412)
        .sort_index()
    )

    assert feature_idx.index.equals(
        original_idx_3412.index
    )

    # Frozen baselines must match exactly.
    pd.testing.assert_frame_equal(
        feature_idx[baseline_cols_3412],
        original_idx_3412[baseline_cols_3412],
        check_exact=True
    )

    # Plan A must preserve activity status and frozen
    # byte-derived features.
    for col in expected_unchanged_features_3412:

        before = original_idx_3412[
            col
        ].to_numpy(dtype=float)

        after = feature_idx[
            col
        ].to_numpy(dtype=float)

        assert np.allclose(
            before,
            after,
            rtol=0,
            atol=1e-9,
            equal_nan=True
        ), f"Unexpected change in {col}: {formula}"

    changed_values = 0

    for col in expected_changed_features_3412:

        before = original_idx_3412[
            col
        ].to_numpy(dtype=float)

        after = feature_idx[
            col
        ].to_numpy(dtype=float)

        same = np.isclose(
            before,
            after,
            rtol=0,
            atol=1e-9,
            equal_nan=True
        )

        changed = int((~same).sum())

        changed_values += changed

        changed_details_3412.append({
            "Formula": formula,
            "Feature": col,
            "Changed_Values": changed
        })

    # Store in memory only.
    feature_variants_3412[formula] = features

    feature_summary_rows_3412.append({
        "Formula": formula,
        "Users": features["user"].nunique(),
        "Eligible_User_Days": len(features),
        "Total_Columns": len(features.columns),
        "Changed_Target_Feature_Values": changed_values,
        "Baseline_Preserved": "PASS",
        "Activity_Indicators_Preserved": "PASS",
        "Byte_Features_Preserved": "PASS"
    })

    print(f"PASS — {formula}")


# ------------------------------------------------------------
# 9. Display rebuilding and change summaries.
# ------------------------------------------------------------

feature_summary_3412 = pd.DataFrame(
    feature_summary_rows_3412
)

feature_changes_3412 = pd.DataFrame(
    changed_details_3412
)

print("\nB. EXPERIMENTAL FEATURE REBUILD SUMMARY")

display(feature_summary_3412)

print("\nC. CHANGED EXISTING FEATURES")

display(
    feature_changes_3412.pivot(
        index="Feature",
        columns="Formula",
        values="Changed_Values"
    )
)

# ------------------------------------------------------------
# 10. Final validation and status.
# ------------------------------------------------------------

assert len(feature_variants_3412) == 3

assert feature_summary_3412[
    "Eligible_User_Days"
].nunique() == 1

assert feature_summary_3412[
    "Total_Columns"
].eq(85).all()

print("\n" + "=" * 78)
print("STEP 3.4.12 — EXISTING FEATURE REBUILD COMPLETED")
print("=" * 78)

print("Existing 85-column feature structure: REBUILT")
print("Baseline values: PRESERVED")
print("Byte-related features: PRESERVED")
print("Workday windows: CAUSAL")
print("New 5/10/20/30 WD features: NOT YET BUILT")
print("Ground Truth: NOT YET FINALIZED")
print("ML training: NOT STARTED")
print("Original datasets: UNCHANGED")
print("No files modified or exported.")


STAGE 3 — STEP 3.4.12: EXISTING FEATURE REBUILD

A. ORIGINAL FEATURE REBUILD
Users: 670
Eligible user-days: 69634
Feature dataset columns: 85
Original feature rebuild: PASS
PASS — A_P95_Addition
PASS — B_Natural_Addition
PASS — C_P95_Headroom

B. EXPERIMENTAL FEATURE REBUILD SUMMARY


,Formula,Users,Eligible_User_Days,Total_Columns,Changed_Target_Feature_Values,Baseline_Preserved,Activity_Indicators_Preserved,Byte_Features_Preserved
0,A_P95_Addition,670,69634,85,97539,PASS,PASS,PASS
1,B_Natural_Addition,670,69634,85,91894,PASS,PASS,PASS
2,C_P95_Headroom,670,69634,85,74255,PASS,PASS,PASS



C. CHANGED EXISTING FEATURES


Formula,A_P95_Addition,B_Natural_Addition,C_P95_Headroom
Feature,,,
email_count_above_p95,7086,3402,3141
email_count_to_p95_ratio,9343,9343,7076
external_email_count_10wd,19083,19083,17195
external_email_count_5wd,15816,15816,13849
http_upload_above_p95,1275,702,563
http_upload_count_10wd,8430,8430,6223
http_upload_count_5wd,5621,5621,4074
http_upload_to_p95_ratio,1533,1533,1084
usb_copy_above_p95,2825,1437,992



STEP 3.4.12 — EXISTING FEATURE REBUILD COMPLETED
Existing 85-column feature structure: REBUILT
Baseline values: PRESERVED
Byte-related features: PRESERVED
Workday windows: CAUSAL
New 5/10/20/30 WD features: NOT YET BUILT
Ground Truth: NOT YET FINALIZED
ML training: NOT STARTED
Original datasets: UNCHANGED
No files modified or exported.



### Step 3.4.12.1 — Original Feature Reference Validation

**Objective:** Independently validate the rebuilt benign behavioral features against the authoritative Data Engineering output `behavior_features_r62_v1.parquet`.

**Validation Tasks:**
1. Load original features for the same 670 users.
2. Compare user-day keys and all 85 feature columns.
3. Check exact equality for categorical and identifier fields.
4. Compare numeric features using strict floating-point tolerance.
5. Report mismatching features and example records.
6. Confirm whether the rebuilt benign reference is suitable for subsequent experimental feature engineering.

**Expected Outcome:** An independent reference-validation report.

**Read-only:** No original or experimental datasets are modified.


In [37]:

# Stage 3 — Step 3.4.12.1: Original Feature Reference Validation
# Goal: Compare all rebuilt benign features with the original
# Data Engineering parquet, using identical users and user-day keys.
# Read-only: No files or existing dataframes are modified.

from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
from IPython.display import display

print("=" * 78)
print("STAGE 3 — STEP 3.4.12.1: FEATURE REFERENCE VALIDATION")
print("=" * 78)

# ------------------------------------------------------------
# 1. Locate the authoritative original feature dataset.
# ------------------------------------------------------------

reference_path_34121 = Path(
    r"C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration"
) / "Data_Engineering" / "Data_engineering" / \
    "data" / "features" / "behavior_features_r62_v1.parquet"

assert reference_path_34121.is_file(), (
    f"Original feature file not found: {reference_path_34121}"
)

# ------------------------------------------------------------
# 2. Read original features for only the scheduled users.
# ------------------------------------------------------------

selected_users_34121 = (
    original_features_3412[["user"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

con_34121 = duckdb.connect()

try:
    con_34121.register(
        "selected_users_34121",
        selected_users_34121
    )

    reference_features_34121 = con_34121.execute(
        """
        SELECT f.*
        FROM read_parquet(?) AS f
        INNER JOIN selected_users_34121 AS u
            ON f.user = u.user
        ORDER BY f.user, f.day
        """,
        [str(reference_path_34121)]
    ).df()

finally:
    con_34121.close()

# ------------------------------------------------------------
# 3. Normalize user-day keys and align both datasets.
# ------------------------------------------------------------

keys_34121 = ["user", "day"]

reference_features_34121["day"] = pd.to_datetime(
    reference_features_34121["day"]
).dt.normalize()

rebuilt_features_34121 = original_features_3412.copy()

rebuilt_features_34121["day"] = pd.to_datetime(
    rebuilt_features_34121["day"]
).dt.normalize()

assert not reference_features_34121.duplicated(
    keys_34121
).any()

assert not rebuilt_features_34121.duplicated(
    keys_34121
).any()

reference_idx_34121 = (
    reference_features_34121
    .set_index(keys_34121)
    .sort_index()
)

rebuilt_idx_34121 = (
    rebuilt_features_34121
    .set_index(keys_34121)
    .sort_index()
)

keys_match_34121 = reference_idx_34121.index.equals(
    rebuilt_idx_34121.index
)

schema_match_34121 = (
    set(reference_features_34121.columns)
    == set(rebuilt_features_34121.columns)
)

print("\nA. REFERENCE DATASET")

print("Source:", reference_path_34121)
print("Reference users:", reference_features_34121["user"].nunique())
print("Reference user-days:", len(reference_features_34121))
print("Rebuilt user-days:", len(rebuilt_features_34121))
print("Reference columns:", len(reference_features_34121.columns))
print("Rebuilt columns:", len(rebuilt_features_34121.columns))
print("User-day keys match:", keys_match_34121)
print("Column names match:", schema_match_34121)

# ------------------------------------------------------------
# 4. Compare values only after verifying keys and schema.
# ------------------------------------------------------------

comparison_details_34121 = []

if keys_match_34121 and schema_match_34121:

    for col in reference_idx_34121.columns:

        expected = reference_idx_34121[col]
        actual = rebuilt_idx_34121[col]

        if (
            pd.api.types.is_numeric_dtype(expected)
            and pd.api.types.is_numeric_dtype(actual)
        ):
            equal = np.isclose(
                expected.to_numpy(dtype=float),
                actual.to_numpy(dtype=float),
                rtol=1e-10,
                atol=1e-9,
                equal_nan=True
            )

        else:
            equal = (
                expected.astype("string")
                .fillna("<MISSING>")
                .to_numpy()
                ==
                actual.astype("string")
                .fillna("<MISSING>")
                .to_numpy()
            )

        mismatches = int((~equal).sum())

        comparison_details_34121.append({
            "Feature": col,
            "Mismatched_Values": mismatches,
            "Status": "PASS" if mismatches == 0 else "REVIEW"
        })

    comparison_34121 = pd.DataFrame(
        comparison_details_34121
    )

    print("\nB. FEATURE VALUE COMPARISON")

    print("Compared feature columns:", len(comparison_34121))
    print(
        "Matching feature columns:",
        int(comparison_34121["Status"].eq("PASS").sum())
    )
    print(
        "Mismatching feature columns:",
        int(comparison_34121["Status"].eq("REVIEW").sum())
    )
    print(
        "Total mismatched values:",
        int(comparison_34121["Mismatched_Values"].sum())
    )

    mismatched_features_34121 = comparison_34121.loc[
        comparison_34121["Mismatched_Values"] > 0
    ]

    if mismatched_features_34121.empty:
        print("\nAll feature values match the original reference.")
    else:
        display(mismatched_features_34121.head(30))

else:
    comparison_34121 = pd.DataFrame()
    print("\nValue comparison skipped due to key/schema mismatch.")

# ------------------------------------------------------------
# 5. Final validation decision.
# ------------------------------------------------------------

reference_validation_pass_34121 = (
    keys_match_34121
    and schema_match_34121
    and not comparison_34121.empty
    and comparison_34121["Mismatched_Values"].sum() == 0
)

print("\n" + "=" * 78)

if reference_validation_pass_34121:
    print("STEP 3.4.12.1 — INDEPENDENT REFERENCE VALIDATION PASSED")
else:
    print("STEP 3.4.12.1 — REFERENCE REVIEW REQUIRED")

print("=" * 78)

print("Original datasets: UNCHANGED")
print("Experimental feature variants: UNCHANGED")
print("New 5/10/20/30 WD features: NOT YET BUILT")
print("No files modified or exported.")


STAGE 3 — STEP 3.4.12.1: FEATURE REFERENCE VALIDATION

A. REFERENCE DATASET
Source: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Data_Engineering\Data_engineering\data\features\behavior_features_r62_v1.parquet
Reference users: 670
Reference user-days: 69634
Rebuilt user-days: 69634
Reference columns: 85
Rebuilt columns: 85
User-day keys match: True
Column names match: True

B. FEATURE VALUE COMPARISON
Compared feature columns: 83
Matching feature columns: 83
Mismatching feature columns: 0
Total mismatched values: 0

All feature values match the original reference.

STEP 3.4.12.1 — INDEPENDENT REFERENCE VALIDATION PASSED
Original datasets: UNCHANGED
Experimental feature variants: UNCHANGED
New 5/10/20/30 WD features: NOT YET BUILT
No files modified or exported.



### Step 3.4.13 — Stealth-Level Formula Comparison

**Objective:** Compare Formulas A, B, and C to select the most suitable injection policy for Mubsir before developing new behavioral features or training an ML model.

**Stealth Objectives:**
- High (0.5): Low-observability injection, preferably remaining within the user's historical P95.
- Medium (1.0): Moderate deviation, with relatively small P95 exceedances.
- Low (1.5): More visible deviation, with larger P95 exceedances.

**Validation Tasks:**
1. Compare natural activity and final injected activity relative to P95.
2. Separate pre-existing natural exceedances from injection-induced exceedances.
3. Analyze final/P95 ratio distributions for every formula and stealth level.
4. Compare the percentages of records in descriptive deviation bands.
5. Examine behavior separately for Email, USB, and HTTP.
6. Measure positive-injection coverage across planned channel-days.
7. Identify whether each formula produces the intended High–Medium–Low hierarchy.

**Important:**
- Descriptive ratio bands are not final stealth thresholds.
- Formula selection is based on simulation quality, not ML performance.
- All three formulas use the same validated attack schedules.
- No datasets, schedules, or source files are modified.

**Expected Outcome:** Evidence supporting the selection of Formula A, Formula B, Formula C, or a hybrid injection policy.


In [38]:

# Stage 3 — Step 3.4.13: Stealth-Level Formula Comparison
# Goal: Compare injection formulas by stealth level, natural P95
# deviations, final P95 deviations, and injection coverage.
# Read-only: No injection, feature modification, or file export.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 78)
print("STAGE 3 — STEP 3.4.13: STEALTH-LEVEL FORMULA COMPARISON")
print("=" * 78)

# ------------------------------------------------------------
# 1. Verify the experimental results from Step 3.4.3.
# ------------------------------------------------------------

formulas_3413 = [
    "A_P95_Addition",
    "B_Natural_Addition",
    "C_P95_Headroom"
]

levels_3413 = ["High", "Medium", "Low"]
channels_3413 = ["Email", "USB", "HTTP"]

audit_3413 = quantity_experiment_343.loc[
    quantity_experiment_343["formula"].isin(formulas_3413)
].copy()

assert len(audit_3413) == 14375 * 3

assert set(audit_3413["formula"]) == set(formulas_3413)

assert audit_3413["effective_stealth"].isin(
    levels_3413
).all()

assert audit_3413["channel"].isin(
    channels_3413
).all()

assert not audit_3413.duplicated(
    ["formula", "user", "day", "channel"]
).any()

assert audit_3413["candidate_p95"].gt(0).all()
assert audit_3413["injected_count"].ge(0).all()

assert (
    audit_3413["simulated_final_count"]
    == audit_3413["natural_count"]
    + audit_3413["injected_count"]
).all()

# ------------------------------------------------------------
# 2. Independently recalculate activity/P95 ratios.
# ------------------------------------------------------------

audit_3413["natural_ratio"] = (
    audit_3413["natural_count"]
    / audit_3413["candidate_p95"]
)

audit_3413["final_ratio"] = (
    audit_3413["simulated_final_count"]
    / audit_3413["candidate_p95"]
)

audit_3413["natural_above_p95"] = (
    audit_3413["natural_count"]
    > audit_3413["candidate_p95"] + 1e-9
)

audit_3413["final_above_p95"] = (
    audit_3413["simulated_final_count"]
    > audit_3413["candidate_p95"] + 1e-9
)

audit_3413["newly_above_p95"] = (
    ~audit_3413["natural_above_p95"]
    & audit_3413["final_above_p95"]
)

audit_3413["positive_injection"] = (
    audit_3413["injected_count"] > 0
)

# ------------------------------------------------------------
# 3. Overall summary by formula and stealth level.
# ------------------------------------------------------------

print("\nA. FORMULA COMPARISON BY STEALTH LEVEL")

summary_3413 = (
    audit_3413
    .groupby(
        ["formula", "effective_stealth"]
    )
    .agg(
        Channel_Days=("user", "size"),
        Users=("user", "nunique"),
        Positive_Injections=("positive_injection", "sum"),
        Natural_Above_P95=("natural_above_p95", "sum"),
        Newly_Above_P95=("newly_above_p95", "sum"),
        Final_Above_P95=("final_above_p95", "sum"),
        Median_Natural_Ratio=("natural_ratio", "median"),
        Median_Final_Ratio=("final_ratio", "median"),
        P25_Final_Ratio=(
            "final_ratio",
            lambda x: x.quantile(0.25)
        ),
        P75_Final_Ratio=(
            "final_ratio",
            lambda x: x.quantile(0.75)
        ),
        P90_Final_Ratio=(
            "final_ratio",
            lambda x: x.quantile(0.90)
        )
    )
    .reset_index()
)

summary_3413["Positive_Injection_%"] = (
    100 * summary_3413["Positive_Injections"]
    / summary_3413["Channel_Days"]
)

summary_3413["Final_Above_P95_%"] = (
    100 * summary_3413["Final_Above_P95"]
    / summary_3413["Channel_Days"]
)

summary_3413["Newly_Above_P95_%"] = (
    100 * summary_3413["Newly_Above_P95"]
    / summary_3413["Channel_Days"]
)

summary_3413["formula"] = pd.Categorical(
    summary_3413["formula"],
    categories=formulas_3413,
    ordered=True
)

summary_3413["effective_stealth"] = pd.Categorical(
    summary_3413["effective_stealth"],
    categories=levels_3413,
    ordered=True
)

summary_3413 = summary_3413.sort_values(
    ["formula", "effective_stealth"]
).reset_index(drop=True)

display(summary_3413.round(3))

# ------------------------------------------------------------
# 4. Classify final activity into descriptive P95 bands.
#
# These ranges are for comparison ONLY.
# They are not approved stealth definitions.
# ------------------------------------------------------------

print("\nB. FINAL/P95 RATIO DISTRIBUTION")

bins_3413 = [
    -np.inf,
    1.0,
    1.10,
    1.25,
    1.50,
    2.00,
    np.inf
]

labels_3413 = [
    "<= 1.00",
    "1.00 - 1.10",
    "1.10 - 1.25",
    "1.25 - 1.50",
    "1.50 - 2.00",
    "> 2.00"
]

audit_3413["ratio_band"] = pd.cut(
    audit_3413["final_ratio"],
    bins=bins_3413,
    labels=labels_3413,
    include_lowest=True
)

band_counts_3413 = pd.crosstab(
    [
        audit_3413["formula"],
        audit_3413["effective_stealth"]
    ],
    audit_3413["ratio_band"]
)

band_counts_3413 = band_counts_3413.reindex(
    pd.MultiIndex.from_product(
        [formulas_3413, levels_3413],
        names=["formula", "effective_stealth"]
    ),
    fill_value=0
).reindex(
    columns=labels_3413,
    fill_value=0
)

band_percentages_3413 = (
    band_counts_3413
    .div(band_counts_3413.sum(axis=1), axis=0)
    * 100
)

print("Percentage of channel-days in each final/P95 band:")
display(band_percentages_3413.round(2))

# ------------------------------------------------------------
# 5. Separate natural exceedances from new exceedances.
# ------------------------------------------------------------

print("\nC. NATURAL VS INJECTION-INDUCED EXCEEDANCES")

cause_3413 = summary_3413[
    [
        "formula",
        "effective_stealth",
        "Channel_Days",
        "Natural_Above_P95",
        "Newly_Above_P95",
        "Final_Above_P95"
    ]
].copy()

cause_3413["Within_P95_Final"] = (
    cause_3413["Channel_Days"]
    - cause_3413["Final_Above_P95"]
)

display(cause_3413)

# ------------------------------------------------------------
# 6. Compare channel-specific behavior.
# ------------------------------------------------------------

print("\nD. COMPARISON BY CHANNEL AND STEALTH")

by_channel_3413 = (
    audit_3413
    .groupby(
        ["formula", "effective_stealth", "channel"]
    )
    .agg(
        Channel_Days=("user", "size"),
        Median_Final_Ratio=("final_ratio", "median"),
        Natural_Above_P95=("natural_above_p95", "sum"),
        Newly_Above_P95=("newly_above_p95", "sum"),
        Final_Above_P95=("final_above_p95", "sum")
    )
    .reset_index()
)

by_channel_3413["Final_Above_P95_%"] = (
    100 * by_channel_3413["Final_Above_P95"]
    / by_channel_3413["Channel_Days"]
)

display(
    by_channel_3413
    .set_index(
        ["formula", "effective_stealth", "channel"]
    )
    .reindex(
        pd.MultiIndex.from_product(
            [formulas_3413, levels_3413, channels_3413],
            names=["formula", "effective_stealth", "channel"]
        )
    )
    .reset_index()
    .round(3)
)

# ------------------------------------------------------------
# 7. Evaluate hierarchy at the distribution-median level.
# This is descriptive, not a final policy decision.
# ------------------------------------------------------------

print("\nE. STEALTH HIERARCHY CHECK")

hierarchy_rows_3413 = []

for formula in formulas_3413:

    part = summary_3413.loc[
        summary_3413["formula"].eq(formula)
    ].set_index("effective_stealth")

    high_median = float(
        part.loc["High", "Median_Final_Ratio"]
    )
    medium_median = float(
        part.loc["Medium", "Median_Final_Ratio"]
    )
    low_median = float(
        part.loc["Low", "Median_Final_Ratio"]
    )

    hierarchy_rows_3413.append({
        "Formula": formula,
        "High_Median": high_median,
        "Medium_Median": medium_median,
        "Low_Median": low_median,
        "Strictly_Increasing": (
            high_median < medium_median < low_median
        )
    })

hierarchy_3413 = pd.DataFrame(hierarchy_rows_3413)

display(hierarchy_3413.round(3))

print(
    "\nNote: Increasing medians do not prove that the "
    "three distributions are well separated."
)

# ------------------------------------------------------------
# 8. Validate output completeness.
# ------------------------------------------------------------

assert len(summary_3413) == 9

assert len(by_channel_3413) == 27

assert (
    band_counts_3413.sum(axis=1).sum()
    == len(audit_3413)
)

assert audit_3413["final_ratio"].ge(
    audit_3413["natural_ratio"]
).all()

print("\n" + "=" * 78)
print("STEP 3.4.13 — FORMULA COMPARISON COMPLETED")
print("=" * 78)

print("Formulas compared: A / B / C")
print("Stealth levels: High / Medium / Low")
print("Final injection formula: NOT YET SELECTED")
print("Descriptive bands: NOT FINAL THRESHOLDS")
print("New behavioral features: NOT YET BUILT")
print("ML training: NOT STARTED")
print("Original datasets: UNCHANGED")
print("No files modified or exported.")


STAGE 3 — STEP 3.4.13: STEALTH-LEVEL FORMULA COMPARISON

A. FORMULA COMPARISON BY STEALTH LEVEL


,formula,effective_stealth,Channel_Days,Users,Positive_Injections,Natural_Above_P95,Newly_Above_P95,Final_Above_P95,Median_Natural_Ratio,Median_Final_Ratio,P25_Final_Ratio,P75_Final_Ratio,P90_Final_Ratio,Positive_Injection_%,Final_Above_P95_%,Newly_Above_P95_%
0,A_P95_Addition,High,4413,217,4413,157,1611,1768,0.5,1.00,0.800,1.250,1.5,100.000,40.063,36.506
1,A_P95_Addition,Medium,5023,226,5023,191,4832,5023,0.5,1.50,1.333,1.750,2.0,100.000,100.000,96.197
2,A_P95_Addition,Low,4939,227,4939,196,4743,4939,0.5,2.00,1.800,2.250,2.5,100.000,100.000,96.032
3,B_Natural_Addition,High,4413,217,4413,157,986,1143,0.5,0.75,0.500,1.116,1.5,100.000,25.901,22.343
4,B_Natural_Addition,Medium,5023,226,5023,191,1793,1984,0.5,1.00,0.533,1.460,2.0,100.000,39.498,35.696
5,B_Natural_Addition,Low,4939,227,4939,196,2762,2958,0.5,1.25,0.750,2.000,2.5,100.000,59.891,55.922
6,C_P95_Headroom,High,4413,217,1409,157,0,157,0.5,0.50,0.500,0.714,1.0,31.928,3.558,0.000
7,C_P95_Headroom,Medium,5023,226,4417,191,0,191,0.5,1.00,1.000,1.000,1.0,87.935,3.803,0.000
8,C_P95_Headroom,Low,4939,227,4827,196,4696,4892,0.5,1.50,1.400,1.500,1.5,97.732,99.048,95.080



B. FINAL/P95 RATIO DISTRIBUTION
Percentage of channel-days in each final/P95 band:


ratio_band                            <= 1.00  1.00 - 1.10  1.10 - 1.25  \
formula            effective_stealth                                      
A_P95_Addition     High                 59.94         2.06        16.27   
                   Medium                0.00         0.02        22.97   
                   Low                   0.00         0.00         0.00   
B_Natural_Addition High                 74.10         0.75        11.04   
                   Medium               60.50         1.17         6.23   
                   Low                  40.11         0.49        14.29   
C_P95_Headroom     High                 96.44         0.20         1.86   
                   Medium               96.20         0.30         2.37   
                   Low                   0.95         0.00         0.97   

ratio_band                            1.25 - 1.50  1.50 - 2.00  > 2.00  
formula            effective_stealth                                    
A_P95_Addition     High                     14.53         6.87    0.34  
                   Medium                   35.70        36.83    4.48  
                   Low                       0.00        57.36   42.64  
B_Natural_Addition High                      7.39         5.82    0.91  
                   Medium                   14.13        14.15    3.80  
                   Low                       8.38        21.22   15.51  
C_P95_Headroom     High                      1.25         0.23    0.02  
                   Medium                    0.92         0.22    0.00  
                   Low                      97.65         0.40    0.02


C. NATURAL VS INJECTION-INDUCED EXCEEDANCES


,formula,effective_stealth,Channel_Days,Natural_Above_P95,Newly_Above_P95,Final_Above_P95,Within_P95_Final
0,A_P95_Addition,High,4413,157,1611,1768,2645
1,A_P95_Addition,Medium,5023,191,4832,5023,0
2,A_P95_Addition,Low,4939,196,4743,4939,0
3,B_Natural_Addition,High,4413,157,986,1143,3270
4,B_Natural_Addition,Medium,5023,191,1793,1984,3039
5,B_Natural_Addition,Low,4939,196,2762,2958,1981
6,C_P95_Headroom,High,4413,157,0,157,4256
7,C_P95_Headroom,Medium,5023,191,0,191,4832
8,C_P95_Headroom,Low,4939,196,4696,4892,47



D. COMPARISON BY CHANNEL AND STEALTH


,formula,effective_stealth,channel,Channel_Days,Median_Final_Ratio,Natural_Above_P95,Newly_Above_P95,Final_Above_P95,Final_Above_P95_%
0,A_P95_Addition,High,Email,2949,1.000,87,948,1035,35.097
1,A_P95_Addition,High,USB,1038,1.000,45,453,498,47.977
2,A_P95_Addition,High,HTTP,426,1.121,25,210,235,55.164
3,A_P95_Addition,Medium,Email,3079,1.500,122,2957,3079,100.000
4,A_P95_Addition,Medium,USB,1417,1.500,44,1373,1417,100.000
5,A_P95_Addition,Medium,HTTP,527,1.583,25,502,527,100.000
6,A_P95_Addition,Low,Email,3315,2.000,134,3181,3315,100.000
7,A_P95_Addition,Low,USB,1044,2.029,45,999,1044,100.000
8,A_P95_Addition,Low,HTTP,580,2.088,17,563,580,100.000
9,B_Natural_Addition,High,Email,2949,0.750,87,672,759,25.738



E. STEALTH HIERARCHY CHECK


,Formula,High_Median,Medium_Median,Low_Median,Strictly_Increasing
0,A_P95_Addition,1.00,1.5,2.00,True
1,B_Natural_Addition,0.75,1.0,1.25,True
2,C_P95_Headroom,0.50,1.0,1.50,True



Note: Increasing medians do not prove that the three distributions are well separated.

STEP 3.4.13 — FORMULA COMPARISON COMPLETED
Formulas compared: A / B / C
Stealth levels: High / Medium / Low
Final injection formula: NOT YET SELECTED
Descriptive bands: NOT FINAL THRESHOLDS
New behavioral features: NOT YET BUILT
ML training: NOT STARTED
Original datasets: UNCHANGED
No files modified or exported.


قررت نكمل الميزات الجديدة اولا ثم نختار افضل معادلة


### Step 3.4.14 — New Behavioral Feature Definitions

**Objective:** Define consistent temporal and cross-channel features for comparing natural behavior with the three experimental injection formulas before selecting the final simulation policy.

**Approved Rolling Windows:** 5, 10, 20, and 30 eligible workdays.

**New Feature Groups:**

1. **Cumulative Deviation:** Rolling sum of positive deviations from each user's historical active-day P50, normalized using the P50-to-P95 range.
2. **Combined Channel Intensity:** Rolling mean of daily channel activity normalized by channel-specific P95.
3. **Dominant Channel Switches:** Number of changes in the dominant observed channel, determined from normalized activity rather than simulated attack labels.
4. **Channel Distribution Shift:** Total variation distance between the rolling distribution of normalized channel activity and the user's historical channel distribution.
5. **Consecutive Elevated Days:** Longest consecutive run of workdays with activity exceeding historical P50 in at least one channel.
6. **Cross-Channel Deviation Days:** Number of workdays with activity exceeding historical P50 in at least two channels.
7. **Rolling Channel Totals:** Additional 20- and 30-workday sums for Email, USB, and HTTP event counts.

**Feature Count:** 24 new temporal/cross-channel features plus 6 additional rolling-count features, giving 30 proposed new features.

**Implementation Requirements:**
- Use current and previous eligible workdays only.
- Do not use simulated primary channels or ground-truth labels as feature inputs.
- Handle missing channel baselines explicitly.
- Track partial-window coverage.
- Apply identical calculations to benign, Formula A, Formula B, and Formula C data.
- Preserve original byte values and historical baselines.

**Expected Outcome:** A fixed feature specification suitable for a fair, model-independent comparison of injection formulas.

**Status:** Definitions proposed; numerical implementation and validation pending.



### Step 3.4.15 — Feature Input & Baseline Coverage Audit

**Objective:** Validate the historical baseline inputs and eligible workday coverage required for the new temporal and cross-channel behavioral features.

**Approved Windows:** 5, 10, 20, and 30 eligible workdays.

**Validation Tasks:**
1. Verify historical P50, P95, active-day mean, active rate, and active-day count for Email, USB, and HTTP.
2. Identify missing, nonpositive, or inconsistent baseline values.
3. Measure the number of usable baseline channels per user.
4. Identify future active channel-days without a usable historical baseline.
5. Evaluate the availability of historical channel-distribution inputs.
6. Measure complete versus partial rolling-window coverage.
7. Confirm that Formula A, B, and C feature datasets use identical eligible user-day keys.

**Methodological Rules:**
- Preserve historical baselines.
- Do not replace missing channel baselines with zero.
- Do not use attack labels, planned primary channels, or future activity.
- Do not infer payload sizes from event counts.
- Do not modify or export any data.

**Expected Outcome:** Establish which new behavioral features can be calculated reliably and identify cases requiring explicit missing-baseline handling.


In [40]:

# Stage 3 — Step 3.4.15: Feature Input & Baseline Coverage Audit
# Goal: Validate historical channel baselines, feature-input
# availability, and rolling-window coverage before building
# new 5/10/20/30 workday behavioral features.
# Corrected: Consistent reference_keys_3415 variable naming.
# Read-only: No data, schedules, features, or files modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 78)
print("STAGE 3 — STEP 3.4.15: FEATURE INPUT & BASELINE COVERAGE")
print("=" * 78)

# ------------------------------------------------------------
# 1. Define baseline columns and activity sources.
# ------------------------------------------------------------

WINDOWS_3415 = [5, 10, 20, 30]

CHANNELS_3415 = {
    "Email": {
        "count": "external_email_count",
        "p50": "email_count_p50",
        "p95": "email_count_p95",
        "mean": "email_count_mean_active",
        "rate": "email_active_rate",
        "days": "email_active_days"
    },
    "USB": {
        "count": "usb_copy_count",
        "p50": "usb_copy_p50",
        "p95": "usb_copy_p95",
        "mean": "usb_copy_mean_active",
        "rate": "usb_active_rate",
        "days": "usb_active_days"
    },
    "HTTP": {
        "count": "www_upload_count",
        "p50": "http_upload_p50",
        "p95": "http_upload_p95",
        "mean": "http_upload_mean_active",
        "rate": "http_upload_active_rate",
        "days": "http_upload_active_days"
    }
}

KEYS_3415 = ["user", "day"]

assert user_baseline["user"].is_unique

required_baseline_3415 = {
    "user", "baseline_workdays"
}

for spec in CHANNELS_3415.values():
    required_baseline_3415.update(
        spec[key]
        for key in ["p50", "p95", "mean", "rate", "days"]
    )

missing_3415 = (
    required_baseline_3415
    - set(user_baseline.columns)
)

assert not missing_3415, (
    f"Missing baseline fields: {sorted(missing_3415)}"
)

# ------------------------------------------------------------
# 2. Prepare original eligible user-days.
# ------------------------------------------------------------

eligible_3415 = original_features_3412[
    ["user", "day", "is_weekend", "logon_count"]
    + [spec["count"] for spec in CHANNELS_3415.values()]
].copy()

eligible_3415["day"] = pd.to_datetime(
    eligible_3415["day"]
).dt.normalize()

eligible_3415 = (
    eligible_3415
    .sort_values(KEYS_3415)
    .reset_index(drop=True)
)

assert len(eligible_3415) == 69634
assert eligible_3415["user"].nunique() == 670

assert not eligible_3415.duplicated(KEYS_3415).any()

assert eligible_3415["day"].ge(
    pd.Timestamp("2011-01-01")
).all()

assert eligible_3415["is_weekend"].eq(0).all()
assert eligible_3415["logon_count"].gt(0).all()

# ------------------------------------------------------------
# 3. Prepare the historical baseline.
# ------------------------------------------------------------

selected_users_3415 = (
    eligible_3415["user"].drop_duplicates()
)

baseline_3415 = user_baseline.loc[
    user_baseline["user"].isin(selected_users_3415),
    sorted(required_baseline_3415)
].copy()

assert len(baseline_3415) == 670
assert baseline_3415["user"].nunique() == 670
assert baseline_3415["baseline_workdays"].gt(0).all()

# ------------------------------------------------------------
# 4. Evaluate historical baselines by channel.
# ------------------------------------------------------------

baseline_parts_3415 = []

for channel, spec in CHANNELS_3415.items():

    part = baseline_3415[
        ["user"] + [
            spec[key]
            for key in ["p50", "p95", "mean", "rate", "days"]
        ]
    ].copy()

    part.columns = [
        "user",
        "p50",
        "p95",
        "mean_active",
        "active_rate",
        "active_days"
    ]

    part["channel"] = channel

    for col in [
        "p50", "p95", "mean_active",
        "active_rate", "active_days"
    ]:
        part[col] = pd.to_numeric(
            part[col],
            errors="coerce"
        )

    p50 = part["p50"]
    p95 = part["p95"]
    mean = part["mean_active"]
    rate = part["active_rate"]
    days = part["active_days"]

    part["valid_p95"] = (
        np.isfinite(p95) & p95.gt(0)
    )

    part["valid_p50_p95"] = (
        np.isfinite(p50)
        & np.isfinite(p95)
        & p50.gt(0)
        & p95.ge(p50)
    )

    part["valid_history"] = (
        np.isfinite(mean)
        & mean.gt(0)
        & np.isfinite(rate)
        & rate.gt(0)
        & rate.le(1)
        & np.isfinite(days)
        & days.gt(0)
    )

    part["usable_intensity"] = part["valid_p95"]

    part["usable_deviation"] = (
        part["valid_p50_p95"]
    )

    part["usable_historical_share"] = (
        part["valid_p95"]
        & part["valid_history"]
    )

    part["zero_percentile_gap"] = (
        part["valid_p50_p95"]
        & np.isclose(
            p50.to_numpy(dtype=float),
            p95.to_numpy(dtype=float),
            rtol=0,
            atol=1e-9,
            equal_nan=False
        )
    )

    baseline_parts_3415.append(part)

channel_baseline_3415 = pd.concat(
    baseline_parts_3415,
    ignore_index=True
)

assert len(channel_baseline_3415) == 670 * 3

assert not channel_baseline_3415.duplicated(
    ["user", "channel"]
).any()

# ------------------------------------------------------------
# 5. Baseline coverage by channel.
# ------------------------------------------------------------

print("\nA. HISTORICAL BASELINE COVERAGE BY CHANNEL")

coverage_3415 = (
    channel_baseline_3415
    .groupby("channel")
    .agg(
        Users=("user", "nunique"),
        Valid_P95=("valid_p95", "sum"),
        Usable_Deviation=("usable_deviation", "sum"),
        Usable_Historical_Share=(
            "usable_historical_share", "sum"
        ),
        Valid_History=("valid_history", "sum"),
        Zero_P50_P95_Gap=("zero_percentile_gap", "sum")
    )
    .reindex(["Email", "USB", "HTTP"])
    .reset_index()
)

coverage_3415["Missing_Or_Invalid_P95"] = (
    coverage_3415["Users"]
    - coverage_3415["Valid_P95"]
)

coverage_3415["P95_Coverage_%"] = (
    100
    * coverage_3415["Valid_P95"]
    / coverage_3415["Users"]
).round(2)

display(coverage_3415)

# ------------------------------------------------------------
# 6. Count usable baseline channels per user.
# ------------------------------------------------------------

print("\nB. USABLE BASELINE CHANNELS PER USER")

user_coverage_3415 = (
    channel_baseline_3415
    .groupby("user")
    .agg(
        Usable_Intensity_Channels=(
            "usable_intensity", "sum"
        ),
        Usable_Deviation_Channels=(
            "usable_deviation", "sum"
        ),
        Usable_Share_Channels=(
            "usable_historical_share", "sum"
        )
    )
    .reset_index()
)

distribution_rows_3415 = []

for field in [
    "Usable_Intensity_Channels",
    "Usable_Deviation_Channels",
    "Usable_Share_Channels"
]:

    for n in range(4):

        distribution_rows_3415.append({
            "Baseline_Type": field,
            "Usable_Channels": n,
            "Users": int(
                user_coverage_3415[field].eq(n).sum()
            )
        })

user_distribution_3415 = pd.DataFrame(
    distribution_rows_3415
)

display(user_distribution_3415)

# ------------------------------------------------------------
# 7. Future activity lacking a usable historical baseline.
# ------------------------------------------------------------

print("\nC. FUTURE ACTIVITY WITHOUT USABLE HISTORICAL BASELINE")

activity_rows_3415 = []

for channel, spec in CHANNELS_3415.items():

    channel_history = channel_baseline_3415.loc[
        channel_baseline_3415["channel"].eq(channel),
        [
            "user",
            "usable_intensity",
            "usable_deviation",
            "usable_historical_share"
        ]
    ]

    daily = eligible_3415[
        ["user", "day", spec["count"]]
    ].merge(
        channel_history,
        on="user",
        how="left",
        validate="many_to_one"
    )

    assert daily["usable_intensity"].notna().all()

    active = daily[spec["count"]].gt(0)

    activity_rows_3415.append({
        "Channel": channel,
        "Eligible_User_Days": len(daily),
        "Naturally_Active_Days": int(active.sum()),
        "Active_Days_Without_P95": int(
            (active & ~daily["usable_intensity"]).sum()
        ),
        "Active_Days_Without_Deviation_Baseline": int(
            (active & ~daily["usable_deviation"]).sum()
        ),
        "Active_Days_Without_Share_Baseline": int(
            (
                active
                & ~daily["usable_historical_share"]
            ).sum()
        ),
        "Users_Active_Without_P95": int(
            daily.loc[
                active & ~daily["usable_intensity"],
                "user"
            ].nunique()
        )
    })

future_coverage_3415 = pd.DataFrame(
    activity_rows_3415
)

display(future_coverage_3415)

# ------------------------------------------------------------
# 8. Inspect historical channel-distribution inputs.
# ------------------------------------------------------------

print("\nD. HISTORICAL CHANNEL-DISTRIBUTION INPUTS")

share_inputs_3415 = channel_baseline_3415.copy()

share_inputs_3415["historical_normalized_activity"] = (
    share_inputs_3415["active_rate"]
    * share_inputs_3415["mean_active"]
    / share_inputs_3415["p95"]
)

share_inputs_3415.loc[
    ~share_inputs_3415["usable_historical_share"],
    "historical_normalized_activity"
] = np.nan

valid_share_values_3415 = (
    share_inputs_3415.loc[
        share_inputs_3415["usable_historical_share"],
        "historical_normalized_activity"
    ]
)

assert np.isfinite(valid_share_values_3415).all()
assert valid_share_values_3415.gt(0).all()

print(
    "Users with all three historical share inputs:",
    int(
        user_coverage_3415[
            "Usable_Share_Channels"
        ].eq(3).sum()
    )
)

print(
    "Users with at least two historical share inputs:",
    int(
        user_coverage_3415[
            "Usable_Share_Channels"
        ].ge(2).sum()
    )
)

print(
    "Users with only one historical share input:",
    int(
        user_coverage_3415[
            "Usable_Share_Channels"
        ].eq(1).sum()
    )
)

print(
    "Users without any historical share input:",
    int(
        user_coverage_3415[
            "Usable_Share_Channels"
        ].eq(0).sum()
    )
)

# ------------------------------------------------------------
# 9. Rolling-window completeness in eligible workdays.
# ------------------------------------------------------------

print("\nE. ROLLING-WINDOW COMPLETENESS")

eligible_3415["eligible_day_position"] = (
    eligible_3415.groupby("user").cumcount() + 1
)

window_rows_3415 = []

for window in WINDOWS_3415:

    complete = (
        eligible_3415["eligible_day_position"] >= window
    )

    window_rows_3415.append({
        "Window_WD": window,
        "Eligible_User_Days": len(eligible_3415),
        "Complete_Windows": int(complete.sum()),
        "Partial_Windows": int((~complete).sum()),
        "Complete_%": round(
            100 * complete.mean(), 2
        )
    })

window_coverage_3415 = pd.DataFrame(
    window_rows_3415
)

display(window_coverage_3415)

# ------------------------------------------------------------
# 10. Verify identical eligible user-day keys for
#     original, Formula A, Formula B, and Formula C.
# ------------------------------------------------------------

print("\nF. EXPERIMENTAL KEY ALIGNMENT")

reference_keys_3415 = pd.MultiIndex.from_frame(
    eligible_3415[KEYS_3415]
)

alignment_rows_3415 = []

for formula, frame in feature_variants_3412.items():

    variant_keys_3415 = frame[KEYS_3415].copy()

    variant_keys_3415["day"] = pd.to_datetime(
        variant_keys_3415["day"]
    ).dt.normalize()

    variant_keys_3415 = pd.MultiIndex.from_frame(
        variant_keys_3415
    )

    same_keys_3415 = (
        variant_keys_3415.is_unique
        and reference_keys_3415.equals(
            variant_keys_3415
        )
    )

    alignment_rows_3415.append({
        "Formula": formula,
        "Users": frame["user"].nunique(),
        "Eligible_User_Days": len(frame),
        "Keys_Aligned": bool(same_keys_3415)
    })

alignment_3415 = pd.DataFrame(
    alignment_rows_3415
)

display(alignment_3415)

assert alignment_3415["Keys_Aligned"].all(), (
    "Experimental user-day keys do not match the benign reference."
)

# ------------------------------------------------------------
# 11. Review missing or zero-gap baseline cases.
# ------------------------------------------------------------

print("\nG. BASELINE CASES REQUIRING REVIEW")

review_3415 = channel_baseline_3415.loc[
    ~channel_baseline_3415["usable_deviation"]
    | ~channel_baseline_3415["usable_historical_share"]
    | channel_baseline_3415["zero_percentile_gap"]
].copy()

print(
    "User-channel combinations requiring review:",
    len(review_3415)
)

if not review_3415.empty:

    display(
        review_3415[
            [
                "user", "channel",
                "p50", "p95",
                "mean_active", "active_rate",
                "active_days",
                "usable_intensity",
                "usable_deviation",
                "usable_historical_share",
                "zero_percentile_gap"
            ]
        ].head(25)
    )

# ------------------------------------------------------------
# 12. Final safeguards and status.
# ------------------------------------------------------------

assert len(coverage_3415) == 3
assert len(user_coverage_3415) == 670
assert len(window_coverage_3415) == 4
assert len(alignment_3415) == 3

print("\n" + "=" * 78)
print("STEP 3.4.15 — BASELINE COVERAGE AUDIT COMPLETED")
print("=" * 78)

print("Historical baselines: AUDITED")
print("Missing-baseline handling: NOT YET FINALIZED")
print("Historical distribution shares: NOT YET CALCULATED")
print("Partial windows: IDENTIFIED")
print("New behavioral features: NOT YET BUILT")
print("Approved windows: 5 / 10 / 20 / 30 WD")
print("Experimental formulas: A / B / C — UNCHANGED")
print("Final injection policy: NOT YET SELECTED")
print("Original datasets: UNCHANGED")
print("No files modified or exported.")


STAGE 3 — STEP 3.4.15: FEATURE INPUT & BASELINE COVERAGE

A. HISTORICAL BASELINE COVERAGE BY CHANNEL


,channel,Users,Valid_P95,Usable_Deviation,Usable_Historical_Share,Valid_History,Zero_P50_P95_Gap,Missing_Or_Invalid_P95,P95_Coverage_%
0,Email,670,670,670,670,670,40,0,100.00
1,USB,670,500,500,500,500,1,170,74.63
2,HTTP,670,339,339,339,339,0,331,50.60



B. USABLE BASELINE CHANNELS PER USER


,Baseline_Type,Usable_Channels,Users
0,Usable_Intensity_Channels,0,0
1,Usable_Intensity_Channels,1,0
2,Usable_Intensity_Channels,2,501
3,Usable_Intensity_Channels,3,169
4,Usable_Deviation_Channels,0,0
5,Usable_Deviation_Channels,1,0
6,Usable_Deviation_Channels,2,501
7,Usable_Deviation_Channels,3,169
8,Usable_Share_Channels,0,0
9,Usable_Share_Channels,1,0



C. FUTURE ACTIVITY WITHOUT USABLE HISTORICAL BASELINE


,Channel,Eligible_User_Days,Naturally_Active_Days,Active_Days_Without_P95,Active_Days_Without_Deviation_Baseline,Active_Days_Without_Share_Baseline,Users_Active_Without_P95
0,Email,69634,46973,0,0,0,0
1,USB,69634,24856,0,0,0,0
2,HTTP,69634,8612,0,0,0,0


... (3 outputs left out)

,Formula,Users,Eligible_User_Days,Keys_Aligned
0,A_P95_Addition,670,69634,True
1,B_Natural_Addition,670,69634,True
2,C_P95_Headroom,670,69634,True



G. BASELINE CASES REQUIRING REVIEW
User-channel combinations requiring review: 542


,user,channel,p50,p95,mean_active,active_rate,active_days,usable_intensity,usable_deviation,usable_historical_share,zero_percentile_gap
19,AFB0904,Email,1.0,1.0,1.0,0.143426,36,True,True,True,True
34,AME1490,Email,1.0,1.0,1.0,0.213178,55,True,True,True,True
62,BDB2163,Email,1.0,1.0,1.0,0.177606,46,True,True,True,True
72,BKF1545,Email,1.0,1.0,1.0,0.177866,45,True,True,True,True
82,BRG3267,Email,1.0,1.0,1.0,0.179283,45,True,True,True,True
91,BYS3626,Email,1.0,1.0,1.0,0.191406,49,True,True,True,True
156,DHL2219,Email,1.0,1.0,1.0,0.159363,40,True,True,True,True
193,ESS3318,Email,1.0,1.0,1.0,0.187251,47,True,True,True,True
195,EYD2871,Email,1.0,1.0,1.0,0.135458,34,True,True,True,True
209,FRB3653,Email,1.0,1.0,1.0,0.119522,30,True,True,True,True



STEP 3.4.15 — BASELINE COVERAGE AUDIT COMPLETED
Historical baselines: AUDITED
Missing-baseline handling: NOT YET FINALIZED
Historical distribution shares: NOT YET CALCULATED
Partial windows: IDENTIFIED
New behavioral features: NOT YET BUILT
Approved windows: 5 / 10 / 20 / 30 WD
Experimental formulas: A / B / C — UNCHANGED
Final injection policy: NOT YET SELECTED
Original datasets: UNCHANGED
No files modified or exported.



### Step 3.4.16 — Cumulative Behavioral Deviation Features

**Objective:** Calculate cumulative behavioral deviations across Email, USB, and HTTP to measure gradual activity changes over time.

**Datasets:** Natural, Formula A, and Formula B.

**Rolling Windows:** 5, 10, 20, and 30 eligible workdays.

**Calculation:**

For each user, channel, and day:

`Daily Deviation = max(0, (Daily Count - Historical P50) / max(P95 - P50, 1))`

The deviations from Email, USB, and HTTP are summed to obtain the daily combined deviation.

The daily values are then accumulated over each rolling window.

**New Features:**
- `cumulative_deviation_5wd`
- `cumulative_deviation_10wd`
- `cumulative_deviation_20wd`
- `cumulative_deviation_30wd`

**Implementation Rules:**
- Use the original historical P50 and P95 values.
- Treat zero activity in a verified historically inactive channel as zero deviation, without inventing a baseline.
- Require a valid baseline for positive channel activity.
- Use current and previous eligible workdays only.
- Return NaN for incomplete rolling windows to avoid comparing different window lengths.
- Apply identical calculations to Natural, A, and B.
- Preserve all 85 original feature columns.
- Do not modify source files or export datasets.

**Expected Outcome:** Four additional behavioral features per dataset, with validation and comparison of their distributions.


In [41]:

# Stage 3 — Step 3.4.16: Cumulative Behavioral Deviation
# Goal: Calculate cumulative deviation features over
# 5, 10, 20, and 30 eligible workdays.
#
# Datasets: Natural, Formula A, Formula B.
# Read-only: Original data and files remain unchanged.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 78)
print("STAGE 3 — STEP 3.4.16: CUMULATIVE BEHAVIORAL DEVIATION")
print("=" * 78)

# ------------------------------------------------------------
# 1. Select the approved experimental datasets.
# ------------------------------------------------------------

datasets_3416 = {
    "Natural": original_features_3412,
    "Formula A": feature_variants_3412["A_P95_Addition"],
    "Formula B": feature_variants_3412["B_Natural_Addition"]
}

WINDOWS_3416 = [5, 10, 20, 30]

CHANNELS_3416 = {
    "email": (
        "external_email_count",
        "email_count_p50",
        "email_count_p95"
    ),
    "usb": (
        "usb_copy_count",
        "usb_copy_p50",
        "usb_copy_p95"
    ),
    "http": (
        "www_upload_count",
        "http_upload_p50",
        "http_upload_p95"
    )
}

KEYS_3416 = ["user", "day"]

NEW_FEATURES_3416 = [
    f"cumulative_deviation_{w}wd"
    for w in WINDOWS_3416
]

# ------------------------------------------------------------
# 2. Create one reusable feature calculation function.
# ------------------------------------------------------------

def build_cumulative_deviation_3416(source):

    # Never modify the original feature dataframe.
    df = source.copy(deep=True)

    original_columns = list(df.columns)

    assert len(original_columns) == 85

    df["day"] = pd.to_datetime(
        df["day"]
    ).dt.normalize()

    df = df.sort_values(
        KEYS_3416
    ).reset_index(drop=True)

    assert not df.duplicated(KEYS_3416).any()

    assert df["user"].nunique() == 670
    assert len(df) == 69634

    # Daily combined deviation starts at zero.
    df["_daily_combined_deviation_3416"] = 0.0

    # --------------------------------------------------------
    # 3. Calculate deviation for each channel.
    # --------------------------------------------------------

    for channel, columns in CHANNELS_3416.items():

        count_col, p50_col, p95_col = columns

        count = pd.to_numeric(
            df[count_col],
            errors="raise"
        )

        p50 = pd.to_numeric(
            df[p50_col],
            errors="raise"
        )

        p95 = pd.to_numeric(
            df[p95_col],
            errors="raise"
        )

        # Valid historical baseline.
        valid_baseline = (
            p50.notna()
            & p95.notna()
            & p50.gt(0)
            & p95.gt(0)
            & p95.ge(p50)
        )

        # Positive activity requires a valid baseline.
        invalid_active = (
            count.gt(0)
            & ~valid_baseline
        )

        assert not invalid_active.any(), (
            f"{channel}: Positive activity without "
            "a valid historical baseline."
        )

        assert count.notna().all()
        assert count.ge(0).all()

        # Handle equal P50/P95 values using a minimum
        # normalization denominator of 1.
        denominator = (
            (p95 - p50).clip(lower=1.0)
        )

        deviation = (
            (count - p50)
            / denominator
        ).clip(lower=0)

        # Missing baseline is acceptable here only
        # when the observed channel count is zero.
        deviation = deviation.where(
            valid_baseline,
            0.0
        )

        assert np.isfinite(deviation).all()
        assert deviation.ge(0).all()

        # Add the channel contribution to the daily total.
        df["_daily_combined_deviation_3416"] += (
            deviation.to_numpy(dtype=float)
        )

    # --------------------------------------------------------
    # 4. Calculate complete causal rolling windows.
    #
    # Window W includes today + previous W-1 eligible
    # workdays for the same user.
    #
    # Incomplete windows remain NaN.
    # --------------------------------------------------------

    daily_col = "_daily_combined_deviation_3416"

    for window in WINDOWS_3416:

        feature_name = (
            f"cumulative_deviation_{window}wd"
        )

        df[feature_name] = (
            df.groupby(
                "user",
                sort=False
            )[daily_col]
            .transform(
                lambda s, w=window: (
                    s.rolling(
                        window=w,
                        min_periods=w
                    ).sum()
                )
            )
        )

        # Validate missing values on partial windows.
        position = (
            df.groupby("user").cumcount() + 1
        )

        expected_complete = position.ge(window)

        assert df.loc[
            expected_complete, feature_name
        ].notna().all()

        assert df.loc[
            ~expected_complete, feature_name
        ].isna().all()

        assert df[feature_name].dropna().ge(0).all()

    # --------------------------------------------------------
    # 5. Preserve original 85 columns unchanged.
    # --------------------------------------------------------

    df = df.drop(
        columns=[daily_col]
    )

    assert len(df.columns) == 89

    assert list(df.columns[:85]) == original_columns

    return df


# ------------------------------------------------------------
# 6. Apply exactly the same calculations to all datasets.
# ------------------------------------------------------------

new_features_3416 = {}

for name, source in datasets_3416.items():

    result = build_cumulative_deviation_3416(source)

    # Independently verify all original 85 columns.
    original_sorted = (
        source.copy()
        .assign(
            day=lambda d: pd.to_datetime(
                d["day"]
            ).dt.normalize()
        )
        .sort_values(KEYS_3416)
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        result[original_sorted.columns],
        original_sorted,
        check_exact=True
    )

    new_features_3416[name] = result

    print(
        f"PASS — {name}: "
        f"{len(result):,} user-days, "
        f"{len(result.columns)} columns"
    )


# ------------------------------------------------------------
# 7. Verify all three datasets have identical keys.
# ------------------------------------------------------------

reference_keys_3416 = (
    new_features_3416["Natural"][KEYS_3416]
    .reset_index(drop=True)
)

for name, frame in new_features_3416.items():

    pd.testing.assert_frame_equal(
        frame[KEYS_3416].reset_index(drop=True),
        reference_keys_3416,
        check_exact=True
    )

print("\nUser-day key alignment: PASS")


# ------------------------------------------------------------
# 8. Summarize each new feature by dataset.
# ------------------------------------------------------------

print("\nA. CUMULATIVE DEVIATION DISTRIBUTIONS")

summary_rows_3416 = []

for name, frame in new_features_3416.items():

    for window in WINDOWS_3416:

        col = f"cumulative_deviation_{window}wd"

        values = frame[col].dropna()

        summary_rows_3416.append({
            "Dataset": name,
            "Window_WD": window,
            "Complete_Rows": len(values),
            "Mean": round(values.mean(), 4),
            "Median": round(values.median(), 4),
            "P90": round(values.quantile(0.90), 4),
            "P95": round(values.quantile(0.95), 4),
            "Max": round(values.max(), 4)
        })

summary_3416 = pd.DataFrame(
    summary_rows_3416
)

display(summary_3416)


# ------------------------------------------------------------
# 9. Compare A and B against Natural on the same days.
# ------------------------------------------------------------

print("\nB. FEATURE CHANGES COMPARED WITH NATURAL")

natural_3416 = new_features_3416["Natural"]

comparison_rows_3416 = []

for name in ["Formula A", "Formula B"]:

    variant_3416 = new_features_3416[name]

    for window in WINDOWS_3416:

        col = f"cumulative_deviation_{window}wd"

        natural_values = natural_3416[col]
        injected_values = variant_3416[col]

        complete = (
            natural_values.notna()
            & injected_values.notna()
        )

        difference = (
            injected_values[complete]
            - natural_values[complete]
        )

        assert difference.ge(-1e-9).all(), (
            f"Unexpected decrease: {name}, {col}"
        )

        comparison_rows_3416.append({
            "Formula": name,
            "Window_WD": window,
            "Compared_Rows": int(complete.sum()),
            "Increased_Rows": int(
                difference.gt(1e-9).sum()
            ),
            "Unchanged_Rows": int(
                np.isclose(
                    difference,
                    0,
                    atol=1e-9,
                    rtol=0
                ).sum()
            ),
            "Mean_Added_Deviation": round(
                difference.mean(), 4
            ),
            "Max_Added_Deviation": round(
                difference.max(), 4
            )
        })

comparison_3416 = pd.DataFrame(
    comparison_rows_3416
)

display(comparison_3416)


# ------------------------------------------------------------
# 10. Final validation.
# ------------------------------------------------------------

assert len(new_features_3416) == 3

assert all(
    len(frame.columns) == 89
    for frame in new_features_3416.values()
)

assert all(
    set(NEW_FEATURES_3416).issubset(frame.columns)
    for frame in new_features_3416.values()
)

print("\n" + "=" * 78)
print("STEP 3.4.16 — CUMULATIVE DEVIATION COMPLETED")
print("=" * 78)

print("Datasets: Natural / Formula A / Formula B")
print("Original 85 columns: PRESERVED")
print("New cumulative-deviation features: 4")
print("Total columns per dataset: 89")
print("Rolling windows: 5 / 10 / 20 / 30 WD")
print("Incomplete windows: NaN")
print("Historical baselines: UNCHANGED")
print("Original datasets: UNCHANGED")
print("No files modified or exported.")


STAGE 3 — STEP 3.4.16: CUMULATIVE BEHAVIORAL DEVIATION
PASS — Natural: 69,634 user-days, 89 columns
PASS — Formula A: 69,634 user-days, 89 columns
PASS — Formula B: 69,634 user-days, 89 columns

User-day key alignment: PASS

A. CUMULATIVE DEVIATION DISTRIBUTIONS


,Dataset,Window_WD,Complete_Rows,Mean,Median,P90,P95,Max
0,Natural,5,66954,1.4560,1.2766,3.0000,3.5135,8.9922
1,Natural,10,63606,2.9080,2.6667,5.2500,6.0833,12.4574
2,Natural,20,56920,5.8128,5.5000,9.5000,10.9634,19.5000
3,Natural,30,50241,8.6672,8.3276,13.5000,15.3759,27.2500
4,Formula A,5,66954,3.4189,1.6667,10.0000,13.6667,41.6264
5,Formula A,10,63606,6.9145,3.5000,19.5000,26.0000,65.6667
6,Formula A,20,56920,14.0716,7.5000,36.8995,48.3670,116.1905
7,Formula A,30,50241,21.3060,12.5000,52.5000,67.6000,150.0476
8,Formula B,5,66954,2.5097,1.5385,6.0000,8.5000,30.3736
9,Formula B,10,63606,5.0558,3.3333,11.9220,15.9038,43.3507



B. FEATURE CHANGES COMPARED WITH NATURAL


,Formula,Window_WD,Compared_Rows,Increased_Rows,Unchanged_Rows,Mean_Added_Deviation,Max_Added_Deviation
0,Formula A,5,66954,16824,50130,1.9629,36.2500
1,Formula A,10,63606,19108,44498,4.0065,62.1905
2,Formula A,20,56920,22572,34348,8.2588,106.7143
3,Formula A,30,50241,24660,25581,12.6388,136.0000
4,Formula B,5,66954,16513,50441,1.0537,26.5000
5,Formula B,10,63606,18868,44738,2.1477,37.7500
6,Formula B,20,56920,22359,34561,4.4243,64.6667
7,Formula B,30,50241,24475,25766,6.7606,83.0952



STEP 3.4.16 — CUMULATIVE DEVIATION COMPLETED
Datasets: Natural / Formula A / Formula B
Original 85 columns: PRESERVED
New cumulative-deviation features: 4
Total columns per dataset: 89
Rolling windows: 5 / 10 / 20 / 30 WD
Incomplete windows: NaN
Historical baselines: UNCHANGED
Original datasets: UNCHANGED
No files modified or exported.



### Step 3.4.17 — Combined Channel Intensity Features

**Objective:** Measure the average normalized activity intensity across Email, USB, and HTTP over multiple workday windows.

**Datasets:** Natural, Formula A, and Formula B.

**Rolling Windows:** 5, 10, 20, and 30 eligible workdays.

**Calculation:**

1. Normalize each channel's daily activity by its historical P95:

   `Channel Intensity = Daily Channel Count / Historical Channel P95`

2. Calculate the daily mean across channels with valid historical P95 values. Valid channels with zero daily activity contribute zero intensity.

3. Calculate the rolling mean of daily combined intensity over each window.

**New Features:**
- `combined_channel_intensity_5wd`
- `combined_channel_intensity_10wd`
- `combined_channel_intensity_20wd`
- `combined_channel_intensity_30wd`

**Implementation Rules:**
- Use the original historical P95 values.
- Exclude channels without a valid historical P95 from the daily average; positive activity without a valid baseline is not permitted.
- Use complete causal rolling windows; incomplete windows remain NaN.
- Apply identical calculations to Natural, Formula A, and Formula B.
- Preserve all 89 existing columns, including the four cumulative deviation features.
- Do not modify or export original datasets.

**Expected Outcome:** Four combined channel intensity features per dataset, with validated comparisons against natural behavior.


In [42]:

# Stage 3 — Step 3.4.17: Combined Channel Intensity
# Goal: Build 5/10/20/30 WD normalized channel-intensity features.
# Datasets: Natural, Formula A, Formula B.
# Read-only: No original data or files modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 78)
print("STAGE 3 — STEP 3.4.17: COMBINED CHANNEL INTENSITY")
print("=" * 78)

# ------------------------------------------------------------
# 1. Use the validated outputs from Step 3.4.16.
# ------------------------------------------------------------

assert set(new_features_3416) == {
    "Natural", "Formula A", "Formula B"
}

WINDOWS_3417 = [5, 10, 20, 30]
KEYS_3417 = ["user", "day"]

CHANNELS_3417 = {
    "Email": ("external_email_count", "email_count_p95"),
    "USB": ("usb_copy_count", "usb_copy_p95"),
    "HTTP": ("www_upload_count", "http_upload_p95")
}

NEW_FEATURES_3417 = [
    f"combined_channel_intensity_{w}wd"
    for w in WINDOWS_3417
]

# ------------------------------------------------------------
# 2. Reusable calculation function.
# ------------------------------------------------------------

def build_combined_intensity_3417(source):

    df = source.copy(deep=True)
    previous_columns = list(df.columns)

    assert len(previous_columns) == 89
    assert not set(NEW_FEATURES_3417).intersection(df.columns)

    df["day"] = pd.to_datetime(
        df["day"]
    ).dt.normalize()

    df = df.sort_values(
        KEYS_3417
    ).reset_index(drop=True)

    assert len(df) == 69634
    assert df["user"].nunique() == 670
    assert not df.duplicated(KEYS_3417).any()

    # Sum normalized channel activities and count channels
    # with valid historical P95 values.
    normalized_total = np.zeros(len(df), dtype=float)
    valid_channel_count = np.zeros(len(df), dtype=np.int64)

    # --------------------------------------------------------
    # 3. Normalize daily activity per channel.
    # --------------------------------------------------------

    for channel, (count_col, p95_col) in CHANNELS_3417.items():

        counts = pd.to_numeric(
            df[count_col], errors="raise"
        ).to_numpy(dtype=float)

        p95 = pd.to_numeric(
            df[p95_col], errors="raise"
        ).to_numpy(dtype=float)

        assert np.isfinite(counts).all(), (
            f"{channel}: non-finite activity counts"
        )

        assert (counts >= 0).all(), (
            f"{channel}: negative activity counts"
        )

        valid_p95 = np.isfinite(p95) & (p95 > 0)

        assert not ((counts > 0) & ~valid_p95).any(), (
            f"{channel}: positive activity without valid P95"
        )

        normalized = np.zeros(len(df), dtype=float)

        np.divide(
            counts,
            p95,
            out=normalized,
            where=valid_p95
        )

        # Values at invalid baselines are excluded from the
        # average, not counted as valid zero-intensity channels.
        normalized_total += normalized
        valid_channel_count += valid_p95.astype(np.int64)

    assert (valid_channel_count >= 1).all(), (
        "At least one valid channel baseline is required."
    )

    # --------------------------------------------------------
    # 4. Daily mean of normalized channel intensities.
    # --------------------------------------------------------

    df["_daily_combined_intensity_3417"] = (
        normalized_total / valid_channel_count
    )

    assert np.isfinite(
        df["_daily_combined_intensity_3417"]
    ).all()

    assert df["_daily_combined_intensity_3417"].ge(0).all()

    # --------------------------------------------------------
    # 5. Rolling means over complete eligible workdays.
    # --------------------------------------------------------

    position = df.groupby("user").cumcount() + 1

    for window in WINDOWS_3417:

        feature = f"combined_channel_intensity_{window}wd"

        df[feature] = (
            df.groupby("user", sort=False)[
                "_daily_combined_intensity_3417"
            ].transform(
                lambda s, w=window: (
                    s.rolling(
                        window=w,
                        min_periods=w
                    ).mean()
                )
            )
        )

        complete = position >= window

        assert df.loc[complete, feature].notna().all()
        assert df.loc[~complete, feature].isna().all()
        assert np.isfinite(
            df.loc[complete, feature]
        ).all()

        assert df[feature].dropna().ge(0).all()

    # --------------------------------------------------------
    # 6. Preserve all previous 89 feature columns.
    # --------------------------------------------------------

    df.drop(
        columns=["_daily_combined_intensity_3417"],
        inplace=True
    )

    assert len(df.columns) == 93
    assert list(df.columns[:89]) == previous_columns

    return df


# ------------------------------------------------------------
# 7. Calculate features for Natural, A, and B.
# ------------------------------------------------------------

new_features_3417 = {}

for name, source in new_features_3416.items():

    result = build_combined_intensity_3417(source)

    original_sorted = (
        source.copy(deep=True)
        .assign(
            day=lambda d: pd.to_datetime(
                d["day"]
            ).dt.normalize()
        )
        .sort_values(KEYS_3417)
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        result[original_sorted.columns],
        original_sorted,
        check_exact=True
    )

    new_features_3417[name] = result

    print(
        f"PASS — {name}: "
        f"{len(result):,} user-days, "
        f"{len(result.columns)} columns"
    )

# ------------------------------------------------------------
# 8. Validate identical user-day alignment.
# ------------------------------------------------------------

reference_keys_3417 = (
    new_features_3417["Natural"][KEYS_3417]
    .reset_index(drop=True)
)

for name, frame in new_features_3417.items():

    pd.testing.assert_frame_equal(
        frame[KEYS_3417].reset_index(drop=True),
        reference_keys_3417,
        check_exact=True
    )

print("\nUser-day key alignment: PASS")

# ------------------------------------------------------------
# 9. Distribution summary.
# ------------------------------------------------------------

print("\nA. COMBINED CHANNEL INTENSITY DISTRIBUTIONS")

summary_rows_3417 = []

for name, frame in new_features_3417.items():

    for window in WINDOWS_3417:

        feature = f"combined_channel_intensity_{window}wd"
        values = frame[feature].dropna()

        summary_rows_3417.append({
            "Dataset": name,
            "Window_WD": window,
            "Complete_Rows": len(values),
            "Mean": round(values.mean(), 4),
            "Median": round(values.median(), 4),
            "P90": round(values.quantile(0.90), 4),
            "P95": round(values.quantile(0.95), 4),
            "Max": round(values.max(), 4)
        })

summary_3417 = pd.DataFrame(summary_rows_3417)
display(summary_3417)

# ------------------------------------------------------------
# 10. Compare injected behavior with Natural.
# ------------------------------------------------------------

print("\nB. FEATURE CHANGES COMPARED WITH NATURAL")

natural_3417 = new_features_3417["Natural"]
comparison_rows_3417 = []

for name in ["Formula A", "Formula B"]:

    injected_3417 = new_features_3417[name]

    for window in WINDOWS_3417:

        feature = f"combined_channel_intensity_{window}wd"

        natural_values = natural_3417[feature]
        injected_values = injected_3417[feature]

        complete = (
            natural_values.notna()
            & injected_values.notna()
        )

        difference = (
            injected_values[complete]
            - natural_values[complete]
        )

        assert difference.ge(-1e-9).all(), (
            f"Unexpected intensity decrease: {name}, {feature}"
        )

        comparison_rows_3417.append({
            "Formula": name,
            "Window_WD": window,
            "Compared_Rows": int(complete.sum()),
            "Increased_Rows": int(
                difference.gt(1e-9).sum()
            ),
            "Unchanged_Rows": int(
                np.isclose(
                    difference,
                    0,
                    rtol=0,
                    atol=1e-9
                ).sum()
            ),
            "Mean_Added_Intensity": round(
                difference.mean(), 4
            ),
            "Max_Added_Intensity": round(
                difference.max(), 4
            )
        })

comparison_3417 = pd.DataFrame(comparison_rows_3417)
display(comparison_3417)

# ------------------------------------------------------------
# 11. Final validation.
# ------------------------------------------------------------

assert len(new_features_3417) == 3

assert all(
    len(frame.columns) == 93
    for frame in new_features_3417.values()
)

assert all(
    set(NEW_FEATURES_3417).issubset(frame.columns)
    for frame in new_features_3417.values()
)

print("\n" + "=" * 78)
print("STEP 3.4.17 — COMBINED CHANNEL INTENSITY COMPLETED")
print("=" * 78)

print("Datasets: Natural / Formula A / Formula B")
print("Existing 89 columns: PRESERVED")
print("New combined intensity features: 4")
print("Total columns per dataset: 93")
print("Rolling windows: 5 / 10 / 20 / 30 WD")
print("Incomplete windows: NaN")
print("Original datasets: UNCHANGED")
print("No files modified or exported.")


STAGE 3 — STEP 3.4.17: COMBINED CHANNEL INTENSITY
PASS — Natural: 69,634 user-days, 93 columns
PASS — Formula A: 69,634 user-days, 93 columns
PASS — Formula B: 69,634 user-days, 93 columns

User-day key alignment: PASS

A. COMBINED CHANNEL INTENSITY DISTRIBUTIONS


,Dataset,Window_WD,Complete_Rows,Mean,Median,P90,P95,Max
0,Natural,5,66954,0.2752,0.2620,0.4367,0.4929,0.8722
1,Natural,10,63606,0.2748,0.2625,0.4114,0.4611,0.7167
2,Natural,20,56920,0.2746,0.2625,0.3944,0.4444,0.6469
3,Natural,30,50241,0.2738,0.2606,0.3877,0.4392,0.6286
4,Formula A,5,66954,0.3677,0.3000,0.6955,0.8667,1.8333
5,Formula A,10,63606,0.3691,0.3067,0.6750,0.8137,1.5806
6,Formula A,20,56920,0.3713,0.3188,0.6406,0.7458,1.4285
7,Formula A,30,50241,0.3721,0.3306,0.6144,0.7050,1.2875
8,Formula B,5,66954,0.3269,0.2936,0.5534,0.6667,1.8250
9,Formula B,10,63606,0.3274,0.2963,0.5288,0.6214,1.4125



B. FEATURE CHANGES COMPARED WITH NATURAL


,Formula,Window_WD,Compared_Rows,Increased_Rows,Unchanged_Rows,Mean_Added_Intensity,Max_Added_Intensity
0,Formula A,5,66954,16825,50129,0.0926,1.2722
1,Formula A,10,63606,19109,44497,0.0943,1.0833
2,Formula A,20,56920,22573,34347,0.0968,0.8975
3,Formula A,30,50241,24661,25580,0.0983,0.7778
4,Formula B,5,66954,16825,50129,0.0517,1.1364
5,Formula B,10,63606,19109,44497,0.0526,0.8841
6,Formula B,20,56920,22573,34347,0.0539,0.6915
7,Formula B,30,50241,24661,25580,0.0547,0.5278



STEP 3.4.17 — COMBINED CHANNEL INTENSITY COMPLETED
Datasets: Natural / Formula A / Formula B
Existing 89 columns: PRESERVED
New combined intensity features: 4
Total columns per dataset: 93
Rolling windows: 5 / 10 / 20 / 30 WD
Incomplete windows: NaN
Original datasets: UNCHANGED
No files modified or exported.



### Step 3.4.18 — Dominant Channel Switches

**Objective:** Measure observable switching between Email, USB, and HTTP using normalized channel activity rather than simulated attack labels.

**Datasets:** Natural, Formula A, and Formula B.

**Rolling Windows:** 5, 10, 20, and 30 eligible workdays.

**Calculation:**

1. Normalize daily activity for each channel:

   `Normalized Activity = Daily Channel Count / Historical Channel P95`

2. Identify the channel with the highest positive normalized activity as the day's dominant channel.

3. Exclude days with no positive activity or ties for the highest normalized activity from the dominant-channel sequence.

4. Within each rolling window, count changes between consecutive valid dominant-channel observations.

**New Features:**
- `dominant_channel_switches_5wd`
- `dominant_channel_switches_10wd`
- `dominant_channel_switches_20wd`
- `dominant_channel_switches_30wd`

**Implementation Rules:**
- Use only observed channel activity and historical P95 values.
- Do not use simulated `primary_channel`, scenario labels, or injection metadata.
- Do not invent baseline values for historically inactive channels.
- Use complete causal rolling windows; incomplete windows remain NaN.
- A complete window with fewer than two identifiable dominant-channel observations has zero observable switches.
- Switching counts may increase or decrease after injection.
- Preserve all 93 existing columns and original datasets.

**Expected Outcome:** Four observable channel-switching features per dataset, including diagnostics for inactive days, ambiguous dominance, and differences between Formula A and Formula B.


In [43]:

# Stage 3 — Step 3.4.18: Dominant Channel Switches
# Goal: Calculate observable channel switches over
# 5, 10, 20, and 30 eligible workdays.
#
# Datasets: Natural, Formula A, Formula B.
# Read-only: No source files or original datasets modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 78)
print("STAGE 3 — STEP 3.4.18: DOMINANT CHANNEL SWITCHES")
print("=" * 78)

# ------------------------------------------------------------
# 1. Define datasets and channels.
# ------------------------------------------------------------

assert set(new_features_3417) == {
    "Natural", "Formula A", "Formula B"
}

WINDOWS_3418 = [5, 10, 20, 30]
KEYS_3418 = ["user", "day"]

CHANNELS_3418 = {
    "Email": (
        "external_email_count",
        "email_count_p95"
    ),
    "USB": (
        "usb_copy_count",
        "usb_copy_p95"
    ),
    "HTTP": (
        "www_upload_count",
        "http_upload_p95"
    )
}

NEW_FEATURES_3418 = [
    f"dominant_channel_switches_{w}wd"
    for w in WINDOWS_3418
]

# ------------------------------------------------------------
# 2. Define the switch-count calculation.
#
# Only dominant channels observed inside the rolling
# window are compared. Inactive and tied days are
# excluded from the sequence.
# ------------------------------------------------------------

def count_rolling_switches_3418(dominant, window):

    n = len(dominant)
    result = np.zeros(n, dtype=float)

    # A negative label means there is no unique
    # positive dominant channel for that day.
    valid_indices = np.flatnonzero(dominant >= 0)

    if len(valid_indices) >= 2:

        valid_labels = dominant[valid_indices]

        # Cumulative number of changes between
        # consecutive valid observations.
        changes = (
            valid_labels[1:] != valid_labels[:-1]
        ).astype(np.int64)

        cumulative = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(changes)
        ))

        positions = np.arange(n)

        starts = np.maximum(
            positions - window + 1,
            0
        )

        # First valid observation inside the window.
        left = np.searchsorted(
            valid_indices,
            starts,
            side="left"
        )

        # Number of valid observations up to today.
        right = np.searchsorted(
            valid_indices,
            positions,
            side="right"
        )

        has_pair = (right - left) >= 2

        # Difference of cumulative counts excludes the
        # transition crossing the window's left boundary.
        result[has_pair] = (
            cumulative[right[has_pair] - 1]
            - cumulative[left[has_pair]]
        )

    # A window requires all W eligible workdays.
    result[:min(window - 1, n)] = np.nan

    return result


# ------------------------------------------------------------
# 3. Build the features for a single dataset.
# ------------------------------------------------------------

def build_dominant_switches_3418(source):

    df = source.copy(deep=True)
    previous_columns = list(df.columns)

    assert len(previous_columns) == 93

    assert not (
        set(NEW_FEATURES_3418) & set(df.columns)
    )

    df["day"] = pd.to_datetime(
        df["day"]
    ).dt.normalize()

    df = (
        df.sort_values(KEYS_3418)
        .reset_index(drop=True)
    )

    assert len(df) == 69634
    assert df["user"].nunique() == 670
    assert not df.duplicated(KEYS_3418).any()

    n = len(df)

    # Rows = days, columns = Email / USB / HTTP.
    normalized_matrix = np.full(
        (n, 3),
        -np.inf,
        dtype=float
    )

    # --------------------------------------------------------
    # 4. Normalize each channel's observed activity.
    # --------------------------------------------------------

    for channel_idx, (channel, cols) in enumerate(
        CHANNELS_3418.items()
    ):

        count_col, p95_col = cols

        count = pd.to_numeric(
            df[count_col],
            errors="raise"
        ).to_numpy(dtype=float)

        p95 = pd.to_numeric(
            df[p95_col],
            errors="raise"
        ).to_numpy(dtype=float)

        assert np.isfinite(count).all()
        assert (count >= 0).all()

        valid_p95 = (
            np.isfinite(p95) & (p95 > 0)
        )

        assert not (
            (count > 0) & ~valid_p95
        ).any(), (
            f"{channel}: positive activity without P95"
        )

        normalized_matrix[
            valid_p95, channel_idx
        ] = (
            count[valid_p95]
            / p95[valid_p95]
        )

    assert np.isfinite(
        normalized_matrix
    ).any(axis=1).all(), (
        "At least one valid channel P95 is required."
    )

    # --------------------------------------------------------
    # 5. Identify the uniquely dominant positive channel.
    #
    # -1 = no unique dominant channel.
    #  0 = Email
    #  1 = USB
    #  2 = HTTP
    # --------------------------------------------------------

    max_intensity = normalized_matrix.max(axis=1)

    positive_activity = max_intensity > 0

    tied_at_max = np.isclose(
        normalized_matrix,
        max_intensity[:, None],
        rtol=1e-9,
        atol=1e-12
    )

    winner_count = tied_at_max.sum(axis=1)

    unique_positive = (
        positive_activity
        & (winner_count == 1)
    )

    ambiguous = (
        positive_activity
        & (winner_count > 1)
    )

    dominant_3418 = np.full(
        n,
        -1,
        dtype=np.int8
    )

    winning_channel = np.argmax(
        normalized_matrix,
        axis=1
    )

    dominant_3418[unique_positive] = (
        winning_channel[unique_positive]
    ).astype(np.int8)

    diagnostics = {
        "Eligible_User_Days": n,
        "Unique_Dominant_Days": int(
            unique_positive.sum()
        ),
        "No_Positive_Activity_Days": int(
            (~positive_activity).sum()
        ),
        "Tied_Dominant_Days": int(
            ambiguous.sum()
        ),
        "Email_Dominant": int(
            (dominant_3418 == 0).sum()
        ),
        "USB_Dominant": int(
            (dominant_3418 == 1).sum()
        ),
        "HTTP_Dominant": int(
            (dominant_3418 == 2).sum()
        )
    }

    assert (
        diagnostics["Unique_Dominant_Days"]
        + diagnostics["No_Positive_Activity_Days"]
        + diagnostics["Tied_Dominant_Days"]
    ) == n

    # --------------------------------------------------------
    # 6. Calculate rolling switches independently per user.
    # --------------------------------------------------------

    user_groups = list(
        df.groupby("user", sort=False).indices.values()
    )

    for window in WINDOWS_3418:

        feature = (
            f"dominant_channel_switches_{window}wd"
        )

        feature_values = np.full(
            n,
            np.nan,
            dtype=float
        )

        for indices in user_groups:

            user_dominant = dominant_3418[indices]

            feature_values[indices] = (
                count_rolling_switches_3418(
                    user_dominant,
                    window
                )
            )

        df[feature] = feature_values

        # Every window contains at most W-1 changes.
        complete = (
            df.groupby("user").cumcount() + 1
        ) >= window

        assert df.loc[
            ~complete, feature
        ].isna().all()

        assert df.loc[
            complete, feature
        ].notna().all()

        valid_values = df[feature].dropna()

        assert valid_values.ge(0).all()
        assert valid_values.le(window - 1).all()

        assert np.allclose(
            valid_values.to_numpy(),
            np.round(valid_values.to_numpy()),
            rtol=0,
            atol=1e-10
        )

    assert len(df.columns) == 97

    assert list(df.columns[:93]) == previous_columns

    return df, diagnostics


# ------------------------------------------------------------
# 7. Apply to Natural, A, and B.
# ------------------------------------------------------------

new_features_3418 = {}
dominance_diagnostics_3418 = []

for name, source in new_features_3417.items():

    result, diagnostics = (
        build_dominant_switches_3418(source)
    )

    previous_sorted = (
        source.copy(deep=True)
        .assign(
            day=lambda d: pd.to_datetime(
                d["day"]
            ).dt.normalize()
        )
        .sort_values(KEYS_3418)
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        result[previous_sorted.columns],
        previous_sorted,
        check_exact=True
    )

    new_features_3418[name] = result

    dominance_diagnostics_3418.append({
        "Dataset": name,
        **diagnostics
    })

    print(
        f"PASS — {name}: "
        f"{len(result):,} user-days, "
        f"{len(result.columns)} columns"
    )


# ------------------------------------------------------------
# 8. Validate user-day alignment.
# ------------------------------------------------------------

reference_keys_3418 = (
    new_features_3418["Natural"][KEYS_3418]
    .reset_index(drop=True)
)

for name, frame in new_features_3418.items():

    pd.testing.assert_frame_equal(
        frame[KEYS_3418].reset_index(drop=True),
        reference_keys_3418,
        check_exact=True
    )

print("\nUser-day key alignment: PASS")


# ------------------------------------------------------------
# 9. Report dominant-channel diagnostics.
# ------------------------------------------------------------

print("\nA. OBSERVED DOMINANT CHANNEL DIAGNOSTICS")

diagnostics_3418 = pd.DataFrame(
    dominance_diagnostics_3418
)

display(diagnostics_3418)


# ------------------------------------------------------------
# 10. Summarize switching features.
# ------------------------------------------------------------

print("\nB. ROLLING DOMINANT CHANNEL SWITCHES")

summary_rows_3418 = []

for name, frame in new_features_3418.items():

    for window in WINDOWS_3418:

        col = (
            f"dominant_channel_switches_{window}wd"
        )

        values = frame[col].dropna()

        summary_rows_3418.append({
            "Dataset": name,
            "Window_WD": window,
            "Complete_Rows": len(values),
            "Mean_Switches": round(
                values.mean(), 4
            ),
            "Median": round(
                values.median(), 4
            ),
            "P90": round(
                values.quantile(0.90), 4
            ),
            "P95": round(
                values.quantile(0.95), 4
            ),
            "Max": int(values.max()),
            "Windows_With_Switches": int(
                values.gt(0).sum()
            )
        })

summary_3418 = pd.DataFrame(
    summary_rows_3418
)

display(summary_3418)


# ------------------------------------------------------------
# 11. Compare A and B with Natural.
#
# Important: switch counts may increase OR decrease.
# ------------------------------------------------------------

print("\nC. SWITCHING CHANGES COMPARED WITH NATURAL")

natural_3418 = new_features_3418["Natural"]

comparison_rows_3418 = []

for name in ["Formula A", "Formula B"]:

    variant = new_features_3418[name]

    for window in WINDOWS_3418:

        col = (
            f"dominant_channel_switches_{window}wd"
        )

        natural_values = natural_3418[col]
        injected_values = variant[col]

        complete = (
            natural_values.notna()
            & injected_values.notna()
        )

        diff = (
            injected_values[complete]
            - natural_values[complete]
        )

        comparison_rows_3418.append({
            "Formula": name,
            "Window_WD": window,
            "Compared_Rows": int(complete.sum()),
            "Increased_Rows": int(
                diff.gt(0).sum()
            ),
            "Decreased_Rows": int(
                diff.lt(0).sum()
            ),
            "Unchanged_Rows": int(
                diff.eq(0).sum()
            ),
            "Mean_Change": round(
                diff.mean(), 4
            ),
            "Max_Absolute_Change": int(
                diff.abs().max()
            )
        })

comparison_3418 = pd.DataFrame(
    comparison_rows_3418
)

display(comparison_3418)


# ------------------------------------------------------------
# 12. Final validation.
# ------------------------------------------------------------

assert len(new_features_3418) == 3

assert all(
    len(frame.columns) == 97
    for frame in new_features_3418.values()
)

assert all(
    set(NEW_FEATURES_3418).issubset(
        frame.columns
    )
    for frame in new_features_3418.values()
)

print("\n" + "=" * 78)
print("STEP 3.4.18 — DOMINANT CHANNEL SWITCHES COMPLETED")
print("=" * 78)

print("Datasets: Natural / Formula A / Formula B")
print("Existing 93 columns: PRESERVED")
print("New dominant-switch features: 4")
print("Total columns per dataset: 97")
print("Rolling windows: 5 / 10 / 20 / 30 WD")
print("Incomplete windows: NaN")
print("Tied and inactive days: NOT ASSIGNED A DOMINANT CHANNEL")
print("Attack labels and primary-channel schedule: NOT USED")
print("Original datasets: UNCHANGED")
print("No files modified or exported.")


STAGE 3 — STEP 3.4.18: DOMINANT CHANNEL SWITCHES
PASS — Natural: 69,634 user-days, 97 columns
PASS — Formula A: 69,634 user-days, 97 columns
PASS — Formula B: 69,634 user-days, 97 columns

User-day key alignment: PASS

A. OBSERVED DOMINANT CHANNEL DIAGNOSTICS


,Dataset,Eligible_User_Days,Unique_Dominant_Days,No_Positive_Activity_Days,Tied_Dominant_Days,Email_Dominant,USB_Dominant,HTTP_Dominant
0,Natural,69634,56708,11650,1276,34966,16344,5398
1,Formula A,69634,56882,11650,1102,35749,15932,5201
2,Formula B,69634,56855,11650,1129,35526,16053,5276



B. ROLLING DOMINANT CHANNEL SWITCHES


,Dataset,Window_WD,Complete_Rows,Mean_Switches,Median,P90,P95,Max,Windows_With_Switches
0,Natural,5,66954,1.2396,1.0,3.0,3.0,4,45764
1,Natural,10,63606,2.8719,3.0,5.0,6.0,9,56069
2,Natural,20,56920,6.1461,6.0,10.0,11.0,17,55540
3,Natural,30,50241,9.4332,9.0,15.0,16.0,25,49890
4,Formula A,5,66954,1.2375,1.0,3.0,3.0,4,45340
5,Formula A,10,63606,2.8639,3.0,5.0,6.0,9,55861
6,Formula A,20,56920,6.1188,6.0,10.0,11.0,18,55585
7,Formula A,30,50241,9.3791,9.0,15.0,16.0,26,49932
8,Formula B,5,66954,1.2454,1.0,3.0,3.0,4,45705
9,Formula B,10,63606,2.8833,3.0,5.0,6.0,9,56056



C. SWITCHING CHANGES COMPARED WITH NATURAL


,Formula,Window_WD,Compared_Rows,Increased_Rows,Decreased_Rows,Unchanged_Rows,Mean_Change,Max_Absolute_Change
0,Formula A,5,66954,2932,2941,61081,-0.0021,4
1,Formula A,10,63606,4272,4320,55014,-0.0079,8
2,Formula A,20,56920,5875,5998,45047,-0.0272,13
3,Formula A,30,50241,6786,7057,36398,-0.0541,15
4,Formula B,5,66954,2236,1992,62726,0.0059,4
5,Formula B,10,63606,3371,3055,57180,0.0114,9
6,Formula B,20,56920,4951,4436,47533,0.0186,10
7,Formula B,30,50241,5866,5301,39074,0.0209,12



STEP 3.4.18 — DOMINANT CHANNEL SWITCHES COMPLETED
Datasets: Natural / Formula A / Formula B
Existing 93 columns: PRESERVED
New dominant-switch features: 4
Total columns per dataset: 97
Rolling windows: 5 / 10 / 20 / 30 WD
Incomplete windows: NaN
Tied and inactive days: NOT ASSIGNED A DOMINANT CHANNEL
Attack labels and primary-channel schedule: NOT USED
Original datasets: UNCHANGED
No files modified or exported.



### Step 3.4.19 — Channel Distribution Shift

**Objective:** Measure changes in the relative distribution of Email, USB, and HTTP activity compared with each user's historical channel behavior.

**Datasets:** Natural, Formula A, and Formula B.

**Rolling Windows:** 5, 10, 20, and 30 eligible workdays.

**Calculation:**

1. Normalize channel activity using its historical P95:

   `Normalized Count = Daily Channel Count / Historical Channel P95`

2. Estimate each user's historical normalized channel weight:

   `Historical Weight = Active Rate × Mean Active-Day Count / P95`

3. Convert historical weights into channel shares that sum to 1.

4. Sum normalized observed channel counts within each rolling window and convert the totals into observed channel shares.

5. Calculate Total Variation Distance (TVD):

   `Distribution Shift = 0.5 × Sum(|Observed Share - Historical Share|)`

**New Features:**
- `channel_distribution_shift_5wd`
- `channel_distribution_shift_10wd`
- `channel_distribution_shift_20wd`
- `channel_distribution_shift_30wd`

**Implementation Rules:**
- Use historical baselines only for reference shares.
- A channel with verified zero historical activity contributes zero historical weight; no artificial P95 is assigned.
- Positive activity without a valid historical P95 is prohibited.
- Windows with no observed channel activity have undefined distribution shift (NaN).
- Incomplete rolling windows remain NaN.
- Use the same calculations for Natural, A, and B.
- Do not use attack labels, simulated primary channels, or future observations.
- Preserve all existing 97 columns.
- Do not modify or export original datasets.

**Expected Outcome:** Four channel-distribution features per dataset, with validation that TVD values remain between 0 and 1 and comparisons against natural behavior.


In [44]:

# Stage 3 — Step 3.4.19: Channel Distribution Shift
# Goal: Compare rolling channel-activity shares with each
# user's historical channel distribution using TVD.
#
# Datasets: Natural, Formula A, Formula B.
# Read-only: No original data or files modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 78)
print("STAGE 3 — STEP 3.4.19: CHANNEL DISTRIBUTION SHIFT")
print("=" * 78)

# ------------------------------------------------------------
# 1. Define datasets, channels, and rolling windows.
# ------------------------------------------------------------

assert set(new_features_3418) == {
    "Natural", "Formula A", "Formula B"
}

WINDOWS_3419 = [5, 10, 20, 30]
KEYS_3419 = ["user", "day"]

CHANNELS_3419 = {
    "Email": ("external_email_count", "email_count_p95"),
    "USB": ("usb_copy_count", "usb_copy_p95"),
    "HTTP": ("www_upload_count", "http_upload_p95")
}

NEW_FEATURES_3419 = [
    f"channel_distribution_shift_{w}wd"
    for w in WINDOWS_3419
]

# ------------------------------------------------------------
# 2. Build historical normalized channel weights.
#
# Expected normalized daily activity:
# (active-day rate * mean active-day count) / P95.
#
# A historically inactive channel gets a zero weight,
# NOT an artificial P95 value.
# ------------------------------------------------------------

baseline_3419 = channel_baseline_3415.copy(deep=True)

assert len(baseline_3419) == 670 * 3
assert not baseline_3419.duplicated(
    ["user", "channel"]
).any()

valid_history_3419 = (
    baseline_3419["usable_historical_share"]
)

verified_inactive_3419 = (
    ~valid_history_3419
    & baseline_3419["active_days"].eq(0)
    & (
        baseline_3419["active_rate"].eq(0)
        | baseline_3419["active_rate"].isna()
    )
)

unresolved_3419 = (
    ~valid_history_3419
    & ~verified_inactive_3419
)

assert not unresolved_3419.any(), (
    "Some missing baselines cannot be confirmed "
    "as historically inactive."
)

baseline_3419["historical_weight"] = 0.0

baseline_3419.loc[
    valid_history_3419, "historical_weight"
] = (
    baseline_3419.loc[
        valid_history_3419, "active_rate"
    ]
    * baseline_3419.loc[
        valid_history_3419, "mean_active"
    ]
    / baseline_3419.loc[
        valid_history_3419, "p95"
    ]
)

assert np.isfinite(
    baseline_3419["historical_weight"]
).all()

assert baseline_3419["historical_weight"].ge(0).all()

# ------------------------------------------------------------
# 3. Convert historical weights to channel shares.
# ------------------------------------------------------------

historical_weights_3419 = (
    baseline_3419
    .pivot(
        index="user",
        columns="channel",
        values="historical_weight"
    )
    .reindex(columns=list(CHANNELS_3419))
)

assert len(historical_weights_3419) == 670
assert historical_weights_3419.notna().all().all()

historical_totals_3419 = (
    historical_weights_3419.sum(axis=1)
)

assert historical_totals_3419.gt(0).all()

historical_shares_3419 = (
    historical_weights_3419.div(
        historical_totals_3419,
        axis=0
    )
)

assert np.isfinite(
    historical_shares_3419.to_numpy()
).all()

np.testing.assert_allclose(
    historical_shares_3419.sum(axis=1).to_numpy(),
    np.ones(670),
    atol=1e-10,
    rtol=0
)

print("\nA. HISTORICAL SHARE REFERENCE")

print("Users with valid historical distributions:", 670)

print(
    "Verified historically inactive user-channel pairs:",
    int(verified_inactive_3419.sum())
)

print(
    "Unresolved historical distributions:",
    int(unresolved_3419.sum())
)

display(
    historical_shares_3419
    .describe()
    .round(4)
)


# ------------------------------------------------------------
# 4. Reusable feature calculation function.
# ------------------------------------------------------------

def build_distribution_shift_3419(source):

    df = source.copy(deep=True)
    previous_columns = list(df.columns)

    assert len(previous_columns) == 97
    assert not (
        set(NEW_FEATURES_3419) & set(df.columns)
    )

    df["day"] = pd.to_datetime(
        df["day"]
    ).dt.normalize()

    df = (
        df.sort_values(KEYS_3419)
        .reset_index(drop=True)
    )

    assert len(df) == 69634
    assert df["user"].nunique() == 670
    assert not df.duplicated(KEYS_3419).any()

    n = len(df)

    # Historical reference shares for each user's rows.
    baseline_share_matrix = np.column_stack([
        df["user"].map(
            historical_shares_3419[channel]
        ).to_numpy(dtype=float)
        for channel in CHANNELS_3419
    ])

    assert np.isfinite(
        baseline_share_matrix
    ).all()

    np.testing.assert_allclose(
        baseline_share_matrix.sum(axis=1),
        np.ones(n),
        atol=1e-10,
        rtol=0
    )

    # --------------------------------------------------------
    # 5. Calculate normalized observed activity per channel.
    # --------------------------------------------------------

    helper_columns = []

    for channel, (count_col, p95_col) in (
        CHANNELS_3419.items()
    ):

        counts = pd.to_numeric(
            df[count_col],
            errors="raise"
        ).to_numpy(dtype=float)

        p95 = pd.to_numeric(
            df[p95_col],
            errors="raise"
        ).to_numpy(dtype=float)

        assert np.isfinite(counts).all()
        assert (counts >= 0).all()

        valid_p95 = np.isfinite(p95) & (p95 > 0)

        assert not (
            (counts > 0) & ~valid_p95
        ).any(), (
            f"{channel}: activity without valid P95"
        )

        normalized = np.zeros(
            n,
            dtype=float
        )

        np.divide(
            counts,
            p95,
            out=normalized,
            where=valid_p95
        )

        helper = f"_normalized_{channel.lower()}_3419"

        df[helper] = normalized
        helper_columns.append(helper)

    position = (
        df.groupby("user").cumcount() + 1
    ).to_numpy()

    zero_activity_windows = {}

    # --------------------------------------------------------
    # 6. Build rolling normalized channel distributions.
    # --------------------------------------------------------

    for window in WINDOWS_3419:

        rolling_channel_totals = []

        for helper in helper_columns:

            values = (
                df.groupby("user", sort=False)[helper]
                .transform(
                    lambda s, w=window: (
                        s.rolling(
                            window=w,
                            min_periods=w
                        ).sum()
                    )
                )
                .to_numpy(dtype=float)
            )

            rolling_channel_totals.append(values)

        rolling_matrix = np.column_stack(
            rolling_channel_totals
        )

        total_observed = (
            rolling_matrix.sum(axis=1)
        )

        complete = position >= window

        valid_window = (
            complete
            & np.isfinite(total_observed)
            & (total_observed > 0)
        )

        zero_activity_windows[window] = int(
            (
                complete
                & (total_observed == 0)
            ).sum()
        )

        # No observable distribution exists when
        # the complete window has zero activity.
        observed_shares = np.zeros_like(
            rolling_matrix,
            dtype=float
        )

        np.divide(
            rolling_matrix,
            total_observed[:, None],
            out=observed_shares,
            where=valid_window[:, None]
        )

        if valid_window.any():

            np.testing.assert_allclose(
                observed_shares[
                    valid_window
                ].sum(axis=1),
                np.ones(valid_window.sum()),
                atol=1e-10,
                rtol=0
            )

        # Total Variation Distance:
        # 0.5 * sum(abs(observed_share - historical_share)).
        tvd = (
            0.5
            * np.abs(
                observed_shares
                - baseline_share_matrix
            ).sum(axis=1)
        )

        feature = (
            f"channel_distribution_shift_{window}wd"
        )

        df[feature] = np.nan

        df.loc[valid_window, feature] = (
            tvd[valid_window]
        )

        valid_values = df[feature].dropna()

        assert np.isfinite(valid_values).all()
        assert valid_values.ge(-1e-10).all()
        assert valid_values.le(1 + 1e-10).all()

        # Remove only negligible floating-point drift.
        df.loc[valid_window, feature] = (
            df.loc[valid_window, feature]
            .clip(lower=0, upper=1)
        )

        # Incomplete windows must stay missing.
        assert df.loc[
            ~complete, feature
        ].isna().all()

        # Complete, positive-activity windows must
        # contain a valid distribution-shift value.
        assert df.loc[
            valid_window, feature
        ].notna().all()

    # --------------------------------------------------------
    # 7. Remove temporary normalized channel columns.
    # --------------------------------------------------------

    df.drop(
        columns=helper_columns,
        inplace=True
    )

    assert len(df.columns) == 101
    assert list(df.columns[:97]) == previous_columns

    return df, zero_activity_windows


# ------------------------------------------------------------
# 8. Apply identical calculations to Natural, A, and B.
# ------------------------------------------------------------

new_features_3419 = {}
zero_windows_rows_3419 = []

for name, source in new_features_3418.items():

    result, zero_windows = (
        build_distribution_shift_3419(source)
    )

    previous_sorted = (
        source.copy(deep=True)
        .assign(
            day=lambda d: pd.to_datetime(
                d["day"]
            ).dt.normalize()
        )
        .sort_values(KEYS_3419)
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        result[previous_sorted.columns],
        previous_sorted,
        check_exact=True
    )

    new_features_3419[name] = result

    for window, count in zero_windows.items():

        zero_windows_rows_3419.append({
            "Dataset": name,
            "Window_WD": window,
            "Complete_Zero_Activity_Windows": count
        })

    print(
        f"PASS — {name}: "
        f"{len(result):,} user-days, "
        f"{len(result.columns)} columns"
    )


# ------------------------------------------------------------
# 9. Verify identical user-day keys.
# ------------------------------------------------------------

reference_keys_3419 = (
    new_features_3419["Natural"][KEYS_3419]
    .reset_index(drop=True)
)

for name, frame in new_features_3419.items():

    pd.testing.assert_frame_equal(
        frame[KEYS_3419].reset_index(drop=True),
        reference_keys_3419,
        check_exact=True
    )

print("\nUser-day key alignment: PASS")


# ------------------------------------------------------------
# 10. Report undefined zero-activity windows.
# ------------------------------------------------------------

print("\nB. ZERO-ACTIVITY WINDOW DIAGNOSTICS")

zero_window_summary_3419 = pd.DataFrame(
    zero_windows_rows_3419
)

display(zero_window_summary_3419)


# ------------------------------------------------------------
# 11. Distribution summary for all three datasets.
# ------------------------------------------------------------

print("\nC. CHANNEL DISTRIBUTION SHIFT SUMMARY")

summary_rows_3419 = []

for name, frame in new_features_3419.items():

    for window in WINDOWS_3419:

        feature = (
            f"channel_distribution_shift_{window}wd"
        )

        values = frame[feature].dropna()

        summary_rows_3419.append({
            "Dataset": name,
            "Window_WD": window,
            "Valid_Rows": len(values),
            "Mean_Shift": round(
                values.mean(), 4
            ),
            "Median": round(
                values.median(), 4
            ),
            "P90": round(
                values.quantile(0.90), 4
            ),
            "P95": round(
                values.quantile(0.95), 4
            ),
            "Max": round(
                values.max(), 4
            )
        })

summary_3419 = pd.DataFrame(
    summary_rows_3419
)

display(summary_3419)


# ------------------------------------------------------------
# 12. Compare changes relative to Natural.
#
# Distribution shift can increase or decrease after
# injection, depending on the resulting channel shares.
# ------------------------------------------------------------

print("\nD. DISTRIBUTION SHIFT CHANGES VS NATURAL")

natural_3419 = new_features_3419["Natural"]
comparison_rows_3419 = []

for name in ["Formula A", "Formula B"]:

    variant = new_features_3419[name]

    for window in WINDOWS_3419:

        feature = (
            f"channel_distribution_shift_{window}wd"
        )

        natural_values = natural_3419[feature]
        injected_values = variant[feature]

        both_valid = (
            natural_values.notna()
            & injected_values.notna()
        )

        diff = (
            injected_values[both_valid]
            - natural_values[both_valid]
        )

        comparison_rows_3419.append({
            "Formula": name,
            "Window_WD": window,
            "Compared_Rows": int(
                both_valid.sum()
            ),
            "Increased_Rows": int(
                diff.gt(1e-9).sum()
            ),
            "Decreased_Rows": int(
                diff.lt(-1e-9).sum()
            ),
            "Unchanged_Rows": int(
                np.isclose(
                    diff,
                    0,
                    rtol=0,
                    atol=1e-9
                ).sum()
            ),
            "Mean_Change": round(
                diff.mean(), 4
            ),
            "Max_Absolute_Change": round(
                diff.abs().max(), 4
            )
        })

comparison_3419 = pd.DataFrame(
    comparison_rows_3419
)

display(comparison_3419)


# ------------------------------------------------------------
# 13. Final safeguards.
# ------------------------------------------------------------

assert len(new_features_3419) == 3

assert all(
    len(frame.columns) == 101
    for frame in new_features_3419.values()
)

assert all(
    set(NEW_FEATURES_3419).issubset(
        frame.columns
    )
    for frame in new_features_3419.values()
)

print("\n" + "=" * 78)
print("STEP 3.4.19 — CHANNEL DISTRIBUTION SHIFT COMPLETED")
print("=" * 78)

print("Datasets: Natural / Formula A / Formula B")
print("Previous 97 columns: PRESERVED")
print("New channel distribution features: 4")
print("Total columns per dataset: 101")
print("Rolling windows: 5 / 10 / 20 / 30 WD")
print("TVD expected range: 0 to 1")
print("Incomplete windows: NaN")
print("Complete zero-activity windows: NaN")
print("Attack labels and future activity: NOT USED")
print("Original datasets: UNCHANGED")
print("No files modified or exported.")


STAGE 3 — STEP 3.4.19: CHANNEL DISTRIBUTION SHIFT

A. HISTORICAL SHARE REFERENCE
Users with valid historical distributions: 670
Verified historically inactive user-channel pairs: 501
Unresolved historical distributions: 0


channel,Email,USB,HTTP
count,670.0000,670.0000,670.0000
mean,0.5718,0.2811,0.1471
std,0.1549,0.2299,0.1614
min,0.1437,0.0000,0.0000
25%,0.4721,0.0000,0.0000
50%,0.5804,0.2705,0.1106
75%,0.6875,0.4681,0.2896
max,0.9581,0.8227,0.5906


PASS — Natural: 69,634 user-days, 101 columns
PASS — Formula A: 69,634 user-days, 101 columns
PASS — Formula B: 69,634 user-days, 101 columns

User-day key alignment: PASS

B. ZERO-ACTIVITY WINDOW DIAGNOSTICS


,Dataset,Window_WD,Complete_Zero_Activity_Windows
0,Natural,5,101
1,Natural,10,4
2,Natural,20,0
3,Natural,30,0
4,Formula A,5,101
5,Formula A,10,4
6,Formula A,20,0
7,Formula A,30,0
8,Formula B,5,101
9,Formula B,10,4



C. CHANNEL DISTRIBUTION SHIFT SUMMARY


,Dataset,Window_WD,Valid_Rows,Mean_Shift,Median,P90,P95,Max
0,Natural,5,66853,0.1685,0.1466,0.3347,0.3965,0.9048
1,Natural,10,63602,0.1182,0.1006,0.2394,0.2881,0.6787
2,Natural,20,56920,0.0847,0.0726,0.1694,0.2035,0.5476
3,Natural,30,50241,0.0707,0.0611,0.1401,0.1678,0.3992
4,Formula A,5,66853,0.1721,0.1509,0.3393,0.3992,0.9048
5,Formula A,10,63602,0.1235,0.1055,0.2491,0.2990,0.6787
6,Formula A,20,56920,0.0921,0.0794,0.1850,0.2184,0.5476
7,Formula A,30,50241,0.0796,0.0691,0.1583,0.1898,0.4348
8,Formula B,5,66853,0.1720,0.1513,0.3384,0.3992,0.9048
9,Formula B,10,63602,0.1224,0.1053,0.2454,0.2956,0.6787



D. DISTRIBUTION SHIFT CHANGES VS NATURAL


,Formula,Window_WD,Compared_Rows,Increased_Rows,Decreased_Rows,Unchanged_Rows,Mean_Change,Max_Absolute_Change
0,Formula A,5,66853,8146,6475,52232,0.0037,0.4522
1,Formula A,10,63602,10831,7521,45250,0.0053,0.3621
2,Formula A,20,56920,13930,8438,34552,0.0074,0.3383
3,Formula A,30,50241,15661,8881,25699,0.0089,0.2525
4,Formula B,5,66853,8012,5579,53262,0.0035,0.2667
5,Formula B,10,63602,10699,7076,45827,0.0042,0.2455
6,Formula B,20,56920,13944,8351,34625,0.0052,0.2334
7,Formula B,30,50241,15456,9075,25710,0.0060,0.1843



STEP 3.4.19 — CHANNEL DISTRIBUTION SHIFT COMPLETED
Datasets: Natural / Formula A / Formula B
Previous 97 columns: PRESERVED
New channel distribution features: 4
Total columns per dataset: 101
Rolling windows: 5 / 10 / 20 / 30 WD
TVD expected range: 0 to 1
Incomplete windows: NaN
Complete zero-activity windows: NaN
Attack labels and future activity: NOT USED
Original datasets: UNCHANGED
No files modified or exported.



### Step 3.4.20 — Consecutive Elevated Days

**Objective:** Measure the longest consecutive sequence of workdays with elevated channel activity to capture persistent behavioral changes associated with low-and-slow exfiltration.

**Datasets:** Natural, Formula A, and Formula B.

**Rolling Windows:** 5, 10, 20, and 30 eligible workdays.

**Calculation:**

1. For each user, channel, and day, identify whether the observed activity exceeds the historical P50.
2. Mark a day as elevated if at least one of Email, USB, or HTTP exceeds its historical P50.
3. For each rolling window, calculate the longest uninterrupted sequence of elevated eligible workdays.

**New Features:**
- `consecutive_elevated_days_5wd`
- `consecutive_elevated_days_10wd`
- `consecutive_elevated_days_20wd`
- `consecutive_elevated_days_30wd`

**Implementation Rules:**
- Use the original historical channel baselines.
- A historically inactive channel with zero observed activity is not elevated.
- Positive activity without a valid baseline is not permitted.
- Consecutive means consecutive eligible workdays, not consecutive calendar days.
- Calculate sequences strictly within each window.
- Incomplete windows remain NaN.
- Do not use attack labels, simulated primary channels, or future activity.
- Apply identical calculations to Natural, A, and B.
- Preserve all existing 101 columns.
- Do not modify or export source files.

**Expected Outcome:** Four new persistence-related features per dataset, with validation and comparison against natural behavior.


In [45]:

# Stage 3 — Step 3.4.20: Consecutive Elevated Days
# Goal: Calculate longest consecutive elevated-day sequences
# within 5, 10, 20, and 30 eligible workday windows.
#
# Datasets: Natural, Formula A, Formula B.
# Read-only: No original data or files modified.

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from IPython.display import display

print("=" * 78)
print("STAGE 3 — STEP 3.4.20: CONSECUTIVE ELEVATED DAYS")
print("=" * 78)

# ------------------------------------------------------------
# 1. Define datasets and input columns.
# ------------------------------------------------------------

assert set(new_features_3419) == {
    "Natural", "Formula A", "Formula B"
}

WINDOWS_3420 = [5, 10, 20, 30]
KEYS_3420 = ["user", "day"]

CHANNELS_3420 = {
    "Email": (
        "external_email_count",
        "email_count_p50",
        "email_count_p95"
    ),
    "USB": (
        "usb_copy_count",
        "usb_copy_p50",
        "usb_copy_p95"
    ),
    "HTTP": (
        "www_upload_count",
        "http_upload_p50",
        "http_upload_p95"
    )
}

NEW_FEATURES_3420 = [
    f"consecutive_elevated_days_{w}wd"
    for w in WINDOWS_3420
]

# ------------------------------------------------------------
# 2. Define the longest-consecutive-run calculation.
#
# The result for each complete window is the longest
# uninterrupted True sequence WITHIN that window.
#
# Partial windows remain NaN.
# ------------------------------------------------------------

def longest_elevated_run_3420(elevated, window):

    elevated = np.asarray(
        elevated,
        dtype=bool
    )

    n = len(elevated)

    result = np.full(
        n,
        np.nan,
        dtype=float
    )

    if n < window:
        return result

    windows = sliding_window_view(
        elevated,
        window_shape=window
    )

    num_windows = len(windows)

    current_run = np.zeros(
        num_windows,
        dtype=np.int32
    )

    longest_run = np.zeros(
        num_windows,
        dtype=np.int32
    )

    for position in range(window):

        current_run = np.where(
            windows[:, position],
            current_run + 1,
            0
        )

        longest_run = np.maximum(
            longest_run,
            current_run
        )

    result[window - 1:] = longest_run

    return result


# ------------------------------------------------------------
# 3. Test the calculation on a known example.
# ------------------------------------------------------------

test_sequence_3420 = np.array(
    [True, True, False, True, True, True]
)

test_result_3420 = longest_elevated_run_3420(
    test_sequence_3420,
    window=5
)

assert np.isnan(test_result_3420[:4]).all()

np.testing.assert_array_equal(
    test_result_3420[4:],
    np.array([2.0, 3.0])
)

print("Longest-run calculation self-test: PASS")


# ------------------------------------------------------------
# 4. Build the new features for one dataset.
# ------------------------------------------------------------

def build_consecutive_elevated_3420(source):

    df = source.copy(deep=True)
    previous_columns = list(df.columns)

    assert len(previous_columns) == 101

    assert not (
        set(NEW_FEATURES_3420)
        & set(previous_columns)
    )

    df["day"] = pd.to_datetime(
        df["day"]
    ).dt.normalize()

    df = (
        df.sort_values(KEYS_3420)
        .reset_index(drop=True)
    )

    assert len(df) == 69634
    assert df["user"].nunique() == 670
    assert not df.duplicated(KEYS_3420).any()

    n = len(df)

    # A day is elevated if at least one channel
    # exceeds its historical active-day P50.
    elevated_any = np.zeros(
        n,
        dtype=bool
    )

    channel_diagnostics = {}

    # --------------------------------------------------------
    # 5. Identify elevated activity in each channel.
    # --------------------------------------------------------

    for channel, columns in CHANNELS_3420.items():

        count_col, p50_col, p95_col = columns

        counts = pd.to_numeric(
            df[count_col],
            errors="raise"
        ).to_numpy(dtype=float)

        p50 = pd.to_numeric(
            df[p50_col],
            errors="raise"
        ).to_numpy(dtype=float)

        p95 = pd.to_numeric(
            df[p95_col],
            errors="raise"
        ).to_numpy(dtype=float)

        assert np.isfinite(counts).all()
        assert (counts >= 0).all()

        valid_baseline = (
            np.isfinite(p50)
            & np.isfinite(p95)
            & (p50 > 0)
            & (p95 > 0)
            & (p95 >= p50)
        )

        assert not (
            (counts > 0)
            & ~valid_baseline
        ).any(), (
            f"{channel}: positive activity without "
            "a valid historical baseline."
        )

        elevated_channel = (
            valid_baseline
            & (counts > p50)
        )

        elevated_any |= elevated_channel

        channel_diagnostics[channel] = int(
            elevated_channel.sum()
        )

    # --------------------------------------------------------
    # 6. Calculate longest runs per user.
    #
    # Users are independent, so sequences must never
    # continue across user boundaries.
    # --------------------------------------------------------

    user_groups = list(
        df.groupby(
            "user",
            sort=False
        ).indices.values()
    )

    position = (
        df.groupby("user").cumcount() + 1
    )

    for window in WINDOWS_3420:

        feature = (
            f"consecutive_elevated_days_{window}wd"
        )

        values = np.full(
            n,
            np.nan,
            dtype=float
        )

        for indices in user_groups:

            user_elevated = (
                elevated_any[indices]
            )

            values[indices] = (
                longest_elevated_run_3420(
                    user_elevated,
                    window
                )
            )

        df[feature] = values

        complete = position >= window

        assert df.loc[
            ~complete, feature
        ].isna().all()

        assert df.loc[
            complete, feature
        ].notna().all()

        valid_values = (
            df[feature].dropna()
        )

        assert valid_values.ge(0).all()
        assert valid_values.le(window).all()

        assert np.allclose(
            valid_values.to_numpy(),
            np.round(
                valid_values.to_numpy()
            ),
            atol=1e-10,
            rtol=0
        )

    # --------------------------------------------------------
    # 7. Preserve all previous feature columns.
    # --------------------------------------------------------

    assert len(df.columns) == 105
    assert list(df.columns[:101]) == previous_columns

    diagnostics = {
        "Eligible_User_Days": n,
        "Elevated_Any_Channel_Days": int(
            elevated_any.sum()
        ),
        "Non_Elevated_Days": int(
            (~elevated_any).sum()
        ),
        "Email_Elevated_Days": (
            channel_diagnostics["Email"]
        ),
        "USB_Elevated_Days": (
            channel_diagnostics["USB"]
        ),
        "HTTP_Elevated_Days": (
            channel_diagnostics["HTTP"]
        )
    }

    return df, diagnostics


# ------------------------------------------------------------
# 8. Apply to Natural, Formula A, and Formula B.
# ------------------------------------------------------------

new_features_3420 = {}
diagnostic_rows_3420 = []

for name, source in new_features_3419.items():

    result, diagnostics = (
        build_consecutive_elevated_3420(source)
    )

    previous_sorted = (
        source.copy(deep=True)
        .assign(
            day=lambda d: pd.to_datetime(
                d["day"]
            ).dt.normalize()
        )
        .sort_values(KEYS_3420)
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        result[previous_sorted.columns],
        previous_sorted,
        check_exact=True
    )

    new_features_3420[name] = result

    diagnostic_rows_3420.append({
        "Dataset": name,
        **diagnostics
    })

    print(
        f"PASS — {name}: "
        f"{len(result):,} user-days, "
        f"{len(result.columns)} columns"
    )


# ------------------------------------------------------------
# 9. Validate identical user-day keys.
# ------------------------------------------------------------

reference_keys_3420 = (
    new_features_3420["Natural"][KEYS_3420]
    .reset_index(drop=True)
)

for name, frame in new_features_3420.items():

    pd.testing.assert_frame_equal(
        frame[KEYS_3420].reset_index(drop=True),
        reference_keys_3420,
        check_exact=True
    )

print("\nUser-day key alignment: PASS")


# ------------------------------------------------------------
# 10. Report elevated-day diagnostics.
# ------------------------------------------------------------

print("\nA. ELEVATED DAY DIAGNOSTICS")

diagnostics_3420 = pd.DataFrame(
    diagnostic_rows_3420
)

display(diagnostics_3420)


# ------------------------------------------------------------
# 11. Summarize consecutive elevated days.
# ------------------------------------------------------------

print("\nB. CONSECUTIVE ELEVATED DAY DISTRIBUTIONS")

summary_rows_3420 = []

for name, frame in new_features_3420.items():

    for window in WINDOWS_3420:

        feature = (
            f"consecutive_elevated_days_{window}wd"
        )

        values = frame[feature].dropna()

        summary_rows_3420.append({
            "Dataset": name,
            "Window_WD": window,
            "Complete_Rows": len(values),
            "Mean_Consecutive_Days": round(
                values.mean(), 4
            ),
            "Median": round(
                values.median(), 4
            ),
            "P90": round(
                values.quantile(0.90), 4
            ),
            "P95": round(
                values.quantile(0.95), 4
            ),
            "Max": int(values.max()),
            "Windows_With_Elevated_Runs": int(
                values.gt(0).sum()
            )
        })

summary_3420 = pd.DataFrame(
    summary_rows_3420
)

display(summary_3420)


# ------------------------------------------------------------
# 12. Compare injected behavior with Natural.
#
# Injecting positive counts cannot turn a previously
# elevated day into a non-elevated day.
# Therefore, longest run lengths cannot decrease.
# ------------------------------------------------------------

print("\nC. CONSECUTIVE-DAY CHANGES VS NATURAL")

natural_3420 = new_features_3420["Natural"]

comparison_rows_3420 = []

for name in ["Formula A", "Formula B"]:

    variant = new_features_3420[name]

    for window in WINDOWS_3420:

        feature = (
            f"consecutive_elevated_days_{window}wd"
        )

        natural_values = (
            natural_3420[feature]
        )

        injected_values = (
            variant[feature]
        )

        complete = (
            natural_values.notna()
            & injected_values.notna()
        )

        diff = (
            injected_values[complete]
            - natural_values[complete]
        )

        assert diff.ge(0).all(), (
            f"Unexpected decrease: {name}, {feature}"
        )

        comparison_rows_3420.append({
            "Formula": name,
            "Window_WD": window,
            "Compared_Rows": int(
                complete.sum()
            ),
            "Increased_Rows": int(
                diff.gt(0).sum()
            ),
            "Unchanged_Rows": int(
                diff.eq(0).sum()
            ),
            "Mean_Added_Consecutive_Days": round(
                diff.mean(), 4
            ),
            "Max_Added_Consecutive_Days": int(
                diff.max()
            )
        })

comparison_3420 = pd.DataFrame(
    comparison_rows_3420
)

display(comparison_3420)


# ------------------------------------------------------------
# 13. Final safeguards.
# ------------------------------------------------------------

assert len(new_features_3420) == 3

assert all(
    len(frame.columns) == 105
    for frame in new_features_3420.values()
)

assert all(
    set(NEW_FEATURES_3420).issubset(
        frame.columns
    )
    for frame in new_features_3420.values()
)

print("\n" + "=" * 78)
print("STEP 3.4.20 — CONSECUTIVE ELEVATED DAYS COMPLETED")
print("=" * 78)

print("Datasets: Natural / Formula A / Formula B")
print("Previous 101 columns: PRESERVED")
print("New consecutive-day features: 4")
print("Total columns per dataset: 105")
print("Rolling windows: 5 / 10 / 20 / 30 WD")
print("Elevated threshold: Daily Count > Historical P50")
print("Incomplete windows: NaN")
print("Attack labels and future activity: NOT USED")
print("Original datasets: UNCHANGED")
print("No files modified or exported.")


STAGE 3 — STEP 3.4.20: CONSECUTIVE ELEVATED DAYS
Longest-run calculation self-test: PASS
PASS — Natural: 69,634 user-days, 105 columns
PASS — Formula A: 69,634 user-days, 105 columns
PASS — Formula B: 69,634 user-days, 105 columns

User-day key alignment: PASS

A. ELEVATED DAY DIAGNOSTICS


,Dataset,Eligible_User_Days,Elevated_Any_Channel_Days,Non_Elevated_Days,Email_Elevated_Days,USB_Elevated_Days,HTTP_Elevated_Days
0,Natural,69634,26756,42878,17268,9409,3386
1,Formula A,69634,33298,36336,23123,11551,4328
2,Formula B,69634,31514,38120,21205,11129,4156



B. CONSECUTIVE ELEVATED DAY DISTRIBUTIONS


,Dataset,Window_WD,Complete_Rows,Mean_Consecutive_Days,Median,P90,P95,Max,Windows_With_Elevated_Runs
0,Natural,5,66954,1.5069,1.0,3.0,4.0,5,59111
1,Natural,10,63606,2.1688,2.0,4.0,5.0,10,62133
2,Natural,20,56920,2.8853,3.0,5.0,6.0,14,56806
3,Natural,30,50241,3.3108,3.0,6.0,7.0,14,50238
4,Formula A,5,66954,2.0111,2.0,5.0,5.0,5,60463
5,Formula A,10,63606,3.2045,2.0,7.0,9.0,10,62411
6,Formula A,20,56920,4.8386,3.0,10.0,13.0,20,56839
7,Formula A,30,50241,6.0787,5.0,12.0,16.0,30,50241
8,Formula B,5,66954,1.8494,1.0,4.0,5.0,5,60373
9,Formula B,10,63606,2.8275,2.0,6.0,7.0,10,62397



C. CONSECUTIVE-DAY CHANGES VS NATURAL


,Formula,Window_WD,Compared_Rows,Increased_Rows,Unchanged_Rows,Mean_Added_Consecutive_Days,Max_Added_Consecutive_Days
0,Formula A,5,66954,13798,53156,0.5042,5
1,Formula A,10,63606,16438,47168,1.0357,10
2,Formula A,20,56920,19482,37438,1.9534,19
3,Formula A,30,50241,21412,28829,2.7679,28
4,Formula B,5,66954,11206,55748,0.3425,5
5,Formula B,10,63606,13903,49703,0.6587,9
6,Formula B,20,56920,16497,40423,1.1589,19
7,Formula B,30,50241,17930,32311,1.5842,28



STEP 3.4.20 — CONSECUTIVE ELEVATED DAYS COMPLETED
Datasets: Natural / Formula A / Formula B
Previous 101 columns: PRESERVED
New consecutive-day features: 4
Total columns per dataset: 105
Rolling windows: 5 / 10 / 20 / 30 WD
Elevated threshold: Daily Count > Historical P50
Incomplete windows: NaN
Attack labels and future activity: NOT USED
Original datasets: UNCHANGED
No files modified or exported.



### Step 3.4.21 — Cross-Channel Deviation Days

**Objective:** Measure the frequency of simultaneous elevated activity across multiple exfiltration channels.

**Datasets:** Natural, Formula A, and Formula B.

**Rolling Windows:** 5, 10, 20, and 30 eligible workdays.

**Calculation:**

1. Compare the observed daily activity in Email, USB, and HTTP with each user's historical active-day P50.
2. Mark a channel as elevated when its daily count exceeds its historical P50.
3. Mark a day as a cross-channel deviation day when at least two channels are elevated simultaneously.
4. Count cross-channel deviation days within each rolling window.

**New Features:**
- `cross_channel_deviation_days_5wd`
- `cross_channel_deviation_days_10wd`
- `cross_channel_deviation_days_20wd`
- `cross_channel_deviation_days_30wd`

**Implementation Rules:**
- Use original historical P50/P95 values without modification.
- Zero activity in a channel without a valid baseline is not elevated.
- Positive activity without a valid baseline is not permitted.
- Require at least two elevated channels on the same day.
- Use complete causal rolling windows; incomplete windows remain NaN.
- Do not use attack labels, planned primary channels, or future activity.
- Apply identical calculations to Natural, Formula A, and Formula B.
- Preserve all 105 existing columns and original datasets.
- Do not modify or export any files.

**Expected Outcome:** Four new cross-channel behavioral features per dataset, with validation and comparison against natural behavior.


In [46]:

# Stage 3 — Step 3.4.21: Cross-Channel Deviation Days
# Goal: Count workdays with elevated activity in at least
# two channels over 5, 10, 20, and 30 WD windows.
#
# Datasets: Natural, Formula A, Formula B.
# Read-only: No original data or files modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 78)
print("STAGE 3 — STEP 3.4.21: CROSS-CHANNEL DEVIATION DAYS")
print("=" * 78)

# ------------------------------------------------------------
# 1. Define validated input datasets.
# ------------------------------------------------------------

assert set(new_features_3420) == {
    "Natural", "Formula A", "Formula B"
}

WINDOWS_3421 = [5, 10, 20, 30]
KEYS_3421 = ["user", "day"]

CHANNELS_3421 = {
    "Email": (
        "external_email_count",
        "email_count_p50",
        "email_count_p95"
    ),
    "USB": (
        "usb_copy_count",
        "usb_copy_p50",
        "usb_copy_p95"
    ),
    "HTTP": (
        "www_upload_count",
        "http_upload_p50",
        "http_upload_p95"
    )
}

NEW_FEATURES_3421 = [
    f"cross_channel_deviation_days_{w}wd"
    for w in WINDOWS_3421
]

# ------------------------------------------------------------
# 2. Reusable feature calculation function.
# ------------------------------------------------------------

def build_cross_channel_deviation_3421(source):

    df = source.copy(deep=True)
    previous_columns = list(df.columns)

    assert len(previous_columns) == 105

    assert not (
        set(NEW_FEATURES_3421)
        & set(previous_columns)
    )

    df["day"] = pd.to_datetime(
        df["day"]
    ).dt.normalize()

    df = (
        df.sort_values(KEYS_3421)
        .reset_index(drop=True)
    )

    assert len(df) == 69634
    assert df["user"].nunique() == 670
    assert not df.duplicated(KEYS_3421).any()

    n = len(df)

    # Number of elevated channels observed per day.
    elevated_channel_count = np.zeros(
        n,
        dtype=np.int8
    )

    elevated_by_channel = {}

    # --------------------------------------------------------
    # 3. Identify elevated activity in each channel.
    # --------------------------------------------------------

    for channel, columns in CHANNELS_3421.items():

        count_col, p50_col, p95_col = columns

        counts = pd.to_numeric(
            df[count_col],
            errors="raise"
        ).to_numpy(dtype=float)

        p50 = pd.to_numeric(
            df[p50_col],
            errors="raise"
        ).to_numpy(dtype=float)

        p95 = pd.to_numeric(
            df[p95_col],
            errors="raise"
        ).to_numpy(dtype=float)

        assert np.isfinite(counts).all()
        assert (counts >= 0).all()

        valid_baseline = (
            np.isfinite(p50)
            & np.isfinite(p95)
            & (p50 > 0)
            & (p95 > 0)
            & (p95 >= p50)
        )

        # No positive activity without a usable baseline.
        assert not (
            (counts > 0)
            & ~valid_baseline
        ).any(), (
            f"{channel}: positive activity "
            "without a valid historical baseline."
        )

        elevated = (
            valid_baseline
            & (counts > p50)
        )

        elevated_channel_count += (
            elevated.astype(np.int8)
        )

        elevated_by_channel[channel] = int(
            elevated.sum()
        )

    assert (
        (elevated_channel_count >= 0)
        & (elevated_channel_count <= 3)
    ).all()

    # --------------------------------------------------------
    # 4. Identify simultaneous cross-channel deviations.
    # --------------------------------------------------------

    cross_channel_day = (
        elevated_channel_count >= 2
    )

    exactly_two = (
        elevated_channel_count == 2
    )

    exactly_three = (
        elevated_channel_count == 3
    )

    assert cross_channel_day.sum() == (
        exactly_two.sum() + exactly_three.sum()
    )

    diagnostics = {
        "Eligible_User_Days": n,
        "No_Elevated_Channels": int(
            (elevated_channel_count == 0).sum()
        ),
        "Exactly_One_Elevated": int(
            (elevated_channel_count == 1).sum()
        ),
        "Exactly_Two_Elevated": int(
            exactly_two.sum()
        ),
        "Exactly_Three_Elevated": int(
            exactly_three.sum()
        ),
        "Cross_Channel_Days": int(
            cross_channel_day.sum()
        ),
        "Email_Elevated_Days": (
            elevated_by_channel["Email"]
        ),
        "USB_Elevated_Days": (
            elevated_by_channel["USB"]
        ),
        "HTTP_Elevated_Days": (
            elevated_by_channel["HTTP"]
        )
    }

    # --------------------------------------------------------
    # 5. Calculate rolling counts per user.
    #
    # Only current and previous eligible workdays
    # are included. Partial windows remain NaN.
    # --------------------------------------------------------

    helper = "_cross_channel_day_3421"

    df[helper] = cross_channel_day.astype(np.int8)

    position = (
        df.groupby("user").cumcount() + 1
    )

    for window in WINDOWS_3421:

        feature = (
            f"cross_channel_deviation_days_{window}wd"
        )

        df[feature] = (
            df.groupby(
                "user",
                sort=False
            )[helper]
            .transform(
                lambda s, w=window: (
                    s.rolling(
                        window=w,
                        min_periods=w
                    ).sum()
                )
            )
        )

        complete = position >= window

        assert df.loc[
            ~complete, feature
        ].isna().all()

        assert df.loc[
            complete, feature
        ].notna().all()

        valid_values = df[feature].dropna()

        assert valid_values.ge(0).all()
        assert valid_values.le(window).all()

        assert np.allclose(
            valid_values.to_numpy(),
            np.round(valid_values.to_numpy()),
            rtol=0,
            atol=1e-10
        )

    # --------------------------------------------------------
    # 6. Remove temporary column and preserve all 105
    #    previously calculated columns.
    # --------------------------------------------------------

    df.drop(
        columns=[helper],
        inplace=True
    )

    assert len(df.columns) == 109
    assert list(df.columns[:105]) == previous_columns

    return df, diagnostics


# ------------------------------------------------------------
# 7. Calculate features for Natural, Formula A, Formula B.
# ------------------------------------------------------------

new_features_3421 = {}
diagnostic_rows_3421 = []

for name, source in new_features_3420.items():

    result, diagnostics = (
        build_cross_channel_deviation_3421(source)
    )

    previous_sorted = (
        source.copy(deep=True)
        .assign(
            day=lambda d: pd.to_datetime(
                d["day"]
            ).dt.normalize()
        )
        .sort_values(KEYS_3421)
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        result[previous_sorted.columns],
        previous_sorted,
        check_exact=True
    )

    new_features_3421[name] = result

    diagnostic_rows_3421.append({
        "Dataset": name,
        **diagnostics
    })

    print(
        f"PASS — {name}: "
        f"{len(result):,} user-days, "
        f"{len(result.columns)} columns"
    )


# ------------------------------------------------------------
# 8. Validate identical user-day keys.
# ------------------------------------------------------------

reference_keys_3421 = (
    new_features_3421["Natural"][KEYS_3421]
    .reset_index(drop=True)
)

for name, frame in new_features_3421.items():

    pd.testing.assert_frame_equal(
        frame[KEYS_3421].reset_index(drop=True),
        reference_keys_3421,
        check_exact=True
    )

print("\nUser-day key alignment: PASS")


# ------------------------------------------------------------
# 9. Report daily cross-channel diagnostics.
# ------------------------------------------------------------

print("\nA. CROSS-CHANNEL ELEVATION DIAGNOSTICS")

diagnostics_3421 = pd.DataFrame(
    diagnostic_rows_3421
)

display(diagnostics_3421)

assert (
    diagnostics_3421[
        [
            "No_Elevated_Channels",
            "Exactly_One_Elevated",
            "Exactly_Two_Elevated",
            "Exactly_Three_Elevated"
        ]
    ].sum(axis=1) == 69634
).all()


# ------------------------------------------------------------
# 10. Summarize rolling cross-channel deviation features.
# ------------------------------------------------------------

print("\nB. CROSS-CHANNEL DEVIATION DISTRIBUTIONS")

summary_rows_3421 = []

for name, frame in new_features_3421.items():

    for window in WINDOWS_3421:

        feature = (
            f"cross_channel_deviation_days_{window}wd"
        )

        values = frame[feature].dropna()

        summary_rows_3421.append({
            "Dataset": name,
            "Window_WD": window,
            "Complete_Rows": len(values),
            "Mean_Days": round(
                values.mean(), 4
            ),
            "Median": round(
                values.median(), 4
            ),
            "P90": round(
                values.quantile(0.90), 4
            ),
            "P95": round(
                values.quantile(0.95), 4
            ),
            "Max": int(values.max()),
            "Windows_With_Cross_Channel_Deviation": int(
                values.gt(0).sum()
            )
        })

summary_3421 = pd.DataFrame(
    summary_rows_3421
)

display(summary_3421)


# ------------------------------------------------------------
# 11. Compare injected values against Natural.
#
# Since both A and B add counts without removing
# natural activity, an already elevated channel
# cannot become non-elevated.
#
# Therefore, cross-channel deviation days
# cannot decrease.
# ------------------------------------------------------------

print("\nC. CROSS-CHANNEL CHANGES VS NATURAL")

natural_3421 = new_features_3421["Natural"]

comparison_rows_3421 = []

for name in ["Formula A", "Formula B"]:

    variant = new_features_3421[name]

    for window in WINDOWS_3421:

        feature = (
            f"cross_channel_deviation_days_{window}wd"
        )

        natural_values = natural_3421[feature]
        injected_values = variant[feature]

        complete = (
            natural_values.notna()
            & injected_values.notna()
        )

        diff = (
            injected_values[complete]
            - natural_values[complete]
        )

        assert diff.ge(0).all(), (
            f"Unexpected cross-channel decrease: "
            f"{name}, {feature}"
        )

        comparison_rows_3421.append({
            "Formula": name,
            "Window_WD": window,
            "Compared_Rows": int(
                complete.sum()
            ),
            "Increased_Rows": int(
                diff.gt(0).sum()
            ),
            "Unchanged_Rows": int(
                diff.eq(0).sum()
            ),
            "Mean_Added_Days": round(
                diff.mean(), 4
            ),
            "Max_Added_Days": int(
                diff.max()
            )
        })

comparison_3421 = pd.DataFrame(
    comparison_rows_3421
)

display(comparison_3421)


# ------------------------------------------------------------
# 12. Final safeguards.
# ------------------------------------------------------------

assert len(new_features_3421) == 3

assert all(
    len(frame.columns) == 109
    for frame in new_features_3421.values()
)

assert all(
    set(NEW_FEATURES_3421).issubset(
        frame.columns
    )
    for frame in new_features_3421.values()
)

print("\n" + "=" * 78)
print("STEP 3.4.21 — CROSS-CHANNEL DEVIATION COMPLETED")
print("=" * 78)

print("Datasets: Natural / Formula A / Formula B")
print("Previous 105 columns: PRESERVED")
print("New cross-channel features: 4")
print("Total columns per dataset: 109")
print("Rolling windows: 5 / 10 / 20 / 30 WD")
print("Deviation rule: At least two channels > P50")
print("Incomplete windows: NaN")
print("Attack labels and future activity: NOT USED")
print("Original datasets: UNCHANGED")
print("No files modified or exported.")


STAGE 3 — STEP 3.4.21: CROSS-CHANNEL DEVIATION DAYS
PASS — Natural: 69,634 user-days, 109 columns
PASS — Formula A: 69,634 user-days, 109 columns
PASS — Formula B: 69,634 user-days, 109 columns

User-day key alignment: PASS

A. CROSS-CHANNEL ELEVATION DIAGNOSTICS


,Dataset,Eligible_User_Days,No_Elevated_Channels,Exactly_One_Elevated,Exactly_Two_Elevated,Exactly_Three_Elevated,Cross_Channel_Days,Email_Elevated_Days,USB_Elevated_Days,HTTP_Elevated_Days
0,Natural,69634,42878,23524,3157,75,3232,17268,9409,3386
1,Formula A,69634,36336,27882,5128,288,5416,23123,11551,4328
2,Formula B,69634,38120,26756,4540,218,4758,21205,11129,4156



B. CROSS-CHANNEL DEVIATION DISTRIBUTIONS


,Dataset,Window_WD,Complete_Rows,Mean_Days,Median,P90,P95,Max,Windows_With_Cross_Channel_Deviation
0,Natural,5,66954,0.2328,0.0,1.0,1.0,4,13242
1,Natural,10,63606,0.4638,0.0,1.0,2.0,7,21147
2,Natural,20,56920,0.9267,1.0,3.0,3.0,9,28971
3,Natural,30,50241,1.3795,1.0,3.0,5.0,12,31114
4,Formula A,5,66954,0.3919,0.0,1.0,2.0,5,19158
5,Formula A,10,63606,0.7876,0.0,2.0,3.0,9,28064
6,Formula A,20,56920,1.5934,1.0,4.0,6.0,14,35822
7,Formula A,30,50241,2.4015,2.0,6.0,8.0,17,37316
8,Formula B,5,66954,0.3442,0.0,1.0,2.0,5,17864
9,Formula B,10,63606,0.6900,0.0,2.0,3.0,7,26893



C. CROSS-CHANNEL CHANGES VS NATURAL


,Formula,Window_WD,Compared_Rows,Increased_Rows,Unchanged_Rows,Mean_Added_Days,Max_Added_Days
0,Formula A,5,66954,7765,59189,0.1591,5
1,Formula A,10,63606,11797,51809,0.3238,8
2,Formula A,20,56920,16470,40450,0.6667,9
3,Formula A,30,50241,19020,31221,1.0220,12
4,Formula B,5,66954,5991,60963,0.1114,4
5,Formula B,10,63606,9664,53942,0.2262,6
6,Formula B,20,56920,14328,42592,0.4666,7
7,Formula B,30,50241,16895,33346,0.7165,8



STEP 3.4.21 — CROSS-CHANNEL DEVIATION COMPLETED
Datasets: Natural / Formula A / Formula B
Previous 105 columns: PRESERVED
New cross-channel features: 4
Total columns per dataset: 109
Rolling windows: 5 / 10 / 20 / 30 WD
Deviation rule: At least two channels > P50
Incomplete windows: NaN
Attack labels and future activity: NOT USED
Original datasets: UNCHANGED
No files modified or exported.



### Step 3.4.22 — Rolling Channel Activity Totals (20 & 30 WD)

**Objective:** Extend existing rolling activity-count features by calculating 20- and 30-workday totals for Email, USB, and HTTP.

**Datasets:** Natural, Formula A, and Formula B.

**Existing Features:** Rolling count totals for 5 and 10 workdays are already available in the original 85-feature dataset and must not be duplicated or modified.

**Calculation:**

For each user and channel:

`Rolling Channel Total = Sum(Daily Channel Count over the current and previous W-1 eligible workdays)`

**New Features:**
- `external_email_count_20wd`
- `usb_copy_count_20wd`
- `http_upload_count_20wd`
- `external_email_count_30wd`
- `usb_copy_count_30wd`
- `http_upload_count_30wd`

**Implementation Rules:**
- Use `external_email_count`, `usb_copy_count`, and `www_upload_count`.
- Group calculations independently by user.
- Include current and previous eligible workdays only.
- Require complete 20- and 30-workday windows; incomplete windows remain NaN.
- Apply identical calculations to Natural, A, and B.
- Preserve all existing 109 columns.
- Verify that injected rolling totals never decrease relative to Natural.
- Do not use attack labels or future activity.
- Do not modify or export source files.

**Expected Outcome:** Six additional rolling-count features per dataset, resulting in 115 columns per dataset and completing the planned new behavioral feature engineering.


In [47]:

# Stage 3 — Step 3.4.22: Rolling Channel Activity Totals
# Goal: Calculate Email, USB, and HTTP activity totals over
# complete 20- and 30-eligible-workday windows.
#
# Datasets: Natural, Formula A, Formula B.
# Read-only: No original data or files modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 78)
print("STAGE 3 — STEP 3.4.22: ROLLING CHANNEL TOTALS")
print("=" * 78)

# ------------------------------------------------------------
# 1. Define datasets and feature specifications.
# ------------------------------------------------------------

assert set(new_features_3421) == {
    "Natural", "Formula A", "Formula B"
}

WINDOWS_3422 = [20, 30]
KEYS_3422 = ["user", "day"]

CHANNELS_3422 = {
    "Email": {
        "source": "external_email_count",
        "prefix": "external_email_count"
    },
    "USB": {
        "source": "usb_copy_count",
        "prefix": "usb_copy_count"
    },
    "HTTP": {
        "source": "www_upload_count",
        "prefix": "http_upload_count"
    }
}

NEW_FEATURES_3422 = [
    f"{spec['prefix']}_{window}wd"
    for window in WINDOWS_3422
    for spec in CHANNELS_3422.values()
]

assert len(NEW_FEATURES_3422) == 6

# ------------------------------------------------------------
# 2. Test rolling calculations on a simple example.
# ------------------------------------------------------------

test_values_3422 = pd.Series(
    [1, 2, 3, 4, 5, 6],
    dtype=float
)

test_rolling_3422 = (
    test_values_3422
    .rolling(window=5, min_periods=5)
    .sum()
)

assert test_rolling_3422.iloc[:4].isna().all()

np.testing.assert_allclose(
    test_rolling_3422.iloc[4:].to_numpy(),
    np.array([15.0, 20.0]),
    rtol=0,
    atol=1e-10
)

print("Rolling-sum calculation self-test: PASS")

# ------------------------------------------------------------
# 3. Reusable feature calculation function.
# ------------------------------------------------------------

def build_rolling_totals_3422(source):

    df = source.copy(deep=True)
    previous_columns = list(df.columns)

    assert len(previous_columns) == 109

    assert not (
        set(NEW_FEATURES_3422)
        & set(previous_columns)
    )

    df["day"] = pd.to_datetime(
        df["day"]
    ).dt.normalize()

    df = (
        df.sort_values(KEYS_3422)
        .reset_index(drop=True)
    )

    assert len(df) == 69634
    assert df["user"].nunique() == 670
    assert not df.duplicated(KEYS_3422).any()

    # Eligible-workday position for each user.
    position = (
        df.groupby("user").cumcount() + 1
    )

    # --------------------------------------------------------
    # 4. Validate daily channel counts.
    # --------------------------------------------------------

    for channel, spec in CHANNELS_3422.items():

        col = spec["source"]

        values = pd.to_numeric(
            df[col],
            errors="raise"
        )

        assert values.notna().all(), (
            f"{channel}: missing activity counts"
        )

        assert np.isfinite(
            values.to_numpy(dtype=float)
        ).all(), (
            f"{channel}: non-finite activity counts"
        )

        assert values.ge(0).all(), (
            f"{channel}: negative activity counts"
        )

    # --------------------------------------------------------
    # 5. Calculate complete 20/30 WD rolling totals.
    # --------------------------------------------------------

    for window in WINDOWS_3422:

        complete = position >= window

        for channel, spec in CHANNELS_3422.items():

            source_col = spec["source"]

            feature = (
                f"{spec['prefix']}_{window}wd"
            )

            df[feature] = (
                df.groupby(
                    "user",
                    sort=False
                )[source_col]
                .transform(
                    lambda s, w=window: (
                        s.rolling(
                            window=w,
                            min_periods=w
                        ).sum()
                    )
                )
            )

            # Partial windows must remain undefined.
            assert df.loc[
                ~complete, feature
            ].isna().all()

            # Complete windows must have valid totals.
            assert df.loc[
                complete, feature
            ].notna().all()

            valid_values = df[feature].dropna()

            assert np.isfinite(
                valid_values.to_numpy(dtype=float)
            ).all()

            assert valid_values.ge(0).all()

            # Rolling sum cannot be less than today's count.
            assert (
                df.loc[complete, feature]
                >= df.loc[complete, source_col] - 1e-9
            ).all()

    # --------------------------------------------------------
    # 6. Preserve existing features.
    # --------------------------------------------------------

    assert len(df.columns) == 115

    assert (
        list(df.columns[:109])
        == previous_columns
    )

    return df


# ------------------------------------------------------------
# 7. Build all three datasets using the same function.
# ------------------------------------------------------------

new_features_3422 = {}

for name, source in new_features_3421.items():

    result = build_rolling_totals_3422(source)

    previous_sorted = (
        source.copy(deep=True)
        .assign(
            day=lambda d: pd.to_datetime(
                d["day"]
            ).dt.normalize()
        )
        .sort_values(KEYS_3422)
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        result[previous_sorted.columns],
        previous_sorted,
        check_exact=True
    )

    new_features_3422[name] = result

    print(
        f"PASS — {name}: "
        f"{len(result):,} user-days, "
        f"{len(result.columns)} columns"
    )


# ------------------------------------------------------------
# 8. Verify identical user-day alignment.
# ------------------------------------------------------------

reference_keys_3422 = (
    new_features_3422["Natural"][KEYS_3422]
    .reset_index(drop=True)
)

for name, frame in new_features_3422.items():

    pd.testing.assert_frame_equal(
        frame[KEYS_3422].reset_index(drop=True),
        reference_keys_3422,
        check_exact=True
    )

print("\nUser-day key alignment: PASS")


# ------------------------------------------------------------
# 9. Summarize rolling totals.
# ------------------------------------------------------------

print("\nA. ROLLING CHANNEL TOTAL DISTRIBUTIONS")

summary_rows_3422 = []

for name, frame in new_features_3422.items():

    for window in WINDOWS_3422:

        for channel, spec in CHANNELS_3422.items():

            feature = (
                f"{spec['prefix']}_{window}wd"
            )

            values = frame[feature].dropna()

            summary_rows_3422.append({
                "Dataset": name,
                "Channel": channel,
                "Window_WD": window,
                "Complete_Rows": len(values),
                "Mean": round(values.mean(), 4),
                "Median": round(values.median(), 4),
                "P90": round(
                    values.quantile(0.90), 4
                ),
                "P95": round(
                    values.quantile(0.95), 4
                ),
                "Max": round(values.max(), 4)
            })

summary_3422 = pd.DataFrame(
    summary_rows_3422
)

display(summary_3422)


# ------------------------------------------------------------
# 10. Compare A and B with Natural.
#
# Injection adds events without removing natural activity.
# Therefore, rolling activity totals must not decrease.
# ------------------------------------------------------------

print("\nB. ROLLING COUNT CHANGES VS NATURAL")

natural_3422 = new_features_3422["Natural"]

comparison_rows_3422 = []

for name in ["Formula A", "Formula B"]:

    variant = new_features_3422[name]

    for window in WINDOWS_3422:

        for channel, spec in CHANNELS_3422.items():

            feature = (
                f"{spec['prefix']}_{window}wd"
            )

            natural_values = natural_3422[feature]
            injected_values = variant[feature]

            complete = (
                natural_values.notna()
                & injected_values.notna()
            )

            diff = (
                injected_values[complete]
                - natural_values[complete]
            )

            assert diff.ge(-1e-9).all(), (
                f"Unexpected decrease: "
                f"{name}, {feature}"
            )

            comparison_rows_3422.append({
                "Formula": name,
                "Channel": channel,
                "Window_WD": window,
                "Compared_Rows": int(
                    complete.sum()
                ),
                "Increased_Rows": int(
                    diff.gt(1e-9).sum()
                ),
                "Unchanged_Rows": int(
                    np.isclose(
                        diff,
                        0,
                        rtol=0,
                        atol=1e-9
                    ).sum()
                ),
                "Mean_Added_Events": round(
                    diff.mean(), 4
                ),
                "Max_Added_Events": round(
                    diff.max(), 4
                )
            })

comparison_3422 = pd.DataFrame(
    comparison_rows_3422
)

display(comparison_3422)


# ------------------------------------------------------------
# 11. Confirm final feature inventory.
# ------------------------------------------------------------

print("\nC. FINAL FEATURE INVENTORY")

feature_groups_3422 = {
    "Original Data Engineering Features": 85,
    "Cumulative Deviation": 4,
    "Combined Channel Intensity": 4,
    "Dominant Channel Switches": 4,
    "Channel Distribution Shift": 4,
    "Consecutive Elevated Days": 4,
    "Cross-Channel Deviation Days": 4,
    "Additional Rolling Channel Totals": 6
}

inventory_3422 = pd.DataFrame(
    [
        {
            "Feature_Group": group,
            "Columns": count
        }
        for group, count in feature_groups_3422.items()
    ]
)

display(inventory_3422)

assert sum(
    feature_groups_3422.values()
) == 115

assert len(new_features_3422) == 3

assert all(
    len(frame.columns) == 115
    for frame in new_features_3422.values()
)

assert all(
    set(NEW_FEATURES_3422).issubset(
        frame.columns
    )
    for frame in new_features_3422.values()
)

# ------------------------------------------------------------
# 12. Final status.
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("STEP 3.4.22 — ROLLING CHANNEL TOTALS COMPLETED")
print("=" * 78)

print("Datasets: Natural / Formula A / Formula B")
print("Original columns: 85")
print("New behavioral columns: 30")
print("Total columns per dataset: 115")
print("Rolling windows: 5 / 10 / 20 / 30 WD")
print("New rolling count windows: 20 / 30 WD")
print("Incomplete new rolling windows: NaN")
print("Historical baselines: UNCHANGED")
print("Attack simulation: UNCHANGED")
print("Original datasets: UNCHANGED")
print("No files modified or exported.")


STAGE 3 — STEP 3.4.22: ROLLING CHANNEL TOTALS
Rolling-sum calculation self-test: PASS
PASS — Natural: 69,634 user-days, 115 columns
PASS — Formula A: 69,634 user-days, 115 columns
PASS — Formula B: 69,634 user-days, 115 columns

User-day key alignment: PASS

A. ROLLING CHANNEL TOTAL DISTRIBUTIONS


,Dataset,Channel,Window_WD,Complete_Rows,Mean,Median,P90,P95,Max
0,Natural,Email,20,56920,30.9192,30.0,48.0,55.0,178.0
1,Natural,USB,20,56920,21.9389,9.0,82.0,105.0,141.0
2,Natural,HTTP,20,56920,10.2637,1.0,35.0,51.0,122.0
3,Natural,Email,30,50241,46.2344,45.0,71.0,81.0,267.0
4,Natural,USB,30,50241,32.8340,13.0,123.0,157.0,200.0
5,Natural,HTTP,30,50241,15.2546,3.0,52.0,74.0,158.0
6,Formula A,Email,20,56920,44.6439,36.0,89.0,113.0,376.0
7,Formula A,USB,20,56920,28.0075,11.0,93.0,116.0,412.0
8,Formula A,HTTP,20,56920,13.8015,1.0,45.0,66.0,287.0
9,Formula A,Email,30,50241,67.1943,55.0,130.0,161.0,557.0



B. ROLLING COUNT CHANGES VS NATURAL


,Formula,Channel,Window_WD,Compared_Rows,Increased_Rows,Unchanged_Rows,Mean_Added_Events,Max_Added_Events
0,Formula A,Email,20,56920,21802,35118,13.7247,270.0
1,Formula A,USB,20,56920,16069,40851,6.0686,280.0
2,Formula A,HTTP,20,56920,10895,46025,3.5378,192.0
3,Formula A,Email,30,50241,23978,26263,20.9599,405.0
4,Formula A,USB,30,50241,17856,32385,9.2319,364.0
5,Formula A,HTTP,30,50241,12318,37923,5.4635,240.0
6,Formula B,Email,20,56920,21802,35118,7.3827,152.0
7,Formula B,USB,20,56920,16069,40851,3.4782,202.0
8,Formula B,HTTP,20,56920,10895,46025,2.1287,139.0
9,Formula B,Email,30,50241,23978,26263,11.2503,208.0



C. FINAL FEATURE INVENTORY


,Feature_Group,Columns
0,Original Data Engineering Features,85
1,Cumulative Deviation,4
2,Combined Channel Intensity,4
3,Dominant Channel Switches,4
4,Channel Distribution Shift,4
5,Consecutive Elevated Days,4
6,Cross-Channel Deviation Days,4
7,Additional Rolling Channel Totals,6



STEP 3.4.22 — ROLLING CHANNEL TOTALS COMPLETED
Datasets: Natural / Formula A / Formula B
Original columns: 85
New behavioral columns: 30
Total columns per dataset: 115
Rolling windows: 5 / 10 / 20 / 30 WD
New rolling count windows: 20 / 30 WD
Incomplete new rolling windows: NaN
Historical baselines: UNCHANGED
Attack simulation: UNCHANGED
Original datasets: UNCHANGED
No files modified or exported.



### Step 3.4.23 — Stealth-Aware Comparison of Formulas A and B

**Objective:** Compare Formula A (P95-Based Addition) and Formula B (Natural-Activity Addition) using injection execution, historical P95 exceedance, and the newly engineered behavioral features.

**Datasets:** Natural, Formula A, and Formula B.

**Scope:** The 670 validated users and their scheduled attack days only.

**Comparison Tasks:**

1. Verify positive injection in every scheduled attack channel-day.
2. Compare natural and final activity relative to historical P95, separately for High, Medium, and Low stealth.
3. Distinguish pre-existing P95 exceedances from exceedances newly caused by injection.
4. Join scheduled attack days with the 115-column feature datasets.
5. Measure feature changes relative to Natural using identical user-day records.
6. Compare all newly engineered feature groups, focusing initially on complete 30-workday windows.
7. Report available complete-window coverage for each stealth level.

**Important:**
- Stealth labels are used only for evaluation, not feature calculation.
- A larger feature change does not automatically mean a better injection formula.
- Dominant channel switches and distribution shift may increase or decrease.
- Incomplete rolling windows are excluded from their respective feature comparisons.
- Formula C is excluded from this comparison because it did not execute all planned channel-days.
- No final formula is selected automatically.
- No source datasets or files are modified.

**Expected Outcome:** A matched, stealth-aware comparison supporting the subsequent selection of the final injection formula.


In [48]:

# Stage 3 — Step 3.4.23: Stealth-Aware Formula Comparison
# Goal: Compare A and B on actual scheduled attack days,
# by stealth level, injection coverage, P95 exceedance,
# and changes in the 30 newly engineered features.
#
# Read-only: No datasets, injections, or files modified.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 78)
print("STAGE 3 — STEP 3.4.23: A VS B — STEALTH COMPARISON")
print("=" * 78)

FORMULAS_3423 = {
    "A_P95_Addition": "Formula A",
    "B_Natural_Addition": "Formula B"
}

LEVELS_3423 = ["High", "Medium", "Low"]
WINDOWS_3423 = [5, 10, 20, 30]
KEYS_3423 = ["user", "day"]

assert set(new_features_3422) == {
    "Natural", "Formula A", "Formula B"
}

# ------------------------------------------------------------
# 1. Audit the exact scheduled channel-day injections.
# ------------------------------------------------------------

quantity_3423 = quantity_experiment_343.loc[
    quantity_experiment_343["formula"].isin(
        FORMULAS_3423
    )
].copy()

quantity_3423["day"] = pd.to_datetime(
    quantity_3423["day"]
).dt.normalize()

assert len(quantity_3423) == 28750

assert (
    quantity_3423.groupby("formula").size()
    .eq(14375)
).all()

assert not quantity_3423.duplicated(
    ["formula", "user", "day", "channel"]
).any()

assert quantity_3423["effective_stealth"].isin(
    LEVELS_3423
).all()

assert quantity_3423["candidate_p95"].gt(0).all()
assert quantity_3423["natural_count"].gt(0).all()
assert quantity_3423["injected_count"].gt(0).all()

np.testing.assert_allclose(
    quantity_3423["simulated_final_count"],
    quantity_3423["natural_count"]
    + quantity_3423["injected_count"],
    rtol=0,
    atol=1e-9
)

quantity_3423["Natural_Above_P95"] = (
    quantity_3423["natural_count"]
    > quantity_3423["candidate_p95"] + 1e-9
)

quantity_3423["Final_Above_P95"] = (
    quantity_3423["simulated_final_count"]
    > quantity_3423["candidate_p95"] + 1e-9
)

quantity_3423["Newly_Above_P95"] = (
    ~quantity_3423["Natural_Above_P95"]
    & quantity_3423["Final_Above_P95"]
)

quantity_3423["Final_to_P95"] = (
    quantity_3423["simulated_final_count"]
    / quantity_3423["candidate_p95"]
)

print("\nA. INJECTION AND P95 COMPARISON BY STEALTH")

injection_rows_3423 = []

for formula, formula_name in FORMULAS_3423.items():

    for level in LEVELS_3423:

        part = quantity_3423.loc[
            quantity_3423["formula"].eq(formula)
            & quantity_3423["effective_stealth"].eq(level)
        ]

        n = len(part)
        assert n > 0

        injection_rows_3423.append({
            "Formula": formula_name,
            "Stealth": level,
            "Channel_Days": n,
            "Positive_Injection_%": round(
                100 * part["injected_count"].gt(0).mean(),
                2
            ),
            "Total_Injected": int(
                part["injected_count"].sum()
            ),
            "Natural_Above_P95_%": round(
                100 * part["Natural_Above_P95"].mean(),
                2
            ),
            "Newly_Above_P95_%": round(
                100 * part["Newly_Above_P95"].mean(),
                2
            ),
            "Final_Above_P95_%": round(
                100 * part["Final_Above_P95"].mean(),
                2
            ),
            "Median_Final_to_P95": round(
                part["Final_to_P95"].median(),
                4
            ),
            "P90_Final_to_P95": round(
                part["Final_to_P95"].quantile(0.90),
                4
            )
        })

injection_comparison_3423 = pd.DataFrame(
    injection_rows_3423
)

display(injection_comparison_3423)

# ------------------------------------------------------------
# 2. Prepare unique scheduled attack-day metadata.
#    Labels are used ONLY for evaluating finished features.
# ------------------------------------------------------------

attack_meta_3423 = quantity_3423.loc[
    quantity_3423["formula"].eq("A_P95_Addition"),
    ["user", "day", "effective_stealth"]
].drop_duplicates().copy()

assert not attack_meta_3423.duplicated(
    KEYS_3423
).any()

assert len(attack_meta_3423) == 12636
assert attack_meta_3423["user"].nunique() == 670

# Both formulas must use identical attack-day metadata.
other_meta_3423 = quantity_3423.loc[
    quantity_3423["formula"].eq("B_Natural_Addition"),
    ["user", "day", "effective_stealth"]
].drop_duplicates()

pd.testing.assert_frame_equal(
    attack_meta_3423.sort_values(
        KEYS_3423
    ).reset_index(drop=True),
    other_meta_3423.sort_values(
        KEYS_3423
    ).reset_index(drop=True),
    check_exact=True
)

attack_meta_3423 = (
    attack_meta_3423
    .rename(columns={
        "effective_stealth": "Stealth"
    })
    .sort_values(KEYS_3423)
    .reset_index(drop=True)
)

print("\nScheduled attack days:", len(attack_meta_3423))

# ------------------------------------------------------------
# 3. Identify all 30 newly engineered features.
# ------------------------------------------------------------

BEHAVIOR_GROUPS_3423 = {
    "Cumulative Deviation": "cumulative_deviation",
    "Combined Intensity": "combined_channel_intensity",
    "Dominant Switches": "dominant_channel_switches",
    "Distribution Shift": "channel_distribution_shift",
    "Consecutive Elevated Days": "consecutive_elevated_days",
    "Cross-Channel Deviation": "cross_channel_deviation_days"
}

FEATURE_SPECS_3423 = []

for group_name, prefix in BEHAVIOR_GROUPS_3423.items():

    for window in WINDOWS_3423:

        FEATURE_SPECS_3423.append({
            "Group": group_name,
            "Window_WD": window,
            "Column": f"{prefix}_{window}wd"
        })

for channel, prefix in [
    ("Email Total", "external_email_count"),
    ("USB Total", "usb_copy_count"),
    ("HTTP Total", "http_upload_count")
]:

    for window in [20, 30]:

        FEATURE_SPECS_3423.append({
            "Group": channel,
            "Window_WD": window,
            "Column": f"{prefix}_{window}wd"
        })

assert len(FEATURE_SPECS_3423) == 30

FEATURE_COLS_3423 = [
    item["Column"]
    for item in FEATURE_SPECS_3423
]

assert len(set(FEATURE_COLS_3423)) == 30

# ------------------------------------------------------------
# 4. Join feature values to actual attack days only.
# ------------------------------------------------------------

attack_features_3423 = {}

for dataset_name, frame in new_features_3422.items():

    assert len(frame.columns) == 115

    assert set(FEATURE_COLS_3423).issubset(
        frame.columns
    )

    selected = frame[
        KEYS_3423 + FEATURE_COLS_3423
    ].copy()

    selected["day"] = pd.to_datetime(
        selected["day"]
    ).dt.normalize()

    assert not selected.duplicated(
        KEYS_3423
    ).any()

    joined = attack_meta_3423.merge(
        selected,
        on=KEYS_3423,
        how="left",
        validate="one_to_one",
        indicator=True
    )

    assert joined["_merge"].eq("both").all()

    joined = joined.drop(
        columns=["_merge"]
    ).sort_values(
        KEYS_3423
    ).reset_index(drop=True)

    assert len(joined) == 12636

    attack_features_3423[dataset_name] = joined

    print(
        f"PASS — {dataset_name}: "
        f"{len(joined):,} scheduled attack days"
    )

reference_3423 = attack_features_3423["Natural"]

for name, frame in attack_features_3423.items():

    pd.testing.assert_frame_equal(
        frame[KEYS_3423 + ["Stealth"]],
        reference_3423[KEYS_3423 + ["Stealth"]],
        check_exact=True
    )

print("Matched attack-day keys and stealth labels: PASS")

# ------------------------------------------------------------
# 5. Report complete rolling-window coverage,
#    separately for each stealth level.
# ------------------------------------------------------------

print("\nB. COMPLETE WINDOW COVERAGE ON ATTACK DAYS")

coverage_rows_3423 = []

for level in LEVELS_3423:

    part = reference_3423.loc[
        reference_3423["Stealth"].eq(level)
    ]

    for window in WINDOWS_3423:

        feature = f"cumulative_deviation_{window}wd"

        complete = part[feature].notna()

        coverage_rows_3423.append({
            "Stealth": level,
            "Window_WD": window,
            "Scheduled_Attack_Days": len(part),
            "Complete_Rows": int(complete.sum()),
            "Partial_Rows": int((~complete).sum()),
            "Complete_%": round(
                100 * complete.mean(), 2
            )
        })

window_coverage_3423 = pd.DataFrame(
    coverage_rows_3423
)

display(window_coverage_3423)

# ------------------------------------------------------------
# 6. Compare each new feature for exactly matched
#    Natural, A, and B attack-day windows.
#
# Some features, especially switching and TVD,
# can increase or decrease after injection.
# ------------------------------------------------------------

feature_rows_3423 = []

for item in FEATURE_SPECS_3423:

    feature = item["Column"]

    for level in LEVELS_3423:

        mask = reference_3423["Stealth"].eq(level)

        natural = attack_features_3423["Natural"].loc[
            mask, feature
        ]

        a = attack_features_3423["Formula A"].loc[
            mask, feature
        ]

        b = attack_features_3423["Formula B"].loc[
            mask, feature
        ]

        valid = (
            natural.notna()
            & a.notna()
            & b.notna()
        )

        n = int(valid.sum())

        if n == 0:
            feature_rows_3423.append({
                "Group": item["Group"],
                "Window_WD": item["Window_WD"],
                "Stealth": level,
                "Compared_Rows": 0,
                "Natural_Mean": np.nan,
                "A_Mean": np.nan,
                "B_Mean": np.nan,
                "A_Mean_Change": np.nan,
                "B_Mean_Change": np.nan
            })
            continue

        nat = natural[valid]
        av = a[valid]
        bv = b[valid]

        feature_rows_3423.append({
            "Group": item["Group"],
            "Window_WD": item["Window_WD"],
            "Stealth": level,
            "Compared_Rows": n,
            "Natural_Mean": nat.mean(),
            "A_Mean": av.mean(),
            "B_Mean": bv.mean(),
            "A_Mean_Change": (av - nat).mean(),
            "B_Mean_Change": (bv - nat).mean()
        })

feature_comparison_3423 = pd.DataFrame(
    feature_rows_3423
)

assert len(feature_comparison_3423) == 90

# ------------------------------------------------------------
# 7. Present the 30 WD comparison.
#    Full 5/10/20/30 results remain in memory.
# ------------------------------------------------------------

print("\nC. NEW FEATURE COMPARISON — 30 WD BY STEALTH")

comparison_30wd_3423 = (
    feature_comparison_3423.loc[
        feature_comparison_3423["Window_WD"].eq(30),
        [
            "Group",
            "Stealth",
            "Compared_Rows",
            "Natural_Mean",
            "A_Mean_Change",
            "B_Mean_Change"
        ]
    ].copy()
)

display(
    comparison_30wd_3423.round(4)
)

# ------------------------------------------------------------
# 8. Final safeguards.
# ------------------------------------------------------------

assert len(attack_features_3423) == 3

assert set(feature_comparison_3423["Stealth"]) == {
    "High", "Medium", "Low"
}

assert feature_comparison_3423[
    "Compared_Rows"
].gt(0).all(), (
    "One or more feature/stealth groups have "
    "no comparable complete windows."
)

print("\n" + "=" * 78)
print("STEP 3.4.23 — STEALTH-AWARE COMPARISON COMPLETED")
print("=" * 78)

print("Compared formulas: A and B only")
print("Planned channel-days per formula: 14,375")
print("Scheduled attack days: 12,636")
print("New features evaluated: 30")
print("Window sizes: 5 / 10 / 20 / 30 WD")
print("Feature datasets: 115 columns each")
print("Stealth levels: High / Medium / Low")
print("Final formula: NOT YET SELECTED")
print("Original datasets: UNCHANGED")
print("No files modified or exported.")


STAGE 3 — STEP 3.4.23: A VS B — STEALTH COMPARISON

A. INJECTION AND P95 COMPARISON BY STEALTH


,Formula,Stealth,Channel_Days,Positive_Injection_%,Total_Injected,Natural_Above_P95_%,Newly_Above_P95_%,Final_Above_P95_%,Median_Final_to_P95,P90_Final_to_P95
0,Formula A,High,4413,100.0,12137,3.56,36.51,40.06,1.00,1.5
1,Formula A,Medium,5023,100.0,25952,3.80,96.20,100.00,1.50,2.0
2,Formula A,Low,4939,100.0,37534,3.97,96.03,100.00,2.00,2.5
3,Formula B,High,4413,100.0,7236,3.56,22.34,25.90,0.75,1.5
4,Formula B,Medium,5023,100.0,13759,3.80,35.70,39.50,1.00,2.0
5,Formula B,Low,4939,100.0,21100,3.97,55.92,59.89,1.25,2.5



Scheduled attack days: 12636
PASS — Natural: 12,636 scheduled attack days
PASS — Formula A: 12,636 scheduled attack days
PASS — Formula B: 12,636 scheduled attack days
Matched attack-day keys and stealth labels: PASS

B. COMPLETE WINDOW COVERAGE ON ATTACK DAYS


,Stealth,Window_WD,Scheduled_Attack_Days,Complete_Rows,Partial_Rows,Complete_%
0,High,5,3892,3776,116,97.02
1,High,10,3892,3572,320,91.78
2,High,20,3892,3091,801,79.42
3,High,30,3892,2564,1328,65.88
4,Medium,5,4405,4317,88,98.00
5,Medium,10,4405,4149,256,94.19
6,Medium,20,4405,3718,687,84.40
7,Medium,30,4405,3220,1185,73.10
8,Low,5,4339,4224,115,97.35
9,Low,10,4339,4018,321,92.60



C. NEW FEATURE COMPARISON — 30 WD BY STEALTH


,Group,Stealth,Compared_Rows,Natural_Mean,A_Mean_Change,B_Mean_Change
9,Cumulative Deviation,High,2564,9.0662,15.3887,8.6622
10,Cumulative Deviation,Medium,3220,9.5693,32.0512,15.9213
11,Cumulative Deviation,Low,3210,8.8857,49.4646,27.2376
21,Combined Intensity,High,2564,0.2714,0.1227,0.0742
22,Combined Intensity,Medium,3220,0.2671,0.2311,0.1196
23,Combined Intensity,Low,3210,0.2486,0.3444,0.1948
33,Dominant Switches,High,2564,10.6821,-0.4863,-0.1587
34,Dominant Switches,Medium,3220,10.6339,-0.5640,-0.1568
35,Dominant Switches,Low,3210,10.3523,-0.1981,0.0274
45,Distribution Shift,High,2564,0.0713,0.0162,0.0117



STEP 3.4.23 — STEALTH-AWARE COMPARISON COMPLETED
Compared formulas: A and B only
Planned channel-days per formula: 14,375
Scheduled attack days: 12,636
New features evaluated: 30
Window sizes: 5 / 10 / 20 / 30 WD
Feature datasets: 115 columns each
Stealth levels: High / Medium / Low
Final formula: NOT YET SELECTED
Original datasets: UNCHANGED
No files modified or exported.



### Step 3.4.24 — Final Formula Selection & Injection Verification

**Objective:** Select Formula B as the final injection policy and verify that its injected activity matches the approved attack schedule.

**Selected Formula:** B — Natural-Activity Addition

`Injected Count = max(1, ceil(Natural Count × Stealth Multiplier))`

**Stealth Multipliers:**
- High: 0.5
- Medium: 1.0
- Low: 1.5

**Selection Rationale:**
Formula B was preferred because it executes every planned injection, produces more moderate activity increases than Formula A, maintains clearer stealth progression, and preserves observed channel-switching behavior more closely.

**Validation Tasks:**
1. Select the existing Formula B injection and 115-column feature dataset.
2. Verify all 14,375 scheduled channel-day injections.
3. Confirm that the injected counts match the differences between the selected and natural datasets.
4. Confirm that there are no injections outside scheduled channels or on Off Days.
5. Preserve all historical baselines and existing features.
6. Prepare the selected data for Ground Truth construction.

**Expected Outcome:** A validated Formula B candidate ready for final Ground Truth generation.

**Safety:** No new injection, feature recalculation, original data modification, or file export.


In [49]:

# Stage 3 — Step 3.4.24: Final Formula Selection & Injection Verification
# Goal: Select Formula B, verify exact injection counts and
# ensure that only scheduled channels were modified.
#
# Read-only: No data files modified or exported.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 75)
print("STAGE 3 — STEP 3.4.24: FINAL FORMULA SELECTION")
print("=" * 75)

# -------------------------------------------------------
# 1. Select Formula B without recalculating any features.
# -------------------------------------------------------

selected_formula_3424 = "B_Natural_Addition"

selected_features_3424 = (
    new_features_3422["Formula B"].copy(deep=True)
)

natural_features_3424 = (
    new_features_3422["Natural"].copy(deep=True)
)

selected_schedule_3424 = (
    validated_schedule_339.copy(deep=True)
)

selected_injection_3424 = (
    quantity_experiment_343.loc[
        quantity_experiment_343["formula"].eq(
            selected_formula_3424
        )
    ].copy(deep=True)
)

assert len(selected_features_3424) == 69634
assert len(selected_features_3424.columns) == 115
assert selected_features_3424["user"].nunique() == 670

assert len(selected_injection_3424) == 14375
assert selected_injection_3424["injected_count"].gt(0).all()

print("\nSelected formula:", selected_formula_3424)
print("Users:", selected_features_3424["user"].nunique())
print("Feature rows:", len(selected_features_3424))
print("Feature columns:", len(selected_features_3424.columns))

# -------------------------------------------------------
# 2. Standardize keys in independent working copies.
# -------------------------------------------------------

keys_3424 = ["user", "day"]

for frame in [
    selected_features_3424,
    natural_features_3424,
    selected_schedule_3424,
    selected_injection_3424
]:
    frame["day"] = pd.to_datetime(
        frame["day"]
    ).dt.normalize()

for frame in [
    selected_features_3424,
    natural_features_3424
]:
    assert not frame.duplicated(keys_3424).any()

selected_features_3424 = (
    selected_features_3424
    .sort_values(keys_3424)
    .reset_index(drop=True)
)

natural_features_3424 = (
    natural_features_3424
    .sort_values(keys_3424)
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    selected_features_3424[keys_3424],
    natural_features_3424[keys_3424],
    check_exact=True
)

print("\nUser-day alignment: PASS")

# -------------------------------------------------------
# 3. Validate planned injection records.
# -------------------------------------------------------

selected_injection_3424["channel"] = (
    selected_injection_3424["channel"]
    .astype(str)
    .str.strip()
    .str.lower()
)

channel_map_3424 = {
    "email": "external_email_count",
    "usb": "usb_copy_count",
    "http": "www_upload_count"
}

assert set(selected_injection_3424["channel"]).issubset(
    channel_map_3424
)

assert not selected_injection_3424.duplicated(
    ["user", "day", "channel"]
).any()

assert (
    selected_injection_3424.groupby(
        ["user", "day"]
    ).ngroups == 12636
)

print("\nA. SELECTED INJECTION SUMMARY")

injection_summary_3424 = (
    selected_injection_3424
    .groupby("effective_stealth")
    .agg(
        Channel_Days=("injected_count", "size"),
        Injected_Events=("injected_count", "sum"),
        Users=("user", "nunique")
    )
    .reindex(["High", "Medium", "Low"])
    .reset_index()
)

display(injection_summary_3424)

print(
    "Total injected events:",
    int(selected_injection_3424["injected_count"].sum())
)

# -------------------------------------------------------
# 4. Reconstruct expected additions from Formula B.
#
# This is verification only, not a new injection.
# -------------------------------------------------------

expected_3424 = (
    selected_injection_3424
    .pivot_table(
        index=keys_3424,
        columns="channel",
        values="injected_count",
        aggfunc="sum",
        fill_value=0
    )
    .rename(
        columns=channel_map_3424
    )
    .reset_index()
)

expected_3424 = (
    natural_features_3424[keys_3424]
    .merge(
        expected_3424,
        on=keys_3424,
        how="left",
        validate="one_to_one"
    )
)

count_cols_3424 = list(channel_map_3424.values())

expected_3424[count_cols_3424] = (
    expected_3424[count_cols_3424]
    .fillna(0)
)

print("\nB. EXACT INJECTION RECONCILIATION")

reconciliation_rows_3424 = []

for channel, count_col in channel_map_3424.items():

    observed_addition = (
        selected_features_3424[count_col].to_numpy(
            dtype=float
        )
        - natural_features_3424[count_col].to_numpy(
            dtype=float
        )
    )

    expected_addition = (
        expected_3424[count_col].to_numpy(
            dtype=float
        )
    )

    np.testing.assert_allclose(
        observed_addition,
        expected_addition,
        rtol=0,
        atol=1e-9,
        err_msg=f"Injection mismatch: {channel}"
    )

    reconciliation_rows_3424.append({
        "Channel": channel.upper(),
        "Expected_Added": int(expected_addition.sum()),
        "Observed_Added": int(observed_addition.sum()),
        "Mismatched_Days": int(
            (
                np.abs(
                    observed_addition - expected_addition
                ) > 1e-9
            ).sum()
        ),
        "Status": "PASS"
    })

reconciliation_3424 = pd.DataFrame(
    reconciliation_rows_3424
)

display(reconciliation_3424)

# -------------------------------------------------------
# 5. Verify no injection occurred on Off Days.
# -------------------------------------------------------

assert not selected_schedule_3424.duplicated(
    keys_3424
).any()

off_days_3424 = (
    selected_schedule_3424.loc[
        selected_schedule_3424["planned_channels"].eq(0),
        keys_3424
    ].copy()
)

assert len(off_days_3424) == 2374

off_check_3424 = off_days_3424.merge(
    expected_3424,
    on=keys_3424,
    how="left",
    validate="one_to_one",
    indicator=True
)

assert off_check_3424["_merge"].eq("both").all()

assert (
    off_check_3424[count_cols_3424]
    .fillna(0)
    .eq(0)
    .all()
    .all()
)

# The exact reconciliation also verifies that
# unscheduled user-day-channel additions are zero.

print("\nC. SCHEDULE COMPLIANCE")

print("Scheduled attack days: 12,636")
print("Scheduled channel-days: 14,375")
print("Off Days checked:", len(off_days_3424))
print("Unexpected injection on Off Days: 0")
print("Unexpected injection outside plan: 0")
print("Schedule compliance: PASS")

# -------------------------------------------------------
# 6. Verify final candidate alignment and preservation.
# -------------------------------------------------------

assert selected_features_3424.shape == (69634, 115)
assert len(selected_injection_3424) == 14375

assert (
    reconciliation_3424["Mismatched_Days"]
    .eq(0)
    .all()
)

assert int(
    reconciliation_3424["Observed_Added"].sum()
) == int(
    selected_injection_3424["injected_count"].sum()
)

print("\n" + "=" * 75)
print("STEP 3.4.24 — FINAL FORMULA CANDIDATE VALIDATED")
print("=" * 75)

print("Selected formula: B_Natural_Addition")
print("Selected feature dataset: selected_features_3424")
print("Selected injection records: selected_injection_3424")
print("Validated schedule: selected_schedule_3424")
print("Total columns: 115")
print("Injection reconciliation: PASS")
print("Off-day protection: PASS")
print("Ground Truth: NOT YET GENERATED")
print("Original datasets: UNCHANGED")
print("No files exported.")


STAGE 3 — STEP 3.4.24: FINAL FORMULA SELECTION

Selected formula: B_Natural_Addition
Users: 670
Feature rows: 69634
Feature columns: 115

User-day alignment: PASS

A. SELECTED INJECTION SUMMARY


,effective_stealth,Channel_Days,Injected_Events,Users
0,High,4413,7236,217
1,Medium,5023,13759,226
2,Low,4939,21100,227


Total injected events: 42095

B. EXACT INJECTION RECONCILIATION


,Channel,Expected_Added,Observed_Added,Mismatched_Days,Status
0,EMAIL,23880,23880,0,PASS
1,USB,11422,11422,0,PASS
2,HTTP,6793,6793,0,PASS



C. SCHEDULE COMPLIANCE
Scheduled attack days: 12,636
Scheduled channel-days: 14,375
Off Days checked: 2374
Unexpected injection on Off Days: 0
Unexpected injection outside plan: 0
Schedule compliance: PASS

STEP 3.4.24 — FINAL FORMULA CANDIDATE VALIDATED
Selected formula: B_Natural_Addition
Selected feature dataset: selected_features_3424
Selected injection records: selected_injection_3424
Validated schedule: selected_schedule_3424
Total columns: 115
Injection reconciliation: PASS
Off-day protection: PASS
Ground Truth: NOT YET GENERATED
Original datasets: UNCHANGED
No files exported.



### Step 3.4.25 — Ground Truth Generation

**Objective:** Generate accurate synthetic ground-truth labels for the selected Formula B simulation using the validated schedule and actual injected counts.

**Selected Formula:** B — Natural-Activity Addition

**Inputs:**
- `selected_schedule_3424`
- `selected_injection_3424`
- `selected_features_3424`
- `natural_features_3424`

**Ground Truth Levels:**

1. **Channel-Level:** Each injected Email, USB, or HTTP channel-day, including the scenario, stealth level, natural count, injected count, and final count.
2. **Scheduled Day-Level:** Each scheduled day, labeled as Attack or Off, with injected activity totals and number of affected channels.
3. **Full User-Day-Level:** All eligible user-days, identifying Attack, Off, and Outside Schedule days.

**Label Definitions:**
- `is_attack_day = 1`: At least one channel received positive synthetic injection.
- `is_off_day = 1`: A scheduled Off day with no synthetic injection.
- `outside_schedule`: An eligible day that is not part of the validated attack schedule.
- `injected_events_total`: Total synthetic events added across Email, USB, and HTTP on that day.

**Validation Requirements:**
- Exactly 14,375 injected channel-days.
- Exactly 12,636 attack days.
- Exactly 2,374 scheduled Off days.
- Exactly 42,095 injected events.
- Zero injections outside the approved schedule.
- Exact agreement between injected counts and observed count differences.
- Consistent scenario IDs and stealth levels.
- Ground-truth labels remain separate from the 115-column feature dataset.

**Important:** Labels represent controlled synthetic count-level injections, not confirmed payload-byte transfers or raw reconstructed attack events.

**Expected Outcome:** Three validated ground-truth datasets ready for final validation and export.

**Safety:** Read-only generation. No source datasets are modified or exported.


In [50]:

# Stage 3 — Step 3.4.25: Final Ground Truth Generation
# Goal: Create three Ground Truth tables for the selected
# Formula B attack simulation:
#
#   1. User-level Ground Truth
#   2. Channel-day Ground Truth
#   3. Full user-day Ground Truth
#
# Read-only:
# No original datasets modified.
# No feature recalculation.
# No files exported.

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 78)
print("STAGE 3 — STEP 3.4.25: FINAL GROUND TRUTH GENERATION")
print("=" * 78)

# ------------------------------------------------------------
# 1. Define approved constants and expected results.
# ------------------------------------------------------------

FORMULA_3425 = "B_Natural_Addition"
SOURCE_3425 = "synthetic_injection"

KEYS_3425 = ["user", "day"]

CHANNEL_MAP_3425 = {
    "email": "external_email_count",
    "usb": "usb_copy_count",
    "http": "www_upload_count"
}

STEALTH_MULTIPLIERS_3425 = {
    "High": 0.5,
    "Medium": 1.0,
    "Low": 1.5
}

EXPECTED_3425 = {
    "users": 670,
    "channel_days": 14375,
    "attack_days": 12636,
    "off_days": 2374,
    "scheduled_days": 15010,
    "user_days": 69634,
    "injected_events": 42095,
    "feature_columns": 115
}

# ------------------------------------------------------------
# 2. Prepare independent working copies.
# ------------------------------------------------------------

schedule_3425 = (
    selected_schedule_3424.copy(deep=True)
)

injection_3425 = (
    selected_injection_3424.copy(deep=True)
)

features_3425 = (
    selected_features_3424.copy(deep=True)
)

natural_3425 = (
    natural_features_3424.copy(deep=True)
)

validated_users_3425 = (
    validated_users_339.copy(deep=True)
)

for frame in [
    schedule_3425,
    injection_3425,
    features_3425,
    natural_3425
]:
    frame["day"] = pd.to_datetime(
        frame["day"]
    ).dt.normalize()

for col in ["window_start", "window_end"]:
    schedule_3425[col] = pd.to_datetime(
        schedule_3425[col]
    ).dt.normalize()

injection_3425["channel"] = (
    injection_3425["channel"]
    .astype(str)
    .str.strip()
    .str.lower()
)

assert len(schedule_3425) == 15010
assert len(injection_3425) == 14375
assert len(features_3425) == 69634
assert features_3425.shape[1] == 115

assert len(validated_users_3425) == 670
assert validated_users_3425["user"].is_unique

assert not schedule_3425.duplicated(
    KEYS_3425
).any()

assert not injection_3425.duplicated(
    KEYS_3425 + ["channel"]
).any()

assert not features_3425.duplicated(
    KEYS_3425
).any()

assert not natural_3425.duplicated(
    KEYS_3425
).any()

assert set(injection_3425["channel"]) == set(
    CHANNEL_MAP_3425
)

assert injection_3425["formula"].eq(
    FORMULA_3425
).all()

assert injection_3425["injected_count"].gt(0).all()

print("\nInput datasets: PASS")


# ------------------------------------------------------------
# 3. Verify Formula B independently.
# ------------------------------------------------------------

print("\nA. FORMULA B VALIDATION")

expected_multiplier_3425 = (
    injection_3425["effective_stealth"]
    .map(STEALTH_MULTIPLIERS_3425)
)

assert expected_multiplier_3425.notna().all()

np.testing.assert_allclose(
    injection_3425["stealth_multiplier"],
    expected_multiplier_3425,
    rtol=0,
    atol=1e-10
)

expected_injection_3425 = np.maximum(
    1,
    np.ceil(
        injection_3425["natural_count"].to_numpy(
            dtype=float
        )
        * expected_multiplier_3425.to_numpy(
            dtype=float
        )
    )
)

np.testing.assert_allclose(
    injection_3425["injected_count"].to_numpy(
        dtype=float
    ),
    expected_injection_3425,
    rtol=0,
    atol=1e-9
)

np.testing.assert_allclose(
    injection_3425["simulated_final_count"],
    injection_3425["natural_count"]
    + injection_3425["injected_count"],
    rtol=0,
    atol=1e-9
)

assert int(
    injection_3425["injected_count"].sum()
) == EXPECTED_3425["injected_events"]

print("Formula B calculation: PASS")
print("Injected events: 42,095")


# ------------------------------------------------------------
# 4. Build Channel-Day Ground Truth.
# ------------------------------------------------------------

ground_truth_channel_days_3425 = (
    injection_3425[
        [
            "user",
            "day",
            "channel",
            "scenario_id",
            "effective_stealth",
            "natural_count",
            "candidate_p95",
            "stealth_multiplier",
            "injected_count",
            "simulated_final_count"
        ]
    ]
    .rename(
        columns={
            "effective_stealth": "stealth_level",
            "candidate_p95": "historical_p95",
            "simulated_final_count": "final_count"
        }
    )
    .copy()
)

ground_truth_channel_days_3425[
    "injection_formula"
] = FORMULA_3425

ground_truth_channel_days_3425 = (
    ground_truth_channel_days_3425
    .sort_values(["user", "day", "channel"])
    .reset_index(drop=True)
)

assert len(
    ground_truth_channel_days_3425
) == EXPECTED_3425["channel_days"]

assert not ground_truth_channel_days_3425.duplicated(
    ["user", "day", "channel"]
).any()

assert ground_truth_channel_days_3425[
    "historical_p95"
].gt(0).all()

assert ground_truth_channel_days_3425[
    "natural_count"
].gt(0).all()

assert ground_truth_channel_days_3425[
    "injected_count"
].gt(0).all()

np.testing.assert_allclose(
    ground_truth_channel_days_3425["final_count"],
    ground_truth_channel_days_3425["natural_count"]
    + ground_truth_channel_days_3425["injected_count"],
    rtol=0,
    atol=1e-9
)

print("\nB. CHANNEL-DAY GROUND TRUTH")

print(
    "Rows:",
    len(ground_truth_channel_days_3425)
)

display(
    ground_truth_channel_days_3425.head(10)
)


# ------------------------------------------------------------
# 5. Validate exact channel selections against schedule.
#
# The schedule uses attack_channels strings such as:
# "email|usb" or "http".
# ------------------------------------------------------------

active_schedule_3425 = schedule_3425.loc[
    schedule_3425["planned_channels"].gt(0),
    KEYS_3425 + ["attack_channels"]
].copy()

planned_channel_keys_3425 = (
    active_schedule_3425
    .assign(
        channel=lambda df: df[
            "attack_channels"
        ].str.split("|", regex=False)
    )
    .explode("channel")
)

planned_channel_keys_3425["channel"] = (
    planned_channel_keys_3425["channel"]
    .astype(str)
    .str.strip()
    .str.lower()
)

planned_channel_keys_3425 = (
    planned_channel_keys_3425[
        KEYS_3425 + ["channel"]
    ]
    .sort_values(
        KEYS_3425 + ["channel"]
    )
    .reset_index(drop=True)
)

actual_channel_keys_3425 = (
    ground_truth_channel_days_3425[
        KEYS_3425 + ["channel"]
    ]
    .sort_values(
        KEYS_3425 + ["channel"]
    )
    .reset_index(drop=True)
)

assert len(planned_channel_keys_3425) == 14375

pd.testing.assert_frame_equal(
    planned_channel_keys_3425,
    actual_channel_keys_3425,
    check_exact=True
)

print("\nScheduled vs injected channel keys: PASS")


# ------------------------------------------------------------
# 6. Aggregate channel injections by user and day.
# ------------------------------------------------------------

daily_totals_3425 = (
    ground_truth_channel_days_3425
    .groupby(KEYS_3425, as_index=False)
    .agg(
        injected_channel_count=(
            "channel", "nunique"
        ),
        injected_events_total=(
            "injected_count", "sum"
        )
    )
)

daily_channels_3425 = (
    ground_truth_channel_days_3425
    .pivot(
        index=KEYS_3425,
        columns="channel",
        values="injected_count"
    )
    .fillna(0)
    .reset_index()
)

daily_channels_3425 = (
    daily_channels_3425.rename(
        columns={
            "email": "injected_email",
            "usb": "injected_usb",
            "http": "injected_http"
        }
    )
)

daily_totals_3425 = daily_totals_3425.merge(
    daily_channels_3425,
    on=KEYS_3425,
    how="left",
    validate="one_to_one"
)

assert len(daily_totals_3425) == 12636

assert (
    daily_totals_3425[
        [
            "injected_email",
            "injected_usb",
            "injected_http"
        ]
    ].sum(axis=1)
    == daily_totals_3425[
        "injected_events_total"
    ]
).all()

print("\nDaily channel aggregation: PASS")


# ------------------------------------------------------------
# 7. Build User-Level Ground Truth.
#
# Use independently validated switching targets
# from Stage 3.3.9.
# ------------------------------------------------------------

user_schedule_3425 = (
    schedule_3425
    .groupby("user", as_index=False)
    .agg(
        scenario_id=("scenario_id", "first"),
        attack_start=("window_start", "first"),
        attack_end=("window_end", "first"),
        duration_days=("day", "size"),
        attack_days=(
            "planned_channels",
            lambda s: int(s.gt(0).sum())
        ),
        off_days=(
            "planned_channels",
            lambda s: int(s.eq(0).sum())
        )
    )
)

user_stealth_3425 = (
    ground_truth_channel_days_3425
    .groupby("user", as_index=False)
    .agg(
        stealth_level=("stealth_level", "first"),
        injected_events_total=(
            "injected_count", "sum"
        )
    )
)

ground_truth_users_3425 = (
    user_schedule_3425
    .merge(
        user_stealth_3425,
        on="user",
        how="left",
        validate="one_to_one"
    )
    .merge(
        validated_users_3425[
            [
                "user",
                "scenario_id",
                "duration",
                "target_switches",
                "realized_switches"
            ]
        ].rename(
            columns={
                "scenario_id": "validated_scenario_id",
                "duration": "validated_duration"
            }
        ),
        on="user",
        how="left",
        validate="one_to_one"
    )
)

assert ground_truth_users_3425[
    "scenario_id"
].eq(
    ground_truth_users_3425[
        "validated_scenario_id"
    ]
).all()

assert (
    ground_truth_users_3425["duration_days"]
    == ground_truth_users_3425["validated_duration"]
).all()

assert (
    ground_truth_users_3425["target_switches"]
    == ground_truth_users_3425["realized_switches"]
).all()

ground_truth_users_3425["planned_switches"] = (
    ground_truth_users_3425["target_switches"]
)

ground_truth_users_3425["injection_formula"] = (
    FORMULA_3425
)

ground_truth_users_3425["ground_truth_source"] = (
    SOURCE_3425
)

ground_truth_users_3425 = (
    ground_truth_users_3425[
        [
            "user",
            "scenario_id",
            "stealth_level",
            "attack_start",
            "attack_end",
            "duration_days",
            "attack_days",
            "off_days",
            "planned_switches",
            "injected_events_total",
            "injection_formula",
            "ground_truth_source"
        ]
    ]
    .sort_values("user")
    .reset_index(drop=True)
)

assert len(ground_truth_users_3425) == 670

assert ground_truth_users_3425[
    "user"
].is_unique

assert (
    ground_truth_users_3425["attack_days"]
    + ground_truth_users_3425["off_days"]
    == ground_truth_users_3425["duration_days"]
).all()

assert (
    ground_truth_users_3425["attack_start"]
    <= ground_truth_users_3425["attack_end"]
).all()

assert int(
    ground_truth_users_3425[
        "injected_events_total"
    ].sum()
) == 42095

print("\nC. USER-LEVEL GROUND TRUTH")

print("Rows:", len(ground_truth_users_3425))

display(
    ground_truth_users_3425.head(10)
)


# ------------------------------------------------------------
# 8. Build Full User-Day Ground Truth.
#
# Every eligible user-day appears exactly once.
# Schedule information and daily injections are joined
# only as labels; the ML features remain separate.
# ------------------------------------------------------------

ground_truth_user_days_3425 = (
    features_3425[KEYS_3425]
    .merge(
        ground_truth_users_3425[
            ["user", "scenario_id", "stealth_level"]
        ],
        on="user",
        how="left",
        validate="many_to_one"
    )
)

assert ground_truth_user_days_3425[
    ["scenario_id", "stealth_level"]
].notna().all().all()

schedule_labels_3425 = (
    schedule_3425[
        KEYS_3425 + ["planned_channels"]
    ].copy()
)

ground_truth_user_days_3425 = (
    ground_truth_user_days_3425
    .merge(
        schedule_labels_3425,
        on=KEYS_3425,
        how="left",
        validate="one_to_one",
        indicator="_schedule_match"
    )
)

ground_truth_user_days_3425["is_scheduled"] = (
    ground_truth_user_days_3425[
        "_schedule_match"
    ].eq("both").astype(np.int8)
)

ground_truth_user_days_3425.drop(
    columns=["_schedule_match"],
    inplace=True
)

ground_truth_user_days_3425 = (
    ground_truth_user_days_3425
    .merge(
        daily_totals_3425,
        on=KEYS_3425,
        how="left",
        validate="one_to_one"
    )
)

daily_numeric_cols_3425 = [
    "planned_channels",
    "injected_channel_count",
    "injected_email",
    "injected_usb",
    "injected_http",
    "injected_events_total"
]

ground_truth_user_days_3425[
    daily_numeric_cols_3425
] = ground_truth_user_days_3425[
    daily_numeric_cols_3425
].fillna(0).astype(np.int64)

ground_truth_user_days_3425[
    "is_attack_day"
] = (
    ground_truth_user_days_3425[
        "injected_events_total"
    ].gt(0).astype(np.int8)
)

ground_truth_user_days_3425["day_status"] = (
    np.select(
        [
            ground_truth_user_days_3425[
                "is_attack_day"
            ].eq(1),
            ground_truth_user_days_3425[
                "is_scheduled"
            ].eq(1)
        ],
        [
            "Attack",
            "Off"
        ],
        default="Outside Schedule"
    )
)

ground_truth_user_days_3425 = (
    ground_truth_user_days_3425[
        [
            "user",
            "day",
            "scenario_id",
            "stealth_level",
            "day_status",
            "is_scheduled",
            "is_attack_day",
            "planned_channels",
            "injected_channel_count",
            "injected_email",
            "injected_usb",
            "injected_http",
            "injected_events_total"
        ]
    ]
    .sort_values(KEYS_3425)
    .reset_index(drop=True)
)

assert len(
    ground_truth_user_days_3425
) == EXPECTED_3425["user_days"]

assert not ground_truth_user_days_3425.duplicated(
    KEYS_3425
).any()

# ------------------------------------------------------------
# 9. Validate daily labels and schedule consistency.
# ------------------------------------------------------------

print("\nD. FULL USER-DAY GROUND TRUTH")

status_counts_3425 = (
    ground_truth_user_days_3425[
        "day_status"
    ].value_counts()
)

assert status_counts_3425["Attack"] == 12636
assert status_counts_3425["Off"] == 2374

assert status_counts_3425[
    "Outside Schedule"
] == 54624

assert ground_truth_user_days_3425[
    "is_scheduled"
].sum() == 15010

assert ground_truth_user_days_3425[
    "is_attack_day"
].sum() == 12636

assert int(
    ground_truth_user_days_3425[
        "injected_events_total"
    ].sum()
) == 42095

# Planned and actual channel counts must agree.
assert (
    ground_truth_user_days_3425[
        "planned_channels"
    ]
    == ground_truth_user_days_3425[
        "injected_channel_count"
    ]
).all()

# No artificial injection on Off or Outside Schedule.
non_attack_3425 = (
    ground_truth_user_days_3425[
        "day_status"
    ].ne("Attack")
)

assert ground_truth_user_days_3425.loc[
    non_attack_3425,
    [
        "injected_email",
        "injected_usb",
        "injected_http",
        "injected_events_total"
    ]
].eq(0).all().all()

day_summary_3425 = (
    ground_truth_user_days_3425
    .groupby("day_status")
    .agg(
        User_Days=("user", "size"),
        Users=("user", "nunique"),
        Injected_Events=("injected_events_total", "sum")
    )
    .reset_index()
)

display(day_summary_3425)


# ------------------------------------------------------------
# 10. Cross-table and feature-label separation checks.
# ------------------------------------------------------------

print("\nE. CROSS-TABLE VALIDATION")

assert (
    ground_truth_channel_days_3425[
        "injected_count"
    ].sum()
    == ground_truth_user_days_3425[
        "injected_events_total"
    ].sum()
    == ground_truth_users_3425[
        "injected_events_total"
    ].sum()
    == 42095
)

assert (
    ground_truth_users_3425[
        "attack_days"
    ].sum()
) == 12636

assert (
    ground_truth_users_3425[
        "off_days"
    ].sum()
) == 2374

assert (
    ground_truth_user_days_3425[
        ["injected_email", "injected_usb", "injected_http"]
    ].sum(axis=1)
    == ground_truth_user_days_3425[
        "injected_events_total"
    ]
).all()

# Labels must never become input features.
label_only_columns_3425 = {
    "scenario_id",
    "stealth_level",
    "day_status",
    "is_scheduled",
    "is_attack_day",
    "planned_channels",
    "injected_channel_count",
    "injected_email",
    "injected_usb",
    "injected_http",
    "injected_events_total",
    "injection_formula",
    "ground_truth_source"
}

assert label_only_columns_3425.isdisjoint(
    selected_features_3424.columns
), "Ground Truth labels found in the ML feature dataset."

assert selected_features_3424.shape == (69634, 115)

print("Unique keys: PASS")
print("Exact planned channel matching: PASS")
print("User-level switch targets: PASS")
print("Formula B: PASS")
print("Injected event reconciliation: PASS")
print("Attack / Off / Outside Schedule: PASS")
print("Feature-label separation: PASS")


# ------------------------------------------------------------
# 11. Final summary.
# ------------------------------------------------------------

print("\nF. FINAL GROUND TRUTH INVENTORY")

inventory_3425 = pd.DataFrame([
    {
        "Dataset": "ground_truth_users_3425",
        "Rows": len(ground_truth_users_3425),
        "Columns": len(ground_truth_users_3425.columns),
        "Primary_Key": "user"
    },
    {
        "Dataset": "ground_truth_channel_days_3425",
        "Rows": len(ground_truth_channel_days_3425),
        "Columns": len(ground_truth_channel_days_3425.columns),
        "Primary_Key": "user + day + channel"
    },
    {
        "Dataset": "ground_truth_user_days_3425",
        "Rows": len(ground_truth_user_days_3425),
        "Columns": len(ground_truth_user_days_3425.columns),
        "Primary_Key": "user + day"
    }
])

display(inventory_3425)

print("\n" + "=" * 78)
print("STEP 3.4.25 — FINAL GROUND TRUTH GENERATION COMPLETED")
print("=" * 78)

print("Selected formula: B_Natural_Addition")
print("Ground Truth tables: 3")
print("Users: 670")
print("Attack days: 12,636")
print("Off days: 2,374")
print("Outside schedule days: 54,624")
print("Injected channel-days: 14,375")
print("Injected events: 42,095")
print("Feature dataset: 69,634 rows × 115 columns")
print("Original datasets: UNCHANGED")
print("No files exported.")


STAGE 3 — STEP 3.4.25: FINAL GROUND TRUTH GENERATION

Input datasets: PASS

A. FORMULA B VALIDATION
Formula B calculation: PASS
Injected events: 42,095

B. CHANNEL-DAY GROUND TRUTH
Rows: 14375


,user,day,channel,scenario_id,stealth_level,natural_count,historical_p95,stealth_multiplier,injected_count,final_count,injection_formula
0,AAF0819,2011-03-03,email,SCN_0007,High,2,4.0,0.5,1,3,B_Natural_Addition
1,AAF0819,2011-03-04,email,SCN_0007,High,1,4.0,0.5,1,2,B_Natural_Addition
2,AAF0819,2011-03-07,usb,SCN_0007,High,1,4.0,0.5,1,2,B_Natural_Addition
3,AAF0819,2011-03-08,email,SCN_0007,High,3,4.0,0.5,2,5,B_Natural_Addition
4,AAF0819,2011-03-09,usb,SCN_0007,High,1,4.0,0.5,1,2,B_Natural_Addition
5,AAP0352,2011-04-06,usb,SCN_0016,High,2,2.1,0.5,1,3,B_Natural_Addition
6,AAP0352,2011-04-07,email,SCN_0016,High,3,4.0,0.5,2,5,B_Natural_Addition
7,AAP0352,2011-04-08,email,SCN_0016,High,2,4.0,0.5,1,3,B_Natural_Addition
8,AAP0352,2011-04-11,email,SCN_0016,High,3,4.0,0.5,2,5,B_Natural_Addition
9,AAP0352,2011-04-11,usb,SCN_0016,High,1,2.1,0.5,1,2,B_Natural_Addition



Scheduled vs injected channel keys: PASS

Daily channel aggregation: PASS

C. USER-LEVEL GROUND TRUTH
Rows: 670


,user,scenario_id,stealth_level,attack_start,attack_end,duration_days,attack_days,off_days,planned_switches,injected_events_total,injection_formula,ground_truth_source
0,AAF0819,SCN_0007,High,2011-03-03,2011-03-09,5,5,0,3,6,B_Natural_Addition,synthetic_injection
1,AAP0352,SCN_0016,High,2011-04-06,2011-04-19,10,9,1,3,12,B_Natural_Addition,synthetic_injection
2,AAP1919,SCN_0040,High,2011-01-31,2011-02-25,20,18,2,4,23,B_Natural_Addition,synthetic_injection
3,ABD3426,SCN_0402,Low,2011-03-31,2011-05-25,40,32,8,7,226,B_Natural_Addition,synthetic_injection
4,ABK3081,SCN_0009,Low,2011-01-25,2011-01-31,5,5,0,3,15,B_Natural_Addition,synthetic_injection
5,ABM3772,SCN_0008,Medium,2011-02-03,2011-02-09,5,5,0,3,9,B_Natural_Addition,synthetic_injection
6,ABP2917,SCN_0041,Medium,2011-04-14,2011-05-11,20,18,2,4,37,B_Natural_Addition,synthetic_injection
7,ABS0677,SCN_0065,Medium,2011-01-13,2011-02-09,20,17,3,4,37,B_Natural_Addition,synthetic_injection
8,ACB3123,SCN_0110,Medium,2011-02-23,2011-04-05,30,26,4,5,50,B_Natural_Addition,synthetic_injection
9,ACP2122,SCN_0003,Low,2011-03-07,2011-03-11,5,5,0,1,27,B_Natural_Addition,synthetic_injection



D. FULL USER-DAY GROUND TRUTH


,day_status,User_Days,Users,Injected_Events
0,Attack,12636,670,42095
1,Off,2374,520,0
2,Outside Schedule,54624,668,0



E. CROSS-TABLE VALIDATION
Unique keys: PASS
Exact planned channel matching: PASS
User-level switch targets: PASS
Formula B: PASS
Injected event reconciliation: PASS
Attack / Off / Outside Schedule: PASS
Feature-label separation: PASS

F. FINAL GROUND TRUTH INVENTORY


,Dataset,Rows,Columns,Primary_Key
0,ground_truth_users_3425,670,12,user
1,ground_truth_channel_days_3425,14375,11,user + day + channel
2,ground_truth_user_days_3425,69634,13,user + day



STEP 3.4.25 — FINAL GROUND TRUTH GENERATION COMPLETED
Selected formula: B_Natural_Addition
Ground Truth tables: 3
Users: 670
Attack days: 12,636
Off days: 2,374
Outside schedule days: 54,624
Injected channel-days: 14,375
Injected events: 42,095
Feature dataset: 69,634 rows × 115 columns
Original datasets: UNCHANGED
No files exported.



### Step 3.4.26 — Final Validation & Export

**Objective:** Validate and export the final Formula B simulation datasets for the Machine Learning team.

**Selected Formula:** B — Natural-Activity Addition

**Output Directory:**
`Attack_Simulation/data/`

**Final Files:**

1. `ground_truth_users.parquet`
   - 670 simulated users.
   - User-level scenario, stealth, schedule, and injection metadata.

2. `ground_truth_channel_days.parquet`
   - 14,375 injected user-day-channel records.
   - Natural, injected, and final event counts.

3. `ground_truth_user_days.parquet`
   - 69,634 eligible user-days.
   - Attack, Off, and Outside Schedule labels.

4. `behavior_features_injected_final.parquet`
   - 69,634 user-days and 115 columns.
   - Original and newly engineered behavioral features.
   - No Ground Truth labels.

**Final Validation:**
- Verify expected row counts and unique primary keys.
- Validate user, scenario, and stealth consistency.
- Reconcile channel-level, day-level, and user-level injection totals.
- Confirm exactly 42,095 injected events.
- Confirm 12,636 Attack days and 2,374 Off days.
- Confirm no injection outside the approved schedule.
- Verify actual feature-count changes against Ground Truth.
- Confirm the five original malicious CERT users remain excluded.
- Verify feature-label separation.
- Save the files in Parquet format and read them back to validate integrity.

**Safety:**
- Do not modify source datasets or historical baselines.
- Do not recalculate or change the approved injection.
- Do not overwrite existing output files.
- Export only after all validation checks pass.

**Expected Outcome:** Four verified Parquet files ready for the Machine Learning team.


In [51]:

# Stage 3 — Step 3.4.26: Final Validation & Export
# Goal: Validate and export the approved Formula B datasets.
#
# Safety:
# - Never modify source datasets.
# - Never overwrite existing files.
# - Validate all datasets before exporting.
# - Read exported files back to verify integrity.

import os
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

print("=" * 78)
print("STAGE 3 — STEP 3.4.26: FINAL VALIDATION & EXPORT")
print("=" * 78)

# ------------------------------------------------------------
# 1. Define the approved output location and filenames.
# ------------------------------------------------------------

OUTPUT_DIR_3426 = Path(
    r"C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration"
    r"\Attack_Simulation\data"
)

EXPECTED_3426 = {
    "users": 670,
    "channel_days": 14375,
    "attack_days": 12636,
    "off_days": 2374,
    "outside_days": 54624,
    "user_days": 69634,
    "injected_events": 42095,
    "feature_columns": 115
}

KEYS_3426 = ["user", "day"]

# ------------------------------------------------------------
# 2. Confirm Step 3.4.25 was completed.
# ------------------------------------------------------------

required_objects_3426 = [
    "ground_truth_users_3425",
    "ground_truth_channel_days_3425",
    "ground_truth_user_days_3425",
    "selected_features_3424",
    "natural_features_3424",
    "selected_schedule_3424"
]

missing_objects_3426 = [
    name
    for name in required_objects_3426
    if name not in globals()
]

if missing_objects_3426:
    raise RuntimeError(
        "Run Step 3.4.25 successfully before exporting. "
        f"Missing objects: {missing_objects_3426}"
    )

# Work only on independent copies.
users_3426 = ground_truth_users_3425.copy(deep=True)
channels_3426 = ground_truth_channel_days_3425.copy(deep=True)
days_3426 = ground_truth_user_days_3425.copy(deep=True)
features_3426 = selected_features_3424.copy(deep=True)
natural_3426 = natural_features_3424.copy(deep=True)
schedule_3426 = selected_schedule_3424.copy(deep=True)

for frame in [
    channels_3426,
    days_3426,
    features_3426,
    natural_3426,
    schedule_3426
]:
    frame["day"] = pd.to_datetime(
        frame["day"]
    ).dt.normalize()

for col in ["attack_start", "attack_end"]:
    users_3426[col] = pd.to_datetime(
        users_3426[col]
    ).dt.normalize()

# ------------------------------------------------------------
# 3. Verify row counts, columns, and unique keys.
# ------------------------------------------------------------

print("\nA. DATASET STRUCTURE VALIDATION")

assert users_3426.shape == (670, 12)
assert channels_3426.shape == (14375, 11)
assert days_3426.shape == (69634, 13)
assert features_3426.shape == (69634, 115)

assert users_3426["user"].is_unique

assert not channels_3426.duplicated(
    ["user", "day", "channel"]
).any()

for name, frame in [
    ("Ground Truth User-Days", days_3426),
    ("Final Features", features_3426),
    ("Natural Features", natural_3426),
    ("Schedule", schedule_3426)
]:
    assert not frame.duplicated(KEYS_3426).any(), (
        f"Duplicate user-day keys: {name}"
    )

assert len(schedule_3426) == 15010

assert set(users_3426["user"]) == set(
    features_3426["user"]
)

assert set(channels_3426["user"]) == set(
    users_3426["user"]
)

print("Expected shapes: PASS")
print("Unique primary keys: PASS")
print("User population alignment: PASS")

# ------------------------------------------------------------
# 4. Confirm original CERT malicious users are excluded.
# ------------------------------------------------------------

excluded_original_users_3426 = {
    "ACM2278",
    "CMP2946",
    "PLJ1771",
    "CDE1846",
    "MBG3183"
}

overlap_3426 = (
    set(users_3426["user"].astype(str))
    & excluded_original_users_3426
)

assert not overlap_3426, (
    f"Original malicious users found: {overlap_3426}"
)

print("Original malicious CERT users excluded: PASS")

# ------------------------------------------------------------
# 5. Validate user-level metadata.
# ------------------------------------------------------------

print("\nB. USER METADATA VALIDATION")

assert users_3426["scenario_id"].notna().all()
assert users_3426["stealth_level"].isin(
    ["High", "Medium", "Low"]
).all()

assert users_3426["injection_formula"].eq(
    "B_Natural_Addition"
).all()

assert users_3426["ground_truth_source"].eq(
    "synthetic_injection"
).all()

assert (
    users_3426["attack_start"]
    <= users_3426["attack_end"]
).all()

assert (
    users_3426["attack_days"]
    + users_3426["off_days"]
    == users_3426["duration_days"]
).all()

assert users_3426["planned_switches"].ge(0).all()

# Confirm channel-level scenario and stealth labels
# match the final user-level Ground Truth.
user_meta_3426 = users_3426[
    ["user", "scenario_id", "stealth_level"]
].rename(columns={
    "scenario_id": "user_scenario",
    "stealth_level": "user_stealth"
})

channel_meta_check_3426 = channels_3426.merge(
    user_meta_3426,
    on="user",
    how="left",
    validate="many_to_one"
)

assert (
    channel_meta_check_3426["scenario_id"].astype(str)
    == channel_meta_check_3426["user_scenario"].astype(str)
).all()

assert (
    channel_meta_check_3426["stealth_level"]
    == channel_meta_check_3426["user_stealth"]
).all()

day_meta_check_3426 = days_3426.merge(
    user_meta_3426,
    on="user",
    how="left",
    validate="many_to_one"
)

assert (
    day_meta_check_3426["scenario_id"].astype(str)
    == day_meta_check_3426["user_scenario"].astype(str)
).all()

assert (
    day_meta_check_3426["stealth_level"]
    == day_meta_check_3426["user_stealth"]
).all()

print("Scenario and stealth consistency: PASS")
print("Injection formula: PASS")
print("Ground Truth source: PASS")

# ------------------------------------------------------------
# 6. Validate Formula B and all channel-level counts.
# ------------------------------------------------------------

print("\nC. INJECTION VALIDATION")

multipliers_3426 = {
    "High": 0.5,
    "Medium": 1.0,
    "Low": 1.5
}

expected_m_3426 = (
    channels_3426["stealth_level"]
    .map(multipliers_3426)
)

assert expected_m_3426.notna().all()

np.testing.assert_allclose(
    channels_3426["stealth_multiplier"],
    expected_m_3426,
    rtol=0,
    atol=1e-10
)

expected_counts_3426 = np.maximum(
    1,
    np.ceil(
        channels_3426["natural_count"].to_numpy(
            dtype=float
        )
        * expected_m_3426.to_numpy(dtype=float)
    )
)

np.testing.assert_allclose(
    channels_3426["injected_count"],
    expected_counts_3426,
    rtol=0,
    atol=1e-9
)

np.testing.assert_allclose(
    channels_3426["final_count"],
    channels_3426["natural_count"]
    + channels_3426["injected_count"],
    rtol=0,
    atol=1e-9
)

assert channels_3426["injected_count"].gt(0).all()

assert set(channels_3426["channel"]) == {
    "email", "usb", "http"
}

assert int(
    channels_3426["injected_count"].sum()
) == 42095

print("Formula B quantities: PASS")
print("Natural + Injected = Final: PASS")
print("Total injected events: 42,095")

# ------------------------------------------------------------
# 7. Validate daily labels and approved schedule.
# ------------------------------------------------------------

print("\nD. DAILY LABEL AND SCHEDULE VALIDATION")

status_3426 = days_3426["day_status"].value_counts()

assert status_3426["Attack"] == 12636
assert status_3426["Off"] == 2374
assert status_3426["Outside Schedule"] == 54624

assert days_3426["is_attack_day"].sum() == 12636
assert days_3426["is_scheduled"].sum() == 15010

assert (
    days_3426["planned_channels"]
    == days_3426["injected_channel_count"]
).all()

assert (
    days_3426[
        ["injected_email", "injected_usb", "injected_http"]
    ].sum(axis=1)
    == days_3426["injected_events_total"]
).all()

# Verify every scheduled day appears in the Ground Truth.
schedule_check_3426 = schedule_3426[
    KEYS_3426 + ["planned_channels"]
].merge(
    days_3426[
        KEYS_3426 + [
            "planned_channels",
            "day_status",
            "is_scheduled"
        ]
    ],
    on=KEYS_3426,
    how="left",
    validate="one_to_one",
    indicator=True,
    suffixes=("_schedule", "_gt")
)

assert schedule_check_3426["_merge"].eq("both").all()

assert (
    schedule_check_3426["planned_channels_schedule"]
    == schedule_check_3426["planned_channels_gt"]
).all()

assert schedule_check_3426["is_scheduled"].eq(1).all()

assert (
    schedule_check_3426["day_status"].eq("Attack")
    == schedule_check_3426[
        "planned_channels_schedule"
    ].gt(0)
).all()

non_attack_3426 = days_3426["day_status"].ne("Attack")

assert days_3426.loc[
    non_attack_3426,
    "injected_events_total"
].eq(0).all()

print("Attack / Off / Outside Schedule counts: PASS")
print("Planned vs actual channels: PASS")
print("No off-plan injection: PASS")

# ------------------------------------------------------------
# 8. Independently reconcile daily channel counts
#    against final feature values.
# ------------------------------------------------------------

print("\nE. FEATURE-TO-GROUND-TRUTH RECONCILIATION")

features_sorted_3426 = (
    features_3426
    .sort_values(KEYS_3426)
    .reset_index(drop=True)
)

natural_sorted_3426 = (
    natural_3426
    .sort_values(KEYS_3426)
    .reset_index(drop=True)
)

days_sorted_3426 = (
    days_3426
    .sort_values(KEYS_3426)
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    features_sorted_3426[KEYS_3426],
    natural_sorted_3426[KEYS_3426],
    check_exact=True
)

pd.testing.assert_frame_equal(
    features_sorted_3426[KEYS_3426],
    days_sorted_3426[KEYS_3426],
    check_exact=True
)

count_mapping_3426 = {
    "external_email_count": "injected_email",
    "usb_copy_count": "injected_usb",
    "www_upload_count": "injected_http"
}

for count_col, gt_col in count_mapping_3426.items():

    observed = (
        features_sorted_3426[count_col].to_numpy(dtype=float)
        - natural_sorted_3426[count_col].to_numpy(dtype=float)
    )

    expected = days_sorted_3426[
        gt_col
    ].to_numpy(dtype=float)

    np.testing.assert_allclose(
        observed,
        expected,
        rtol=0,
        atol=1e-9,
        err_msg=f"Mismatch for {count_col}"
    )

    print(f"{count_col}: PASS")

# Three independent total reconciliations.
assert (
    int(channels_3426["injected_count"].sum())
    == int(days_3426["injected_events_total"].sum())
    == int(users_3426["injected_events_total"].sum())
    == 42095
)

# Ensure no labels are inside ML features.
label_columns_3426 = {
    "scenario_id",
    "stealth_level",
    "day_status",
    "is_scheduled",
    "is_attack_day",
    "planned_channels",
    "injected_channel_count",
    "injected_email",
    "injected_usb",
    "injected_http",
    "injected_events_total",
    "injection_formula",
    "ground_truth_source"
}

assert label_columns_3426.isdisjoint(
    features_3426.columns
)

print("Cross-table injected totals: PASS")
print("Feature-label separation: PASS")

# ------------------------------------------------------------
# 9. Prepare the four final outputs.
# ------------------------------------------------------------

exports_3426 = {
    "ground_truth_users.parquet": users_3426,
    "ground_truth_channel_days.parquet": channels_3426,
    "ground_truth_user_days.parquet": days_3426,
    "behavior_features_injected_final.parquet": features_3426
}

inventory_rows_3426 = []

for filename, frame in exports_3426.items():

    inventory_rows_3426.append({
        "Filename": filename,
        "Rows": len(frame),
        "Columns": len(frame.columns)
    })

inventory_3426 = pd.DataFrame(inventory_rows_3426)

print("\nF. FINAL EXPORT INVENTORY")
display(inventory_3426)

# ------------------------------------------------------------
# 10. Protect existing output files.
#
# This code refuses to overwrite any existing file.
# ------------------------------------------------------------

existing_files_3426 = [
    filename
    for filename in exports_3426
    if (OUTPUT_DIR_3426 / filename).exists()
]

if existing_files_3426:
    raise FileExistsError(
        "Export stopped to protect existing files: "
        f"{existing_files_3426}"
    )

# All validation checks have passed.
# Create the destination folder if necessary.
OUTPUT_DIR_3426.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 11. Write temporary Parquet files, read them back,
#     and verify the saved contents before finalizing.
# ------------------------------------------------------------

print("\nG. PARQUET EXPORT AND READ-BACK VALIDATION")

with tempfile.TemporaryDirectory(
    prefix="_stage3_validation_",
    dir=str(OUTPUT_DIR_3426)
) as temp_dir_3426:

    temp_dir_3426 = Path(temp_dir_3426)

    # Write and independently read back every file.
    for filename, frame in exports_3426.items():

        temp_path = temp_dir_3426 / filename

        frame.reset_index(drop=True).to_parquet(
            temp_path,
            engine="pyarrow",
            index=False
        )

        loaded = pd.read_parquet(
            temp_path,
            engine="pyarrow"
        )

        pd.testing.assert_frame_equal(
            frame.reset_index(drop=True),
            loaded.reset_index(drop=True),
            check_dtype=False,
            check_exact=True
        )

        print(f"PASS — {filename}")

    # One more overwrite safeguard before finalizing.
    for filename in exports_3426:
        final_path = OUTPUT_DIR_3426 / filename

        if final_path.exists():
            raise FileExistsError(
                f"Export stopped: {final_path}"
            )

    # Move validated files to their final location.
    for filename in exports_3426:

        os.rename(
            temp_dir_3426 / filename,
            OUTPUT_DIR_3426 / filename
        )

# ------------------------------------------------------------
# 12. Confirm final files exist.
# ------------------------------------------------------------

saved_rows_3426 = []

for filename, frame in exports_3426.items():

    path = OUTPUT_DIR_3426 / filename

    assert path.is_file()

    saved_rows_3426.append({
        "File": filename,
        "Rows": len(frame),
        "Columns": len(frame.columns),
        "Size_MB": round(
            path.stat().st_size / (1024 ** 2),
            3
        )
    })

saved_inventory_3426 = pd.DataFrame(
    saved_rows_3426
)

print("\nH. SAVED FILES")
display(saved_inventory_3426)

print("\n" + "=" * 78)
print("STAGE 3 — FINAL VALIDATION & EXPORT COMPLETED")
print("=" * 78)

print("Formula: B_Natural_Addition")
print("Ground Truth files: 3")
print("Final feature files: 1")
print("Simulated users: 670")
print("Injected channel-days: 14,375")
print("Attack days: 12,636")
print("Off days: 2,374")
print("Injected events: 42,095")
print("Final features: 115 columns")
print("Validation: PASS")
print("Parquet read-back: PASS")
print("Source datasets: UNCHANGED")
print("Output directory:", OUTPUT_DIR_3426)
print("STAGE 3 STATUS: COMPLETED")


STAGE 3 — STEP 3.4.26: FINAL VALIDATION & EXPORT

A. DATASET STRUCTURE VALIDATION
Expected shapes: PASS
Unique primary keys: PASS
User population alignment: PASS
Original malicious CERT users excluded: PASS

B. USER METADATA VALIDATION
Scenario and stealth consistency: PASS
Injection formula: PASS
Ground Truth source: PASS

C. INJECTION VALIDATION
Formula B quantities: PASS
Natural + Injected = Final: PASS
Total injected events: 42,095

D. DAILY LABEL AND SCHEDULE VALIDATION
Attack / Off / Outside Schedule counts: PASS
Planned vs actual channels: PASS
No off-plan injection: PASS

E. FEATURE-TO-GROUND-TRUTH RECONCILIATION
external_email_count: PASS
usb_copy_count: PASS
www_upload_count: PASS
Cross-table injected totals: PASS
Feature-label separation: PASS

F. FINAL EXPORT INVENTORY


,Filename,Rows,Columns
0,ground_truth_users.parquet,670,12
1,ground_truth_channel_days.parquet,14375,11
2,ground_truth_user_days.parquet,69634,13
3,behavior_features_injected_final.parquet,69634,115



G. PARQUET EXPORT AND READ-BACK VALIDATION
PASS — ground_truth_users.parquet
PASS — ground_truth_channel_days.parquet
PASS — ground_truth_user_days.parquet
PASS — behavior_features_injected_final.parquet

H. SAVED FILES


,File,Rows,Columns,Size_MB
0,ground_truth_users.parquet,670,12,0.021
1,ground_truth_channel_days.parquet,14375,11,0.069
2,ground_truth_user_days.parquet,69634,13,0.088
3,behavior_features_injected_final.parquet,69634,115,7.035



STAGE 3 — FINAL VALIDATION & EXPORT COMPLETED
Formula: B_Natural_Addition
Ground Truth files: 3
Final feature files: 1
Simulated users: 670
Injected channel-days: 14,375
Attack days: 12,636
Off days: 2,374
Injected events: 42,095
Final features: 115 columns
Validation: PASS
Parquet read-back: PASS
Source datasets: UNCHANGED
Output directory: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Attack_Simulation\data
STAGE 3 STATUS: COMPLETED
